# CAL01 공정 시계열 기반 이상 구간 탐지 및 공정 신호·정비 이력 관계 분석
**담당: 정수진 · Issue: #65**  
**분석 단위: LOT별 관측 구간 · 작업 WO · 실제 시각의 교집합**

2024년 **14 LOT 관측 구간**에서 운전 전환, 반복 CP 이탈, 같은 강종의 수준 차이와 정비 후 변화를 구분했다. 확인된 변화를 점검 우선순위와 후속 검증 항목으로 연결한다. 고장·품질 예측 모델은 만들지 않는다.

1. [Executive Summary — 핵심 결과](#overview)
2. [분석 목적 및 데이터 특성](#data-scope)
3. [전처리·탐지 방법과 판단 범위](#methods)
4. [전체 시계열 특성과 탐지 결과](#screening)
5. [주요 LOT 상세 분석과 계획수리 대조](#cases)
6. [공정별 시계열 특성 종합](#process-summary)
7. [종합 판단 및 현장 적용 방안](#conclusion)
8. [부록 — 14 LOT 전체 검토·재현 코드](#detail)

**읽는 연결:** 전체 관찰 → 탐지 근거 → 고정 구간 번호(예: 240024 (a)) → 같은 번호의 확대·동반 반응 → 정비와의 시간 관계 → 판단. 각 확대 상단에 전체 LOT의 같은 신호와 현재 위치를 다시 표시한다.

시계열 구간 표시색은 **빨강=CP 원본 이탈 / 주황=시간순 Z / 회색=운전 전환 참고 / 보라=정비 감지 대조**다. IQR 수염 밖 점은 청록이다. 색은 고장 등급이 아니며 모든 그림은 원본 CSV에서 코드로 생성한다.

**중심 연구 질문**

1. CP-Z4와 Z3~Z6의 TC·OP 변화가 사건마다 반복되는가?
2. 변화는 정비 감지·착수·완료의 어느 쪽에 위치하며 작업 대상과 연결되는가?
3. CP가 없는 반례와 운전 전환을 대조했을 때 점검할 근거가 남는가?

**코드 읽는 법:** 핵심 그림의 matplotlib 함수는 해당 절의 실행 셀에 있다. 공통 색·축·파일 저장·통계 전처리만 앞에서 재사용한다. 이미지를 읽어 붙이지 않고 CSV에서 다시 그린다.

관계 분석 히트맵의 빨강·파랑은 변화량 증가·감소를 뜻하며 해당 그림 범례를 따른다. [5.6 사건별 관계](#relation-events) → [5.7 CP 전후 파형](#relation-lags) → [5.7.1 TC-OP 관계](#relation-tc-op) → [5.7.2 출력 뒤 온도 복귀](#tc-op-response) → [5.7.3 품질 검증 근거](#tc-quality-evidence) → [5.8 CP 이탈 유무 비교](#relation-controls) → [5.9 채널·정비](#relation-maintenance) 순서로 연결한다.


In [ ]:
from pathlib import Path
# 이 칸은 그대로 실행하세요. 데이터 경로와 그래프 한글 폰트를 잡아둡니다.
# 이걸 기준으로 코드 작성할 것
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager, rcParams

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/T-CR1-CAL01_온도.csv").exists() and (p / "notebooks").is_dir())
DATA_DIR = str(PROJECT_ROOT / "data")
FIG_DIR = str(PROJECT_ROOT / "figures")
os.makedirs(FIG_DIR, exist_ok=True)

TEMP_CSV = os.path.join(DATA_DIR, "T-CR1-CAL01_온도.csv")
EVENT_CSV = os.path.join(DATA_DIR, "G-02_조업이벤트.csv")
REPAIR_CSV = os.path.join(DATA_DIR, "G-01_정기수리캘린더.csv")
EQP_CSV = os.path.join(DATA_DIR, "G-01_설비마스터.csv")
PART_CSV = os.path.join(DATA_DIR, "G-01_부품마스터.csv")
MAINT_CSV = os.path.join(DATA_DIR, "G-02_정비이력.csv")
PARTCHG_CSV = os.path.join(DATA_DIR, "G-02_부품교체이력.csv")

ENC = "utf-8-sig"

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

installed = {f.name for f in font_manager.fontManager.ttflist}
for cand in ["Malgun Gothic", "AppleGothic", "NanumGothic", "NanumBarunGothic", "Noto Sans CJK KR"]:
    if cand in installed:
        rcParams["font.family"] = cand
        break
else:
    print("경고: 한글 폰트를 못 찾았습니다. 그래프의 한글이 □ 로 보일 수 있습니다.")

rcParams["axes.unicode_minus"] = False
rcParams["figure.dpi"] = 110
"""Shared presentation rules for the five CAL01 time-series notebooks.

Only artists and display labels are changed here. Sensor values, thresholds,
aggregation windows and axis limits belong to the notebooks.
"""


import html
import re
from pathlib import Path

from matplotlib import rcParams
from matplotlib.lines import Line2D

EXPORT_DPI = 180
EXPORT_PADDING = 0.20
INK = '#1F2937'
MUTED = '#64748B'

# A color always identifies the same member of the same domain. Different
# domains use separate legends. LOT colors contrast most within the same grade.
PROCESS_COLORS = {'세정': '#2563EB', '건조': '#D97706', '가열·균열': '#DC4C3E', '과시효': '#059669', '냉각': '#7C3AED'}
GRADE_COLORS = {'SGCC': '#B9D6D9', 'SPFH590': '#B4634E', 'SPCE': '#3F909A', 'SPCEN': '#8676AC', 'SPHC': '#A48258', 'DP590': '#2B527A', 'SPCD': '#3E75A8', 'SPCC': '#9AA7B5'}
LOT_COLORS = {240001: '#334155', 240024: '#2563EB', 240108: '#F97316', 240037: '#1D4ED8', 240121: '#D946EF', 240133: '#10B981', 240038: '#0891B2', 240114: '#DC2626', 240044: '#0F766E', 240061: '#D97706', 240069: '#7C3AED', 240091: '#DB2777', 240058: '#92400E', 240117: '#475569'}
MAINTENANCE_COLORS = {'BM': '#C81E3A', 'CBM': '#6D28D9', 'TBM': '#0E7490'}
MAINTENANCE_LABELS = {"BM": "BM · 고장 후 정비", "CBM": "CBM · 상태 기준 정비", "TBM": "TBM · 주기 기준 정비"}
MAINTENANCE_HATCHES = {"BM": "///", "CBM": "...", "TBM": "xxx"}
ZONE_COLORS = {1: '#475569', 2: '#B45309', 3: '#7C3AED', 4: '#2563EB', 5: '#F97316', 6: '#059669', 7: '#DB2777', 8: '#0891B2', 9: '#A16207'}
SENSOR_COLORS = {'CLN-IN': '#0F766E', 'CP-Z4': '#E11D48', 'CP-Z4M': '#64748B', 'TC-OUT1': '#4F46E5', 'TC-OUT2': '#CA8A04'}

# LOT is encoded by color. Measured OP uses a dashed line; TC/CP use solid lines.
LOT_LINESTYLES = {240001: '-', 240024: '-', 240037: '-', 240038: '-', 240044: '-', 240058: '-', 240061: '-', 240069: '-', 240091: '-', 240108: '-', 240114: '-', 240117: '-', 240121: '-', 240133: '-'}

# Full original explanations remain available immediately below each figure.
# Short figure text describes the target, comparison rule and reading purpose.
_HEADINGS = {
    "연간 공정 흐름": ("2024년 CAL01 관측·정비 흐름", "2024년 관측창과 정비 시각을 대조해 비교 범위 확인"),
    "LOT별 자료 상태 · 분석에 쓸 수 있는 구간과 빠진 곳": ("CAL01 14 LOT 관측 품질 비교", "LOT별 기록량과 결측을 대조해 분석 가능 구간 확인"),
    "측정 기록 공백·센서 NaN 비교": ("CAL01 측정 공백·센서 결측 비교", "같은 LOT의 TC·OP를 대조해 결측 종류 구분"),
    "운전 상태별 급변 — 정지·승온을 따로 떼어야 하는 이유": ("CAL01 운전 상태별 급변 크기", "신호별 안정 운전 기준으로 정지·승온의 급변 비교"),
    "정비 일정 전후 정지·재가동 — 설명됨의 근거": ("CAL01 정비 전후 정지·재가동", "14 LOT의 온도·출력을 일정과 대조해 전환 구간 확인"),
    "CP-Z4 · 14 LOT 원본에서 기준(±0.05)을 넘은 곳": ("CAL01 14 LOT의 CP 기준 이탈", "CP 편차(무차원) ±0.05 이탈의 위치·횟수 비교"),
    "탐지된 구간이 어디로 갔나 — 근거가 있어 빼는 것과 03으로 넘기는 것": ("CAL01 탐지 구간의 검토 대상 선별", "탐지 구간을 설명 근거별로 나눠 상세 검토 대상 확인"),
    "안정 참고 구간 · 가열·균열 Z3 수준": ("CAL01 강종별 Z3 온도·출력 수준", "같은 강종의 안정 참고 중앙값으로 LOT 수준 비교"),
    "안정 참고 구간 · Z3 제어출력과 온도를 같이 보기": ("CAL01 LOT별 Z3 온도·출력 반응", "LOT별 안정 참고 구간에서 온도와 출력의 동행 확인"),
    "출력이 0%·100%에 붙을 때 Z3 온도는 어떻게 움직이나": ("CAL01 Z3 출력 한계·온도 반응", "출력 0%·100% 지속 구간의 온도 반응을 함께 확인"),
    "세정 입구 온도 (CLN-IN)": ("CAL01 LOT별 세정 입구 온도", "LOT 중앙값과 같은 강종의 흐름으로 세정 온도 비교"),
    "정비 전후 흐름 · LOT 사이 정비와 LOT별 상태 지표": ("CAL01 정비 시각·LOT 지표 비교", "관측 날짜 순 지표와 LOT 사이 정비의 시간 관계 확인"),
    "볼 것 고르기 — 평소 흔들림 대비 편차": ("CAL01 평소 변동 대비 검토 우선순위", "신호별 평소 변동으로 나눈 편차로 검토 대상 비교"),
    "CP의 평소 관계 — 14 LOT에서 먼저 확인": ("CAL01 14 LOT의 CP 동시 관계", "안정 참고 1초 변화 상관으로 CP와 센서의 동행 확인"),
    "어느 LOT이 CP의 평소 관계에서 벗어났나 — 14 LOT 비교": ("CAL01 LOT별 CP 변동·동조 비교", "안정 참고 구간의 이탈·변동·동조로 검토 LOT 비교"),
    "CP 관점에서 확인할 LOT · 같은 강종부터 비교": ("CAL01 강종별 CP 검토 LOT 비교", "같은 강종의 시간당 이탈 빈도와 크기로 후보 비교"),
    "가스 분석계 셀 교체와 CP 신호 상태": ("CAL01 분석계 셀 교체·CP 상태", "CP 계측 부품의 교체 시각과 시간별 신호 상태 대조"),
    "같은 강종끼리 대조한 LOT별 요약 · 과시효 Z7·Z8": ("CAL01 동일 강종의 Z7·Z8 비교", "같은 강종 다른 LOT 중앙값 대비 온도·출력 차이 확인"),
    "14 LOT × 공정 · 확인 이슈와 근거 한눈에 보기": ("CAL01 14 LOT 공정별 검토 근거", "공정별 비교 가능성과 근거로 추가 확인할 LOT 선별"),
    "과시효 Z7·Z8 · 같은 강종 LOT끼리 비교": ("CAL01 동일 강종의 과시효 온도 비교", "안정 참고 1분 중앙값 분포로 LOT 수준 차이 확인"),
}


def configure():
    rcParams.update({
        "font.size": 11.5, "axes.titlesize": 13, "axes.labelsize": 11.5,
        "xtick.labelsize": 10.5, "ytick.labelsize": 10.5,
        "savefig.dpi": EXPORT_DPI, "savefig.pad_inches": EXPORT_PADDING,
        "svg.fonttype": "none", "pdf.fonttype": 42,
    })


def sensor_color(sensor):
    if sensor.startswith(("TC-Z", "OP-Z")):
        return ZONE_COLORS[int(sensor.rsplit("Z", 1)[1])]
    return SENSOR_COLORS[sensor]


def lot_linestyle(lot):
    return '-'


def process_title_color(title, default=INK):
    return INK


def palette_preview(path):
    """Read the guide in order: signal type, zone color, then compared LOTs."""
    import matplotlib.pyplot as plt
    from matplotlib.patches import FancyBboxPatch
    from IPython.display import display
    fig = plt.figure(figsize=(16, 10.2), facecolor="#F5F7FB")
    fig.text(.045, .955, "CAL01 시계열 · 색상과 선 표기", fontsize=25, weight="bold", color=INK, va="top")
    fig.text(.045, .909, "TC는 실선 · OP는 점선  |  같은 존은 같은 색  |  LOT 비교는 같은 강종 안에서 색으로 구분", fontsize=12, color=MUTED, va="top")

    def card(x, y, w, h, title, subtitle=""):
        fig.add_artist(FancyBboxPatch((x,y),w,h,boxstyle="round,pad=0.008,rounding_size=0.012",
                       transform=fig.transFigure,fc="white",ec="#E2E8F0",lw=.8,zorder=0))
        ax=fig.add_axes([x+.014,y+.014,w-.028,h-.028],zorder=1)
        ax.set_xlim(0,1);ax.set_ylim(0,1);ax.axis("off")
        ax.text(0,.96,title,va="top",fontsize=14,weight="bold",color=INK)
        if subtitle:ax.text(0,.79,subtitle,va="top",fontsize=10.5,color=MUTED)
        return ax

    ax=card(.045,.661,.438,.207,"1  선 모양 = 측정 신호","TC·OP는 색을 바꾸지 않고 선 모양으로 구분")
    for y,s,label in [(.55,"TC-Z4","TC 온도 (℃) · 실선"),(.29,"OP-Z4","OP 제어출력 (%) · 점선")]:
        ax.plot([.015,.27],[y,y],color=ZONE_COLORS[4],lw=2.8,ls=signal_linestyle(s))
        ax.text(.32,y,label,va="center",fontsize=12.5,color=INK)
    ax.text(.015,.06,"CP는 실선 · 무차원",fontsize=10.5,color=MUTED)

    ax=card(.504,.661,.451,.207,"2  선 색 = 존(ZONE)","같은 ZONE의 TC와 OP는 아래 두 선처럼 같은 색")
    for i,(z,col) in enumerate(ZONE_COLORS.items()):
        row,column=divmod(i,3);x=column*.337;y=.56-row*.20
        ax.text(x,y,f"Z{z}",va="center",fontsize=11.5,color=INK)
        ax.plot([x+.085,x+.175],[y+.027,y+.027],color=col,lw=2.1)
        ax.plot([x+.085,x+.175],[y-.027,y-.027],color=col,lw=2.1,ls="--")

    for x,w,title,colors in [(.045,.291,"공정 · 축 옆 구획색",PROCESS_COLORS),
                              (.357,.291,"강종 · 그룹 표식",{k:GRADE_COLORS[k] for k in ["SPCC","SPCD","SPCE","SPCEN","SPHC","SPFH590","DP590","SGCC"]}),
                              (.669,.286,"정비 · 실제 작업 구간",MAINTENANCE_COLORS)]:
        ax=card(x,.476,w,.151,title)
        columns=3 if len(colors)>5 else 2 if len(colors)>3 else 3
        for i,(label,col) in enumerate(colors.items()):
            row,column=divmod(i,columns);xx=column/columns;yy=.61-row*.235
            ax.plot([xx+.01,xx+.048],[yy,yy],color=col,lw=6,solid_capstyle="round")
            ax.text(xx+.073,yy,label,va="center",fontsize=9.8,color=INK)
        if title.startswith("정비"):
            ax.text(.01,.18,"BM 고장후  ·  CBM 상태기준  ·  TBM 주기기준",fontsize=9,color=MUTED)

    fig.text(.045,.437,"3  LOT 비교 · 아래 같은 강종 안에서 선 색을 대조",fontsize=14,weight="bold",color=INK)
    groups=[("SGCC",[240001]),("SPFH590",[240024,240108]),("SPCE",[240037,240121,240133]),
            ("SPCEN",[240038,240114]),("SPHC",[240044,240061,240069,240091]),
            ("DP590",[240058]),("SPCD",[240117])]
    for i,(grade,ids) in enumerate(groups):
        row,column=divmod(i,4);x=.045+column*.232;y=.237-row*.196
        ax=card(x,y,.213,.171,grade)
        for j,k in enumerate(ids):
            yy=.69-j*.175
            ax.plot([.015,.29],[yy,yy],color=LOT_COLORS[k],lw=2.6,ls="-")
            ax.text(.35,yy,str(k),va="center",fontsize=11.5,color=INK)
    ax=card(.741,.041,.213,.171,"비교 그림 읽는 법")
    ax.text(.01,.67,"대상 LOT  굵게\n대조 LOT  얇게\nTC 실선 / OP 점선",va="top",fontsize=11.5,color=MUTED,linespacing=1.65)
    Path(path).parent.mkdir(parents=True,exist_ok=True)
    fig.savefig(path,dpi=EXPORT_DPI,facecolor=fig.get_facecolor(),bbox_inches="tight",pad_inches=.18)
    fig.set_dpi(72);display(fig);plt.close(fig)



def signal_linestyle(signal):
    return "--" if str(signal).startswith("OP-") else "-"


def _signal_kind(label, axis_text=""):
    label=str(label)
    if re.search(r"\bOP-Z\d",label) or label.startswith("OP "):return "OP"
    if re.search(r"\bTC-(?:Z\d|OUT)",label) or label=="CLN-IN" or label.startswith("TC "):return "TC"
    if label.startswith("CP-"):return "CP"
    if "출력" in label and "온도" not in label:return "OP"
    # A LOT label carries color identity; its panel supplies the measurement.
    if "제어출력" in axis_text or "출력[%]" in axis_text or "[%]" in axis_text:return "OP"
    if "온도" in axis_text or "[℃]" in axis_text:return "TC"
    if "무차원" in axis_text and "CP" in axis_text:return "CP"
    return None


def style_signal_lines(fig):
    """Color identifies a zone/LOT; a dashed measured curve identifies OP."""
    for ax in fig.axes:
        titles=" ".join([ax.title.get_text(),ax._left_title.get_text(),ax._right_title.get_text()])
        context=ax.get_ylabel()+" "+titles
        for line in ax.lines:
            if len(line.get_xdata())<3 or line.get_linestyle() in ["None","none","",None]:continue
            if line.get_transform()!=ax.transData:continue
            label=line.get_label();kind=_signal_kind(label,context)
            if kind:line.set_linestyle("--" if kind=="OP" else "-")
            if re.fullmatch(r"(?:TC-Z[1-9]|OP-Z[1-8]|TC-OUT[12]|CP-Z4M?|CLN-IN)",str(label)):
                line.set_color(sensor_color(label))
        # A restrained axis rail carries process context; titles stay dark.
        found=[p for p in PROCESS_COLORS if p in titles or (p=="가열·균열" and ("가열" in titles or "균열" in titles))]
        if len(found)==1 and ax.axison:
            ax.spines["left"].set_color(PROCESS_COLORS[found[0]])
            ax.spines["left"].set_alpha(.65)
            ax.spines["left"].set_linewidth(1.0)
        for title in [ax.title,ax._left_title,ax._right_title]:
            if title.get_text() and str(title.get_color()).lower() not in ["#c83745","#d62728","#e11d48"]:
                title.set_color(INK)
        legend=ax.get_legend()
        if legend is not None:_style_legend(legend,context)
    for legend in fig.legends:_style_legend(legend)


def _style_legend(legend, context=""):
    handles=getattr(legend,"legend_handles",getattr(legend,"legendHandles",[]))
    for handle,text in zip(handles,legend.get_texts()):
        if not isinstance(handle,Line2D) or handle.get_linestyle() in ["None","none","",None]:continue
        label=text.get_text()
        if any(word in label for word in ["기준","중앙값","경계","감지","정비","수리","범위"]):continue
        kind=_signal_kind(label,context)
        if kind:handle.set_linestyle("--" if kind=="OP" else "-")
        if re.fullmatch(r"(?:TC-Z[1-9]|OP-Z[1-8]|TC-OUT[12]|CP-Z4M?|CLN-IN)",label):
            handle.set_color(sensor_color(label))


def _heading(title, subtitle):
    if title in _HEADINGS:
        return _HEADINGS[title]
    patterns = [
        (r"CAL01 정비 이력 (\d+)건", lambda m: (f"CAL01 정비 {m[1]}건의 작업 흐름", "작업별 감지·착수·완료와 관측창을 대조해 해석 범위 확인")),
        (r"남은 짧은 변화 · (\d+)", lambda m: (f"LOT {m[1]} 미설명 변화 확대", "같은 LOT의 1초 원본으로 탐지 구간의 동시 변화 확인")),
        (r"같은 강종 LOT 전체 비교 · (\w+) · (.+)", lambda m: (f"{m[1]} LOT별 공정 신호 비교", "같은 강종·같은 신호의 경과시간으로 LOT 간 차이 확인")),
        (r"같은 강종 LOT 비교 · (\w+) · 대상 (\d+) · Z4", lambda m: (f"LOT {m[2]} 동일 강종 Z4 비교", "안정 참고 시작을 맞춰 같은 강종의 온도·출력 비교")),
        (r"(\d+) 정비 전·중·후", lambda m: (f"LOT {m[1]} 정비 전후 신호 비교", "같은 LOT의 실제 작업 경계와 온도·출력 변화를 대조")),
        (r"(\d+) CP 변화·정비", lambda m: (f"LOT {m[1]} CP·정비 시각 대조", "같은 LOT의 CP 극값과 정비·교체 기록의 시간 관계 확인")),
        (r"(\d+) · 언제 CP", lambda m: (f"LOT {m[1]} CP 동조 변화 구간", "안정 참고의 시간별 변동과 1초 동조로 변화 시점 확인")),
        (r"CP 펄스 뒤 무엇이 달라졌나 · (.+)", lambda m: (f"CAL01 CP 펄스 전후 반응 {m[1]}", "같은 LOT의 펄스 없는 안정 참고 시각과 전후 반응 비교")),
    ]
    for pattern, make in patterns:
        match = re.search(pattern, title)
        if match:
            return make(match)
    return title, subtitle


def header(fig, title, subtitle="", handles=(), ncol=4):
    original_title, original_subtitle = title, subtitle
    title, subtitle = _heading(title, subtitle)
    fig.suptitle(title, x=.035, y=.975, ha="left", va="top",
                 fontsize=23, fontweight="bold", color=INK)
    sub = fig.text(.035, .924, subtitle, ha="left", va="top", fontsize=12, color=MUTED)
    fig._report_subtitle = sub
    fig._report_original = {"title": original_title, "subtitle": original_subtitle}
    fig._report_details = []
    if handles:
        # Legend order is the caller's comparison order, never an alphabetic sort.
        labels = [h.get_label() for h in handles]
        fig.legend(handles=handles, labels=labels, loc="upper left", bbox_to_anchor=(.031, .879),
                   ncol=ncol, frameon=False, fontsize=11, handlelength=2.4,
                   columnspacing=1.5, labelspacing=.65)


def _panel_titles(fig, renderer):
    """Keep the identifying clause on the chart and retain the full caption."""
    for ax in fig.axes:
        title = ax._left_title if ax._left_title.get_text() else ax.title
        original = title.get_text()
        if title.get_color().lower() in [INK, "#17344d"]:
            title.set_color(process_title_color(original, title.get_color()))
        if not original or "\n" in original:
            continue
        if len(original) <= 66 and title.get_window_extent(renderer).width <= ax.bbox.width:
            continue
        # A long explanatory clause belongs below the figure. Do not cut a
        # label in the middle of a sensor name, quantity or comparison value.
        short = re.split(r" — | · (?=회색|위 |아래 |점 =|빨간|파란|평소|숫자|칸 =|세로)| \| ", original, maxsplit=1)[0]
        if short == original or len(short) > 66:
            continue
        fig._report_details.append(original)
        title.set_text(short)


def _legend_outside(fig, renderer):
    """Move an inside legend to a measured free title-row space when it fits."""
    moved = 0
    for ax in fig.axes:
        leg = ax.get_legend()
        if leg is None or not leg.get_window_extent(renderer).overlaps(ax.bbox):
            continue
        handles, labels = ax.get_legend_handles_labels()
        if not labels:
            continue
        candidate = ax.legend(handles, labels, loc="lower right", bbox_to_anchor=(1, 1.025),
                              borderaxespad=0, ncol=min(4, len(labels)), frameon=False,
                              fontsize=10.5, handlelength=2, columnspacing=1.2)
        fig.canvas.draw()
        box = candidate.get_window_extent(renderer)
        obstacles = [a.bbox for a in fig.axes if a is not ax and a.bbox != ax.bbox]
        texts = list(fig.texts)
        for other in fig.axes:
            texts += [other._left_title, other._right_title, other.title, other.xaxis.label]
            texts += list(other.texts)
        obstacles += [t.get_window_extent(renderer) for t in texts if t.get_visible() and t.get_text()]
        if any(box.overlaps(b) for b in obstacles) or box.y1 > fig.bbox.y1:
            candidate.remove()
            ax.add_artist(leg)
            ax.legend_ = leg
        else:
            candidate._detail_legend = True
            moved += 1
    return moved


def prepare(fig):
    """Arrange the figure header in physical units, preserving data limits."""
    if getattr(fig, "_report_prepared", False):
        return
    fig._report_prepared = True
    style_signal_lines(fig)
    fig._report_details = getattr(fig, "_report_details", [])
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    _panel_titles(fig, renderer)
    if hasattr(fig, "_report_subtitle"):
        height = fig.bbox.height
        gap = 9 * fig.dpi / 72
        title = fig._suptitle
        title.set_y(1 - .20 / fig.get_figheight())
        title.set_fontsize(min(25, title.get_fontsize()))
        fig.canvas.draw()
        sub = fig._report_subtitle
        sub.set_position((.035, (title.get_window_extent(renderer).y0 - gap) / height))
        sub.set_fontsize(12)
        sub.set_va("top")
        fig.canvas.draw()
        cursor = sub.get_window_extent(renderer).y0 - gap
        for legend in fig.legends:
            legend.set_bbox_to_anchor((.031, cursor / height), transform=fig.transFigure)
            fig.canvas.draw()
            cursor = legend.get_window_extent(renderer).y0 - gap
        # Existing charts already allocate a header. Grow it only if the
        # measured text requires additional room, including secondary axes.
        if fig.axes:
            positions = {ax: ax.get_position().frozen() for ax in fig.axes}
            low = min(p.y0 for p in positions.values())
            high = max(p.y1 for p in positions.values())
            visible_top = max((ax.get_tightbbox(renderer).y1 for ax in fig.axes if ax.get_visible()), default=high*height)
            extra = max(0, visible_top - high*height)
            target = (cursor - extra) / height
            target = min(target, high + .55 / fig.get_figheight())
            if abs(target-high) > .003:
                scale = (target-low) / (high-low)
                if scale < .55:
                    raise RuntimeError("그림 머리글에 비해 본문 높이가 부족합니다.")
                for ax, p in positions.items():
                    ax.set_position([p.x0, low+(p.y0-low)*scale, p.width, p.height*scale])
                for text in fig.texts:
                    if text in [title, sub]:
                        continue
                    x, y = text.get_position()
                    if low <= y <= high:
                        text.set_position((x, low+(y-low)*scale))
    fig.canvas.draw()
    fig._report_legends_moved = _legend_outside(fig, fig.canvas.get_renderer())


def details_html(fig):
    original = getattr(fig, "_report_original", {})
    details = [original.get("subtitle", ""), *getattr(fig, "_report_details", [])]
    details = list(dict.fromkeys(text for text in details if text))
    if not details:
        return ""
    body = "".join(f"<p>{html.escape(text)}</p>" for text in details)
    return '<details data-report-visuals="caption"><summary>그림 상세 기준·해석 펼치기</summary>' + body + "</details>"


# 각 노트북 안에서 완결되는 공통 그림 설정
from types import SimpleNamespace
report_viz = SimpleNamespace(
    EXPORT_DPI=EXPORT_DPI,
    EXPORT_PADDING=EXPORT_PADDING,
    PROCESS_COLORS=PROCESS_COLORS,
    GRADE_COLORS=GRADE_COLORS,
    LOT_COLORS=LOT_COLORS,
    MAINTENANCE_COLORS=MAINTENANCE_COLORS,
    MAINTENANCE_LABELS=MAINTENANCE_LABELS,
    MAINTENANCE_HATCHES=MAINTENANCE_HATCHES,
    ZONE_COLORS=ZONE_COLORS,
    SENSOR_COLORS=SENSOR_COLORS,
    LOT_LINESTYLES=LOT_LINESTYLES,
    header=header,
    prepare=prepare,
    details_html=details_html,
    sensor_color=sensor_color,
    lot_linestyle=lot_linestyle,
    process_title_color=process_title_color,
    palette_preview=palette_preview,
    signal_linestyle=signal_linestyle,
    style_signal_lines=style_signal_lines,
)
configure()

In [ ]:
NOTEBOOK_NO = "06"
from pathlib import Path
from IPython.display import display, Markdown
import matplotlib.dates as mdates
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator
from scipy.signal import find_peaks

files = {"온도": TEMP_CSV, "조업": EVENT_CSV, "수리": REPAIR_CSV,
         "설비": EQP_CSV, "부품": PART_CSV, "정비": MAINT_CSV,
         "교체": PARTCHG_CSV}
frames = {name: pd.read_csv(path, encoding=ENC) for name, path in files.items()}
temp, event, repair, eqp = [frames[k] for k in ["온도", "조업", "수리", "설비"]]
part, maint, partchg = [frames[k] for k in ["부품", "정비", "교체"]]
for frame, columns in [(temp, ["MEAS_DT"]), (event, ["EVT_DT"]),
                       (repair, ["STRT_DT", "END_DT"]),
                       (maint, ["DETC_DT", "STRT_DT", "END_DT"])]:
    for column in columns:
        frame[column] = pd.to_datetime(frame[column])
CAL_TAG = "P1-CR1-CAL01"
YEAR_START, YEAR_END = pd.Timestamp("2024-01-01"), pd.Timestamp("2025-01-01")
grade_evt = event.loc[event.PLANT_CD.eq("CR1") & event.EVT_TYPE.eq("강종변경")].sort_values("EVT_DT")
assert not grade_evt.EVT_DT.duplicated().any()
lots = temp.groupby("LOT_NO", as_index=False).agg(
    START=("MEAS_DT", "min"), END=("MEAS_DT", "max"), N_ROWS=("MEAS_DT", "size"))
lots = lots.merge(grade_evt[["EVT_DT", "AFT_VAL"]], left_on="START", right_on="EVT_DT",
                  how="left", validate="one_to_one").rename(columns={"AFT_VAL": "GRADE"})
lots = lots.sort_values("START").reset_index(drop=True)
assert lots.GRADE.notna().all()
lots["MINUTES"] = (lots.END - lots.START).dt.total_seconds() / 60
lot_meta = lots.set_index("LOT_NO")
cal_work = maint.loc[maint.EQP_TAG.eq(CAL_TAG)].sort_values("STRT_DT").copy()
assert cal_work.WO_NO.is_unique and cal_work.END_DT.ge(cal_work.STRT_DT).all()
cal_parts = (partchg.merge(cal_work[["WO_NO", "EQP_TAG", "MNT_TYPE", "DETC_DT", "STRT_DT", "END_DT", "REMARK"]],
                           on="WO_NO", validate="many_to_one")
             .merge(part, on="PART_CD", validate="many_to_one").sort_values("END_DT"))
plans = repair.loc[repair.PLANT_CD.eq("CR1")].sort_values("STRT_DT").copy()
plans["END_EXCL"] = plans.END_DT.dt.normalize() + pd.Timedelta(days=1)
TC = ["CLN-IN"] + [f"TC-Z{i}" for i in range(1, 10)] + ["TC-OUT1", "TC-OUT2"]
OP = [f"OP-Z{i}" for i in range(1, 9)]
SIGNALS = TC + OP + ["CP-Z4", "CP-Z4M"]
HEAT = [f"TC-Z{i}" for i in range(3, 7)]
PROCESS = {"CLN-IN": "세정", **{f"TC-Z{i}": "건조" if i < 3 else "가열·균열" if i < 7 else "과시효" for i in range(1, 10)},
           "TC-OUT1": "냉각", "TC-OUT2": "냉각",
           **{f"OP-Z{i}": "건조" if i < 3 else "가열·균열" if i < 7 else "과시효" for i in range(1, 9)},
           "CP-Z4": "가열·균열", "CP-Z4M": "가열·균열"}
LOT_COLOR = report_viz.LOT_COLORS.copy()
COL = {"LOT": "#087f9c", "GRADE": "#477eb5", "BM": report_viz.MAINTENANCE_COLORS["BM"],
       "CBM": report_viz.MAINTENANCE_COLORS["CBM"], "TBM": report_viz.MAINTENANCE_COLORS["TBM"], "PART": "#7540a8", "PLAN": "#b8e2f2",
       "WORK": "#9a6324", "DETECT": "#34495e", "FOCUS": "#c83745", "QUALITY": "#bfc8cf"}
context_rows = []
for r in lots.itertuples():
    tasks = cal_work.loc[(cal_work.STRT_DT < r.END) & (cal_work.END_DT > r.START)]
    detected = cal_work.loc[cal_work.DETC_DT.between(r.START, r.END)]
    relationships = []
    for p in plans.itertuples():
        before = lots.loc[lots.END < p.STRT_DT].tail(1)
        after = lots.loc[lots.START >= p.END_EXCL].head(1)
        if len(before) and before.LOT_NO.iloc[0] == r.LOT_NO:
            relationships.append("수리 전 마지막 관측")
        if len(after) and after.LOT_NO.iloc[0] == r.LOT_NO:
            relationships.append("수리 후 첫 관측")
    context_rows.append({"LOT": int(r.LOT_NO), "GRADE": r.GRADE, "START": r.START, "END": r.END,
                         "MINUTES": r.MINUTES, "CAL_WORK": len(tasks), "CAL_DETECT": len(detected),
                         "PLAN_RELATION": " · ".join(relationships)})
context = pd.DataFrame(context_rows).sort_values("START")
OUT_DIR = Path(FIG_DIR) / "05_all_data_timeserise_jinny"
OUT_DIR.mkdir(parents=True, exist_ok=True)
figure_checks = []
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 30)

def note(text):
    display(Markdown(event_text(text)))

def table(frame, columns=None, opened=False):
    """계산 결과표는 접어 둔다(눌러서 펼침). 본문에는 그림과 글로 쓴 결론·판단표만 펼쳐 둔다."""
    from IPython.display import HTML
    shown = frame if columns is None else frame[columns]
    if isinstance(shown, pd.Series):
        shown = shown.to_frame()
    shown = event_frame(shown)
    # 작은 CP-Z4M도 0으로 반올림하지 않고 과학 표기법으로 보여준다.
    with pd.option_context("display.float_format", lambda value: f"{value:.4g}"):
        body = shown.to_html()
    display(HTML(f"<details{' open' if opened else ''}><summary>표 펼쳐 보기 · {len(shown)}행</summary>{body}</details>"))

# 정비 종류는 색으로도 구분한다.
COL.update({"BM":report_viz.MAINTENANCE_COLORS["BM"],"CBM":report_viz.MAINTENANCE_COLORS["CBM"],"TBM":report_viz.MAINTENANCE_COLORS["TBM"]})
rcParams.update({"font.size": 11.5, "axes.titlesize": 12.5, "axes.labelsize": 11.5,
                 "xtick.labelsize": 10, "ytick.labelsize": 10.5, "savefig.dpi": 160})

def unit(signal):
    if signal in TC:
        return "온도[℃]"
    if signal in OP:
        return "제어출력[%]"
    if signal == "CP-Z4M":
        return "CP-Z4M (기록값)"
    if signal == "CP-Z4":
        return "CP-Z4 (기록값)"
    return signal

def style(ax, xlabel="관측 시작 후 경과시간[분]"):
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color("#a9bac7")
    ax.grid(color="#e4ebf0", linewidth=.6)
    ax.set_axisbelow(True)
    ax.set_xlabel(xlabel, labelpad=8)
    ax.tick_params(pad=5)

def header(fig, title, subtitle="", handles=(), ncol=4):
    return report_viz.header(fig, title, subtitle, handles, ncol)

def lot_handles(ids):
    return [Line2D([0], [0], color=LOT_COLOR[int(k)], lw=2,
                   label=f"{int(k)} · {lot_meta.loc[int(k), 'GRADE']} · {lot_meta.loc[int(k), 'START']:%m/%d}", ls=report_viz.lot_linestyle(int(k))) for k in ids]

def text_box(artist, renderer):
    patch = artist.get_bbox_patch() if hasattr(artist, "get_bbox_patch") else None
    return patch.get_window_extent(renderer) if patch is not None else artist.get_window_extent(renderer)


# 설명은 글자 높이와 짧은 간격에 맞춰 모으고, 확보한 공간을 그래프에 돌려준다.
layout_reviews = []

def compact_explanations(fig):
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    axes = list(fig.axes)
    visible = [ax for ax in axes if ax.axison]
    area_before = sum(ax.get_position().width * ax.get_position().height for ax in visible)
    report = {"CAPTIONS": 0, "AREA_BEFORE": area_before, "AREA_AFTER": area_before,
              "GAIN_PCT": 0.0, "LIMITS_PRESERVED": True}
    if not axes or not visible:
        return report
    positions = {ax: ax.get_position().frozen() for ax in axes}
    bottom = min(p.y0 for p in positions.values())
    top = max(p.y1 for p in positions.values())
    captions = [text for text in fig.texts
                if text is not fig._suptitle and text.get_visible() and text.get_text()
                and text.get_position()[1] < bottom - .003]
    if not captions:
        return report
    # 표·설명 상자처럼 전체 배치가 하나인 그림은 그대로 둔다.
    for artist in fig.artists:
        if hasattr(artist, "get_xy") and hasattr(artist, "get_height"):
            if artist.get_y() < bottom and artist.get_height() > .02:
                return report
    figure_height = fig.get_figheight()
    gap = 7 / 72 / figure_height
    cursor = 10 / 72 / figure_height
    caption_ids = {id(text) for text in captions}
    heights = {id(text): text_box(text, renderer).height / fig.bbox.height for text in captions}
    for text in sorted(captions, key=lambda t: t.get_position()[1]):
        text.set_position((text.get_position()[0], cursor))
        text.set_verticalalignment("bottom")
        cursor += heights[id(text)] + gap
    # 축 아래의 눈금·단위가 설명과 겹치지 않도록 실제 글자 크기를 반영한다.
    depth = 0.0
    for ax in visible:
        if positions[ax].y0 > bottom + .025:
            continue
        labels = [ax.xaxis.label] + ax.get_xticklabels()
        boxes = [text_box(t, renderer) for t in labels if t.get_visible() and t.get_text()]
        if boxes:
            depth = max(depth, bottom - min(b.y0 for b in boxes) / fig.bbox.height)
    target_bottom = max(.065, cursor + depth + gap)
    limits = [(ax.get_xlim(), ax.get_ylim()) for ax in axes]
    if target_bottom < bottom:
        scale = (top - target_bottom) / (top - bottom)
        for ax, old in positions.items():
            ax.set_position([old.x0, target_bottom + (old.y0-bottom)*scale,
                             old.width, old.height*scale])
        for text in fig.texts:
            if id(text) in caption_ids:
                continue
            x, y = text.get_position()
            if bottom <= y <= top:
                text.set_position((x, target_bottom + (y-bottom)*scale))
    # 알파벳은 축 높이와 무관하게 파형 위 8pt 위치에 놓는다.
    for ax in axes:
        height = fig.get_figheight() * ax.get_position().height
        for text in ax.texts:
            if text.get_transform() == ax.transAxes and text.get_text() in ["(a)","(b)","(c)"]:
                text.set_y(1 + 8 / 72 / height)
                text.set_verticalalignment("bottom")
        for line in ax.lines:
            if line.get_transform() != ax.transAxes:
                continue
            y = np.asarray(line.get_ydata(), dtype=float)
            if len(y) == 2 and np.all(y > 1) and np.max(y) < 1.2:
                line.set_ydata([1 + 1 / 72 / height, 1 + 7 / 72 / height])
    assert limits == [(ax.get_xlim(), ax.get_ylim()) for ax in axes], "배치 변경으로 축 범위가 달라짐"
    fig.canvas.draw()
    area_after = sum(ax.get_position().width * ax.get_position().height for ax in visible)
    report.update({"CAPTIONS": len(captions), "AREA_AFTER": area_after,
                   "GAIN_PCT": 100*(area_after/area_before-1)})
    return report

# 저장 파일과 노트북 화면 모두 그림 네 변의 여백을 같게(0.2인치) 둔다.
PAD = report_viz.EXPORT_PADDING
rcParams["savefig.pad_inches"] = PAD

def finish(fig, name, svg=False, save=True):
    """저장할 크기에서 모든 글자와 범례의 겹침·잘림을 확인한다. save=False면 파일로 저장하지 않고 노트북에만 보인다."""
    fig.set_dpi(report_viz.EXPORT_DPI)
    report_viz.prepare(fig)
    layout = compact_explanations(fig)
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    for ax in fig.axes:
        for tag in ax.texts:
            if getattr(tag, "_review_label", False):
                assert not text_box(tag, renderer).overlaps(ax.get_window_extent(renderer)), "확인 표시가 파형을 가림"
    artists = list(fig.texts) + list(fig.legends)
    for ax in fig.axes:
        if not ax.axison:
            continue
        artists += [ax.title, ax._left_title, ax.xaxis.label, ax.yaxis.label,
                    ax.xaxis.offsetText, ax.yaxis.offsetText] + list(ax.texts)
        for axis in [ax.xaxis, ax.yaxis]:
            low, high = sorted(axis.get_view_interval())
            for tick in axis.get_major_ticks():
                if low <= tick.get_loc() <= high:
                    artists += [tick.label1, tick.label2]
    unique = {}
    for artist in artists:
        if not artist.get_visible():
            continue
        if hasattr(artist, "get_text") and not artist.get_text():
            continue
        unique[id(artist)] = artist
    boxes = [(artist, text_box(artist, renderer)) for artist in unique.values()]
    for artist, box in boxes:
        if box.x0 < -.5 or box.y0 < -.5 or box.x1 > fig.bbox.x1 + .5 or box.y1 > fig.bbox.y1 + .5:
            raise RuntimeError(f"글자 잘림: {name}: {getattr(artist, 'get_text', lambda: '범례')()}")
    for i, (artist, box) in enumerate(boxes):
        for other, other_box in boxes[i + 1:]:
            if box.overlaps(other_box):
                raise RuntimeError(f"글자 겹침: {name}: {getattr(artist, 'get_text', lambda: '범례')()} / {getattr(other, 'get_text', lambda: '범례')()}")
    for legend in fig.legends:
        box = legend.get_window_extent(renderer)
        if any(box.overlaps(ax.get_window_extent(renderer)) for ax in fig.axes if ax.axison):
            raise RuntimeError(f"범례가 그래프를 가림: {name}")
    # 여백을 잘라 저장할 때 그림 위쪽 범례만 따로 밀리지 않도록, 범례 위치를 그림 좌표에 다시 묶는다.
    for legend in fig.legends:
        anchor = legend.get_bbox_to_anchor().transformed(fig.transFigure.inverted())
        legend.set_bbox_to_anchor(anchor, transform=fig.transFigure)
    path = OUT_DIR / name
    if save:
        path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(path, dpi=report_viz.EXPORT_DPI, facecolor="white", bbox_inches="tight", pad_inches=PAD)
    if svg and save:
        vector = path.with_suffix(".svg")
        fig.savefig(vector, facecolor="white", bbox_inches="tight", pad_inches=PAD)
        vector.write_text("\n".join(line.rstrip() for line in vector.read_text(encoding="utf-8").splitlines()) + "\n", encoding="utf-8")
    figure_checks.append({"FILE": name, "TITLE": fig._suptitle.get_text(), "N_AXES": len(fig.axes),
                          "TEXT_OVERLAP": "통과", "TEXT_BOUNDS": "통과", "LEGEND_BOUNDS": "통과", "SAVED": "저장" if save else "노트북만"})
    layout_reviews.append({"FILE": name, **layout})
    # 저장한 파일을 다시 불러오지 않고, 방금 그린 그림을 노트북에 바로 그린다.
    # 파일은 위에서 원본 해상도(180dpi)로 저장했고, 노트북 화면용으로만 해상도를 낮춘다.
    fig.set_dpi(72)
    display_report_figure(fig)
    caption = report_viz.details_html(fig)
    if caption:
        from IPython.display import HTML
        display(HTML(caption))
    plt.close(fig)
import matplotlib

# 시간축 태그: 그림 오른쪽 위에 이 그림을 어떤 기준으로 읽는지 적는다.
TAG_REAL = "TIME AXIS · 실제 측정 날짜·시각"
TAG_ELAPSED = "TIME AXIS · 관측 시작 후 경과시간[분]"
TAG_REL = "TIME AXIS · LOT 시작 기준 상대시간[시간]"

def axis_tag(fig, text):
    fig.text(.975, .978, text, ha="right", va="top", fontsize=13, color="#17344d",
             bbox=dict(boxstyle="round,pad=.45", fc="#eef3f7", ec="#9fb3c3", lw=.9))

def compare_tag(target):
    return f"COMPARISON · {target}"

def elapsed_top(ax, start, label="경과[분]"):
    """실제 시각 축 위에 같은 LOT의 관측 시작 후 경과 분을 함께 적는다."""
    base = mdates.date2num(start)
    top = ax.secondary_xaxis("top", functions=(lambda x: (x - base) * 1440, lambda m: base + m / 1440))
    top.set_xlabel(label, fontsize=11, color="#526d80", labelpad=6)
    top.tick_params(labelsize=10, colors="#526d80")
    top.xaxis.set_major_locator(MaxNLocator(6))
    return top

def compact_header(fig, sub_y, legend_y, title_size=25, sub_size=14, legend_size=13):
    """키 큰 그림에서 제목·부제·범례 사이 빈 공간을 줄이고 글자를 키운다."""
    fig._suptitle.set_fontsize(title_size)
    for text in fig.texts:
        if np.isclose(text.get_position()[1], .924):
            text.set_position((.035, sub_y))
            text.set_fontsize(sub_size)
    if fig.legends:
        fig.legends[0].set_bbox_to_anchor((.031, legend_y))
        for text in fig.legends[0].get_texts():
            text.set_fontsize(legend_size)

def shades(color, n=4):
    """LOT 색 하나를 진한 색 → 밝은 색으로 나눈다. 구역 순서를 같은 LOT 안에서 구분한다."""
    rgb = np.array(matplotlib.colors.to_rgb(color))
    dark, light = rgb * .55, rgb * .62 + .38
    return [tuple(dark * (1 - t) + light * t) for t in np.linspace(0, 1, n)]

In [ ]:
# 1초 기대 간격은 원본 시각 차이의 최빈값으로 확인한다. 결측은 보간하지 않는다.
raw, profile, quality, expected, states = {}, {}, {}, {}, {}
quality_rows, state_rows = [], []
STATE_COLOR = {"안정 참고": "#009e73", "초기 승온": "#e69f00", "종료 전환": "#d55e00"}
for r in lots.itertuples():
    k = int(r.LOT_NO)
    g = temp.loc[temp.LOT_NO.eq(k)].sort_values("MEAS_DT").copy()
    assert not g.MEAS_DT.duplicated().any()
    gaps_s = g.MEAS_DT.diff().dt.total_seconds().dropna()
    assert gaps_s.mode().iloc[0] == 1 and gaps_s.ge(1).all()
    g["ELAPSED"] = (g.MEAS_DT - r.START).dt.total_seconds() / 60
    g["MINUTE"] = np.floor(g.ELAPSED).astype(int)
    grid = pd.RangeIndex(int(np.floor(r.MINUTES)) + 1, name="MINUTE")
    p = g.groupby("MINUTE")[SIGNALS].agg(["median", "min", "max", "count"]).reindex(grid)
    exp = pd.Series(np.minimum(60, int((r.END-r.START).total_seconds()) + 1 - grid.to_numpy()*60), index=grid)
    counts = g.groupby("MINUTE").size().reindex(grid, fill_value=0)
    q = p.xs("count", axis=1, level=1).fillna(0).div(exp, axis=0)
    raw[k], profile[k], quality[k], expected[k] = g, p, q, exp
    for s in SIGNALS:
        quality_rows.append({"LOT": k, "GRADE": r.GRADE, "SENSOR": s, "ROWS": len(g),
                             "EXPECTED_SECONDS": int(exp.sum()), "MISSING_TIMES_S": int(exp.sum()-len(g)),
                             "SENSOR_NAN": int(g[s].isna().sum()), "VALID_RATIO": g[s].count()/exp.sum(),
                             "LOW_90_WINDOWS": int(q[s].lt(.9).sum())})
    # 사후 형태 분류다. 실제 운전 모드나 온라인 모델 입력으로 사용하지 않는다.
    st = pd.Series("안정 참고", index=grid, dtype="object")
    early = g.loc[g.ELAPSED.lt(5), HEAT].median()
    later = g.loc[g.ELAPSED.between(80,100), HEAT].median()
    rise = later - early
    warm_end, tail_start = np.nan, np.nan
    if rise.notna().all() and rise.min() >= 20 and g.ELAPSED.between(80,100).shape[0] >= 600:
        near = pd.concat([(p[(s,"median")]-later[s]).abs().le(5 if s=="TC-Z3" else 2) for s in HEAT], axis=1).all(axis=1)
        slow = pd.concat([p[(s,"median")].diff(5).abs().le(2) for s in HEAT[1:]], axis=1).all(axis=1)
        run = (near & slow & grid.to_series().ge(5)).rolling(5, min_periods=5).sum().eq(5)
        warm_end = int(run.index[run][0])+1 if run.any() else len(grid)
        st.loc[grid < warm_end] = "초기 승온"
    middle = g.loc[g.ELAPSED.between(r.MINUTES*.45,r.MINUTES*.6), HEAT].median()
    tail = g.loc[g.ELAPSED.ge(max(0,r.MINUTES-5)), HEAT].median()
    if (tail-middle).notna().all() and (tail-middle).max() <= -10:
        midmask = g.ELAPSED.between(r.MINUTES*.45,r.MINUTES*.6)
        drop = pd.concat([(p[(s,"median")]-middle[s]).le(-10) for s in HEAT], axis=1).sum(axis=1).ge(3)
        cp_drop = (p[("CP-Z4","median")]-g.loc[midmask,"CP-Z4"].median()).le(-.05).rolling(3).sum().eq(3)
        op = ["OP-Z4","OP-Z5","OP-Z6"]
        op_drop = (p.loc[:,[(s,"median") for s in op]].median(axis=1)-g.loc[midmask,op].median().median()).le(-10).rolling(3).sum().eq(3)
        hit = (drop | cp_drop | op_drop) & grid.to_series().ge(r.MINUTES*.6)
        if hit.any():
            tail_start = int(hit.index[hit][0])
            st.loc[grid >= tail_start] = "종료 전환"
    states[k] = st
    state_rows.append({"LOT": k, "GRADE": r.GRADE, "RISE_MIN_C": rise.min(), "RISE_MAX_C": rise.max(),
                       "WARM_END_MIN": warm_end, "TAIL_START_MIN": tail_start,
                       "STABLE_WINDOWS": int(st.eq("안정 참고").sum())})
quality_summary = pd.DataFrame(quality_rows)
state_summary = pd.DataFrame(state_rows)
def runs(mask, minimum=1):
    """연속 True의 [시작, 끝 제외) 경과 분을 돌려준다."""
    a = np.flatnonzero(mask.to_numpy(bool))
    if not len(a):
        return []
    groups = np.split(a, np.flatnonzero(np.diff(a)>1)+1)
    return [(int(mask.index[z[0]]), int(mask.index[z[-1]])+1) for z in groups if len(z)>=minimum]

# 관측 범위와 실제 CAL01 작업의 교집합을 경과 분으로 계산한다. 같은 시각의 작업은 한 번만 센다.
def merge_ranges(intervals):
    merged = []
    for start, end in sorted(intervals):
        if end <= start:
            continue
        if merged and start <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(end, merged[-1][1]))
        else:
            merged.append((start, end))
    return merged

work_ranges = {}
for r in lots.itertuples():
    k = int(r.LOT_NO)
    tasks = cal_work.loc[cal_work.STRT_DT.lt(r.END) & cal_work.END_DT.gt(r.START)]
    work_ranges[k] = merge_ranges([((max(r.START, t.STRT_DT) - r.START).total_seconds() / 60,
                                    (min(r.END, t.END_DT) - r.START).total_seconds() / 60) for t in tasks.itertuples()])
assert np.isclose(sum(b - a for a, b in work_ranges[240117]), 304)

NONCYCLIC = [s for s in TC if s not in ["TC-Z9","TC-OUT1","TC-OUT2"]]

relation = context.set_index("LOT").PLAN_RELATION
expected_rows = []
for k in lots.LOT_NO.astype(int):
    for state, kind in [("종료 전환", "수리 전 하강 전환"), ("초기 승온", "수리 후 승온 형태")]:
        for a, b in runs(states[k].eq(state)):
            near_plan = bool(relation.get(k))
            expected_rows.append({"LOT": k, "GRADE": lot_meta.loc[k, "GRADE"], "종류": kind if near_plan else state + " · 일정 없음",
                                  "START_MIN": float(a), "END_MIN": float(min(b, lot_meta.loc[k, "MINUTES"])),
                                  "시작": lot_meta.loc[k, "START"] + pd.Timedelta(minutes=float(a)),
                                  "끝": lot_meta.loc[k, "START"] + pd.Timedelta(minutes=float(min(b, lot_meta.loc[k, "MINUTES"]))),
                                  "수리 관계": relation.get(k) or "-"})
expected = pd.DataFrame(expected_rows)

def in_expected(k, minute):
    d = expected.loc[expected.LOT.eq(k)]
    return bool(((d.START_MIN <= minute) & (d.END_MIN > minute)).any())




# 강종 수준 대조는 수리 전의 완만한 냉각도 보수적으로 제외한다.
# 2℃는 사후 대조 구간을 자르는 기술 기준이며 설비 이상 임계값이 아니다.
comparison_states={k:st.copy() for k,st in states.items()}
for k in [240058,240114]:
    r=lot_meta.loc[k];p=profile[k]
    ref=raw[k].loc[raw[k].ELAPSED.between(r.MINUTES*.2,r.MINUTES*.6),HEAT[1:]].median()
    falling=pd.concat([(p[(s,'median')]-ref[s]).le(-2) for s in HEAT[1:]],axis=1).all(axis=1)
    hit=falling.rolling(3,min_periods=3).sum().eq(3)&pd.Series(p.index,index=p.index).gt(r.MINUTES*.5)
    if hit.any():comparison_states[k].loc[p.index>=max(0,int(hit.index[hit][0])-2)]='종료 전환'

level_rows = []
for grade, sub in lots.groupby("GRADE", sort=False):
    ids = sub.LOT_NO.astype(int).tolist()
    for s in SIGNALS:
        med = {k: profile[k][(s, "median")].where(comparison_states[k].eq("안정 참고") & profile[k][(s,"count")].ge(54)).median() for k in ids}
        for k in ids:
            peers = [med[o] for o in ids if o != k and pd.notna(med[o])]
            level_rows.append({"LOT": k, "GRADE": grade, "SENSOR": s, "LOT_MEDIAN": med[k],
                               "PEER_MEDIAN": np.median(peers) if peers else np.nan, "N_PEERS": len(peers)})
level_gap = pd.DataFrame(level_rows)
level_gap["GAP"] = level_gap.LOT_MEDIAN - level_gap.PEER_MEDIAN



# 동종 LOT 수준 차이의 표시 범위. 시간순 Z 점수·고장 판정과 별개다.
LEVEL_TC=[f'TC-Z{i}' for i in range(1,9)]
LEVEL_OP=[f'OP-Z{i}' for i in range(1,9)]
LEVEL_SIGNALS=LEVEL_TC+LEVEL_OP
LEVEL_TC_LIMIT,LEVEL_OP_LIMIT=2.5,4.0
level_check=level_gap.loc[level_gap.SENSOR.isin(LEVEL_SIGNALS)].copy()
level_check['DISPLAY_LIMIT']=np.where(level_check.SENSOR.isin(LEVEL_TC),LEVEL_TC_LIMIT,LEVEL_OP_LIMIT)
level_check['DISPLAY_HIT']=level_check.N_PEERS.gt(0)&level_check.GAP.abs().ge(level_check.DISPLAY_LIMIT)
level_check['STATUS']=np.where(level_check.N_PEERS.eq(0),'동일 강종 대조 없음',np.where(level_check.DISPLAY_HIT,'수준 차이 표시 대상','표시 경계 미만'))
# 기존 우선 검토 5 LOT의 그림 범위는 이제 위 16신호 계산 결과에서 결정한다.
LEVEL_TARGETS={k:level_check.loc[level_check.LOT.eq(k)&level_check.DISPLAY_HIT,'SENSOR'].tolist()
               for k in [240038,240061,240091,240114,240121]}
level_inventory=level_gap.copy()
level_inventory['비교 용도']=level_inventory.SENSOR.map(lambda s:'TC·OP 수준 비교' if s in LEVEL_SIGNALS else 'LOT 내부 국소 변화' if s=='CLN-IN' else '주기·위상 별도 확인' if s in ['TC-Z9','TC-OUT1','TC-OUT2'] else 'CP 원본·계측 정의 별도 확인')
assert len(level_check)==len(lots)*16 and all(LEVEL_TARGETS.values())

from matplotlib.patches import Rectangle
SELECT_COLOR="#c83745"

def sensor_color(s):
    return report_viz.sensor_color(s)

def remark_text(t):return str(t.REMARK).strip() if pd.notna(t.REMARK) else "작업 내용 미기재"

def parts_for(wo):
    return [f"{p.PART_NM} [{p.PART_CD}] ×{p.QTY:g} ({p.CHG_RSN})" for p in cal_parts.loc[cal_parts.WO_NO.eq(wo)].itertuples()]

def linked_work(k,pad_hours=0):
    r=lot_meta.loc[k]
    return cal_work.loc[cal_work.STRT_DT.lt(r.END+pd.Timedelta(hours=pad_hours))&cal_work.END_DT.gt(r.START-pd.Timedelta(hours=pad_hours))].sort_values("STRT_DT")

# 품질 판단과 본문이 같은 온도 변화 점수 객체를 사용한다.
Z_WINDOW,Z_MIN,Z_LIMIT=30,20,3
zstats={};z_rows=[]
for k in lots.LOT_NO.astype(int):
    for s in NONCYCLIC+OP:
        x=profile[k][(s,'median')].where(profile[k][(s,'count')].ge(54))
        past=x.shift(1);center=past.rolling(Z_WINDOW,min_periods=Z_MIN).median()
        sd=past.rolling(Z_WINDOW,min_periods=Z_MIN).std().clip(lower=.5 if s in TC else 2.)
        z=(x-center)/sd;zstats[k,s]=pd.DataFrame({'X':x,'MEDIAN':center,'SD':sd,'Z':z})
    d=pd.DataFrame({s:zstats[k,s].Z for s in NONCYCLIC})
    for m,v in d.abs().max(axis=1).items():z_rows.append({'LOT':k,'MINUTE':m,'MAX_ABS_Z':v})
zscreen=pd.DataFrame(z_rows)


# IQR은 각 LOT·신호의 전환 제외 유효 1분 중앙값 분포로 계산한다.
# 강종을 섞거나 긴 LOT의 모든 기록을 합쳐 정상 기준을 만들지 않는다.
iqr_rows=[];iqr_intervals=[];iqr_data={};iqr_masks={}
for r in lots.itertuples():
    k=int(r.LOT_NO)
    for sensor in LEVEL_SIGNALS:
        x=profile[k][(sensor,'median')].where(comparison_states[k].eq('안정 참고')&profile[k][(sensor,'count')].ge(54))
        usable=x.dropna();q1,q3=usable.quantile([.25,.75]);iq=q3-q1;lo,hi=q1-1.5*iq,q3+1.5*iq
        flag=x.lt(lo)|x.gt(hi) if iq>0 else pd.Series(False,index=x.index)
        iqr_data[k,sensor]=usable;iqr_masks[k,sensor]=flag
        groups=runs(flag)
        iqr_rows.append(dict(LOT=k,GRADE=r.GRADE,SENSOR=sensor,N=len(usable),MEAN=usable.mean(),MEDIAN=usable.median(),
                             Q1=q1,Q3=q3,IQR=iq,LOW=lo,HIGH=hi,OUT=int(flag.sum()),RUNS=len(groups),STATUS='탐색 가능' if iq>0 else 'IQR 0: 판정 보류'))
        for aa,bb in groups:
            y=x.loc[(x.index>=aa)&(x.index<bb)]
            iqr_intervals.append(dict(LOT=k,GRADE=r.GRADE,SENSOR=sensor,START=lot_meta.loc[k,'START']+pd.Timedelta(minutes=float(aa)),
                END=min(lot_meta.loc[k,'START']+pd.Timedelta(minutes=float(bb)),r.END),MINUTES=int(flag.loc[(flag.index>=aa)&(flag.index<bb)].sum()),
                LOW=lo,HIGH=hi,MIN=y.min(),MAX=y.max()))
iqr_summary=pd.DataFrame(iqr_rows);iqr_intervals=pd.DataFrame(iqr_intervals)
assert len(iqr_summary)==14*16


In [ ]:
# CP: 전체 LOT 중앙값을 기준으로 한 사후 탐색이다. 온라인 경보 기준으로 사용하지 않는다.
CP_GROUP, CP_LIMIT = .02, .05
RED, AMBER, GREY = "#B11226", "#b8863b", "#8496a5"

# 음영·강조선은 구간 분류, 센서 파형은 기존 센서별 색을 사용한다.
EVENT_COLORS={'CP':'#B11226','Z':'#D17A00','전환':GREY,'감지':'#7540A8','참고':'#477B94'}
EVENT_LABELS={'CP':'CP 원본 이탈','Z':'국소 변화(Z)','전환':'승온·하강 전환','감지':'정비 감지 대조'}
EVENT_LINES={'CP':':','Z':'--','전환':'-.','감지':':','참고':':'}
assert len({EVENT_COLORS[k] for k in EVENT_LABELS})==4

cp_rows=[]
for k in lots.LOT_NO.astype(int):
    s=raw[k].set_index('MEAS_DT')['CP-Z4'].dropna()
    dev=s-s.median(); hit=dev.abs().gt(CP_GROUP)
    groups=(hit.ne(hit.shift()) | s.index.to_series().diff().dt.total_seconds().gt(5)).cumsum()
    # 최대 5초 간격의 기록은 한 사건으로 묶되 보간하지 않는다. 5초 초과 공백에서 분리한다.
    for _, block in dev[hit].groupby(groups[hit]):
        peak=block.abs().idxmax()
        if abs(block.loc[peak])>=CP_LIMIT:
            a,b,m=[(t-lot_meta.loc[k,'START']).total_seconds()/60 for t in (block.index[0],block.index[-1],peak)]
            cp_rows.append(dict(LOT=k,PEAK=peak,DEV=block.loc[peak],A=a,B=b,M=m,EXP=in_expected(k,m)))
cp_events=pd.DataFrame(cp_rows)
cp_real=cp_events.loc[~cp_events.EXP].copy()
cp_count=cp_real.groupby('LOT').size()
CP_REVIEW=[240037,240069,240117]  # 반복 2회 이상 또는 작업 완료 뒤의 변화 사례
LEVEL_REVIEW=list(LEVEL_TARGETS)
assert set(cp_count[cp_count.ge(2)].index)=={240037,240069}
assert set(CP_REVIEW)|set(LEVEL_REVIEW)=={240037,240038,240061,240069,240091,240114,240117,240121}
assert len(lots)==14 and len(cal_work)==44 and len(SIGNALS)==22

def show_table(frame):
    from IPython.display import HTML
    display(HTML(event_frame(frame).to_html(index=False,escape=True,float_format=lambda x:f'{x:.3g}')))


import textwrap
from matplotlib.transforms import Bbox

def report_page(main,sub,rows,width=20,info=None,event_key=False,cards=None):
    """제목·카드·설명·범례·파형을 독립 영역에 두어 실제 렌더링 경계를 검증한다."""
    main,sub=event_text(main),event_text(sub)
    info=[event_text(line) for line in info] if info else info
    cards=[{**c,'title':event_text(c['title']),'value':event_text(c['value']),
             'lines':[event_text(line) for line in c.get('lines',[])]} for c in cards] if cards else cards
    specs=[]
    for row in rows:
        if isinstance(row,dict):specs.append((row['panels'],row.get('height',1.4),row.get('captions')))
        else:specs.append((row,1.4,None))
    card_lines=[]
    if cards:
        for c in cards:
            lines=[]
            for line in c.get('lines',[]):lines.extend(textwrap.wrap(line,width=40) or [''])
            card_lines.append(lines)
        card_h=.85+.24*max(map(len,card_lines))
    else:card_h=0
    title_h=1.25+len(info or [])*.28+(.40 if event_key else 0)+(card_h+.18 if cards else 0)
    tick_h,gap=.55,.12
    height=title_h+sum((1.6 if captions else .58)+h+tick_h+gap for _,h,captions in specs)+.12
    fig=plt.figure(figsize=(width,height),facecolor='white')
    fig.suptitle(main,x=.07,y=1-.16/height,ha='left',va='top',fontsize=19,fontweight='bold',color='#17344d')
    fig.text(.07,1-.55/height,textwrap.fill(sub,width=125),ha='left',va='top',fontsize=10.5,color='#526d80',linespacing=1.5)
    offset=0
    if cards:
        from matplotlib.patches import FancyBboxPatch
        left,right,hgap=.075,.98,.022;w=(right-left-(len(cards)-1)*hgap)/len(cards)
        for j,(c,lines) in enumerate(zip(cards,card_lines)):
            x=left+j*(w+hgap);top=1.02;color=c.get('color','#477B94')
            fig.add_artist(FancyBboxPatch((x,1-(top+card_h)/height),w,card_h/height,
                boxstyle='round,pad=0.004,rounding_size=0.006',transform=fig.transFigure,
                facecolor=c.get('face','#f4f7fa'),edgecolor=color,linewidth=.7,zorder=0))
            for depth,text,size,bold,tint in [(top+.10,c['title'],11,True,color),(top+.37,c['value'],14,True,color)]+[
                (top+.77+q*.24,line,10.5,False,'#17344d') for q,line in enumerate(lines)]:
                fig.text(x+.012,1-depth/height,text,ha='left',va='top',fontsize=size,
                         fontweight='bold' if bold else 'normal',color=tint)
        offset=card_h+.18
    for i,line in enumerate(info or []):
        fig.text(.075,1-(1.12+offset+i*.28)/height,line,ha='left',va='top',fontsize=10.5,color='#17344d')
    if event_key:
        from matplotlib.patches import Patch
        handles=[Patch(facecolor=EVENT_COLORS[k],edgecolor=EVENT_COLORS[k],alpha=.28,label=label) for k,label in EVENT_LABELS.items()]
        fig.legend(handles=handles,loc='upper left',bbox_to_anchor=(.075,1-(1.12+offset+len(info or [])*.28)/height),
                   ncol=4,fontsize=10.5,frameon=False,handlelength=2.,columnspacing=2.,borderaxespad=0)
    cursor=title_h;cells=[];left,right,hgap=.075,.98,.065
    for panels,data_h,captions in specs:
        head_h=1.6 if captions else .58
        count=len(panels);w=(right-left-(count-1)*hgap)/count;rowcells=[]
        for col,(label,sensors) in enumerate(panels):
            x=left+col*(w+hgap)
            head=fig.add_axes([x,1-(cursor+head_h)/height,w,head_h/height]);head.axis('off');head._has_note_gutter=bool(captions)
            original_label=label
            label=event_text(label,include_lot=False)
            display_label_checks.append((original_label,label))
            head.text(0,.98,label,ha='left',va='top',fontsize=12,fontweight='bold',color='#17344d',transform=head.transAxes)
            if captions:
                head.text(0,.73,textwrap.fill(event_text(captions[col]),width=105 if count==1 else 64 if count==2 else 44),
                          ha='left',va='top',fontsize=9.5,color='#526d80',linespacing=1.35,transform=head.transAxes)
            ax=fig.add_axes([x,1-(cursor+head_h+data_h)/height,w,data_h/height])
            ax._note_gutter=.45 if captions else 0
            style(ax,'');ax.ticklabel_format(axis='y',style='plain',useOffset=False);ax.tick_params(labelsize=9,pad=3)
            ax.yaxis.set_major_locator(MaxNLocator(4));ax.xaxis.label.set_fontsize(10);ax.yaxis.label.set_fontsize(10)
            rowcells.append((ax,head,sensors))
        cells.append(rowcells);cursor+=head_h+data_h+tick_h+gap
    fig._report_cells=cells
    return fig,cells


def panel_legend(head,ax,extra=()):
    handles,labels=ax.get_legend_handles_labels()
    handles+=list(extra);labels+=[h.get_label() for h in extra]
    if handles:
        head.legend(handles,labels,loc='lower left',bbox_to_anchor=(0,.34 if getattr(head,'_has_note_gutter',False) else .1),ncol=min(4,len(handles)),
                    fontsize=9,frameon=False,handlelength=2.,columnspacing=1.6,borderaxespad=0)

def clock_ticks(ax,start=None,end=None):
    if start is not None and end is not None:ax.set_xlim(start,end if end>start else start+pd.Timedelta(seconds=1))
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=3,maxticks=5))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d\n%H:%M'))
    ax.tick_params(axis='x',labelsize=9,pad=4)

def display_unit(s):
    return '온도[℃]' if s in TC else '출력[%]' if s in OP else 'CP-Z4M [기록값 x 1e-10]' if s=='CP-Z4M' else 'CP-Z4 [기록값]'

def report_signal(ax,k,s,lo=None,hi=None,fine=False,**kwargs):
    if fine or s.startswith('CP'):
        y=raw[k].set_index('MEAS_DT')[s].asfreq('s')
        if lo is not None:y=y.loc[lo:hi]
        xx=y.index
    else:
        y=profile[k][(s,'median')].where(profile[k][(s,'count')].ge(54));xx=at(k,y.index+.5)
    ax.plot(xx,y.to_numpy()*(1e10 if s=='CP-Z4M' else 1),color=sensor_color(s),lw=1.2,label=s,**kwargs)

def note_box(ann,renderer):
    return ann.get_bbox_patch().get_window_extent(renderer)


def label_hits_line(ax,box):
    # 실선·점선·수평 기준선의 실제 경로와 상자 경계를 대조한다. NaN 공백은 연결하지 않는다.
    for line in ax.lines:
        if not line.get_visible():continue
        path=line.get_path().transformed(line.get_transform())
        if path.intersects_bbox(box,filled=False):return True
    return False


def near_note(ax,x,y,label,color='#17344d'):
    """지점에서 가까운 빈 공간을 탐색한다. 글상자가 파형을 덮는 위치는 사용하지 않는다."""
    label=event_text(label,include_lot=False)
    if not np.isfinite(y):return
    fig=ax.figure
    if not hasattr(fig,'_near_notes'):fig._near_notes=[]
    ax.margins(y=.5)
    fig.canvas.draw();renderer=fig.canvas.get_renderer()
    existing=[note_box(a,renderer).expanded(1.04,1.08) for a in fig._near_notes if a.axes is ax]
    offsets=[(dx,dy) for radius in [4,6,10,16,24,34,46] for dx,dy in [(radius,radius),(-radius,radius),(radius,-radius),(-radius,-radius),(radius,0),(-radius,0),(0,radius),(0,-radius)]]
    for dx,dy in offsets:
        ann=ax.annotate(label,xy=(x,y),xytext=(dx,dy),textcoords='offset points',
            ha='right' if dx<0 else 'left' if dx>0 else 'center',va='top' if dy<0 else 'bottom',
            fontsize=9.5,color=color,bbox=dict(boxstyle='round,pad=.32',fc='white',ec=color,lw=.8),
            arrowprops=dict(arrowstyle='->',color=color,lw=.85,shrinkA=3,shrinkB=2),zorder=9)
        ann.update_positions(renderer);ann.update_bbox_position_size(renderer)
        box=note_box(ann,renderer).expanded(1.02,1.06)
        inside=ax.bbox.contains(box.x0,box.y0) and ax.bbox.contains(box.x1,box.y1)
        px,py=ax.transData.transform((mdates.date2num(x),y))
        distance=np.hypot(max(box.x0-px,px-box.x1,0),max(box.y0-py,py-box.y1,0))
        inside=inside and distance<=100*fig.dpi/150
        if inside and not label_hits_line(ax,box) and not any(box.overlaps(old) for old in existing):
            fig._near_notes.append(ann);return ann
        ann.remove()
    # 경계의 최고점은 같은 x 부근의 전용 상단 여백을 쓴다. 멀리 떨어진 모서리로 보내지 않는다.
    if getattr(ax,'_note_gutter',0):
        px,py=ax.transData.transform((mdates.date2num(x),y));dy=(ax.bbox.y1+5-py)*72/fig.dpi
        for ha in ['left','right','center']:
            ann=ax.annotate(label,xy=(x,y),xytext=(0,dy),textcoords='offset points',ha=ha,va='bottom',fontsize=9.5,
                color=color,bbox=dict(boxstyle='round,pad=.32',fc='white',ec=color,lw=.8),
                arrowprops=dict(arrowstyle='->',color=color,lw=.85,shrinkA=3,shrinkB=2),zorder=9)
            ann.update_positions(renderer);ann.update_bbox_position_size(renderer);box=note_box(ann,renderer)
            distance=np.hypot(max(box.x0-px,px-box.x1,0),max(box.y0-py,py-box.y1,0))
            if box.x0>=ax.bbox.x0 and box.x1<=ax.bbox.x1 and box.y1<=ax.bbox.y1+ax._note_gutter*fig.dpi and distance<=100*fig.dpi/150 and not label_hits_line(ax,box):
                fig._near_notes.append(ann);return ann
            ann.remove()
    raise AssertionError(f'가까운 빈 주석 위치 없음: {label}')


def validate_near_notes(fig):
    renderer=fig.canvas.get_renderer()
    for ann in getattr(fig,'_near_notes',[]):
        box=note_box(ann,renderer).expanded(1.01,1.02)
        assert not label_hits_line(ann.axes,box),f'주석이 파형을 가림: {ann.get_text()}'
        point=ann.axes.transData.transform((mdates.date2num(ann.xy[0]),ann.xy[1]))
        dx=max(box.x0-point[0],point[0]-box.x1,0);dy=max(box.y0-point[1],point[1]-box.y1,0)
        assert np.hypot(dx,dy)<=100, f'주석이 지점에서 너무 멂: {ann.get_text()}'


def save_lot_sheet(pixels,name,axes_count):
    # 저장 PNG를 읽지 않는다. 방금 코드로 그린 Figure의 캔버스를 수직으로 묶는다.
    from PIL import Image
    width=max(a.shape[1] for a in pixels);height=sum(a.shape[0] for a in pixels)
    sheet=Image.new('RGB',(width,height),'white');top=0
    for array in pixels:
        frame=Image.fromarray(array).convert('RGB');sheet.paste(frame,(0,top));top+=frame.height
    path=OUT_DIR/name;path.parent.mkdir(parents=True,exist_ok=True);sheet.save(path)
    figure_checks.append(dict(FILE=name,TITLE='모든 구간 동등 상세',N_AXES=axes_count,SAVED='저장',TEXT_OVERLAP='통과',LEGEND_BOUNDS='통과'))


def checked_emit(fig,name,save=True,record=True):
    """모든 제목/범례/눈금의 실제 렌더링 경계를 검사한다. 축 사각형만 검사하지 않는다."""
    fig.set_dpi(150);fig.canvas.draw();renderer=fig.canvas.get_renderer()
    artists=[]
    for t in fig.texts:
        if t.get_visible() and t.get_text():artists.append((t,t.get_window_extent(renderer)))
    for ax in fig.axes:
        for t in ax.texts:
            if t.get_visible() and t.get_text():artists.append((t,t.get_window_extent(renderer)))
        leg=ax.get_legend()
        if leg is not None:artists.append((leg,leg.get_window_extent(renderer)))
        if ax.axison:
            for t in [ax.xaxis.label,ax.yaxis.label,ax.xaxis.offsetText,ax.yaxis.offsetText]:
                if t.get_visible() and t.get_text():artists.append((t,t.get_window_extent(renderer)))
            for axis in [ax.xaxis,ax.yaxis]:
                low,high=sorted(axis.get_view_interval())
                for tick in axis.get_major_ticks():
                    if low-1e-7<=tick.get_loc()<=high+1e-7:
                        for t in [tick.label1,tick.label2]:
                            if t.get_visible() and t.get_text():artists.append((t,t.get_window_extent(renderer)))
    for legend in fig.legends:
        artists.append((legend,legend.get_window_extent(renderer)))
        for body in [p[0] for row in getattr(fig,'_report_cells',[]) for p in row]:
            overlap=Bbox.intersection(legend.get_window_extent(renderer),body.bbox)
            assert overlap is None or overlap.height<=1 or overlap.width<=1, f'구간 범례가 파형을 가림: {name}'
    for a,bb in artists:
        assert bb.x0>=-.5 and bb.y0>=-.5 and bb.x1<=fig.bbox.x1+.5 and bb.y1<=fig.bbox.y1+.5, f'글자 잘림: {name} / {getattr(a,"get_text",lambda:"범례")()}'
    for i,(a,bb) in enumerate(artists):
        for other,ob in artists[i+1:]:
            inter=Bbox.intersection(bb,ob)
            if inter is not None and inter.width>1 and inter.height>1:
                raise AssertionError(f'글자/범례 겹침: {name} / {getattr(a,"get_text",lambda:"범례")()} / {getattr(other,"get_text",lambda:"범례")()}')
    # 범례가 데이터 파형 영역을 덮는지도 별도로 검사한다.
    for ax in fig.axes:
        leg=ax.get_legend()
        if leg is not None:
            for body in [p[0] for row in getattr(fig,'_report_cells',[]) for p in row]:
                overlap=Bbox.intersection(leg.get_window_extent(renderer),body.bbox)
                assert overlap is None or overlap.height<=1 or overlap.width<=1, f'범례가 파형을 가림: {name}'
    validate_near_notes(fig)
    pixels=np.asarray(fig.canvas.buffer_rgba()).copy() if not save else None
    path=OUT_DIR/name
    if save:
        path.parent.mkdir(parents=True,exist_ok=True)
        fig.savefig(path,dpi=150,facecolor='white')
    if record:figure_checks.append({'FILE':name,'TITLE':fig._suptitle.get_text(),'N_AXES':len(fig.axes),'SAVED':'저장','TEXT_OVERLAP':'통과','LEGEND_BOUNDS':'통과'})
    fig.set_dpi(80)
    if not getattr(fig,'_silent',False):display_report_figure(fig)
    plt.close(fig)
    return pixels

emit=checked_emit


def at(k,m):return lot_meta.loc[k,'START']+pd.to_timedelta(m,unit='min')



def draw_boxes(ax,head,sensor,ids,highlight=None):
    values=[iqr_data[k,sensor].to_numpy() for k in ids]
    box=ax.boxplot(values,positions=np.arange(len(ids)),widths=.58,patch_artist=True,whis=1.5,
                   showmeans=True,meanprops=dict(marker='D',markersize=3.5,markerfacecolor='#17344d',markeredgecolor='#17344d'),
                   medianprops=dict(color='#17344d',linewidth=1.6),flierprops=dict(marker='o',markersize=2,markerfacecolor='#147f9c',markeredgecolor='none',alpha=.55))
    for patch,k in zip(box['boxes'],ids):patch.set_facecolor('#147f9c' if k==highlight else '#dce8ee');patch.set_alpha(.65)
    ax.set_xticks(range(len(ids)),[f'{k}\n{lot_meta.loc[k,"GRADE"]}' for k in ids],fontsize=8)
    ax.set_ylabel(display_unit(sensor));ax.set_xlim(-.7,len(ids)-.3)
    panel_legend(head,ax,[Line2D([0],[0],color='#17344d',label='중앙값'),Line2D([0],[0],marker='D',lw=0,color='#17344d',label='평균'),
                          Line2D([0],[0],marker='o',lw=0,color='#147f9c',label='1.5 IQR 밖 1분창')])


def display_report_figure(fig):
    label=getattr(fig,'_folded',None)
    if not label:display(fig);return
    from io import BytesIO
    from base64 import b64encode
    from html import escape
    from IPython.display import HTML
    data=BytesIO();fig.savefig(data,format='png',dpi=90,facecolor='white')
    display(HTML('<details><summary><b>'+escape(label)+'</b></summary><img style="max-width:100%;height:auto" src="data:image/png;base64,'+b64encode(data.getvalue()).decode()+'"></details>'))



def captured_details(label,captured):
    from IPython.display import HTML
    from html import escape
    import mistune
    pieces=[]
    for item in captured.outputs:
        data=item.data
        if 'text/html' in data:pieces.append(data['text/html'])
        elif 'image/png' in data:pieces.append('<img style="max-width:100%" src="data:image/png;base64,'+data['image/png']+'">')
        elif 'text/markdown' in data:pieces.append(mistune.html(data['text/markdown']))
        elif 'text/plain' in data:pieces.append('<pre>'+escape(data['text/plain'])+'</pre>')
    display(HTML('<details><summary><b>'+escape(label)+'</b></summary>'+''.join(pieces)+'</details>'))


In [ ]:
# 모든 LOT에 같은 검토 순서를 적용한다. 변화 점수를 고장 라벨로 바꾸지 않는다.
FLOW_ROWS=[('세정 CLN-IN',['CLN-IN'],None),('건조 Z1·Z2',['TC-Z1','TC-Z2'],['OP-Z1','OP-Z2']),
 ('가열 초입 Z3',['TC-Z3'],['OP-Z3']),('가열·균열 Z4–Z6',['TC-Z4','TC-Z5','TC-Z6'],['OP-Z4','OP-Z5','OP-Z6']),
 ('CP 기록',['CP-Z4'],['CP-Z4M']),('과시효 Z7·Z8',['TC-Z7','TC-Z8'],['OP-Z7','OP-Z8']),('Z9 / 냉각',['TC-Z9'],['TC-OUT1','TC-OUT2'])]
TYPE_COLOR=EVENT_COLORS

def review_events(k):
    duration=float(lot_meta.loc[k,'MINUTES']);out=[]
    ev=cp_real.loc[cp_real.LOT.eq(k)].sort_values('PEAK')
    for _,g in ev.groupby(ev.M.diff().gt(45).cumsum()):
        peak=g.loc[g.DEV.abs().idxmax()]
        out.append(dict(kind='CP',a=max(0,float(g.A.min())-8),b=min(duration,float(g.B.max())+10),point=peak.PEAK,
                        sensor='CP-Z4',deviation=float(peak.DEV),pulses=g.copy(),title=f'CP {len(g)}개 사건 · 최대 {peak.DEV:+.3f}'))
    for e in expected.loc[expected.LOT.eq(k)].itertuples():
        out.append(dict(kind='전환',a=max(0,float(e.START_MIN)-10),b=min(duration,float(e.END_MIN)+5),point=e.시작,
                        sensor='TC-Z3',title=e.종류))
    # 1분창 종료 시각 기준으로 표시한다. 원본의 순간 최고점이라는 뜻은 아니다.
    for signals in [NONCYCLIC,['CLN-IN']]:
        z=pd.DataFrame({s:zstats[k,s].Z for s in signals}).where(states[k].eq('안정 참고'),axis=0)
        ranked=z.abs().stack().sort_values(ascending=False)
        if ranked.empty or ranked.iloc[0]<Z_LIMIT:continue
        (m,sensor),strength=next(iter(ranked.items()))
        if any(e['a']<=m+1<=e['b'] for e in out):continue
        out.append(dict(kind='Z',a=max(0,float(m)-12),b=min(duration,float(m)+16),point=at(k,float(m)+1),
                        sensor=sensor,score=float(z.loc[m,sensor]),minute=int(m),title=f'{sensor} |Z|={strength:.2f} · 국소 변화'))
    for w in cal_work.loc[cal_work.DETC_DT.between(lot_meta.loc[k,'START'],lot_meta.loc[k,'END'])].itertuples():
        m=(w.DETC_DT-lot_meta.loc[k,'START']).total_seconds()/60
        if not any(e['a']<=m<=e['b'] for e in out):
            out.append(dict(kind='감지',a=max(0,m-20),b=min(duration,m+20),point=w.DETC_DT,sensor='CP-Z4',title=f'{w.MNT_TYPE} {w.WO_NO[-5:]} 감지 시각 대조'))
    out.sort(key=lambda e:e['a'])
    for i,e in enumerate(out):
        e['tag']=f'({chr(97+i)})';e['color']=TYPE_COLOR[e['kind']]
        e['transition_context']=any(e['a']<float(t.END_MIN) and e['b']>float(t.START_MIN) for t in expected.loc[expected.LOT.eq(k)].itertuples())
    return out

LOT_EVENTS={int(k):review_events(int(k)) for k in lots.LOT_NO}

def wo_distance(w,r):
    workgap=max((r.START-w.END_DT).total_seconds(),(w.STRT_DT-r.END).total_seconds(),0)
    detgap=max((r.START-w.DETC_DT).total_seconds(),(w.DETC_DT-r.END).total_seconds(),0)
    return min(workgap,detgap)

def lot_works(k):
    r=lot_meta.loc[k]
    direct=cal_work.loc[(cal_work.STRT_DT.lt(r.END)&cal_work.END_DT.gt(r.START))|cal_work.DETC_DT.between(r.START,r.END)]
    if len(direct):return direct.sort_values('STRT_DT')
    distances=cal_work.apply(lambda w:wo_distance(w,r),axis=1)
    return cal_work.loc[[distances.idxmin()]]

def primary_work(k):
    ws=lot_works(k)
    # 240117은 두 실제 교집합 WO 중 CP 이전 마지막 완료 CBM을 기준으로 전후를 나눈다.
    ev=cp_real.loc[cp_real.LOT.eq(k)]
    if len(ev):
        peak=ev.loc[ev.DEV.abs().idxmax(),'PEAK']
        prev=ws.loc[ws.END_DT.le(peak)]
        if len(prev):return prev.sort_values('END_DT').iloc[-1]
    return ws.iloc[0]

def observation_phases(w,pad_hours=72):
    # 같은 LOT 안에서도 공백은 채우지 않는다. 전·후는 가장 가까운 실제 관측 1 LOT만 쓴다.
    before=temp.loc[temp.MEAS_DT.lt(w.STRT_DT)&temp.MEAS_DT.ge(w.STRT_DT-pd.Timedelta(hours=pad_hours))]
    after=temp.loc[temp.MEAS_DT.ge(w.END_DT)&temp.MEAS_DT.le(w.END_DT+pd.Timedelta(hours=pad_hours))]
    during=temp.loc[temp.MEAS_DT.ge(w.STRT_DT)&temp.MEAS_DT.lt(w.END_DT)]
    if len(before):
        last=before.MEAS_DT.max();key=int(before.loc[before.MEAS_DT.idxmax(),'LOT_NO'])
        before=before.loc[before.LOT_NO.eq(key)&before.MEAS_DT.ge(last-pd.Timedelta(minutes=45))]
    if len(after):
        first=after.MEAS_DT.min();key=int(after.loc[after.MEAS_DT.idxmin(),'LOT_NO'])
        after=after.loc[after.LOT_NO.eq(key)&after.MEAS_DT.le(first+pd.Timedelta(minutes=45))]
    return {'착수 전':before,'작업 중':during,'완료 후':after}

LOT_PHASES={k:observation_phases(primary_work(k)) for k in LOT_EVENTS}

def degree_zone(s):
    return int(s.split('-Z')[1]) if '-Z' in s and s.split('-Z')[1].isdigit() else None

def paint_event(ax,k,e,shade=True):
    if shade:ax.axvspan(at(k,e['a']),at(k,e['b']),color=e['color'],alpha=.18 if e['kind']!='전환' else .12,lw=0)
    ax.axvline(e['point'],color=e['color'],lw=1.15,ls=EVENT_LINES[e['kind']])

def clock_in_lot(ax,k):clock_ticks(ax,lot_meta.loc[k,'START'],lot_meta.loc[k,'END'])

def work_clock(ax,head,k,expanded=False):
    r=lot_meta.loc[k];ws=lot_works(k) if expanded else cal_work.loc[(cal_work.STRT_DT.lt(r.END)&cal_work.END_DT.gt(r.START))|cal_work.DETC_DT.between(r.START,r.END)]
    ax.plot([r.START,r.END],[0,0],color='#147f9c',lw=6,solid_capstyle='butt',label='LOT 관측창')
    for i,w in enumerate(ws.itertuples(),1):
        color=COL[w.MNT_TYPE]
        ax.plot([w.DETC_DT,w.STRT_DT],[i,i],color=color,ls=':',lw=1.6)
        ax.plot([w.STRT_DT,w.END_DT],[i,i],color=color,lw=6,solid_capstyle='butt')
        for t,mark in [(w.DETC_DT,'D'),(w.STRT_DT,'|'),(w.END_DT,'s')]:ax.plot(t,i,marker=mark,color=EVENT_COLORS['감지'] if mark=='D' else color,ms=6)
    ax.set_yticks(list(range(len(ws)+1)),[f'LOT {k}']+[f'{w.MNT_TYPE} {w.WO_NO[-5:]}' for w in ws.itertuples()],fontsize=9)
    ax.set_ylim(-.6,len(ws)+.6)
    if expanded:
        times=[r.START,r.END]+[t for w in ws.itertuples() for t in [w.DETC_DT,w.STRT_DT,w.END_DT]]
        a,b=min(times),max(times);pad=max((b-a)*.035,pd.Timedelta(minutes=10));clock_ticks(ax,a-pad,b+pad)
    else:clock_in_lot(ax,k)
    panel_legend(head,ax,[Line2D([0],[0],marker='D',color=EVENT_COLORS['감지'],lw=0,label='감지 DETC'),Line2D([0],[0],marker='|',color=GREY,lw=0,label='착수 STRT'),Line2D([0],[0],marker='s',color=GREY,lw=0,label='완료 END')])


def add_work_remarks(ax,ws):
    # 각 작업 행의 막대 바로 위에 REMARK를 둔다. 범례·센서 파형 영역과 분리한다.
    lo,hi=ax.get_xlim();span=hi-lo
    for i,w in enumerate(ws.itertuples(),1):
        center=(mdates.date2num(w.STRT_DT)+mdates.date2num(w.END_DT))/2
        center=np.clip(center,lo+.20*span,hi-.20*span)
        ax.text(center,i+.26,remark_text(w),ha='center',va='bottom',fontsize=9,color=COL[w.MNT_TYPE],
                bbox=dict(boxstyle='round,pad=.18',fc='white',ec='none',alpha=.95))
    ax.set_ylim(-.6,len(ws)+1.)


In [ ]:
# 계산·링크용 ID와 보고서의 구간 표기를 분리한다.
def event_id(k,e):
    return f"{k}-{e['tag'][1]}"


def event_text(value,include_lot=True):
    """전체 구간 (a), 그 안의 개별 사건 (a-1). LOT 간 표에서는 LOT도 보인다."""
    if not isinstance(value,str):return value
    pattern=r'(?<![A-Za-z0-9_/#-])(\d{6})-\(?([a-z])(?:-?([1-9]\d*))?\)?(?![A-Za-z0-9_/-])'
    def replace(match):
        lot,letter,number=match.groups()
        tag='('+letter+('-'+number if number else '')+')'
        return (lot+' ' if include_lot else '')+tag
    value=re.sub(pattern,replace,value)
    # 해석·카드에 쓰던 a1 / b-1도 같은 표기로 맞춘다. 기존 괄호·앵커는 보존한다.
    return re.sub(r'(?<![A-Za-z0-9_(\-/])([abcABC])-?([1-9]\d*)(?![A-Za-z0-9_)])',
                  lambda m:'('+m[1]+'-'+m[2]+')',value)


def event_frame(frame):
    """원본 키를 바꾸지 않고 출력용 사본만 정리한다."""
    shown=frame.to_frame() if isinstance(frame,pd.Series) else frame.copy()
    return shown.map(event_text)


locator_checks=[]
overview_trigger_checks=[]
zoom_presentations=[]
locator_visual_checks=[]
zoom_alignment_checks=[]
display_label_checks=[]


In [ ]:
PLAN_CASES=[(0,240058,240061,'june'),(1,240114,240117,'november')]












# 시계열 특성: 확대창 폭과 실제 이탈 지속을 구분한다. 결측은 연결·보간하지 않는다.
cp_time_rows=[]
for k,events in LOT_EVENTS.items():
    y=raw[k].set_index('MEAS_DT')['CP-Z4'].asfreq('s');base=y.median();previous=None
    for event in events:
        if event['kind']!='CP':continue
        for pulse in event['pulses'].sort_values('PEAK').itertuples():
            start,end=at(k,pulse.A).round('s'),at(k,pulse.B).round('s')
            segment=y.loc[start:end];after=y.loc[end+pd.Timedelta(seconds=1):]
            # 복귀 확인은 사후 기술 규칙이다. 60개 연속 1초 실측이 ±0.02 안일 때의 첫 시각.
            inside=after.notna()&(after-base).abs().le(CP_GROUP)
            stable=inside.rolling(60,min_periods=60).sum().eq(60)
            confirmed=stable.index[stable][0] if stable.any() else pd.NaT
            back=confirmed-pd.Timedelta(seconds=59) if pd.notna(confirmed) else pd.NaT
            cp_time_rows.append(dict(LOT=k,EVENT_ID=event_id(k,event),START=start,END=end,PEAK=pulse.PEAK,DEV=pulse.DEV,
                SPAN_SEC=(end-start).total_seconds(),POINTS=int(segment.notna().sum()),
                PEAK_GAP_MIN=(pulse.PEAK-previous).total_seconds()/60 if previous is not None else np.nan,
                RETURN_START=back,RETURN_CONFIRMED=confirmed,RETURN_DELAY_SEC=(confirmed-pulse.PEAK).total_seconds() if pd.notna(confirmed) else np.nan))
            previous=pulse.PEAK
cp_time_features=pd.DataFrame(cp_time_rows).sort_values(['LOT','PEAK']).reset_index(drop=True)
interval_time_rows=[]
for k,events in LOT_EVENTS.items():
    for event in events:
        duration=np.nan;description='지속시간 정의 없음';recovery='별도 판단'
        if event['kind']=='CP':
            features=cp_time_features.loc[cp_time_features.EVENT_ID.eq(event_id(k,event))]
            duration=features.SPAN_SEC.max();description=f"CP {len(features)}사건 / 최장 원본 첫·끝 간격 {duration:g}초"
            recovery=f"60개 연속 실측 복귀 확인 {features.RETURN_CONFIRMED.notna().sum()}/{len(features)}사건"
        elif event['kind']=='Z':
            z=zstats[k,event['sensor']].Z;hit=z.abs().ge(Z_LIMIT)
            run=next((pair for pair in runs(hit) if pair[0]<=event['minute']<pair[1]),None)
            duration=float(hit.loc[(hit.index>=run[0])&(hit.index<run[1])].sum()) if run else np.nan
            description=f"선택 신호의 연속 |Z|≥3 {duration:g}개 1분창"
            good=z.notna()&z.abs().lt(Z_LIMIT);good.loc[good.index<=event['minute']]=False
            ok=good.rolling(3,min_periods=3).sum().eq(3)
            recovery=f"3개 연속 |Z|<3 확인 {at(k,int(ok.index[ok][0])+1):%m/%d %H:%M}" if ok.any() else '3개 연속 |Z|<3 미확인'
        elif event['kind']=='전환':
            transition=expected.loc[expected.LOT.eq(k)].iloc[0];duration=float(transition.END_MIN-transition.START_MIN)
            description=f'전환 참고 구간 {duration:g}분';recovery='운전 모드·SV 추가 확인'
        interval_time_rows.append(dict(LOT=k,ID=event_id(k,event),TYPE=event['kind'],SENSOR=event['sensor'],POINT=event['point'],
            WINDOW_START=at(k,event['a']),WINDOW_END=at(k,event['b']),WINDOW_MIN=event['b']-event['a'],DURATION_DESCRIPTION=description,RECOVERY=recovery))
interval_time_features=pd.DataFrame(interval_time_rows)
assert len(cp_time_features)==len(cp_real)==14
assert len(interval_time_features)==sum(len(events) for events in LOT_EVENTS.values())






report_case_checks=[]


# 240069: 피크별 반응과 반례를 직접 비교한다. 상관계수·고장 분류 모델은 만들지 않는다.



















# CP 피크별 Z4~Z6의 같은 시각 반응과 근접 최대편차를 함께 확인한다.

<a id="overview"></a>
## 01 Executive Summary — 핵심 결과와 후속 확인

In [ ]:
note('### 보고서에서 먼저 볼 세 가지 결과')
show_table(pd.DataFrame([
 {'확인한 사실':'같은 SPHC의 OP-Z4 수준 차이','근거':'240061 중앙값 54.13%, 다른 3 LOT 41.82·42.12·44.37%','판단':'설정·부하 확인 대상. 온도 이상·고장 확정 아님'},
 {'확인한 사실':'240069 CP 원본 7사건','근거':'최대 +0.459, 이후 BM REMARK: 1호기 재발·절연 저하','판단':'시간상 선행. 원인·수리 효과는 미확인'},
 {'확인한 사실':'240117 CP 하강은 CBM 완료 뒤','근거':'REMARK 마모량 초과, 완료 12:44 → CP 최저 13:04:50','판단':'해당 CBM을 유발한 사전 경보로 해석 불가'}]))
rows=[{'panels':[('같은 SPHC | TC-Z4 분포',['TC-Z4']),('같은 SPHC | OP-Z4 분포',['OP-Z4'])],'height':2.1},
      {'panels':[('240069 | CP 반복 이탈 → 이후 BM',['CP-Z4']),('240117 | CBM 완료 뒤 CP 하강',['CP-Z4'])],'height':2.1}]
fig,cells=report_page('CAL01 · 보고서 핵심 근거 3가지',
 '위: 같은 강종의 온도·출력 분포 / 아래: CP 원본과 실제 정비 시각. 빨강은 CP 탐색 경계 이탈이며 고장 확정이 아니다.',rows,width=24)
ids=lots.loc[lots.GRADE.eq('SPHC'),'LOT_NO'].astype(int).tolist()
for ax,head,sigs in cells[0]:draw_boxes(ax,head,sigs[0],ids,highlight=240061)
for (ax,head,_),k in zip(cells[1],[240069,240117]):
    r=lot_meta.loc[k];report_signal(ax,k,'CP-Z4',fine=True)
    base=raw[k]['CP-Z4'].median()
    for value in [base-.05,base+.05]:ax.axhline(value,color=EVENT_COLORS['CP'],ls='--',lw=.8)
    events=cp_real.loc[cp_real.LOT.eq(k)];peak=events.loc[events.DEV.abs().idxmax()]
    clock_ticks(ax,r.START,r.END);ax.set_ylabel('CP-Z4 [기록값]');panel_legend(head,ax)
    near_note(ax,peak.PEAK,base+peak.DEV,f'{peak.PEAK:%H:%M:%S}\nCP 편차 {peak.DEV:+.3f}',EVENT_COLORS['CP'])
    if k==240117:
        done=cal_work.loc[cal_work.WO_NO.eq('WO-2024-01854'),'END_DT'].iloc[0]
        ax.axvline(done,color=COL['CBM'],ls='-.',lw=1)
emit(fig,'01_개요/00_보고서_핵심근거.png')
note('**읽은 결과:** 240061은 TC-Z4와 OP-Z4를 분리해 조건 차이를 확인해야 한다. 240069는 CP가 BM 전에 발생했지만 작업 후 관측이 없다. 240117은 CP가 CBM 후에 발생했다. 아래는 이 판단을 만든 일정·A/B 수리 대조와 구간별 근거다.')

**먼저 판단할 것:** 240069·240037의 CP는 반복성·지속·복귀를, 240117은 실제 CBM 완료와의 선후 관계를, 240061은 같은 강종의 온도·출력 수준을 확인한다. 계획수리 전후의 공통 하강·승온은 운전 전환 가설을 우선 검토한다. 핵심 사례와 남은 가설은 5절, 점검·데이터 확보 항목은 7절로 연결한다.

**추가 관계 분석의 핵심 결과**

- CP 14사건에서 Z3뿐 아니라 Z4~Z6도 모두 비교했다. CP 피크 시각의 TC-Z4~Z6 변화는 상대적으로 작지만, CP 이탈이 없던 같은 LOT의 다른 때보다 이탈 때 변화폭이 큰 경우가 TC-Z4 **10/12**, TC-Z5 **9/12**, TC-Z6 **8/12**였다. 작은 반복 반응을 함께 확인할 근거다.
- CP 하강 2사건에서는 TC-Z3 최고점이 CP 최저점보다 약 6분 앞에 있고 OP-Z3는 낮아졌다가 올라오는 양상이 반복됐다. CP 최저 부근의 TC-Z4·Z6는 앞 기준보다 낮고 TC-Z5는 높은 방향이다. 하강 사례 2개에서 본 관측이며 원인·정상 여부를 확정하지 않는다.
- 정상 제어·계측 변화·고장 원인을 구분하려면 CP 정의·SV·속도·부하·정비 진단과 조치 직전직후 자료가 필요하다. [사건별 관계](#relation-events)·[CP 전후 파형](#relation-lags)·[CP 이탈 유무 비교](#relation-controls)·[채널·정비](#relation-maintenance)에 그래프와 코드, 관찰·반례를 붙였다.
- **TC–OP 관계는 [5.7.1 독립 절](#relation-tc-op)에서 확인한다.** 같은 존의 온도·출력은 14 LOT의 짧은 동시 변화에서 주로 역방향이었다. 실제 파형과 전체 LOT 표를 함께 보며 선후·원인 방향은 분리하지 못했다.


TC–OP의 방향성 다음에는 [출력을 높인 뒤 온도 복귀](#tc-op-response)를 확인한다. ‘고출력인데 TC가 내려갔다’는 모양의 반복성과 240069 두 반응의 복귀 차이를 구분한다. SV 없는 복귀는 과거 관측 수준에 대한 묘사이며, [품질 검증에 필요한 근거](#tc-quality-evidence)는 별도로 정리했다.


<a id="data-scope"></a>
## 02 분석 목적 및 데이터 특성

**목적:** 어떤 시각·신호가 평소 또는 다른 LOT과 다르게 관측됐는지 찾고, 지속·반복·복귀와 정비의 실제 시각으로 변화의 성격을 좁힌다.

**관측 범위:** 연간 전체를 연속 측정한 자료가 아니다. LOT 사이 관측 공백은 온도 추세로 연결하거나 정상·정지로 채우지 않는다. LOT 관측창과 정비 감지·착수·완료의 교집합만 실제 기록으로 비교한다.

**분석팀이 확인한 미정의 항목:** CP-Z4·CP-Z4M의 물리량·단위·측정/계산 방식과 계측점 대응이 현재 정의되지 않은 상태다. INS-006(가스 분석계 셀)과 각 CP 채널의 연결도 확인되지 않았다. CP의 통계는 제공된 기록값을 기준으로 읽는다.

In [ ]:
gaps=pd.concat([g.MEAS_DT.diff().dt.total_seconds().dropna() for g in raw.values()])
show_table(pd.DataFrame([
 {'항목':'온도 자료','범위':f'{len(temp):,}행 · {len(lots)} LOT · {lots.GRADE.nunique()}강종 · {len(SIGNALS)}신호'},
 {'항목':'관측 날짜 범위','범위':f'{temp.MEAS_DT.min():%Y/%m/%d}–{temp.MEAS_DT.max():%Y/%m/%d}의 분리된 LOT 관측창'},
 {'항목':'수집 간격','범위':f'LOT 내 양의 시각 차이 최빈값 {gaps.loc[gaps.gt(0)].mode().iloc[0]:g}초 · 누락 시각은 보간하지 않음'},
 {'항목':'CAL01 정비','범위':f'{len(cal_work)} WO · TBM 22 / BM 12 / CBM 10'},
 {'항목':'실제 작업과 관측 교집합','범위':'240117의 2 WO · 합집합 304분'},
 {'항목':'정기수리 중 온도','범위':'6월·11월 계획수리 날짜 안에는 제공 온도 0행'},
 {'항목':'추가로 필요한 자료','범위':'정상 기준·SV·라인 속도·소재 두께/부하·존 간 거리·품질 결과'}]))
table(lots[['LOT_NO','GRADE','START','END','MINUTES','N_ROWS']])

<a id="methods"></a>
## 03 전처리·탐지 방법과 판단 범위

| 경로 | 입력·기준 | 찾는 것 | 이 경로만으로 판단하지 않는 것 |
|---|---|---|---|
| LOT 추세 | 실제 기록 54개 이상인 1분창 중앙값 | 분 단위 중심 수준·함께 움직임 | 순간 최고·최저 |
| 시간순 Z | 이전 30분 중앙값·SD, 최소 20개 과거 유효창, 절대 Z≥3 | 이전 구간 대비 국소 변화 | 검증된 고장·정상 확률 |
| CP 원본 | LOT 원본 중앙값 편차 0.02 초과 묶음 중 최대 0.05 이상 | 짧은 펄스·반복 이탈 | 계측 오류·공정 원인 확정 |
| IQR | LOT·신호별 전환 제외 1분 중앙값, Q1−1.5IQR~Q3+1.5IQR | LOT 내부 분포의 드문 창·분포 폭 | 같은 강종의 정상 한계 |
| 같은 강종 수준 | LOT별 중앙값을 동일 가중으로 대조 | 분포 전체의 높낮이 차이 | 운전 조건 일치·에너지 증가·고장 |

**지속시간을 구분한다.** 확대창의 폭은 설명을 위한 문맥이며 이상 지속시간이 아니다. CP는 원본 이탈의 첫·끝 기록 간격과 실측 점 수를 함께 제시한다. 피크 간격은 반복의 기술량이며 주기 확정이 아니다.

**복귀도 관측 규칙으로만 기술한다.** CP는 이후 60개 연속 실제 1초 기록이 LOT 기준 ±0.02 안에 들어온 첫 확인 시각을 제시한다. Z는 3개 연속 유효 1분창이 절대 Z<3으로 들어오는지 확인한다. 두 규칙은 사후 비교를 위한 기술 기준이며 고장 수리 완료·정상 보증이 아니다. Z 기준선 자체의 이동도 복귀 판독에 영향을 준다.

In [ ]:
from IPython.utils.capture import capture_output
with capture_output() as captured:
    
    # 1) 본문과 동일한 9개 비주기 온도 채널의 1분 Z를 사용한다.
    # "계산 가능"은 최소 1개 채널에 점수가 있다는 뜻이며, 모든 채널의 정상 보장이 아니다.
    quality_effect_rows=[]
    for k in lots.LOT_NO.astype(int):
        z=pd.DataFrame({s:zstats[k,s].Z for s in NONCYCLIC})
        counts=profile[k].xs('count',axis=1,level=1)[NONCYCLIC]
        current_ok=counts.ge(54).any(axis=1)
        scored=z.notna().any(axis=1);flag=z.abs().ge(Z_LIMIT).any(axis=1)
        # 품질 조건을 생략한 경우도 같은 식으로 계산해 후보 위치 변화만 비교한다.
        unfiltered=[]
        for s in NONCYCLIC:
            y=profile[k][(s,'median')];past=y.shift(1)
            center=past.rolling(Z_WINDOW,min_periods=Z_MIN).median()
            sd=past.rolling(Z_WINDOW,min_periods=Z_MIN).std().clip(lower=.5)
            unfiltered.append(((y-center)/sd).abs().ge(Z_LIMIT))
        unfiltered_flag=pd.concat(unfiltered,axis=1).any(axis=1)
        quality_effect_rows.append({'LOT':k,'GRADE':lot_meta.loc[k,'GRADE'],'전체 창':len(z),
            '계산 후 3미만':int((scored&~flag).sum()),'3이상 후보':int(flag.sum()),
            '현재 기록 부족':int((~scored&~current_ok).sum()),'과거 기준 부족':int((~scored&current_ok).sum()),
            '조건 미적용 후보':int(unfiltered_flag.sum()),'조건 적용 후 제외':int((unfiltered_flag&~flag).sum()),
            '조건 적용 후 추가':int((flag&~unfiltered_flag).sum())})
    quality_effect=pd.DataFrame(quality_effect_rows)
    status_columns=['계산 후 3미만','3이상 후보','현재 기록 부족','과거 기준 부족']
    assert quality_effect[status_columns].sum(axis=1).eq(quality_effect['전체 창']).all()
    assert quality_effect['3이상 후보'].sum()==zscreen.MAX_ABS_Z.ge(Z_LIMIT).sum()
    qt=quality_effect.sum(numeric_only=True)
    
    note('**계산 가능 범위:** 기록 부족과 초기 기준 부족은 판단을 보류한다')
    fig,cells=report_page('온도 변화: 실제로 계산한 창과 판정을 보류한 창',
        '색별 길이=상태별 창 수 합계 · 실제 발생 시각은 8절 LOT에서 확인 · 9개 비주기 온도의 계산된 Z 사용 · 3미만도 정상 확정은 아님',
        [{'panels':[('14 LOT · 각 1분 집계창을 계산 결과로 분류',[])],'height':5.6}])
    ax,head,_=cells[0][0];yy=np.arange(len(quality_effect));left=np.zeros(len(yy))
    status_colors=['#4a92aa',EVENT_COLORS['Z'],'#b6c1cb','#e4d1ab']
    status_labels=['계산된 신호 3미만','3이상 검토 후보','기록 부족: 보류','과거 기준 부족: 보류']
    for col,color,label in zip(status_columns,status_colors,status_labels):
        values=quality_effect[col].to_numpy();ax.barh(yy,values,left=left,height=.66,color=color,label=label);left+=values
    for i,r in enumerate(quality_effect.to_dict('records')):
        held=r['현재 기록 부족']+r['과거 기준 부족']
        ax.text(r['전체 창']+12,i,f"후보 {r['3이상 후보']} / 보류 {held}",ha='left',va='center',fontsize=9,color='#34495e')
    ax.set_yticks(yy,[f"{r.LOT} · {r.GRADE}" for r in quality_effect.itertuples()],fontsize=9)
    ax.set_ylim(len(yy)-.4,-.6);ax.set_xlim(0,float(left.max())*1.32);ax.set_xlabel('1분 집계창[개] · 끝부분의 짧은 창 포함');panel_legend(head,ax)
    fig._folded='계산 가능한 관측창·보류 범위·Z 후보 수 확인 펼치기'
    emit(fig,'02_선별/01_관측품질.png')
    note(f"**이 그림의 결과:** 전체 {int(qt['전체 창']):,}개 집계창 중 {int(qt['계산 후 3미만']+qt['3이상 후보']):,}개에서 최소 한 온도 채널의 Z를 계산했다. {int(qt['현재 기록 부족'])}개는 기록 수가 부족하고, {int(qt['과거 기준 부족'])}개는 과거 기준이 부족해 판정을 보류했다. 계산된 창 중 **{int(qt['3이상 후보'])}개가 3 이상 검토 후보**다.")
    note('**그래서 무엇을 하는가:** 회색·베이지 구간을 정상으로 채우거나 그 구간에서 고장이 없었다고 결론 내리지 않는다. 특히 각 LOT 초기 20분은 온도 기록이 있어도 이전 기준이 없어 Z를 판정할 수 없다. 주황 구간은 뒤의 LOT 원본 확대와 정비 시각 대조 대상으로 넘긴다. 이 그림은 모든 9개 센서가 동시에 평가되었다는 뜻은 아니다.')
    note(f"**품질 조건이 실제로 바꾼 것:** 조건 미적용 시 {int(qt['조건 미적용 후보'])}개 후보였으나, 기록 조건을 반영하면 {int(qt['3이상 후보'])}개다. 기존 후보 {int(qt['조건 적용 후 제외'])}개가 빠지고 {int(qt['조건 적용 후 추가'])}개가 추가된다. 현재 값뿐 아니라 과거 중앙값·표준편차도 달라지기 때문이다. 이를 진짜·가짜 이상 개수로 해석하지 않고, 근거가 있는 점수와 계산 보류 범위를 구분하는 데 사용한다.")
    table(quality_effect)
    
    
    # 240069의 CP 후보에 필요한 기록 수만 검증한다. 별도 공백 그림은 만들지 않는다.
    gap_lot=240069
    cp_window_counts=profile[gap_lot][('CP-Z4','count')]
    candidates=cp_real.loc[cp_real.LOT.eq(gap_lot)]
    peak_counts=pd.Series({r.PEAK:int(cp_window_counts.loc[int(r.M)]) for r in candidates.itertuples()})
    assert len(peak_counts)==7 and peak_counts.ge(54).all()
    
    
    # 2) 실제 CP 사건으로 평균/중앙값의 의미와 짧은 피크를 별도 처리하는 이유를 보여 준다.
    # 최대값이 든 한 분만이 아니라 사건이 걸친 모든 유효한 분의 중앙값을 검사한다.
    cp_aggregation_rows=[]
    for ev in cp_real.itertuples():
        base=float(raw[int(ev.LOT)]['CP-Z4'].median())
        bins=profile[int(ev.LOT)].loc[int(ev.A):int(ev.B)]
        usable=bins[('CP-Z4','count')].ge(54)
        med=bins[('CP-Z4','median')].where(usable)
        cp_aggregation_rows.append({'LOT':int(ev.LOT),'PEAK':ev.PEAK,'RAW_DEV':float(ev.DEV),
            'USABLE_MINUTES':int(usable.sum()),'MEDIAN_ABS_MAX':float((med-base).abs().max()),
            'MEDIAN_EVENT_HIT':bool((med-base).abs().ge(CP_LIMIT).any())})
    cp_aggregation_review=pd.DataFrame(cp_aggregation_rows)
    assert cp_aggregation_review.USABLE_MINUTES.gt(0).all()
    missed=cp_aggregation_review.loc[~cp_aggregation_review.MEDIAN_EVENT_HIT]
    
    cp_quality_evidence=cp_aggregation_review.copy()
captured_details('계산 가능 범위·기록 조건과 후보 수 검증 펼치기',captured)

**계산에 반영한 조건:** 기록 부족 242개 창과 과거 기준 부족 280개 창은 온도 Z를 판정하지 않는다. CP는 모든 LOT의 원본에 같은 0.05 기준을 적용한다. 계산 정의는 마지막의 접힌 설명에서 확인한다.

### 3.1 왜 기본 검토선을 3으로 두는가?

일반적인 관리도는 **관리 상태의 평균을 중심으로 ±3σ**를 출발점으로 쓴다. 4를 기본값으로 삼을 검증 근거가 없어, 이 노트북의 **국소 변화 기본 검토 배수도 3**으로 바꾼다. 4·5는 민감도 비교로 유지한다. [NIST: 관리도의 중심선과 3σ 한계](https://www.itl.nist.gov/div898/handbook/pmc/section3/pmc32.htm)

**전통적인 3σ와 이 분석의 차이를 구분한다.** 이 분석은 요청한 시간순 중앙값 비교를 유지한다. 현재 분의 1분 중앙값을 \(x_t\), 현재 분을 제외한 **이전 30분의 중앙값**을 \(m_t\), 그 과거 1분 값들의 표준편차를 \(s_t\)로 둔다. 과거 유효 분이 20개 이상일 때만 계산한다.

\[
Z_t=\frac{x_t-m_t}{\max(s_t,s_{\min})},\qquad
L_t=m_t-3\max(s_t,s_{\min}),\quad U_t=m_t+3\max(s_t,s_{\min})
\]

따라서 그림의 범위는 **중앙값 중심 ±3SD 검토선**이다. 독립적으로 확보한 정상 평균과 σ로 만든 고정 관리한계가 아니며, 일반 평균 Z나 MAD 기반 robust Z도 아니다. 이를 근거로 정상 99.73% 보장·고장 확률·검증된 오경보율을 주장하지 않는다.

`shift(1)`로 현재 분을 과거 기준에서 제외한다. LOT 시작 20분은 기준이 부족해 비워 둔다. 작은 분산이 점수를 과도하게 키우지 않도록 표준편차 하한을 온도 0.5℃, 출력 2%p로 둔다. 이 하한도 탐색용 설정이며 센서 사양이나 공정 허용오차가 아니다. σ는 원본 1초 값이 아니라 과거의 **1분 중앙값들**에서 계산한다.

| 검토 기준 | 쓰는 목적 | 판단 범위 |
|---|---|---|
| 중앙값 중심 절대 Z ≥ 3 | LOT 내부 국소 변화의 기본 표시 | 이상 의심 후보, 고장 확정 아님 |
| 배수 4·5 | 3과 비교한 후보 수 변화 | 기본 후보를 숨기는 조건으로 쓰지 않음 |
| CP 원본과 LOT 중앙값 차이 ≥ 0.05 | 짧은 CP 펄스의 별도 검토 | 3σ가 아닌 기록값 단위의 보조 기준, 전체 LOT를 쓴 사후 탐색 |
| CP 사건 묶음 | 0.02 초과 구간 중 0.05 도달, 5초 초과 공백에서 분리 | 보간하지 않음, 사건 길이는 기록 범위 |
| 같은 강종 수준 | 안정 참고의 LOT별 중앙값 비교 | 설정·속도·두께·부하가 일치한 대조는 아님 |
| 온도 2.5℃ / 출력 4%p 차이 | LOT 간 수준 차이의 보조 경계 | 3σ와 별개이며 공정 허용 한계 아님 |

3을 넘었다는 사실은 **확인할 위치를 찾은 것**이다. 이후 LOT 원본 확대에서 지속·반복·회복 양상과 TC–OP 반응을 보고, 실제 WO 감지·착수·완료 전후 관측으로 해석한다. 정지·승온과 강종 차이가 같은 분산을 따른다고 가정하지 않는다.

**안정 참고:** LOT 전체의 승온·하강 형태를 사후 분리하고 남은 구간이다. 검증된 정상 구간이나 실제 조업 모드가 아니며, 전환 분류에 미래 관측을 사용했다.

### 3.2 임계값 민감도와 사후 선별의 한계

3·4·5 배수와 CP 0.04·0.05·0.06의 후보 수 비교는 4.2절의 접힌 계산표에 제공한다.

**관찰:** 기본 검토선 |Z|≥3은 14 LOT의 356개 1분 집계창·155개 연속 구간을 표시한다. 비교 기준 |Z|≥4에서는 12 LOT의 93개 1분 집계창·52개 구간이다. CP 0.05는 전환 후보를 제외한 6 LOT의 14개 사건이다. 따라서 임계 초과 LOT 수와 결합 근거 우선 검토 8 LOT은 같은 집계가 아니다. 8 LOT은 반복 CP·작업 완료 뒤 CP·같은 강종 수준 차이를 함께 고려하여 사후에 지정한 검토 우선순위이며, 자동으로 정상·이상을 나눈 결과가 아니다. 모든 14 LOT은 뒤에서 같은 순서로 검토한다.

경계를 높이면 후보 수가 줄어드는 정도를 수치로 확인한다. ‘표시 1분창’은 계산된 비주기 온도 신호 중 하나라도 경계를 넘은 집계창 수이며, ‘사건 수’는 이 분들의 연속 구간 수다. CP는 원본 펄스 사건 수이므로 Z 사건 수와 직접 비교하지 않는다.

**가설 검토:** ‘3 또는 0.05가 이 자료에서 통계적으로 최적인 값이다’는 가설을 검증할 정답 데이터가 없다. 임계값 선택에 민감한 사례와 여러 기준에서 유지되는 사례를 나누어 원본 검토 우선순위에 쓴다. 반복 CP(2회 이상), 작업 완료 뒤 CP, 큰 강종 수준 차이를 함께 고려하여 검토 우선순위를 정했다. 240108도 같은 온도 기준으로 검사하고 부록 상세 검토에 포함한다. 본문 사례는 변화의 성격과 정비 시각을 설명하는 역할로 골랐다.

**판단 범위:** 이 자료를 본 뒤 정한 수치와 선별 결과다. 민감도는 오경보율 검증이나 독립 검증이 아니다. 정규성·자기상관을 무시한 ‘3σ 확률’로 해석하지 않는다.

<a id="screening"></a>
## 04 전체 시계열 특성과 탐지 결과

전체 관측 위치와 정비 기록을 먼저 확인한 뒤 14 LOT 검토표로 상세 사례를 선별한다. 우선 검토 사례는 반복 CP, 작업 완료 뒤의 CP, 같은 강종 수준 차이, 계획수리 전후 전환이라는 서로 다른 질문을 대표한다. 다른 LOT을 정상으로 판정해 제외한 것은 아니다.

### 4.1 연간 관측·정비와 공정별 검토

In [ ]:
focus = set(context.loc[context.CAL_WORK.gt(0)|context.CAL_DETECT.gt(0)|context.PLAN_RELATION.ne(""),"LOT"])
# 행과 주석 사이에 여백을 확보하되, 연간 날짜축의 가로 폭은 유지한다.
fig, ax = plt.subplots(figsize=(30,19.5))
fig.subplots_adjust(left=.14,right=.977,top=.83,bottom=.12)
rows = {"LOT":9,"GRADE":8,"BM":7,"CBM":6,"TBM":5,"DETECT":4,"INS-006":3,"SEL-006":2,"SEL-009":1,"PLAN":0}
for label,y in rows.items():
    color = COL.get(label,COL["PART"])
    ax.axhspan(y-.09,y+.09,color=color,alpha=.055,lw=0)
for p in plans.itertuples():
    ax.axvspan(p.STRT_DT,p.END_EXCL,color=COL["PLAN"],alpha=.65,lw=0)
    ax.plot([p.STRT_DT,p.END_EXCL],[0,0],color=COL["LOT"],lw=5,solid_capstyle="butt")
for r in grade_evt.itertuples():
    ax.plot(r.EVT_DT,8,marker="|",color=COL["GRADE"],ms=8,mew=.9)
for t in cal_work.itertuples():
    y = rows[t.MNT_TYPE]
    ax.plot([t.STRT_DT,t.END_DT],[y,y],color=COL[t.MNT_TYPE],lw=2)
    ax.plot(t.STRT_DT,y,"x",color=COL[t.MNT_TYPE],ms=6)
    ax.plot(t.DETC_DT,4,"v",color=COL["DETECT"],ms=5,alpha=.75)
# 동일 WO의 정비와 부품교체 연결은 LOT 시간 겹침과 별도로 유지한다.
# 한 WO에 부품이 둘이어도 세로 점선 하나를 쓰고 해당 부품 행에 마름모를 찍는다.
annual_part_links = []
for wo, group in cal_parts.groupby("WO_NO",sort=False):
    shown = group.loc[group.PART_CD.isin(rows)]
    if shown.empty:
        continue
    assert shown.MNT_TYPE.nunique()==1 and shown.END_DT.nunique()==1
    maintenance = shown.MNT_TYPE.iloc[0]
    completed = shown.END_DT.iloc[0]
    bottom = min(rows[code] for code in shown.PART_CD)
    ax.plot([completed,completed],[rows[maintenance],bottom],
            color=COL["PART"],lw=1.8,ls="--",alpha=.85,zorder=1.8,
            label=f"_wo_part_link_{wo}")
    for p in shown.itertuples():
        ax.plot(p.END_DT,rows[p.PART_CD],"D",color=COL["PART"],ms=7,zorder=5)
    annual_part_links.append({
        "WO":wo,"정비 종류":maintenance,"WO 완료 시각":completed,
        "연결 부품":" · ".join(f"{p.PART_NM} [{p.PART_CD}] × {p.QTY}" for p in shown.itertuples()),
        "교체 기록[행]":len(shown),"REMARK":shown.REMARK.fillna("미기재").iloc[0]})
annual_part_summary = pd.DataFrame(annual_part_links)
assert len(annual_part_links)==cal_parts.WO_NO.nunique()
work_by_wo = cal_work.set_index("WO_NO")
labels = []
for i,r in enumerate(lots.itertuples()):
    color = COL["FOCUS"] if r.LOT_NO in focus else COL["LOT"]

    # 온도 관측 범위만 배경으로 표시한다. 실제 정비 겹침은 빈 원과 교집합 표로 확인한다.
    ax.fill_betweenx(
        [rows["PLAN"], rows["LOT"]], r.START, r.END,
        color=COL["LOT"], alpha=0.10, zorder=0.5,
    )
    ax.vlines(
        [r.START, r.END], ymin=rows["PLAN"], ymax=rows["LOT"],
        color=COL["LOT"], linestyle="--", linewidth=0.9,
        alpha=0.35, zorder=1.0,
    )

    ax.plot([r.START,r.END],[9,9],color=color,lw=3)
    ax.plot(r.START,9,"o",color=color,ms=6)
    y = 10.4 if i%2==0 else 12.3
    label_text = f"{r.LOT_NO}\n{r.GRADE}\n{r.START:%m/%d}"
    label = ax.text(r.START,y,label_text,ha="center",va="center",fontsize=14.5,color=color,
                    bbox=dict(boxstyle="round,pad=.30",fc="white",ec=color,lw=.65))
    labels.append((label,r.START,y,color))

# 한 LOT당 세로 안내선 하나를 사용한다.
# 빈 원만 LOT 관측창과 실제로 겹친 기록을 뜻하며, 지나가는 선만으로는 겹침을 뜻하지 않는다.
# 원은 실제 사건/교집합 대표시각에 찍고, 정확한 시간·구간은 아래 표에 보존한다.
annual_links, annual_overlap_rows, annual_guides = [], [], []

def clock_union_minutes(intervals):
    merged = []
    for start, end in sorted(intervals):
        if merged and start <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(end, merged[-1][1]))
        else:
            merged.append((start, end))
    return sum((end-start).total_seconds()/60 for start,end in merged)

def mark_lot_record(lot, category, at_time, start, end, key=""):
    color = COL[category]
    # 원의 위치는 실제 시각 그대로 두어, 서로 다른 사건을 같은 순간으로 옮기지 않는다.
    ax.plot(at_time, rows[category], "o", mfc="white", mec=color, ms=9, mew=2, zorder=5)
    assert lot_meta.loc[lot,"START"] <= at_time <= lot_meta.loc[lot,"END"]
    annual_links.append({"LOT":lot, "연결 기록":category, "키":key,
                         "겹침 시작":start, "겹침 끝":end, "빈 원 표시 시각":at_time,
                         "구간 교집합[분]":(end-start).total_seconds()/60})

for r in lots.itertuples():
    k = int(r.LOT_NO)
    ge = grade_evt.loc[grade_evt.EVT_DT.between(r.START,r.END)]
    tasks = cal_work.loc[(cal_work.STRT_DT < r.END) & (cal_work.END_DT > r.START)]
    detected = cal_work.loc[cal_work.DETC_DT.between(r.START,r.END)]
    for event_row in ge.itertuples():
        at_time = event_row.EVT_DT
        mark_lot_record(k, "GRADE", at_time, at_time, at_time,
                        key=f"{event_row.BEF_VAL}→{event_row.AFT_VAL}")
    work_intervals = []
    for task in tasks.itertuples():
        start,end = max(task.STRT_DT,r.START),min(task.END_DT,r.END)
        at_time = start + (end-start)/2
        mark_lot_record(k, task.MNT_TYPE, at_time, start, end, key=task.WO_NO)
        work_intervals.append((start,end))
    for task in detected.itertuples():
        at_time = task.DETC_DT
        mark_lot_record(k, "DETECT", at_time, at_time, at_time, key=task.WO_NO)
    annual_overlap_rows.append({
        "LOT":k, "강종":r.GRADE, "온도 관측 시작":r.START, "온도 관측 끝":r.END,
        "강종변경 동시 기록[건]":len(ge),
        "실제 CAL01 작업[WO]":len(tasks),
        "실제 작업 교집합 합집합[분]":clock_union_minutes(work_intervals),
        "관측 중 감지 기록[건]":len(detected)})
annual_overlap_summary = pd.DataFrame(annual_overlap_rows)
annual_interval_links = pd.DataFrame(annual_links)

# 연간 날짜축에서는 한 LOT의 짧은 관측창을 같은 세로 위치에서 읽도록 묶는다.
# 아래쪽 끝은 마지막으로 겹친 기록 행이며, 겹치지 않은 행에는 빈 원을 그리지 않는다.
GUIDE_COLOR = "#607080"
for r in lots.itertuples():
    k = int(r.LOT_NO)
    linked = annual_interval_links.loc[annual_interval_links.LOT.eq(k)]
    if linked.empty:
        continue
    bottom_y = min(rows[category] for category in linked["연결 기록"])
    guide, = ax.plot([r.START,r.START], [rows["LOT"],bottom_y],
                     color=GUIDE_COLOR,lw=2.2,alpha=.75,zorder=2,
                     label=f"_lot_guide_{k}",solid_capstyle="butt")
    ax.plot(r.START,rows["LOT"],"o",mfc="none",mec=GUIDE_COLOR,ms=10,mew=1.5,zorder=5)
    annual_guides.append({"LOT":k,"안내선 기준":r.START,"끝 행":bottom_y,
                          "겹친 기록 종류":",".join(linked["연결 기록"].drop_duplicates())})

assert annual_overlap_summary.LOT.nunique() == len(lots)
assert annual_overlap_summary["강종변경 동시 기록[건]"].sum() == 14
assert annual_overlap_summary.loc[annual_overlap_summary["실제 CAL01 작업[WO]"].gt(0),"LOT"].tolist() == [240117]
assert np.isclose(annual_overlap_summary.set_index("LOT").loc[240117,"실제 작업 교집합 합집합[분]"],304)
assert annual_overlap_summary.loc[annual_overlap_summary["관측 중 감지 기록[건]"].gt(0),"LOT"].tolist() == [240117,240133]

# 글상자 폭을 측정하여 같은 층에서 겹치는 라벨만 조금 이동한다.
fig.canvas.draw()
renderer = fig.canvas.get_renderer()
for y in [10.4,12.3]:
    previous = None
    for label,x,level,color in labels:
        if level != y:
            continue
        box = text_box(label,renderer)
        if previous is not None and box.x0 < previous+7:
            delta = previous+7-box.x0
            screen = ax.transData.transform((mdates.date2num(x),y))
            label.set_x(ax.transData.inverted().transform((screen[0]+delta,screen[1]))[0])
            fig.canvas.draw()
            box = text_box(label,renderer)
        previous = box.x1
for label,x,y,color in labels:
    ax.plot([x,label.get_position()[0]],[9,y-.45],color=color,alpha=.45,lw=.7)
counts = cal_work.MNT_TYPE.value_counts()
axis_labels = ["CR1 계획수리 · 2회","O링 세트 · SEL-009","오일씰 · SEL-006","가스 분석계 셀 · INS-006",
               "정비 감지 [DETC_DT]",f"TBM 시간기반 · {counts['TBM']} WO",
               f"CBM 상태기반 · {counts['CBM']} WO",f"BM 사후 · {counts['BM']} WO",
               f"강종변경 · {len(grade_evt)}건",f"온도 관측 · {len(lots)} LOT"]
ax.set_yticks(range(10),axis_labels)
axis_colors = [COL["LOT"],COL["PART"],COL["PART"],COL["PART"],COL["DETECT"],
               COL["TBM"],COL["CBM"],COL["BM"],COL["GRADE"],COL["LOT"]]
for label,color in zip(ax.get_yticklabels(),axis_colors):
    label.set_color(color)
    label.set_fontweight("bold")
    label.set_fontsize(14.5)
ax.set_ylim(-.65,13.8)
ax.set_xlim(YEAR_START-pd.Timedelta(days=2),YEAR_END+pd.Timedelta(days=3))
ax.set_xticks(pd.date_range("2024-01-01","2024-12-01",freq="MS"))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m월"))
style(ax,"2024년 기록 시각[월]")
ax.set_ylabel("")
handles = [Line2D([0],[0],color=COL["FOCUS"],marker="o",label="작업 겹침·감지 포함·수리 전후 LOT"),
           Line2D([0],[0],color=COL["LOT"],marker="o",label="기타 온도 관측 LOT"),
           Line2D([0],[0],color=COL["GRADE"],marker="|",label="CR1 강종변경"),
           Line2D([0],[0],color=GUIDE_COLOR,lw=2.2,label="LOT별 세로 안내선"),
           Line2D([0],[0],color="#34495e",marker="o",mfc="white",ls="None",ms=8,label="빈 원 · LOT 관측창과 겹친 기록"),
           *[Line2D([0],[0],color=COL[k],marker="x",label=k+" 작업") for k in ["BM","CBM","TBM"]],
           Line2D([0],[0],color=COL["DETECT"],marker="v",label="DETC_DT 감지 기록 · 방법 미제공"),
           Line2D([0],[0],color=COL["PART"],marker="D",ls="--",lw=1.8,label="동일 WO 정비→부품교체 · 완료 기준"),
           Patch(facecolor=COL["PLAN"],label="계획수리 날짜"),
           Patch(facecolor=COL["LOT"],alpha=.15,label="온도 관측 범위, 경계 점선과 음영")]
# 연간 축에는 주요 REMARK만 표시한다. 전체 WO 내용은 월별 상세 그림에서 확인한다.
for wo,x,y,align,prefix in [
    ("WO-2024-00960","2024-06-17",5.5,"right","6월 수리 TBM 6건 · 대표 REMARK"),
    ("WO-2024-01094","2024-07-10",7.5,"right","07/08 BM"),
    ("WO-2024-01859","2024-11-07",5.5,"right","수리 TBM 5건 이후 · 11/13 TBM"),
    ("WO-2024-01854","2024-11-09",6.5,"right","11/13 CBM"),
    ("WO-2024-02105","2024-12-27",6.5,"right","12/28 CBM · 셀 3개 교체 연결")]:
    t=work_by_wo.loc[wo]
    ax.annotate(f"{prefix}\n{str(t.REMARK).strip()}",xy=(t.STRT_DT,rows[t.MNT_TYPE]),
                xytext=(pd.Timestamp(x),y),ha=align,va="center",fontsize=14,color=COL[t.MNT_TYPE],
                bbox=dict(boxstyle="round,pad=.42",fc="white",ec=COL[t.MNT_TYPE],lw=.8),
                arrowprops=dict(arrowstyle="->",color=COL[t.MNT_TYPE],lw=.9),zorder=8)
for p in plans.itertuples():
    scope = p.SCOPE
    ax.annotate(f"{p.STRT_DT:%m/%d}–{p.END_DT:%m/%d}\n{scope}",xy=(p.STRT_DT,0),
                xytext=(p.END_EXCL+pd.Timedelta(days=3),.5),ha="left",va="center",fontsize=14,color=COL["LOT"],
                bbox=dict(boxstyle="round,pad=.42",fc="white",ec=COL["LOT"],lw=.8),
                arrowprops=dict(arrowstyle="->",color=COL["LOT"],lw=.9),zorder=8)
for y,text in [
    (.081,"빈 원: 강종변경 14 LOT · 실제 작업 240117(TBM·CBM) · 감지 240117·240133"),
    (.053,"240117: 실제 작업 합집합 304분(TBM 115+CBM 230-동시 41)"),
    (.025,"선 구분: 빈 원이 있는 행만 LOT 관측창과 겹침 · 보라 점선은 같은 WO의 정비→부품교체 | 7월 BM·12월 작업은 관측 종료 뒤")]:
    fig.text(.14,y,text,ha="left",va="center",fontsize=14.2,color="#34495e")
header(fig,"2024년 CAL01 관측·정비 흐름","TIME AXIS · 실제 측정 날짜·시각 | 회색 직선·빈 원=LOT 관측창 겹침 · 보라 점선·◆=동일 WO 정비→부품교체",handles,ncol=4)
ax.tick_params(axis="x",labelsize=14)
ax.xaxis.label.set_fontsize(14.5)
fig._suptitle.set_fontsize(24)
fig.texts[-1].set_fontsize(14)
# 제목·설명·범례는 위쪽에 모으고, 본문과 범례 사이에 별도 여백을 둔다.
fig.texts[-1].set_position((.035,.94))
fig.legends[0].set_bbox_to_anchor((.031,.907))
for text in fig.legends[0].get_texts(): text.set_fontsize(13)
# LOT당 선 하나이며 모두 직선이고, 화면 안에 표시됐는지 확인한다.
fig.canvas.draw()
guide_lines=[line for line in ax.lines if line.get_label().startswith("_lot_guide_")]
assert len(guide_lines)==len(annual_guides)==len(lots)
for guide in guide_lines:
    points=guide.get_transform().transform_path(guide.get_path()).vertices
    assert points.shape==(2,2) and np.isclose(points[0,0],points[1,0]), "LOT 안내선이 직선이 아님"
    assert np.all(points[:,0]>=ax.bbox.x0-1) and np.all(points[:,0]<=ax.bbox.x1+1), "안내선이 날짜축 밖으로 나감"
    assert np.all(points[:,1]>=ax.bbox.y0-1) and np.all(points[:,1]<=ax.bbox.y1+1), "안내선이 그림 밖으로 나감"

part_lines=[line for line in ax.lines if line.get_label().startswith("_wo_part_link_")]
assert len(part_lines)==len(annual_part_links)==cal_parts.WO_NO.nunique()
for link in part_lines:
    points=link.get_transform().transform_path(link.get_path()).vertices
    assert points.shape==(2,2) and np.isclose(points[0,0],points[1,0]), "WO 교체 연결선이 직선이 아님"
    assert np.all(points[:,0]>=ax.bbox.x0-1) and np.all(points[:,0]<=ax.bbox.x1+1), "WO 연결선이 날짜축 밖으로 나감"

finish(fig,"01_개요/01_연간_관측정비.png")
table(annual_overlap_summary)
table(annual_interval_links)
table(annual_part_summary)

**관찰:** 134건의 CR1 강종변경 중 온도 자료는 14개 LOT 관측창으로 제공된다. CAL01 44 WO 가운데 실제 작업이 관측창과 겹치는 것은 240117의 TBM·CBM 2건이다. 교집합 합계는 동시 작업을 한 번만 세어 **304분(115 + 230 - 41)**이다.

**가설 검토:** ‘정비일 부근 온도 변화는 정비 작업 중 변화다’라는 해석은 성립하지 않는다. 7월 BM은 240069 관측 뒤, 12월 CBM 작업·교체는 240133 관측 뒤에 있다.

**판단 범위:** 빈 원이 있는 행에서 실제 겹침을 확인한다. 관측 사이의 흰 구간은 온도 자료가 없는 기간이다. 부품 ◆ 위치는 동일 WO의 완료 시각을 대신 사용했으며 실제 교체 순간은 알 수 없다.

### 4.1.1 14 LOT × 공정 검토표

**읽을 질문:** 반복 펄스·LOT 전체 수준 차이·운전 전환 후보를 공정별로 어디에서 구분할 수 있는가?

In [ ]:
from matplotlib.patches import Rectangle
PROC=[('정지·승온','일정과 형태 대조'),('세정\nCLN-IN','LOT 내 시간 변화'),('건조\nZ1·Z2','같은 강종 대조'),
      ('가열·균열\nZ3–Z6','TC–OP · 같은 강종'),('CP\nCP-Z4','1초 기록의 이탈'),('과시효\nZ7·Z8','같은 강종 대조'),('Z9·냉각','주기 파형 구분')]
STATUS={'CP 이탈':('#fbe4e7',EVENT_COLORS['CP']),'국소 변화(Z)':('#fff1d6',EVENT_COLORS['Z']),'전환 후보':('#e9edf0',EVENT_COLORS['전환']),'조건 대조':('#e6f0f8','#086788'),'대조 없음':('#f4f6f7','#8c9aa4')}
grid={};ids=lots.LOT_NO.astype(int).tolist()
for k in ids:
    if (lots.GRADE==lot_meta.loc[k,'GRADE']).sum()==1:
        for j in [2,3,5]:grid[k,j]=('대조 없음','같은 강종 LOT 없음')
for r in expected.itertuples():grid[r.LOT,0]=('전환 후보',f'{r.종류}\n{r.시작:%m/%d %H:%M}')
for k,n in cp_count.items():
    d=cp_real.loc[cp_real.LOT.eq(k),'DEV'];peak=d.loc[d.abs().idxmax()]
    grid[k,4]=('CP 이탈',f'{n}개 사건 · 최대 {peak:+.3f}')
for k in cp_events.loc[cp_events.EXP,'LOT'].unique():grid[k,4]=('전환 후보','하강 전환에 포함\n별도 검토')

for k in LEVEL_REVIEW:
    for column,zones in [(2,{1,2}),(3,{3,4,5,6}),(5,{7,8})]:
        d=level_check.loc[level_check.LOT.eq(k)&level_check.DISPLAY_HIT&level_check.SENSOR.map(lambda s:int(s.split('-Z')[1]) in zones)]
        if d.empty:continue
        items=[f'{r.SENSOR} {r.GAP:+.1f}'+('℃' if r.SENSOR in LEVEL_TC else '%p') for r in d.itertuples()]
        lines=[' / '.join(items[i:i+2]) for i in range(0,len(items),2)]
        grid[k,column]=('조건 대조','\n'.join(lines)+f'\n대조 {int(d.N_PEERS.min())} LOT')

grid[240108,1]=('국소 변화(Z)','짧은 국소 변화\n부록 240108 확대 확인')
grid[240117,4]=('CP 이탈','CBM 완료 뒤 CP -0.083\n원인·효과 미확정')
H=15;fig,ax=plt.subplots(figsize=(28,H));fig.subplots_adjust(left=.11,right=.99,top=.82,bottom=.035)
for i,k in enumerate(ids):
    for j in range(7):
        st,text=grid.get((k,j),('', '—'));face,ink=STATUS.get(st,('white','#aebbc5'))
        ax.add_patch(Rectangle((j+.025,i+.05),.95,.9,facecolor=face,edgecolor='#d3dce3',lw=.8,hatch='///' if st=='대조 없음' else None))
        ax.text(j+.5,i+.5,text,ha='center',va='center',fontsize=10.5,color=ink,linespacing=1.3)
    hot=k in CP_REVIEW
    ax.text(-.07,i+.5,f"{k} · {lot_meta.loc[k,'GRADE']}\n{lot_meta.loc[k,'START']:%m/%d}",ha='right',va='center',fontsize=12,color=RED if hot else '#17344d',fontweight='bold' if hot else 'normal')
for j,(name,rule) in enumerate(PROC):
    ax.text(j+.5,-.65,name,ha='center',va='center',fontsize=14,color='#17344d',fontweight='bold')
    ax.text(j+.5,-.13,rule,ha='center',va='bottom',fontsize=10,color='#526d80')
ax.set(xlim=(0,7),ylim=(14,-1.25));ax.axis('off')
header(fig,'CAL01 14 LOT 공정별 검토 근거','빨강 = CP · 주황 = 국소 변화(Z) · 회색 = 전환 · 파랑 = 강종 조건 대조 · 빈 칸 = 요약 미선정',
       [Patch(facecolor=v[0],label=k) for k,v in STATUS.items()]+[Patch(facecolor='white',edgecolor='#d3dce3',label='— 요약 미선정')],ncol=6)
compact_header(fig,.94,.90,title_size=25,sub_size=13,legend_size=13)
finish(fig,'01_개요/02_14LOT_공정별검토.png')

**관찰:** 빨간 칸은 CP 원본 이탈이 있는 6 LOT이다. 그중 반복·작업 완료 뒤 사례 3 LOT을 우선 검토한다. 파란 칸은 기존 수준 검토 5 LOT에서 TC-Z1–TC-Z8·OP-Z1–OP-Z8의 표시 경계 이상 차이를 요약한 **운전 조건 대조**다. OP-Z4 외 OP-Z8 등 다른 차이도 포함하며, 4.2절의 접힌 표에서 전체 22신호 수치를 확인한다. 수리 전후 승온·하강을 제외한 부분에서 계산하며, 그 차이도 고장 판정은 아니다. SPCEN은 2 LOT뿐이므로 서로 다른 수준이라는 사실과 어느 쪽이 정상인지는 분리해야 한다. 표의 출력 차이는 **대상 LOT 중앙값 - 같은 강종의 다른 LOT 중앙값들의 중앙값**이다.

**가설 검토:** ‘LOT 안에서 평탄하면 특이점이 없다’는 가설은 강종 대조에서 반례를 보인다. 반대로 정비 전후의 큰 하강·승온은 운전 전환이라는 대안 설명을 먼저 검토한다.

**판단 범위:** 표는 검토 경로 안내다. 모든 신호·임계 초과를 빠짐없이 채운 고장 판정표는 아니다. 임계 초과·두 LOT 차이·사후 전환 분류를 고장 정답으로 해석하지 않는다. Z3의 반복 응답과 Z9·냉각의 주기 파형은 한 번의 큰 변화만으로 선정하지 않는다.

### 4.2 전체 14 LOT의 선별 근거와 후속 경로

같은 온도·CP 기준을 모든 LOT에 적용한 결과다. 반복성·정비의 실제 시각·같은 강종 수준 차이를 함께 읽는다.

In [ ]:
from IPython.utils.capture import capture_output
with capture_output() as captured:
    sensitivity=[]
    for limit in [3,4,5]:
        flags=zscreen.MAX_ABS_Z.ge(limit)
        nrun=sum(len(runs(g.set_index('MINUTE').MAX_ABS_Z.ge(limit))) for _,g in zscreen.groupby('LOT'))
        sensitivity.append({'기준':f'온도 |Z| ≥ {limit}','선정 LOT':zscreen.loc[flags,'LOT'].nunique(),'표시 1분창[개]':int(flags.sum()),'사건 수':nrun})
    # CP 사건은 묶음 기준 0.02를 고정하고 최고 편차 경계만 바꾼다.
    for limit in [.04,.05,.06]:
        found=[]
        for k in lots.LOT_NO.astype(int):
            s=raw[k].set_index('MEAS_DT')['CP-Z4'].dropna();dev=s-s.median();hit=dev.abs().gt(.02)
            grp=(hit.ne(hit.shift())|s.index.to_series().diff().dt.total_seconds().gt(5)).cumsum()
            for _,d in dev[hit].groupby(grp[hit]):
                peak=d.abs().idxmax();m=(peak-lot_meta.loc[k,'START']).total_seconds()/60
                if abs(d.loc[peak])>=limit and not in_expected(k,m):found.append(k)
        sensitivity.append({'기준':f'CP 최대 편차 ≥ {limit:.2f}','선정 LOT':len(set(found)),'표시 1분창[개]':np.nan,'사건 수':len(found)})
    sensitivity=pd.DataFrame(sensitivity)
    table(sensitivity)
    
    # 아래 8 LOT은 결합 근거에 따른 검토 우선순위다. 14 LOT의 계산·상세 검토를 제한하지 않는다.
    selected_rows=[]
    for k in CP_REVIEW:
        d=cp_real.loc[cp_real.LOT.eq(k)];r=d.loc[d.DEV.abs().idxmax()]
        selected_rows.append({'LOT':k,'강종':lot_meta.loc[k,'GRADE'],'유형':'CP','근거':f'{len(d)}개 사건, 최대 {r.DEV:+.3f}','시간 범위':r.PEAK.strftime('%m/%d %H:%M:%S')})
    for k in LEVEL_REVIEW:
        sensors=LEVEL_TARGETS[k]
        d=level_gap.loc[level_gap.LOT.eq(k)&level_gap.SENSOR.isin(sensors)]
        selected_rows.append({'LOT':k,'강종':lot_meta.loc[k,'GRADE'],'유형':'강종 대조','근거':' / '.join(f'{r.SENSOR} {r.GAP:+.1f}'+('℃' if r.SENSOR in TC else '%p') for r in d.itertuples()),'시간 범위':'전환 제외 구간의 LOT 중앙값'})
    selected=pd.DataFrame(selected_rows).sort_values('LOT')
    selected_by_lot=selected.set_index('LOT')
    screening_rows=[]
    for r in lots.itertuples():
        k=int(r.LOT_NO);c=context.set_index('LOT').loc[k]
        nz=int((zscreen.LOT.eq(k)&zscreen.MAX_ABS_Z.ge(Z_LIMIT)).sum())
        nc=int(cp_count.get(k,0));parts=[]
        if c.PLAN_RELATION:parts.append(c.PLAN_RELATION)
        if c.CAL_WORK:parts.append(f'실제 작업 {int(c.CAL_WORK)} WO 겹침')
        if c.CAL_DETECT:parts.append(f'감지 시각 {int(c.CAL_DETECT)}건 포함')
        if k in selected_by_lot.index:
            d=selected_by_lot.loc[k]
            reason=('반복 CP 원본·동반 반응' if k in [240037,240069] else 'CBM 완료 뒤 CP 하강' if k==240117 else '동일 강종의 조건·수준 대조')+' · '+d['근거']
            priority='결합 근거 우선 검토'
        else:
            reason=('수리 전 하강의 전환 설명 확인' if k==240058 else f'단발 CP {nc}건 원본·TC–OP 대조' if nc else '국소 변화의 원본·TC–OP 대조')
            if k==240133:reason+=' · 감지 후 착수는 관측 종료 뒤'
            priority='해당 구간 확인'
        screening_rows.append({'LOT·강종':f'{k} · {r.GRADE}','온도 |Z|≥3[1분창]':nz,'CP[사건]':nc,'정비·전환 문맥':' / '.join(parts) or '관측 중 작업·감지 교집합 없음','검토 경로':priority,'확인할 이유':reason})
    screening_result=pd.DataFrame(screening_rows)
    note('### 4.2 전체 14 LOT의 검토 근거와 후속 경로')
    note('온도 Z 창 수에는 수리 전후 전환도 포함한다. CP 사건 수는 전환 후보를 제외한 값이다. 3σ 초과만으로는 14 LOT 모두가 표시되므로, 검토 우선순위는 반복성·정비 시각·강종 수준 차이와 함께 읽는다.')
    table(screening_result)
    assert len(screening_result)==14 and screening_result['CP[사건]'].sum()==len(cp_real)
    assert screening_result['온도 |Z|≥3[1분창]'].sum()==zscreen.MAX_ABS_Z.ge(Z_LIMIT).sum()
    
    note('**같은 강종 비교의 범위:** 같은 강종 대조 LOT이 있으면 22신호 모두의 중앙값 차이를 계산하고, 단독 강종은 비교를 보류한다. 수준 차이 표시는 TC-Z1–TC-Z8·OP-Z1–OP-Z8의 16신호를 사용한다. CLN-IN은 LOT 내부 변화, TC-Z9·TC-OUT1·TC-OUT2는 주기·위상, CP는 원본 사건과 계측 정의를 따로 확인한다. 그림은 기존 우선 검토 5 LOT의 상세 사례이며, 모든 수준 차이를 고장으로 선별한 결과가 아니다.')
    table(level_inventory)
captured_details('3SD·CP 민감도와 신호별 상세 수치 펼치기',captured)
show_table(screening_result)

<a id="grade-iqr"></a>
### 4.3 강종·공정별 분포 — 수준 차이와 순간 이탈을 나누어 본다

각 LOT의 전환 제외 유효 1분 중앙값으로 **평균 ◆·중앙값·Q1~Q3 상자·1.5 IQR 수염과 밖의 점**을 계산했다. **평균 ◆는 이 유효 1분 중앙값들의 산술평균**이다. 공정별 그림을 펼치면 16신호 전체를 강종별로 묶어 비교할 수 있다. 상자 밖 전체를 이상으로 세지 않는다. 후보 경계는 `Q1−1.5×IQR`, `Q3+1.5×IQR`이다. [정의: NIST](https://www.itl.nist.gov/div898/handbook/eda/section3/boxplot.htm)

**읽을 것:** 상자 전체가 이동하면 LOT 간 수준 차이, 상자가 넓으면 변동 폭 차이, 수염 밖 점은 LOT 내부의 드문 1분창이다. 강종 대조는 SPHC 4 LOT·SPCE 3 LOT·그 외 1~2 LOT로 정상 한계를 확정하기 어렵다. 아래 IQR은 LOT별 탐색 기준이며, 같은 강종 정상 기준이나 3σ와 같은 확률 경계가 아니다.

In [ ]:
grade_order=lots.GRADE.drop_duplicates().tolist()
box_ids=[int(k) for grade in grade_order for k in lots.loc[lots.GRADE.eq(grade),'LOT_NO']]
for label,zones,number in [('건조',[1,2],2),('가열균열',[3,4,5,6],3),('과시효',[7,8],4)]:
    rows=[{'panels':[(f'TC-Z{z} · 강종별 LOT 분포',[f'TC-Z{z}']),(f'OP-Z{z} · 강종별 LOT 분포',[f'OP-Z{z}'])],'height':2.0} for z in zones]
    fig,cells=report_page(f'강종별 {label} · LOT 내부 분포와 LOT 간 수준',
        '평균 ◆ · 중앙값 실선 · 상자=가운데 50% · 수염=1.5 IQR 안 실제 끝값 · 점=수염 밖 유효 1분 중앙값',rows,width=26)
    for row in cells:
        for ax,head,sigs in row:draw_boxes(ax,head,sigs[0],box_ids)
    fig._folded=f'{label}: 전체 LOT·각 TC/OP의 평균·중앙값·IQR 비교 펼치기'
    emit(fig,f'02_선별/{number:02d}_강종분포_{label}.png')
summary=iqr_summary.groupby(['LOT','GRADE'],sort=False).agg(유효창_최소=('N','min'),수염밖_신호창=('OUT','sum'),연속구간_신호별합=('RUNS','sum')).reset_index()
table(summary)
table(iqr_summary)
example=iqr_summary.loc[iqr_summary.SENSOR.eq('OP-Z4')&iqr_summary.GRADE.eq('SPHC')]
note('**실제 비교 결과:** SPHC의 OP-Z4 LOT 중앙값은 '+ ' / '.join(f'{r.LOT} {r.MEDIAN:.2f}%' for r in example.itertuples())+'. 240061은 분포 중심이 다른 SPHC LOT보다 높다. 자기 LOT 내부 수염 밖 점만 세면 이 수준 차이를 설명할 수 없다. TC-Z4와 OP-Z4를 함께 비교해 온도 수준과 출력 수준의 차이를 구분한다.')
note('**실제 시각 연결:** 각 LOT의 상세 검토에는 수염 밖 값의 신호·시작/끝·지속 창 수를 접힌 표로 제공한다. 이 구간이 기존 CP·Z 확대와 겹치는지 확인한다. IQR이 넘었다는 이유만으로 새 고장 사건을 추가하지 않는다.')

In [ ]:
note('**시계열 특성 전체 검토:** CP 원본 14사건의 시작·종료·피크 간격·복귀 확인을 같은 정의로 계산했다. 아래 표를 펼치면 순간 이탈과 확대창 폭을 구분할 수 있다.')
table(cp_time_features)
table(interval_time_features)

<a id="cases"></a>
## 05 주요 LOT 상세 시계열 분석

각 사례는 **분석 질문 → 선정 근거 → 같은 번호의 원본·동반 신호 → 지속·반복·복귀와 정비 시각 → 잠정 판단** 순서다. 그래프 상단의 작은 전체 신호가 현재 확대 위치를 반복해서 알려 준다. 표시된 동시 반응은 동일 소재 위치의 반응으로 정렬한 결과가 아니다.
각 절은 **질문 → 선정 근거 → 생성 코드·그래프 → 관찰 결과 → 반례·해석** 순서로 읽는다. 함수 정의는 최초 사용 셀에 두고, 같은 레이아웃을 쓰는 다음 LOT은 그 함수를 재사용한다.

| 질문 | 직접 볼 그림·생성 코드 | 판독 범위 |
|---|---|---|
| CP 사건에서 어느 존이 움직였는가? | 5.1.1의 피크별 원본, 5.6의 전체 사건 TC·OP 변화량 | 반복 방향·크기·다른 사건과의 차이 |
| CP 하강 전후 주변 존은 어떤 모양인가? | 5.7의 CP·TC-Z3~Z6·OP-Z3~Z6 동시 파형 | CP 하강 2사건의 선후·반복 양상, 상승 사건 대조 |
| 같은 존의 TC와 OP는 어떤 관계인가? | [5.7.1 TC–OP](#relation-tc-op)의 실제 동시 파형·14 LOT 변화 방향 표 | 온도·출력의 반대 움직임, 선후 방향 미분리 |
| CP가 없을 때도 같은 반응인가? | 5.1.2의 원본 반례, 5.8의 같은 LOT·서로 다른 두 시간대 파형 | CP 사건의 구별 가능성 |
| 정비 전후 언제 발생했는가? | 5.3·5.5의 실제 작업과 A/B 원본, 5.9의 감지·작업·REMARK 표 | 선후 관계·비교 가능 범위 |
| LOT 간 수준 차이는 무엇인가? | 5.4의 SPHC TC·OP 분포와 부록 동일 강종 원본 | 설정·부하 차이 가설, 동일 운전 조건 미확인 |

| 실제 생성 함수 | 정의·최초 실행 위치 | 재사용 위치 |
|---|---|---|
| `draw_report_case`, `draw_cp69_overview` | 5.1 | 5.2·5.3·5.4 |
| `draw_cp69_peaks`, `draw_cp69_zone_peaks` | 5.1.1 | CP 피크별 Z3~Z6 대조 |
| `draw_cp69_b`, `draw_cp69_c` | 각각 5.1.2·5.1.3 | 해당 그림에서 정의 후 실행 |
| `draw_plan_case` | 5.5의 6월 코드 | 11월 코드 |
| 관계 분석 6개 그림 | 5.6·5.7·5.7.1·5.7.2·5.8·5.9의 직접 작성 코드 | 각 절에서 계산·그림·해석 연결 |
| `draw_lot_overview`, `draw_lot_zoom`, `event_evidence`, `draw_lot_maintenance` | 8.1의 각 그림 직전 | 이후 13 LOT |
| `draw_lot_level` | 8.4 동일 강종 대조 | 동일 강종 수준 차이 LOT |


<a id="case-240069"></a>
### 5.1 LOT 240069 — 반복 CP 이탈

**분석 질문:** 반복 CP가 어떤 지속·복귀 양상을 보이며 이후 BM 기록보다 앞서는가?

**선정 근거:** 전환 후보를 제외한 CP 원본 7사건, 최대 편차 +0.459. (a)는 CP 6사건, (b)는 TC-Z3의 국소 Z 변화, (c)는 가장 큰 CP 사건이다. (b)에 CP가 없다는 이유로 (b)를 생략하지 않는다.

**정비 대조:** 가까운 BM REMARK는 “1호기 재발, 절연 저하”. 감지 07/08 08:49, 작업 11:11–19:04이며 LOT 관측은 그 전에 끝난다.

**읽는 순서:** 먼저 (a)(b)(c)의 전체 위치와 CP·TC-Z3·OP-Z3를 같은 행에서 보고, 아래에서 (a)의 피크 6개, (b)의 첫 반응과 약 34분 뒤 별도 반응, (c) 이후 변동성과 30일 내 WO를 각각 확인한다.

**잠정 판단:** 반복 CP와 동반 반응은 관찰되지만, TC-Z3 상승 하나로 모든 CP를 설명하기 어렵다. 07/25 가스 분석계 셀 손상 교체는 계측 경로 확인의 단서이며 CP와의 부품 연결·인과는 미확정이다.

**관계 분석의 현재 범위:** CP 두 채널의 정의가 미확인이므로 기록값의 이탈·반복, 다른 신호의 동반 변화와 시간 순서를 확인한다. 이를 가스 성분·온도·전기 상태의 변화 또는 특정 부품의 고장으로 번역할 근거는 부족하다. 후속 WO와 부품 교체는 같은 설비의 작업 기록이며, CP 원인으로 선정한 결과가 아니다. 채널 정의가 확보되면 물리적으로 대응하는 TC·OP와 계측·제어 경로를 먼저 정하고 다시 비교해야 한다.

In [ ]:
# 이 셀에서 아래 그래프의 matplotlib 생성 함수를 정의하고 바로 실행한다.
CP69=240069
cp69_raw=raw[CP69].set_index('MEAS_DT').asfreq('s')
cp69_base=cp69_raw['CP-Z4'].median()
cp69_pulses=cp_real.loc[cp_real.LOT.eq(CP69)].sort_values('PEAK').reset_index(drop=True).copy()
cp69_pulses['ID']=[f'240069-a{i}' if i<=6 else '240069-c1' for i in range(1,8)]
cp69_pulses['ONSET']=[at(CP69,v).round('s') for v in cp69_pulses.A]
cp69_rows=[]
for pulse in cp69_pulses.itertuples():
    for sensor in [s for s in SIGNALS if s!='CP-Z4']:
        y=cp69_raw[sensor]
        ref=y.loc[pulse.ONSET-pd.Timedelta(minutes=5):pulse.ONSET-pd.Timedelta(minutes=1)]
        q=y.loc[pulse.ONSET-pd.Timedelta(minutes=2):pulse.ONSET+pd.Timedelta(minutes=2)]
        dev=(q-ref.median()).dropna()
        t=dev.abs().idxmax() if len(dev) else pd.NaT
        cp69_rows.append(dict(ID=pulse.ID,SENSOR=sensor,CP_START=pulse.ONSET,CP_PEAK=pulse.PEAK,CP_DEV=pulse.DEV,
            BASE=ref.median(),MAX_DEV=dev.loc[t] if pd.notna(t) else np.nan,MAX_AT=t,
            OFFSET_SEC=(t-pulse.ONSET).total_seconds() if pd.notna(t) else np.nan,N_BASE=ref.count(),N_WINDOW=q.count()))
cp69_signalstats=pd.DataFrame(cp69_rows)
cp69_by_peak=[]
for pulse in cp69_pulses.itertuples():
    d=cp69_signalstats.loc[cp69_signalstats.ID.eq(pulse.ID)].set_index('SENSOR')
    row={'피크':pulse.ID,'CP 시작':pulse.ONSET.strftime('%m/%d %H:%M:%S'),'CP 피크':pulse.PEAK.strftime('%H:%M:%S'),'CP 편차':pulse.DEV}
    for sensor in ['TC-Z3','OP-Z3','TC-Z4','OP-Z4']:
        item=d.loc[sensor];unit='℃' if sensor in TC else '%p'
        row[sensor]=f'{item.MAX_DEV:+.2f}{unit} / {item.OFFSET_SEC:+.0f}초'
    cp69_by_peak.append(row)
cp69_by_peak=pd.DataFrame(cp69_by_peak)
cp69_b=next(e for e in LOT_EVENTS[CP69] if e['tag']=='(b)')
cp69_c=cp69_pulses.iloc[-1]
cp69_later_point=cp69_raw['TC-Z3'].loc[cp69_b['point']:cp69_b['point']+pd.Timedelta(hours=1)].idxmax()
cp69_b_compare=[]
for label,t,m in [('b-1 · 최대 Z',cp69_b['point'],cp69_b['minute']),
                  ('b-2 · 뒤 1시간 내 원본 최고',cp69_later_point,int((cp69_later_point-lot_meta.loc[CP69,'START']).total_seconds()//60))]:
    d=zstats[CP69,'TC-Z3'].loc[m]
    cp=cp69_raw['CP-Z4'].loc[t-pd.Timedelta(minutes=2):t+pd.Timedelta(minutes=2)]
    cp69_b_compare.append(dict(LABEL=label,POINT=t,MINUTE=m,X=d.X,REF=d.MEDIAN,SD=d.SD,Z=d.Z,
                              CP_DEV=(cp-cp69_base).abs().max()))
cp69_b_compare=pd.DataFrame(cp69_b_compare)
cp69_var=[]
for span in [25,50]:
    lo,hi=(5,30) if span==25 else (10,60)
    for phase,a,b in [('전',-hi,-lo),('후',lo,hi)]:
        q=cp69_raw.loc[cp69_c.PEAK+pd.Timedelta(minutes=a):cp69_c.PEAK+pd.Timedelta(minutes=b)]
        cp69_var.append(dict(WINDOW=f'전후 {span}분',PHASE=phase,START=q.index.min(),END=q.index.max(),
            TC3_SD=q['TC-Z3'].std(),TC3_MIN=q['TC-Z3'].min(),TC3_MAX=q['TC-Z3'].max(),
            OP3_LOW_PCT=100*q['OP-Z3'].lt(10).sum()/q['OP-Z3'].notna().sum(),N=q['TC-Z3'].count()))
cp69_variability=pd.DataFrame(cp69_var)
cp69_sd=cp69_raw['TC-Z3'].rolling(600,min_periods=540).std()
cp69_works=cal_work.loc[cal_work.DETC_DT.between(cp69_c.PEAK,cp69_c.PEAK+pd.Timedelta(days=30))].sort_values('DETC_DT').copy()
cp69_after_rows=[]
for w in cp69_works.itertuples():
    cp69_after_rows.append({'WO':w.WO_NO,'종류':w.MNT_TYPE,'감지':w.DETC_DT,'착수':w.STRT_DT,'완료':w.END_DT,
        'REMARK':remark_text(w),'부품·교체 사유':' / '.join(parts_for(w.WO_NO)) or '교체 기록 없음',
        'CP 최대→감지[시간]':(w.DETC_DT-cp69_c.PEAK).total_seconds()/3600})
cp69_after_work=pd.DataFrame(cp69_after_rows)
cp69_peak_checks=[]
assert len(cp69_signalstats)==7*(len(SIGNALS)-1)==147
assert len(cp69_works)==3

def event_reason(k,e):
    if e['kind']=='CP':return f"CP-Z4 · 원본 {len(e['pulses'])}사건 · 최대 편차 {e['deviation']:+.4f} (±0.05 초과)"
    if e['kind']=='Z':return f"{e['sensor']} · 1분 중앙값 Z={e['score']:+.2f} (|Z|≥3)"
    if e['kind']=='전환':return 'TC·OP 동반 하강/승온 · 수리 일정과 전환 형태 대조'
    return '정비 DETC 기록 · 센서 자동 경보라는 뜻은 아님'

def draw_event_locator(ax,head,k,e):
    # 위와 아래 모두 같은 1초 원본·센서색이다. 음영과 테두리만 구간 유형색이다.
    sensor=e['sensor'] if e['kind']=='Z' else 'TC-Z3' if e['kind']=='전환' else 'CP-Z4'
    y=raw[k].set_index('MEAS_DT')[sensor].asfreq('s');xx=y.index
    factor=1e10 if sensor=='CP-Z4M' else 1
    ax.plot(xx,y.to_numpy()*factor,color='#aebac3',lw=.75,label=f'{sensor} · 전체 LOT')
    for other in LOT_EVENTS[k]:
        chosen=other['tag']==e['tag']
        ax.axvspan(at(k,other['a']),at(k,other['b']),color=e['color'] if chosen else '#9aabb8',
                   alpha=.17 if chosen else .035,ec=e['color'] if chosen else 'none',lw=1.3 if chosen else 0)
    a,b=at(k,e['a']).round('s'),at(k,e['b']).round('s');mask=(xx>=a)&(xx<=b)
    selected,=ax.plot(xx[mask],y.to_numpy()[mask]*factor,color=sensor_color(sensor),lw=1.7,label=f'{sensor} · 아래 확대 원본')
    clock_in_lot(ax,k);ax.set_ylabel(display_unit(sensor),fontsize=9);panel_legend(head,ax)
    assert selected.get_color()==sensor_color(sensor)
    locator_visual_checks.append(dict(LOT=k,TAG=e['tag'],SENSOR=sensor,LINE=selected.get_color(),START=a,END=b))
    locator_checks.append((k,e['tag']))


def raw_window(k,s,a,b):
    return raw[k].set_index('MEAS_DT')[s].asfreq('s').loc[a:b]

def zoom_clock(ax,a,b):
    clock_ticks(ax,a,b)
    minutes=(b-a).total_seconds()/60
    if minutes<=80:
        step=next(step for step in [1,2,3,5,10,15,20,30] if minutes/step<=5)
        ax.xaxis.set_major_locator(mdates.MinuteLocator(byminute=range(0,60,step)))


def event_focus_ranges(k,e):
    if e['kind']=='Z':return [(e['point']-pd.Timedelta(minutes=1),e['point'])]
    if e['kind']=='CP':return [(at(k,p.A).round('s'),at(k,p.B).round('s')) for p in e['pulses'].itertuples()]
    if e['kind']=='감지':return [(e['point']-pd.Timedelta(seconds=30),e['point']+pd.Timedelta(seconds=30))]
    return [(row.시작,row.끝) for row in expected.loc[expected.LOT.eq(k)].itertuples()]


def shade_event_focus(ax,k,e):
    for a,b in event_focus_ranges(k,e):ax.axvspan(a,b,color=e['color'],alpha=.16 if e['kind']!='전환' else .12,lw=0)
    if e['kind']=='감지':ax.axvline(e['point'],color=e['color'],ls=EVENT_LINES['감지'],lw=1.1)


def z_window_marker(ax,k,e,annotate=False):
    # 1분 요약값은 실제 창 중앙에 별도 마커로 놓는다. 종료 시각의 원본 값처럼 표시하지 않는다.
    start,end=event_focus_ranges(k,e)[0];middle=start+(end-start)/2
    value=float(zstats[k,e['sensor']].loc[e['minute'],'X'])
    ax.plot([start,end],[value,value],color=e['color'],lw=1.8,zorder=7)
    ax.plot(middle,value,marker='D',ms=4,mfc='white',mec=e['color'],ls='',zorder=8,label='선정 1분창 중앙값')
    if annotate:
        near_note(ax,middle,value,f'{start:%H:%M}–{end:%H:%M} 중앙값 {value:.2f}\nZ={e["score"]:+.2f}',e['color'])


def plot_event_trace(ax,head,k,e,a,b,large=False):
    s=e['sensor'] if e['kind']=='Z' else 'CP-Z4' if e['kind'] in ['CP','감지'] else 'TC-Z3'
    report_signal(ax,k,s,a,b,fine=True);t=e['point'];y=raw_window(k,s,a,b)
    shade_event_focus(ax,k,e)
    if e['kind']=='CP':
        base=raw[k]['CP-Z4'].median()
        for v in [base-.05,base+.05]:ax.axhline(v,color=EVENT_COLORS['CP'],ls='--',lw=.9)
        ax.axhline(base,color=GREY,ls=':',lw=.75)
        for pulse in e['pulses'].itertuples():
            q=y.loc[at(k,pulse.A).round('s'):at(k,pulse.B).round('s')]
            ax.plot(q.index,q,color=e['color'],lw=2.2,zorder=5)
            if a<=pulse.PEAK<=b:
                ax.scatter(pulse.PEAK,y.get(pulse.PEAK,np.nan),s=25,facecolors='white',edgecolors=e['color'],lw=1.5,zorder=6)
                ax.vlines(pulse.PEAK,base,y.get(pulse.PEAK,np.nan),color=e['color'],ls=':',lw=1.)
    elif e['kind']=='Z':
        d=zstats[k,s];xx=at(k,d.index+1);mask=(xx>=a)&(xx<=b)
        for values in [d.MEDIAN-3*d.SD,d.MEDIAN+3*d.SD]:ax.step(xx[mask],values[mask],where='pre',color=e['color'],ls='--',lw=.9)
        z_window_marker(ax,k,e,annotate=False)
    ax.set_ylabel(display_unit(s));zoom_clock(ax,a,b);panel_legend(head,ax)
    if large and e['kind']=='CP':
        value=y.get(t,np.nan)
        near_note(ax,t,value,f'{t:%H:%M:%S} · {s} {value:.4f}\n편차 {e["deviation"]:+.4f}',e['color'])
    elif large and e['kind']=='Z':
        start,end=event_focus_ranges(k,e)[0];middle=start+(end-start)/2
        value=float(zstats[k,s].loc[e['minute'],'X'])
        near_note(ax,middle,value,f'{start:%H:%M}–{end:%H:%M} 중앙값 {value:.2f}\nZ={e["score"]:+.2f}',e['color'])


def case_interval_table(k):
    frame=interval_time_features.loc[interval_time_features.LOT.eq(k)].copy()
    out=pd.DataFrame({'구간 번호':frame.ID,'선정 신호':frame.SENSOR,'기준 시각':frame.POINT.dt.strftime('%m/%d %H:%M:%S'),
        '지속·반복':frame.DURATION_DESCRIPTION,'복귀 기록':frame.RECOVERY})
    show_table(out)

def draw_report_case(k):
    r=lot_meta.loc[k];events=LOT_EVENTS[k]
    if k==240061:
        rows=[{'panels':[(f'TC-Z{zone} · 같은 SPHC의 온도',[f'TC-Z{zone}']),(f'OP-Z{zone} · 같은 SPHC의 출력',[f'OP-Z{zone}'])],'height':2.1} for zone in [4,8]]
        fig,cells=report_page('Case 04 · 240061 | 같은 강종의 운전 수준 차이',
            '승온·종료 전환 제외 후 분포. TC와 OP를 별도로 비교한다. 평균 ◆ · 중앙값 · IQR · 수염 밖 1분창.',rows,width=20,
            info=['OP-Z4뿐 아니라 OP-Z8도 함께 확인한다. 분포 이동과 LOT 내부의 드문 1분창은 서로 다른 근거다.'])
        peers=lots.loc[lots.GRADE.eq('SPHC'),'LOT_NO'].astype(int).tolist()
        for row in cells:
            for ax,head,sigs in row:draw_boxes(ax,head,sigs[0],peers,highlight=k)
    elif k==240069:
        fig=draw_cp69_overview()
    else:
        rows=[{'panels':[(event_id(k,e)+' · 전체 LOT 위치',[]) for e in events],'height':.8},
              {'panels':[(event_id(k,e)+' · '+e['sensor'],[]) for e in events],'height':2.0,
               'captions':[event_reason(k,e) for e in events]},
              {'panels':[(e['tag']+' 동시 온도 · TC-Z3–TC-Z6',HEAT) for e in events],'height':1.25},
              {'panels':[(e['tag']+' 동시 출력 · OP-Z3–OP-Z6',[f'OP-Z{i}' for i in range(3,7)]) for e in events],'height':1.25}]
        number={240069:1,240037:2,240117:3}[k]
        fig,cells=report_page(f'Case {number:02d} · {k} {r.GRADE} | 선정 지점과 동반 반응',
            '각 열은 동일한 구간 번호다. 전체 위치 → 선정 신호 확대 → 같은 시각의 TC·OP를 읽는다. 최대값 하나로 나머지 구간을 정상 처리하지 않는다.',rows,width=24,
            info=[f'구간 {len(events)}개 · 정확한 번호·시각·지속·복귀는 아래 표, 22신호 상세는 부록의 같은 번호로 연결된다.',
                  {240069:'BM 01094 · 1호기 재발, 절연 저하 | 감지 07/08 08:49 · 작업 11:11–19:04 · LOT 관측 뒤의 기록',
                   240037:'LOT 관측 중 작업·감지 교집합 없음. 가까운 WO와 CP 펄스의 원인 관계는 확인되지 않음',
                   240117:'CBM 01854 · 마모량 초과 | 완료 12:44 → CP 최저 13:04:50 (+20.83분) · TBM 01859 계획 점검 07:40–09:35'}[k]],event_key=True)
        for (ax,head,_),event in zip(cells[0],events):draw_event_locator(ax,head,k,event)
        for (ax,head,_),event in zip(cells[1],events):plot_event_trace(ax,head,k,event,at(k,event['a']),at(k,event['b']),large=True)
        for row in cells[2:]:
            for (ax,head,sensors),event in zip(row,events):
                a,b=at(k,event['a']),at(k,event['b'])
                for sensor in sensors:report_signal(ax,k,sensor,a,b,fine=True)
                ax.axvspan(event['point']-pd.Timedelta(seconds=30),event['point']+pd.Timedelta(seconds=30),color=event['color'],alpha=.13,lw=0)
                ax.set_ylabel(display_unit(sensors[0]));zoom_clock(ax,a,b);panel_legend(head,ax)
    emit(fig,f'01_개요/사례_{k}_보고서.png')
    if k==240061:
        show_table(level_check.loc[level_check.LOT.eq(k)&level_check.SENSOR.isin(['TC-Z4','OP-Z4','TC-Z8','OP-Z8']),
            ['SENSOR','LOT_MEDIAN','PEER_MEDIAN','GAP','N_PEERS']].rename(columns={'SENSOR':'신호','LOT_MEDIAN':'240061 중앙값','PEER_MEDIAN':'다른 SPHC LOT 중앙값의 중앙값','GAP':'차이','N_PEERS':'대조 LOT 수'}))
    else:case_interval_table(k)
    if k in [240069,240037,240117]:
        features=cp_time_features.loc[cp_time_features.LOT.eq(k)]
        show_table(features[['EVENT_ID','START','END','SPAN_SEC','POINTS','DEV','PEAK_GAP_MIN','RETURN_CONFIRMED']].rename(columns={
            'EVENT_ID':'구간 번호','START':'CP 이탈 시작','END':'끝','SPAN_SEC':'첫·끝 간격[초]','POINTS':'실측 점 수','DEV':'최대 편차','PEAK_GAP_MIN':'이전 피크 간격[분]','RETURN_CONFIRMED':'60개 연속 복귀 확인'}))
    report_case_checks.append(k)

def draw_cp69_overview():
    events=LOT_EVENTS[CP69]
    rows=[{'panels':[(event_id(CP69,e)+' · 전체 LOT 위치',[]) for e in events],'height':.8},
          {'panels':[(e['tag']+' CP-Z4 · 원본 이탈' if e['kind']=='CP' else e['tag']+' CP-Z4 · TC 변화 때의 대조',[]) for e in events],
           'height':1.6,'captions':[event_reason(CP69,e) if e['kind']=='CP' else '선정 이유는 TC-Z3의 최대 Z. 이 시간의 CP 원본도 직접 확인한다.' for e in events]},
          {'panels':[(e['tag']+' TC-Z3 · 가열 초입 온도',[]) for e in events],'height':1.5},
          {'panels':[(e['tag']+' OP-Z3 · 대응 출력',['OP-Z3']) for e in events],'height':1.4},
          {'panels':[(e['tag']+' TC-Z4·TC-Z5·TC-Z6',['TC-Z4','TC-Z5','TC-Z6']) for e in events],'height':1.2},
          {'panels':[(e['tag']+' OP-Z4·OP-Z5·OP-Z6',['OP-Z4','OP-Z5','OP-Z6']) for e in events],'height':1.2}]
    fig,cells=report_page('Case 01 · 240069 SPHC | CP와 TC-Z3·OP-Z3의 시간 관계',
        '모든 열의 행 순서는 같다. CP → TC-Z3 → OP-Z3 → 다른 가열·균열 신호. 같은 TC-Z3를 위아래에 중복해 그리지 않는다.',rows,width=24,
        info=['(a) 6개 CP 피크는 아래에서 각각 비교한다. (b) 최대 Z와 뒤의 더 큰 온도 변화를 함께 대조한다. (c) 전후 변동 폭·이후 WO를 확인한다.',
              'BM 01094 · 1호기 재발, 절연 저하 | 감지 07/08 08:49 · 작업 11:11–19:04 · 실제 작업은 LOT 관측 뒤'],event_key=True)
    for (ax,head,_),e in zip(cells[0],events):draw_event_locator(ax,head,CP69,e)
    for (ax,head,_),e in zip(cells[1],events):
        ax.set_ylim(cp69_base-.15,cp69_base+cp69_pulses.DEV.max()+.14)
        a,b=at(CP69,e['a']),at(CP69,e['b'])
        if e['kind']=='CP':plot_event_trace(ax,head,CP69,e,a,b,large=True)
        else:
            report_signal(ax,CP69,'CP-Z4',a,b,fine=True);ax.set_ylabel('CP-Z4 [기록값]');zoom_clock(ax,a,b);panel_legend(head,ax)
    for (ax,head,_),e in zip(cells[2],events):
        ax.set_ylim(848,873)
        a,b=at(CP69,e['a']),at(CP69,e['b'])
        if e['kind']=='Z':plot_event_trace(ax,head,CP69,e,a,b,large=True)
        else:
            report_signal(ax,CP69,'TC-Z3',a,b,fine=True);ax.set_ylabel('온도 [℃]');zoom_clock(ax,a,b);panel_legend(head,ax)
    for row in cells[3:]:
        for (ax,head,sensors),e in zip(row,events):
            a,b=at(CP69,e['a']),at(CP69,e['b'])
            for sensor in sensors:report_signal(ax,CP69,sensor,a,b,fine=True)
            ax.axvspan(e['point']-pd.Timedelta(seconds=30),e['point']+pd.Timedelta(seconds=30),color=e['color'],alpha=.12,lw=0)
            ax.set_ylabel(display_unit(sensors[0]));zoom_clock(ax,a,b);panel_legend(head,ax)
            if sensors==['OP-Z3']:ax.set_ylim(-3,103)
    return fig

draw_report_case(240069)

#### 5.1.1 (a) 피크별로 CP와 Z3·Z4~Z6의 TC·OP를 비교한다

**비교 범위:** 그래프는 CP 이탈 첫 기록 전 3분~후 4분, 표의 최대편차는 시작 전후 2분이다. 근접 반응을 대조한 결과이며, 더 긴 지연 관계까지 배제하는 분석은 아니다. 첫 6개 CP는 밀집해 있어 긴 반응창이 서로 겹치므로 각 피크의 영향을 독립적으로 분리하기 어렵다.

**Z3만 원인 후보로 고른 것은 아니다.** 기존 최대 시간순 Z를 설명하는 예시로 Z3를 먼저 보였다. 아래에서 Z3와 Z4~Z6를 같은 CP 피크·비교 기준으로 각각 확인한다. Z4~Z6 그림은 채널별 평소 높낮이를 빼고 변화량으로 겹쳐 보여, 다른 기본 출력 수준에 변화가 가려지지 않게 했다.

**번호 표기:** (a)는 전체 비교 구간, **(a-1)~(a-6)**은 그 안의 개별 CP 피크다. 같은 번호의 CP·TC·OP를 세로로 읽는다. (b-1)·(b-2)는 서로 다른 시각의 두 온도 반응, (c-1)은 (c)의 CP 사건이다. 여러 LOT이 섞인 표에서는 `240069 (a-1)`처럼 LOT을 함께 표시한다.

In [ ]:
# 이 셀에서 아래 그래프의 matplotlib 생성 함수를 정의하고 바로 실행한다.
CP69_ZONE_SENSORS=[f'{prefix}-Z{zone}' for prefix in ['TC','OP'] for zone in [4,5,6]]
cp69_zone_rows=[]
for pulse in cp69_pulses.itertuples():
    for sensor in CP69_ZONE_SENSORS:
        metric=cp69_signalstats.loc[cp69_signalstats.ID.eq(pulse.ID)&cp69_signalstats.SENSOR.eq(sensor)].iloc[0]
        q=cp69_raw[sensor].loc[pulse.PEAK-pd.Timedelta(seconds=15):pulse.PEAK+pd.Timedelta(seconds=15)]
        cp69_zone_rows.append(dict(ID=pulse.ID,SENSOR=sensor,CP_DEV=pulse.DEV,AT_PEAK=q.median()-metric.BASE,
            N_PEAK=q.count(),MAX_DEV=metric.MAX_DEV,OFFSET_SEC=metric.OFFSET_SEC,BASE=metric.BASE))
cp69_zone_metrics=pd.DataFrame(cp69_zone_rows)
cp69_zone_peak_checks=[]
cp69_zone_limits={}
for prefix in ['TC','OP']:
    values=[]
    for pulse in cp69_pulses.iloc[:6].itertuples():
        for sensor in [f'{prefix}-Z{i}' for i in [4,5,6]]:
            base=cp69_zone_metrics.loc[cp69_zone_metrics.ID.eq(pulse.ID)&cp69_zone_metrics.SENSOR.eq(sensor),'BASE'].iloc[0]
            values.append(cp69_raw[sensor].loc[pulse.ONSET-pd.Timedelta(minutes=3):pulse.ONSET+pd.Timedelta(minutes=4)]-base)
    amplitude=pd.concat(values).abs().max()
    cp69_zone_limits[prefix]=(-amplitude*1.15,amplitude*1.15)

def cp69_event_panel(ax,head,pulse,sensor,annotate=False):
    t=pulse.ONSET;a=t-pd.Timedelta(minutes=3);b=t+pd.Timedelta(minutes=4)
    report_signal(ax,CP69,sensor,a,b,fine=True)
    ax.axvspan(t,at(CP69,pulse.B).round('s'),color=EVENT_COLORS['CP'],alpha=.12,lw=0)
    ax.set_xlim(a,b);ax.set_xticks([t+pd.Timedelta(minutes=m) for m in [-2,0,2,4]],['-2','0','+2','+4'])
    ax.set_xlabel('CP 이탈 시작 대비 [분]');ax.set_ylabel(display_unit(sensor));panel_legend(head,ax)
    if sensor=='CP-Z4':
        ax.set_ylim(cp69_base-.12,cp69_base+cp69_pulses.iloc[:6].DEV.max()+.10)
        for level in [cp69_base-.05,cp69_base+.05]:ax.axhline(level,color=EVENT_COLORS['CP'],ls='--',lw=.7)
        y=cp69_raw.loc[pulse.PEAK,sensor];ax.scatter(pulse.PEAK,y,s=25,facecolors='white',edgecolors=EVENT_COLORS['CP'],zorder=7)
        if annotate:near_note(ax,pulse.PEAK,y,f'{pulse.PEAK:%H:%M:%S}  +{pulse.DEV:.3f}',EVENT_COLORS['CP'])
    else:
        # 6개 피크에 같은 세로축을 사용해 진폭을 비교한다.
        windows=[cp69_raw[sensor].loc[p.ONSET-pd.Timedelta(minutes=3):p.ONSET+pd.Timedelta(minutes=4)] for p in cp69_pulses.iloc[:6].itertuples()]
        values=pd.concat(windows).dropna();lo,hi=values.min(),values.max();pad=max((hi-lo)*.12,.1)
        ax.set_ylim(lo-pad,hi+pad)

def draw_cp69_peaks():
    rows=[]
    for start in [0,3]:
        group=list(cp69_pulses.iloc[start:start+3].itertuples())
        for sensor in ['CP-Z4','TC-Z3','OP-Z3']:
            panels=[(f'{event_text(p.ID,include_lot=False)} {sensor}',[sensor]) for p in group]
            captions=[]
            for p in group:
                if sensor=='CP-Z4':captions.append(f'시작 {p.ONSET:%H:%M:%S} · 피크 {p.PEAK:%H:%M:%S} · CP 편차 +{p.DEV:.3f}')
                else:
                    d=cp69_signalstats.loc[cp69_signalstats.ID.eq(p.ID)&cp69_signalstats.SENSOR.eq(sensor)].iloc[0]
                    captions.append(f'기준 대비 최대편차 {d.MAX_DEV:+.2f}'+('℃' if sensor in TC else '%p')+f' · CP 시작 {d.OFFSET_SEC:+.0f}초 위치 (반응 시작 시각 아님)')
            rows.append({'panels':panels,'height':1.5,'captions':captions if sensor=='CP-Z4' else None})
    fig,cells=report_page('240069-a | 6개 CP 피크를 하나씩 같은 시간 폭으로 대조',
        '각 열에서 CP → TC-Z3 → OP-Z3. 0=CP 이탈 첫 기록이며 모든 패널은 전 3분~후 4분이다. 7번째 큰 사건은 c에서 별도 확인한다.',rows,width=24,
        info=['6개 피크에 신호별 같은 세로축을 쓴다. 음영은 원본 CP 편차 0.02 초과 묶음의 첫·끝 기록 범위이며 공백은 채우지 않는다.',
              '전체 21개 다른 신호의 원본 편차·최대편차 시각은 아래 표에 남겼다. 특정 채널을 생략했다고 정상으로 판정하지 않는다.'])
    for row_index,row in enumerate(cells):
        start=0 if row_index<3 else 3;sensor=['CP-Z4','TC-Z3','OP-Z3'][row_index%3]
        for (ax,head,_),pulse in zip(row,cp69_pulses.iloc[start:start+3].itertuples()):
            cp69_event_panel(ax,head,pulse,sensor,annotate=sensor=='CP-Z4')
            if sensor=='CP-Z4':cp69_peak_checks.append(pulse.ID)
    emit(fig,'01_개요/사례_240069_a_피크별대조.png')
    show_table(cp69_by_peak)
    note('**표의 읽기:** 기준은 CP 시작 전 5~1분 중앙값, 비교는 시작 전후 2분이다. 부호는 최대 절대편차의 방향, 초는 그 값의 위치다. 반응 시작·인과 시차 추정이 아니다. 가까운 CP 사건은 비교창이 서로 겹치므로 독립 실험 7회로 해석하지 않는다.')
    table(cp69_signalstats.rename(columns={'MAX_DEV':'기준 대비 최대편차','OFFSET_SEC':'CP 시작 대비 최대편차 위치[초]'}))
    draw_cp69_zone_peaks()

def cp69_b_observations():
    # 서로 다른 두 반응을 한 연속 창에서 확인한다. 기존 (b)의 추가 배율이 아니다.
    a=cp69_b['point'].floor('min')-pd.Timedelta(minutes=14)
    b=cp69_later_point.ceil('min')+pd.Timedelta(minutes=17)
    q=cp69_raw.loc[a:b]
    cp=q['CP-Z4'].dropna();dev=cp-cp69_base
    points=[]
    for number,d in enumerate(cp69_b_compare.itertuples(),1):
        end=at(CP69,d.MINUTE+1)
        minute=cp69_raw['TC-Z3'].loc[end-pd.Timedelta(minutes=1):end-pd.Timedelta(seconds=1)].dropna()
        tc_at=minute.idxmax()
        op=cp69_raw['OP-Z3'].loc[d.POINT-pd.Timedelta(minutes=5):d.POINT+pd.Timedelta(minutes=5)].dropna()
        points.append(dict(ID=f'b-{number}',END=end,TC_AT=tc_at,TC=float(minute.loc[tc_at]),
                           OP_AT=op.idxmin(),OP=float(op.min())))
    return dict(START=a,END=b,DATA=q,POINTS=points,CP_MAX_AT=dev.abs().idxmax(),
                CP_MAX_DEV=float(dev.abs().max()),CP_HITS=int(dev.abs().gt(.05).sum()),
                CP_VALID=len(cp),SECONDS=len(q))

def cp69_results(section):
    if section=='a':
        d=cp69_signalstats.loc[cp69_signalstats.ID.ne('240069-c1')]
        tc=d.loc[d.SENSOR.eq('TC-Z3')];op=d.loc[d.SENSOR.eq('OP-Z3')]
        return f'''**이 그림으로 확인한 것**

- (a)는 CP 피크 6개를 각각 확인한다. 최대편차는 피크마다 다르고 TC-Z3의 최대 절대편차 방향은 증가 {(tc.MAX_DEV>0).sum()}개·감소 {(tc.MAX_DEV<0).sum()}개다. OP-Z3도 감소 {(op.MAX_DEV<0).sum()}개·증가 {(op.MAX_DEV>0).sum()}개로 한 방향으로 반복되지 않는다.
- CP 시작 앞뒤에서 TC·OP가 움직이는 시각도 다르다. 따라서 **TC-Z3 상승 하나로 6개 CP 상승을 모두 설명하기 어렵다.** 크기·방향·최대편차 위치는 반복 반응을 찾는 단서이며, 실제 반응 시작 또는 인과를 뜻하지 않는다.
- **왜 Z3부터 보았는가:** 기존 온도 선별의 최대 시간순 Z가 TC-Z3에서 관측되어 설명 예시로 먼저 확대했다. CP와 가장 강하게 연결됐다는 검정이나 물리적 대응을 근거로 고른 것은 아니다. 바로 위 Z4~Z6 그림과 같은 기준으로 함께 판단한다.
- TC-Z1–Z9·전체 OP·세정·냉각·CP-Z4M까지 CP 외 21신호를 모두 같은 기준으로 계산했다. 접힌 표의 147행은 개별 신호를 다시 확대할 위치를 찾는 목록이다. 단위가 다른 편차를 한 크기로 순위 매기지 않는다.
- CP 계측 정의·원시 변환식·센서 보정·SV·제어/라인 조건이 없어 **CP가 튄 물리적 원인은 아직 미확정**이다. CP-Z4M도 같은 단위의 독립 검증 센서라고 가정할 수 없다.'''
    if section=='b':
        left,right=list(cp69_b_compare.itertuples());evidence=cp69_b_observations()
        gap=(right.POINT-left.POINT).total_seconds()/60
        return f'''**결과: TC-Z3 온도 상승과 OP-Z3 저하가 보여도 CP가 반드시 튀지는 않는다.**

- **b-2는 b-1을 더 확대한 것이 아니다.** b-1은 {left.POINT:%m/%d %H:%M}의 Z 선별 지점이고, b-2는 약 {gap:.0f}분 뒤 {right.POINT:%H:%M:%S}에 나타난 별도의 반응이다. 뒤의 더 높은 온도를 생략하지 않기 위해 함께 확인했다. 그래프는 {evidence['START']:%H:%M}–{evidence['END']:%H:%M}를 끊지 않고 보여준다.
- **그림에서 직접 읽는 것:** 두 반응 모두 TC-Z3가 상승한 뒤 하강하고, OP-Z3는 낮아졌다가 복귀한다. 그동안 기록된 CP의 LOT 중앙값 대비 최대 절대편차는 **{evidence['CP_MAX_DEV']:.4f}로 0.05보다 작다.** 비교창의 CP 경계 초과 기록은 {evidence['CP_HITS']}초다. 관측된 {evidence['CP_VALID']:,}초를 기준으로 하며 미관측 {evidence['SECONDS']-evidence['CP_VALID']:,}초를 정상으로 채우지 않았다.
- **이 대조가 필요한 이유:** (a)·(c)에서는 CP 피크 주변의 TC·OP를 보았다. 여기서는 반대로 **TC·OP가 크게 움직인 때에도 CP 피크가 나타나는가**를 확인한다. 두 사례에서 CP 피크가 없으므로, TC-Z3 상승 또는 OP-Z3 저하 하나만으로 CP 피크를 설명하거나 선별하기 어렵다. 같은 시각의 관측 범위에 대한 결과이며 장시간 지연이나 다른 조건에서의 관계를 배제한 것은 아니다.
- **왜 앞의 작은 온도를 먼저 골랐는가:** b-1은 최고 온도가 아니라 과거 대비 Z가 가장 컸다. 1분 중앙값 {left.X:.2f}℃·과거 SD {left.SD:.2f}℃에서 Z={left.Z:+.2f}였다. 뒤의 b-2는 원본 최고 {cp69_raw.loc[cp69_later_point,'TC-Z3']:.2f}℃·1분 중앙값 {right.X:.2f}℃로 더 높지만, 과거 SD도 {right.SD:.2f}℃로 커져 Z={right.Z:+.2f}였다. **온도의 크기와 이전 대비 이례성은 다르다.** 상세 계산은 접힌 표에서 확인한다.
- **여기서 결론 내리지 않는 것:** CP가 튄 물리적 원인, 두 온도 반응의 정상 여부, OP 저하와 TC 상승 사이의 인과다. 이 그림은 CP 원인 검토에 쓰는 대조 근거이며 설비 정상 판정의 근거는 아니다.'''
    before=cp69_variability.loc[cp69_variability.PHASE.eq('전')].set_index('WINDOW')
    after=cp69_variability.loc[cp69_variability.PHASE.eq('후')].set_index('WINDOW')
    return f'''**CP 이후 TC-Z3가 흔들리는 것은 어떤 관계인가?**

- CP 최고 {cp69_c.PEAK:%m/%d %H:%M:%S} 뒤 TC-Z3·OP-Z3의 큰 반응이 관측된다. 주석은 후 0~15분·15~30분 각각의 TC 최고·OP 최저까지의 시간 차이를 표시한다. 뒤의 반복 반응도 함께 확인한다. 이는 선후 관계이며 전달 지연이나 원인을 추정한 값은 아니다.
- 피크 전 30~5분과 후 5~30분의 TC-Z3 SD는 {before.loc['전후 25분','TC3_SD']:.2f}→{after.loc['전후 25분','TC3_SD']:.2f}℃다. 전 60~10분과 후 10~60분으로 넓히면 {before.loc['전후 50분','TC3_SD']:.2f}→{after.loc['전후 50분','TC3_SD']:.2f}℃다. **비교창에 따라 증가 폭이 달라진다.** OP-Z3<10% 비율은 0 부근의 출력을 요약하는 기술량이며 공정 허용 한계가 아니다.
- 전체 LOT의 이동 SD를 보면 CP 전에 이미 큰 흔들림이 있다. 특히 07/07 23시대의 원본 TC-Z3 SD는 {cp69_raw['TC-Z3'].loc['2024-07-07 23:00':'2024-07-07 23:59:59'].std():.2f}℃다. CP가 변동성을 처음 만들었다고 볼 근거가 부족하다. 독립된 동시 변화, 공유된 운전 조건, 계측·제어 경로라는 설명을 구분해 확인해야 한다.

**뒤의 정비가 CP 원인을 설명하는가?**

- CP 최대 이후 30일 안의 CAL01 감지 기록은 3 WO다. 07/08 BM은 “1호기 재발, 절연 저하”와 O링 3개 예방 교체, 07/10 TBM은 REMARK 미기재, 07/25 BM은 가스 분석계 셀 1개 손상 교체다. 정비 건수가 많다는 인상만으로 CP와 연결하지 않는다.
- 특히 **가스 분석계 셀 손상 교체는 계측 경로를 확인할 단서**지만, CP 최대에서 감지까지 약 {(cp69_works.iloc[-1].DETC_DT-cp69_c.PEAK).total_seconds()/86400:.1f}일 떨어져 있다. CP 채널과 INS-006의 연결, 손상 진단·보정 기록이 없어 원인으로 확정할 수 없다. 07/08 절연 저하도 CP와 동일 원인인지 확인되지 않았다.
- 모든 작업은 240069 관측 뒤다. 다음 온도 관측은 09/08의 240091이며 그동안 운전 조건도 달라질 수 있어 수리로 개선됐는지 검증할 근접 관측이 없다.

**현재 가장 필요한 검증:** CP-Z4·CP-Z4M의 측정/계산 정의와 INS-006 연결 → 각 CP 시작 전후 제어·가스/전기·라인 로그 → 07/08·07/25 실제 작업 진단과 보정 결과 → 조치 직전·직후 같은 조건의 관측 순서로 확인한다. 단일 상관계수로 원인을 단정하지 않는다.'''

def draw_cp69_zone_peaks():
    rows=[]
    for start in [0,3]:
        group=list(cp69_pulses.iloc[start:start+3].itertuples())
        rows.append({'panels':[(event_text(p.ID,include_lot=False)+' CP-Z4',['CP-Z4']) for p in group],'height':1.45,
            'captions':[f'시작 {p.ONSET:%H:%M:%S} · CP 피크 {p.PEAK:%H:%M:%S} · 편차 +{p.DEV:.3f}' for p in group]})
        for prefix in ['TC','OP']:
            sensors=[f'{prefix}-Z{z}' for z in [4,5,6]]
            rows.append({'panels':[(event_text(p.ID,include_lot=False)+' '+prefix+'-Z4·Z5·Z6',sensors) for p in group],'height':1.6})
    fig,cells=report_page('240069-a | CP 피크별 Z4·Z5·Z6의 온도·출력 반응',
        '각 열은 같은 CP 피크다. CP 원본 → TC-Z4~Z6 변화 → OP-Z4~Z6 변화. Z3 비교와 같은 전 3분~후 4분을 사용한다.',rows,width=24,
        info=['TC·OP는 CP 시작 전 5~1분 중앙값을 뺀 변화량이다. 0은 해당 구간의 비교 기준이며 정상 한계가 아니다.',
              '각 신호군은 6개 피크에 같은 세로축이다. 음영=CP 이탈 기록 범위 · 세로 점선=CP 피크 · 색 점=그 시각의 실제 기록값 변화.'])
    for index,row in enumerate(cells):
        start=0 if index<3 else 3;kind=['CP','TC','OP'][index%3]
        for (ax,head,sensors),pulse in zip(row,cp69_pulses.iloc[start:start+3].itertuples()):
            if kind=='CP':cp69_event_panel(ax,head,pulse,'CP-Z4',annotate=True);continue
            a=pulse.ONSET-pd.Timedelta(minutes=3);b=pulse.ONSET+pd.Timedelta(minutes=4)
            for sensor in sensors:
                base=cp69_zone_metrics.loc[cp69_zone_metrics.ID.eq(pulse.ID)&cp69_zone_metrics.SENSOR.eq(sensor),'BASE'].iloc[0]
                y=cp69_raw[sensor].loc[a:b]-base
                ax.plot(y.index,y,color=sensor_color(sensor),lw=1.15,label=sensor)
                value=cp69_raw.loc[pulse.PEAK,sensor]-base
                ax.scatter(pulse.PEAK,value,s=20,color=sensor_color(sensor),edgecolors='white',linewidths=.4,zorder=6)
                cp69_zone_peak_checks.append((pulse.ID,sensor))
            ax.axhline(0,color=GREY,lw=.7,ls='--');ax.axvline(pulse.PEAK,color=EVENT_COLORS['CP'],ls=':',lw=.8)
            ax.axvspan(pulse.ONSET,at(CP69,pulse.B).round('s'),color=EVENT_COLORS['CP'],alpha=.12,lw=0)
            ax.set_xlim(a,b);ax.set_ylim(*cp69_zone_limits[kind])
            ax.set_xticks([pulse.ONSET+pd.Timedelta(minutes=m) for m in [-2,0,2,4]],['-2','0','+2','+4'])
            ax.set_xlabel('CP 이탈 시작 대비 [분]');ax.set_ylabel('기준 대비 변화 [℃]' if kind=='TC' else '기준 대비 변화 [%p]');panel_legend(head,ax)
    emit(fig,'01_개요/사례_240069_a_Z4-Z6_피크별대조.png')
    for prefix in ['TC','OP']:
        d=cp69_zone_metrics.loc[cp69_zone_metrics.SENSOR.str.startswith(prefix)]
        frame=d.pivot(index='ID',columns='SENSOR',values='AT_PEAK').reindex(cp69_pulses.ID).reset_index()
        frame.insert(1,'CP 편차',cp69_pulses.DEV.to_numpy())
        note('**CP 피크 전후 15초 중앙값의 기준 대비 변화 · '+('TC [℃]' if prefix=='TC' else 'OP [%p]')+'**')
        show_table(frame.rename(columns={'ID':'피크'}))
    note('**두 표의 기준:** CP 시작 전 5~1분 중앙값을 빼서 계산했다. 같은 CP 피크 전후 15초를 대조한 값이다. 그래프의 색 점은 피크 시각의 1초 실제 값이므로 31초 중앙값과 구분한다. c1은 아래 c 상세의 큰 CP 사건이다.')
    table(cp69_zone_metrics[['ID','SENSOR','MAX_DEV','OFFSET_SEC','N_PEAK']].rename(columns={
        'ID':'피크','SENSOR':'신호','MAX_DEV':'시작 전후 2분 최대절대편차의 부호 있는 값','OFFSET_SEC':'그 값의 위치[시작 대비 초]','N_PEAK':'피크 전후 15초 실측 수'}))
    note(cp69_zone_result())
    assert len(cp69_zone_metrics)==42 and cp69_zone_metrics.N_PEAK.ge(24).all()
    assert len(cp69_zone_peak_checks)==36 and set(cp69_zone_peak_checks)=={(p.ID,s) for p in cp69_pulses.iloc[:6].itertuples() for s in CP69_ZONE_SENSORS}

def cp69_zone_result():
    d=cp69_zone_metrics.pivot(index='ID',columns='SENSOR',values='AT_PEAK')
    r1,r2,r7=d.loc['240069-a1'],d.loc['240069-a2'],d.loc['240069-c1']
    near=cp69_zone_metrics.loc[cp69_zone_metrics.ID.ne('240069-c1')]
    mt=near.loc[near.SENSOR.str.startswith('TC'),'MAX_DEV'].abs().max()
    mo=near.loc[near.SENSOR.str.startswith('OP'),'MAX_DEV'].abs().max()
    return f'''**Z4~Z6에서 확인한 관계와 남는 질문**

- a1의 CP 편차는 +{cp69_pulses.iloc[0].DEV:.3f}다. 같은 피크 부근의 기준 대비 온도는 TC-Z4 {r1['TC-Z4']:+.3f}℃, TC-Z5 {r1['TC-Z5']:+.3f}℃, TC-Z6 {r1['TC-Z6']:+.3f}℃다. Z4와 Z5·Z6가 같은 방향으로 움직인 사례가 아니다.
- a2에서도 CP는 +{cp69_pulses.iloc[1].DEV:.3f} 상승했지만, TC-Z4 {r2['TC-Z4']:+.3f}℃·TC-Z5 {r2['TC-Z5']:+.3f}℃·TC-Z6 {r2['TC-Z6']:+.3f}℃로 모두 기준보다 낮다. a5·a6의 반응도 표와 곡선으로 함께 확인한다. **양의 CP 피크마다 같은 온도 상승이 반복되지는 않는다.**
- 첫 6개 사건의 시작 전후 2분에서 Z4~Z6 최대 절대편차는 온도 {mt:.3f}℃, 출력 {mo:.3f}%p다. 이는 탐색창의 관측 크기이며 정상·이상의 공정 한계가 아니다. OP 변화의 방향과 최대편차 시각도 피크마다 다르다.
- 가장 큰 c1의 CP 편차는 +{cp69_c.DEV:.3f}인데, 피크 부근 TC-Z4 {r7['TC-Z4']:+.3f}℃·TC-Z5 {r7['TC-Z5']:+.3f}℃·TC-Z6 {r7['TC-Z6']:+.3f}℃다. 이 사건도 Z4~Z6 전체의 동시 상승 하나로 설명할 근거가 부족하다. Z3 외 신호를 생략해서 도출한 결론이 아니다.
- 현재 확인한 것은 **정해진 근접 시간창의 동반 반응이 일정하지 않다**는 사실이다. 6개 사건이 밀집해 있어 긴 지연 응답이 겹친다. 이 결과로 더 긴 지연 관계나 공통 운전 조건의 영향을 배제하지 않는다.
- CP-Z4라는 이름만으로 TC-Z4와 같은 계측점이거나 물리적으로 직접 연결됐다고 가정하지 않는다. 채널 정의·계측점·SV·제어 로그가 확인되면 대응 채널과 지연 시간부터 정해 다시 검증해야 한다.'''

draw_cp69_peaks()
note(cp69_results("a"))

#### 5.1.2 (b) 두 차례 TC-Z3·OP-Z3 반응에서 CP를 대조한다

**(b-1)과 (b-2)는 약 34분 떨어진 별도의 반응이다.** 앞 구간을 더 확대한 것이 아니다. 두 반응 사이를 포함한 연속 시간축에서 TC-Z3·OP-Z3와 CP를 대조하고, 아래 해석에서 이 결과가 CP 원인 검토에 어떤 의미가 있는지 설명한다.

In [ ]:
# 이 셀에서 아래 그래프의 matplotlib 생성 함수를 정의하고 바로 실행한다.
def draw_cp69_b():
    evidence=cp69_b_observations();a,b=evidence['START'],evidence['END']
    first,second=list(cp69_b_compare.itertuples())
    assert evidence['CP_HITS']==0 and evidence['CP_MAX_DEV']<.05
    assert evidence['POINTS'][1]['TC_AT']==cp69_later_point
    cards=[dict(title='b-1 · 먼저 선별한 반응',value=f'{first.POINT:%H:%M} · 과거 대비 Z 최대',
                lines=[f'1분 중앙값 {first.X:.2f}℃ · Z={first.Z:+.2f}',
                       '직전보다 얼마나 이례적인지로 선별'],color=EVENT_COLORS['Z']),
           dict(title='b-2 · 약 34분 뒤 별도 반응',value=f'{cp69_later_point:%H:%M:%S} · {cp69_raw.loc[cp69_later_point,"TC-Z3"]:.2f}℃',
                lines=['b-1 뒤 1시간 안의 원본 TC-Z3 최고',
                       'b-1을 더 확대한 그림이 아님'],color=EVENT_COLORS['Z']),
           dict(title='두 반응에서 확인한 CP 결과',value='CP 탐색 경계 초과 0초',
                lines=[f'전체 비교창 최대 절대편차 {evidence["CP_MAX_DEV"]:.4f} < 0.05',
                       f'CP 실측 {evidence["CP_VALID"]:,}/{evidence["SECONDS"]:,}초 · 미관측은 판정 제외'],color='#477B94')]
    rows=[dict(panels=[(f'전체 LOT에서 아래 비교 범위의 위치 | 07/07 {a:%H:%M}–{b:%H:%M}',['TC-Z3'])],height=.65),
          dict(panels=[('TC-Z3 | 두 차례의 온도 상승과 이후 하강',['TC-Z3'])],height=2.05),
          dict(panels=[('OP-Z3 | 같은 시각의 출력 저하와 복귀',['OP-Z3'])],height=1.55),
          dict(panels=[('CP-Z4 | 두 반응과 그 사이 구간을 같은 경계로 대조',['CP-Z4'])],height=1.45)]
    fig,cells=report_page('240069-(b) | 두 차례 TC-Z3 반응에도 CP 탐색 경계 초과는 없었다',
        'b-1과 b-2는 서로 다른 시각의 반응이다. 아래 세 그래프는 하나의 연속 시간축이며, TC·OP 변화만으로 CP 피크를 설명할 수 있는지 대조한다.',
        rows,width=20,cards=cards,
        info=['주황 음영=각 선정 시점의 1분창 · 지점 주석=원본 값 · CP 점선=LOT 중앙값 ±0.05(탐색 기준, 공정 허용 한계 아님).'])
    ax,head,_=cells[0][0]
    whole=cp69_raw['TC-Z3']
    ax.plot(whole.index,whole,color='#aebac3',lw=.65,label='TC-Z3 · 전체 LOT')
    selected=whole.loc[a:b]
    ax.axvspan(a,b,color=EVENT_COLORS['Z'],alpha=.14,lw=0)
    ax.plot(selected.index,selected,color=EVENT_COLORS['Z'],lw=1.15,label='아래 연속 비교창')
    clock_in_lot(ax,CP69);ax.set_ylabel('TC-Z3 [℃]');panel_legend(head,ax)
    for row,sensor in zip(cells[1:],['TC-Z3','OP-Z3','CP-Z4']):
        ax,head,_=row[0]
        report_signal(ax,CP69,sensor,a,b,fine=True)
        zoom_clock(ax,a,b);ax.set_ylabel(display_unit(sensor));ax.set_xlabel('07/07 실제 측정 시각 · 같은 가로축')
        for point in evidence['POINTS']:
            ax.axvspan(point['END']-pd.Timedelta(minutes=1),point['END'],color=EVENT_COLORS['Z'],alpha=.16,lw=0)
        if sensor=='TC-Z3':
            ax.set_ylim(846,876)
            for point in evidence['POINTS']:
                ax.plot(point['TC_AT'],point['TC'],'o',ms=4,mfc='white',mec=EVENT_COLORS['Z'])
                near_note(ax,point['TC_AT'],point['TC'],
                          f'{point["ID"]} · {point["TC_AT"]:%H:%M:%S}\n원본 {point["TC"]:.2f}℃',EVENT_COLORS['Z'])
        elif sensor=='OP-Z3':
            ax.set_ylim(-18,119)
            for point in evidence['POINTS']:
                ax.plot(point['OP_AT'],point['OP'],'o',ms=4,mfc='white',mec=EVENT_COLORS['Z'])
                near_note(ax,point['OP_AT'],point['OP'],
                          f'{point["ID"]} 부근 · {point["OP_AT"]:%H:%M:%S}\nOP-Z3 최저 {point["OP"]:.1f}%',EVENT_COLORS['Z'])
        else:
            for v in [cp69_base-.05,cp69_base+.05]:ax.axhline(v,color=EVENT_COLORS['CP'],ls='--',lw=.85)
            ax.set_ylim(cp69_base-.085,cp69_base+.085)
            t=evidence['CP_MAX_AT'];value=cp69_raw.loc[t,'CP-Z4']
            near_note(ax,t,value,f'최대 절대편차 {evidence["CP_MAX_DEV"]:.4f} < 0.05',color='#477B94')
        panel_legend(head,ax,extra=[Line2D([0],[0],color=EVENT_COLORS['CP'],ls='--',lw=.85,label='LOT 중앙값 ±0.05')] if sensor=='CP-Z4' else ())
    emit(fig,'01_개요/사례_240069_b_Z와후속온도.png')
    comparison=cp69_b_compare.copy()
    comparison['Z 계산 1분창 끝']=[at(CP69,m+1) for m in comparison.MINUTE]
    table(comparison.rename(columns={'LABEL':'서로 다른 두 반응','POINT':'선정 기준 시각','X':'1분 중앙값[℃]','REF':'과거 중앙값[℃]',
        'SD':'과거 SD[℃]','Z':'시간순 Z','CP_DEV':'선정 시각 ±2분 CP 최대절대편차'}).drop(columns=['MINUTE']))

draw_cp69_b()
note(cp69_results("b"))

#### 5.1.3 (c) 전후 변동성과 이후 30일의 정비를 대조한다

**정비 이력의 해석:** 가스 분석계 셀 손상 교체와 CP 변화의 연결은 현재 보류한다. 부품명과 CP라는 채널명만으로 관련 계측 경로라고 가정하지 않는다. 정의와 설비 대응이 확인된 뒤 작업 진단·보정 결과 및 조치 전후 자료로 검증할 항목이다.

In [ ]:
# 이 셀에서 아래 그래프의 matplotlib 생성 함수를 정의하고 바로 실행한다.
def draw_cp69_c():
    e=next(e for e in LOT_EVENTS[CP69] if e['tag']=='(c)');t=e['point']
    a=t-pd.Timedelta(minutes=20);b=t+pd.Timedelta(minutes=30)
    rows=[{'panels':[('전체 LOT | TC-Z3의 10분 이동 표준편차',[])],'height':1.4},
          {'panels':[('(c-1) CP-Z4 원본',['CP-Z4']),('CP-Z4M · 별도 정의의 계측값',['CP-Z4M'])],'height':2.0},
          {'panels':[('TC-Z3 · CP 전후의 온도',['TC-Z3']),('OP-Z3 · CP 전후의 대응 출력',['OP-Z3'])],'height':1.9},
          {'panels':[('TC-Z4·TC-Z5·TC-Z6',['TC-Z4','TC-Z5','TC-Z6']),('OP-Z4·OP-Z5·OP-Z6',['OP-Z4','OP-Z5','OP-Z6'])],'height':1.3},
          {'panels':[('별도 날짜축 | CP 최대 이후 30일의 CAL01 실제 WO·REMARK',[])],'height':2.3}]
    fig,cells=report_page('240069-c | CP 뒤의 TC-Z3 변화와 정비를 어떻게 해석하는가?',
        '가운데 3행은 같은 실제 시각이다. 맨 위는 LOT 전체 변동성, 맨 아래는 이후 30일의 별도 날짜축이다. 선후와 원인을 구분한다.',rows,width=24,
        info=['온도 변동은 CP 이전에도 있다. 전후 비교창을 바꾸면 SD 차이도 달라지므로 CP가 변동성 증가를 일으켰다고 결론 내리지 않는다.',
              '10분 SD는 원본 600초 중 실측 540개 이상에서만 표시한다. OP·CP의 크기와 직접 비교하는 점수가 아니다.'])
    ax,head,_=cells[0][0];ax.plot(cp69_sd.index,cp69_sd,color=sensor_color('TC-Z3'),lw=1.1,label='TC-Z3 · 10분 SD')
    ax.axvspan(a,b,color=EVENT_COLORS['CP'],alpha=.12,lw=0)
    for pulse in cp69_pulses.itertuples():ax.scatter(pulse.PEAK,cp69_sd.get(pulse.PEAK,np.nan),color=EVENT_COLORS['CP'],s=18,zorder=5)
    peak=cp69_sd.idxmax();near_note(ax,peak,cp69_sd.loc[peak],f'{peak:%H:%M} · SD {cp69_sd.loc[peak]:.2f}℃')
    clock_in_lot(ax,CP69);ax.set_ylabel('TC-Z3 SD [℃]');panel_legend(head,ax,[Line2D([0],[0],marker='o',color=EVENT_COLORS['CP'],lw=0,label='7개 CP 피크 시각')])
    for row in cells[1:4]:
        for ax,head,sensors in row:
            if sensors==['CP-Z4']:plot_event_trace(ax,head,CP69,e,a,b,large=True)
            else:
                for sensor in sensors:report_signal(ax,CP69,sensor,a,b,fine=True)
                ax.axvspan(t-pd.Timedelta(seconds=30),t+pd.Timedelta(seconds=30),color=EVENT_COLORS['CP'],alpha=.15,lw=0)
                zoom_clock(ax,a,b);ax.set_ylabel(display_unit(sensors[0]));panel_legend(head,ax)
                if sensors in [['TC-Z3'],['OP-Z3']]:
                    sensor=sensors[0]
                    for lo,hi in [(0,15),(15,30)]:
                        q=cp69_raw[sensor].loc[t+pd.Timedelta(minutes=lo):t+pd.Timedelta(minutes=hi)].dropna()
                        pt=q.idxmax() if sensor=='TC-Z3' else q.idxmin();value=q.loc[pt]
                        near_note(ax,pt,value,f'{pt:%H:%M:%S} · {value:.2f}'+('℃' if sensor in TC else '%')+chr(10)+f'CP 피크 뒤 {(pt-t).total_seconds()/60:.2f}분')
    ax,head,_=cells[-1][0];r=lot_meta.loc[CP69]
    ax.plot([r.START,r.END],[0,0],color='#147f9c',lw=6,label='온도 관측창')
    for i,w in enumerate(cp69_works.itertuples(),1):
        color=COL[w.MNT_TYPE];ax.plot([w.DETC_DT,w.STRT_DT],[i,i],color=color,ls=':',lw=1.3)
        ax.plot([w.STRT_DT,w.END_DT],[i,i],color=color,lw=6);ax.scatter(w.DETC_DT,i,color=EVENT_COLORS['감지'],marker='D',s=25,zorder=6)
        parts=' / '.join(parts_for(w.WO_NO));label=f'{w.STRT_DT:%m/%d %H:%M} · {remark_text(w)}'+(' · '+parts if parts else '')
        ax.text(w.END_DT+pd.Timedelta(hours=8),i,label,ha='left',va='center',fontsize=9.5,color=color)
    ax.set_yticks(range(4),['LOT 240069']+[f'{w.MNT_TYPE} {w.WO_NO[-5:]}' for w in cp69_works.itertuples()],fontsize=9)
    ax.set_ylim(3.6,-.6);clock_ticks(ax,r.START-pd.Timedelta(hours=6),t+pd.Timedelta(days=30));ax.set_xlabel('실제 기록 날짜 [월/일] · 위의 분 단위 확대와 다른 축')
    panel_legend(head,ax,[Line2D([0],[0],marker='D',color=EVENT_COLORS['감지'],lw=0,label='정비 감지'),Line2D([0],[0],color=COL['BM'],lw=4,label='실제 작업')])
    emit(fig,'01_개요/사례_240069_c_선후와정비.png')
    cp69_peak_checks.append('240069-c1')
    show_table(cp69_variability.rename(columns={'WINDOW':'비교창','PHASE':'전후','TC3_SD':'TC-Z3 SD[℃]','TC3_MIN':'최저[℃]','TC3_MAX':'최고[℃]',
        'OP3_LOW_PCT':'OP-Z3<10% 기록 비율[%]','N':'실측 수'}))
    show_table(cp69_after_work)

draw_cp69_c()
note(cp69_results("c"))

<a id="case-240037"></a>
### 5.2 LOT 240037 — 짧은 CP 펄스와 국소 변화

**분석 질문:** 짧은 CP가 반복되는 동안 TC·OP도 같은 폭으로 이탈하는가?

**선정 근거:** CP 3사건. (a)의 2사건과 (c)의 1사건을 모두 원본으로 확인하고, (b)의 TC-Z3 국소 변화는 별도 근거로 대조한다. 1분 중앙값만으로 짧은 CP를 검사하면 일부 사건이 작게 요약된다.

**정비 대조:** 관측 중 실제 작업·감지 교집합은 없다. 가까운 WO라는 이유로 펄스 원인과 연결하지 않는다.

**잠정 판단:** CP 원본·동시 TC/OP·계측 정의를 함께 확인할 대상이다. 반복 자체가 고장 확정은 아니다.

In [ ]:
# draw_report_case 생성 함수 정의: 5.1. 같은 레이아웃을 재사용한다.
draw_report_case(240037)

<a id="case-240117"></a>
### 5.3 LOT 240117 — CBM 완료 뒤 CP 하강

**분석 질문:** 이 CP 하강이 해당 CBM을 유발한 사전 경보인가?

**선정 근거와 시간 순서:** (a) 수리 후 승온, (b) CP 하강, (c) CLN-IN 국소 변화를 구분한다. CBM 01854 REMARK는 “마모량 초과”, 완료 12:44다. (b)의 CP 최저 13:04:50은 완료 후 20.83분이다. TBM 01859 “계획 점검 수행”도 07:40–09:35에 진행됐다.

**잠정 판단:** 이 CBM의 사전 경보라는 설명과 시간 순서가 맞지 않는다. 수리 완료 뒤 발생했다는 이유만으로 수리 실패를 확정하지도 않는다. 실제 작업 대상·완료 점검·재가동 조건과 CP 계측 로그가 필요하다.

In [ ]:
# draw_report_case 생성 함수 정의: 5.1. 같은 레이아웃을 재사용한다.
draw_report_case(240117)

<a id="case-240061"></a>
### 5.4 LOT 240061 — 같은 SPHC에서 출력 수준 차이

**분석 질문:** 비슷한 온도 중심 수준에서 출력 분포가 다른가, 일부 점만 튀는가?

**선정 근거:** 승온을 제외한 OP-Z4·OP-Z8이 다른 SPHC LOT 대비 표시 경계를 넘는다. TC-Z4·TC-Z8도 함께 보여 온도 자체의 이상으로 오해하지 않게 한다. 같은 강종 4 LOT의 분포 위치·평균·중앙값·폭을 확인한다.

**정비·조건 대조:** 6월 계획수리 후 첫 관측이다. 승온은 제외했지만 설정·속도·두께·부하·보정이 같은지는 확인되지 않았다. 전후 강종도 다르므로 수준 차이를 수리 효과로 비교하지 않는다.

**잠정 판단:** 운전 조건·설정 확인 대상이다. 출력 차이를 고장·에너지 증가로 확정하지 않는다. 이 LOT의 (a)(b)(c) 원본 상세는 부록의 같은 번호로 보존했다.

In [ ]:
# draw_report_case 생성 함수 정의: 5.1. 같은 레이아웃을 재사용한다.
draw_report_case(240061)

### 5.5 계획수리 전(A)·후(B) — 전환과 남는 국소 변화

수리 기간 안에는 온도 관측이 없다. 캘린더·실제 WO·REMARK를 먼저 확인하고, A의 하강과 B의 승온을 원본으로 대조한다. (B-1)은 전환 뒤 남는 TC-Z3 반응 또는 CP 이탈을 확인하기 위한 창이다. 전후 강종이 달라 온도·출력 높낮이를 개선 효과로 해석하지 않는다.

<a id="repair-june"></a>
#### 6월 정기수리 · 일정 → 전후 원본 → 같은 시각 확대

In [ ]:
# 이 셀에서 아래 그래프의 matplotlib 생성 함수를 정의하고 바로 실행한다.
def plan_case_data(idx,pre,post):
    plan=plans.iloc[idx];pr,po=lot_meta.loc[pre],lot_meta.loc[post]
    ws=cal_work.loc[cal_work.STRT_DT.lt(po.END)&cal_work.END_DT.gt(pr.START)].sort_values('STRT_DT')
    before_a=max(pr.START,pr.END-pd.Timedelta(minutes=100));before_b=pr.END
    after_a,after_b=po.START,po.END
    # 승온 뒤 남은 단발 응답을 본다. 11월은 CBM 뒤 CP 사건과 TC·OP를 같은 시각에 대조한다.
    if post==240117:
        hit=cp_real.loc[cp_real.LOT.eq(post)].iloc[0];point=hit.PEAK
        zoom_a,zoom_b=point-pd.Timedelta(minutes=18),point+pd.Timedelta(minutes=12)
    else:
        d=zstats[post,'TC-Z3'].Z.where(states[post].eq('안정 참고'))
        m=int(d.abs().idxmax());point=at(post,m+1)
        zoom_a,zoom_b=point-pd.Timedelta(minutes=10),point+pd.Timedelta(minutes=16)
    return dict(plan=plan,pre=pr,post=po,ws=ws,pre_a=before_a,pre_b=before_b,post_a=after_a,post_b=after_b,point=point,zoom_a=zoom_a,zoom_b=zoom_b)

def draw_plan_case(idx,pre,post,slug):
    d=plan_case_data(idx,pre,post);plan,pr,po,ws=d['plan'],d['pre'],d['post'],d['ws']
    month='6월' if idx==0 else '11월'
    focus_kind='CP' if post==240117 else 'Z'
    focus_color=EVENT_COLORS[focus_kind]
    rows=[{'panels':[('① 정기수리 일정 · 강종 · 관측창 · 실제 WO 작업',[])],'height':2.3+len(ws)*.34},
          {'panels':[(f'수리 전(A) {pre} · {pr.GRADE} | 온도',HEAT),(f'수리 후(B) {post} · {po.GRADE} | 온도',HEAT)],'height':2.0},
          {'panels':[(f'수리 전(A) {pre} | 출력',[f'OP-Z{i}' for i in range(3,7)]),(f'수리 후(B) {post} | 출력',[f'OP-Z{i}' for i in range(3,7)])],'height':2.0},
          {'panels':[('(B-1) 수리 후 확인 구간 · 온도',HEAT),('(B-1) 같은 시각 · 출력',[f'OP-Z{i}' for i in range(3,7)]),('(B-1) 같은 시각 · CP',['CP-Z4'])],'height':1.8}]
    info=[f'계획: {plan.STRT_DT:%m/%d}–{plan.END_DT:%m/%d} (종료일 포함) · {plan.SCOPE} | 이 기간 온도 기록 0행',
          f'전 {pre} {pr.GRADE} {pr.START:%m/%d %H:%M}–{pr.END:%H:%M} / 후 {post} {po.GRADE} {po.START:%m/%d %H:%M}–{po.END:%H:%M}',
          f'전체 시각 범위에 실제 작업 {len(ws)} WO · ▼=감지 / 실선=착수~완료 / ◆=교체 WO 완료(교체 시각 대용) · 음영 안을 보간하지 않음']
    fig,cells=report_page(f'{month} 정기수리 | 일정 → 전후 원본 → 같은 구간 확대',
        '먼저 일정과 다신호의 함께 움직임을 확인하고, 승온·하강 이후 남는 변화를 따로 본다. 전후 강종이 달라 개선 효과로 해석하지 않는다.',rows,width=26,info=info,event_key=True)
    ax,head,_=cells[0][0]
    lo,hi=pr.START-pd.Timedelta(hours=4),po.END+pd.Timedelta(hours=4)
    ax.axvspan(plan.STRT_DT,plan.END_EXCL,color=COL['PLAN'],alpha=.48,lw=0)
    changes=grade_evt.loc[grade_evt.EVT_DT.gt(lo)&grade_evt.EVT_DT.lt(hi)]
    earlier=grade_evt.loc[grade_evt.EVT_DT.le(lo)].tail(1)
    points=[lo]+changes.EVT_DT.tolist()+[hi]
    grades=([earlier.AFT_VAL.iloc[0]] if len(earlier) else ['미기록'])+changes.AFT_VAL.tolist()
    grade_colors={'DP590':'#3f6385','SPCC':'#8e9da9','SPHC':'#ac8c62','SPCEN':'#8e7bb7','SPCE':'#4c929b','SPFH590':'#b66d55','SPCD':'#4b7fb3'}
    for a,b,g in zip(points[:-1],points[1:],grades):
        ax.plot([a,b],[0,0],lw=15,color=grade_colors.get(g,GREY),solid_capstyle='butt')
        if (b-a)/(hi-lo)>.055:ax.text(a+(b-a)/2,0,g,ha='center',va='center',fontsize=10,color='white',fontweight='bold')
    for key in [pre,post]:
        r=lot_meta.loc[key];ax.plot([r.START,r.END],[1,1],color=LOT_COLOR[key],lw=7,solid_capstyle='butt')
        # 짧은 관측 막대의 날짜·시각은 상단 정보에 모아 표시한다.
    for tag,key,align in [('A',pre,'left'),('B',post,'right')]:
        rr=lot_meta.loc[key];xx=rr.START if tag=='A' else rr.END
        ax.text(xx,1.33,f'{tag} · '+('수리 전' if tag=='A' else '수리 후'),ha=align,va='center',fontsize=11,fontweight='bold',color='#17344d')
    for i,w in enumerate(ws.itertuples(),2):
        color=COL[w.MNT_TYPE]
        ax.plot([w.DETC_DT,w.STRT_DT],[i,i],color=color,ls=':',lw=1.2)
        ax.plot([w.STRT_DT,w.END_DT],[i,i],color=color,lw=6,solid_capstyle='butt')
        if lo<=w.DETC_DT<=hi:ax.scatter(w.DETC_DT,i,marker='v',s=32,color=EVENT_COLORS['감지'],zorder=5)
        if parts_for(w.WO_NO):ax.scatter(w.END_DT,i,marker='D',s=32,color=COL['PART'],zorder=5)
        label=f'{remark_text(w)} · {(w.END_DT-w.STRT_DT).total_seconds()/60:.0f}분'
        after=w.END_DT+pd.Timedelta(minutes=35)
        if (hi-after)/(hi-lo)>.25:where,align=after,'left'
        else:where,align=w.DETC_DT-pd.Timedelta(minutes=35),'right'
        ax.text(where,i,label,fontsize=9.5,color=color,ha=align,va='center')
    ax.set_yticks(range(len(ws)+2),['강종 구간','관측 A / B']+[f'{w.MNT_TYPE} {w.WO_NO[-5:]}' for w in ws.itertuples()],fontsize=9)
    ax.set_ylim(len(ws)+1.6,-.8);clock_ticks(ax,lo,hi)
    ax.xaxis.set_major_locator(mdates.DayLocator());ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))
    panel_legend(head,ax,[Patch(facecolor=COL['PLAN'],label='계획수리 날짜'),Line2D([0],[0],color=COL['LOT'],lw=5,label='온도 관측창'),Line2D([0],[0],color=COL['TBM'],lw=5,label='TBM 작업')]+([Line2D([0],[0],color=COL['CBM'],lw=5,label='CBM 작업')] if ws.MNT_TYPE.eq('CBM').any() else []))
    for rownum in [1,2]:
        for j,(ax,head,sigs) in enumerate(cells[rownum]):
            key=pre if j==0 else post;a,b=(d['pre_a'],d['pre_b']) if j==0 else (d['post_a'],d['post_b'])
            for sensor in sigs:report_signal(ax,key,sensor,a,b,fine=True)
            for tr in expected.loc[expected.LOT.eq(key)].itertuples():ax.axvspan(tr.시작,tr.끝,color=GREY,alpha=.15,lw=0)
            if j==1:
                ax.axvspan(d['zoom_a'],d['zoom_b'],color=focus_color,alpha=.15,lw=0)
                for w in ws.loc[ws.STRT_DT.lt(po.END)&ws.END_DT.gt(po.START)].itertuples():
                    for when,ls in [(w.STRT_DT,'--'),(w.END_DT,'-.')]:ax.axvline(when,color=COL[w.MNT_TYPE],ls=ls,lw=.9)
            ax.set_ylabel('온도[℃]' if rownum==1 else '출력[%]');clock_ticks(ax,a,b);panel_legend(head,ax)
        low=min(c[0].get_ylim()[0] for c in cells[rownum]);high=max(c[0].get_ylim()[1] for c in cells[rownum])
        for c in cells[rownum]:c[0].set_ylim(low,high)
    for ax,head,sigs in cells[3]:
        for sensor in sigs:report_signal(ax,post,sensor,d['zoom_a'],d['zoom_b'],fine=True)
        ax.axvline(d['point'],color=focus_color,ls=EVENT_LINES[focus_kind],lw=1.5)
        ax.axvspan(d['point']-pd.Timedelta(minutes=1),d['point']+pd.Timedelta(minutes=1),color=focus_color,alpha=.18,lw=0)
        if 'CP-Z4' in sigs:
            base=raw[post]['CP-Z4'].median()
            for y in [base-.05,base+.05]:ax.axhline(y,color=EVENT_COLORS['CP'],ls='--',lw=.9)
        ax.set_ylabel(display_unit(sigs[0]));clock_ticks(ax,d['zoom_a'],d['zoom_b']);panel_legend(head,ax)
    for ax,head,sigs in cells[3]:
        sensor='CP-Z4' if 'CP-Z4' in sigs else 'TC-Z3' if 'TC-Z3' in sigs else 'OP-Z3'
        y=raw_window(post,sensor,d['zoom_a'],d['zoom_b']).dropna()
        point=(y-raw[post][sensor].median()).abs().idxmax() if sensor=='CP-Z4' else y.idxmax() if sensor=='TC-Z3' else y.idxmin()
        unit='℃' if sensor in TC else '%' if sensor in OP else ''
        near_note(ax,point,y.loc[point],f'{point:%H:%M:%S}\n{sensor} {y.loc[point]:.3f}{unit}',focus_color)
    emit(fig,f'03_상세/04_정비대조/정기수리_{month}_일정_전후_확대.png')
    show_table(pd.DataFrame([{'WO':w.WO_NO,'유형':w.MNT_TYPE,'감지':w.DETC_DT,'착수':w.STRT_DT,'완료':w.END_DT,'작업분':(w.END_DT-w.STRT_DT).total_seconds()/60,'내용':remark_text(w),'교체 기록':' / '.join(parts_for(w.WO_NO)) or '-'} for w in ws.itertuples()]))
    return {'계획수리 시작':plan.STRT_DT,'계획수리 끝(날짜)':plan.END_DT,'전 LOT':pre,'후 LOT':post,'수리 중 온도 행':int(temp.MEAS_DT.between(plan.STRT_DT,plan.END_EXCL,inclusive='left').sum())}

plan_compare=[]
plan_compare.append(draw_plan_case(0,240058,240061,'june'))

**이 그림에서 확인한 사실**

1. **수리 기간과 관측을 분리했다.** 계획수리 안에는 온도 0행이다. 위 파란 날짜 범위에서는 작업 WO와 강종 이벤트를 확인하며, 온도는 수리 전(A) 240058(DP590)·수리 후(B) 240061(SPHC)에서만 관측된다. 전체 범위 실제 작업은 6 WO다. 기간 안의 온도·정지 상태를 추정해서 그리지 않았다. ◆는 교체 WO 완료 시각을 대용하며 실제 교체 순간은 아니다.
2. **일정과 맞는 전환은 먼저 설명한다.** 전 LOT 하강 06/09 07:37–09:02, 후 LOT 승온 06/15 01:04–02:40를 회색으로 표시했다. 여러 TC·OP의 동시 변화와 수리 전후라는 위치는 운전 전환 가설을 뒷받침한다. 사후 형태 분류여서 실제 모드·SV·정지 명령 확인은 남는다.
3. **수리 후(B)의 남는 응답을 (B-1)로 확대했다.** (B-1) 06/15 03:05–03:31의 TC·OP·CP는 같은 시간축이다. 확인 시각 전 10~5분과 후 5~10분 TC-Z3 중앙값은 858.63℃ → 858.47℃다. 원본의 복귀·지속과 함께 읽는다. 이 창의 CP 최대 절대 편차는 0.0209다. 국소 응답 근거이며 정상 범위의 검증 자료는 아니다.

**어떻게 이상이 아닐 가능성을 검토하는가?** 일정과 다신호 동시 하강·승온 → 기준 수준으로 접근·복귀 → 같은 TC–OP의 반복 제어 응답 → 별도 CP 이탈 잔존 여부 순서로 본다. 근거가 있는 전환은 **고장 후보와 분리한 설명 가능 구간**으로 둔다. 구간 전체의 정상 보증으로 바꾸지 않는다. 전환 뒤의 단발 응답·CP 사건도 별도 검토한다.

**수리 효과로 결론 내릴 수 없는 이유:** 강종 DP590→SPHC, 관측 시점과 운전 조건이 다르고 수리 중 온도도 없다. 출력·온도 높낮이를 수리의 개선·악화 효과로 해석하지 않는다.

**왜 (B-1)을 골랐는가?** 승온 참고 구간 이후 TC-Z3의 절대 Z가 가장 큰 창을 코드로 선택했다. 모든 온도 또는 전체 수리 효과가 이상이라는 뜻은 아니다.

**수리를 했는데 왜 다시 변화가 있는가? — 현재 자료로 나눠 볼 가설**

- **재가동 전환:** 공통 승온은 재가동 형태로 설명할 수 있어 수준 비교에서 제외했다. (B-1)은 그 이후의 반응을 따로 확인한다.
- **국소 제어 응답:** TC-Z3의 단발 상승·하강과 OP-Z3의 저하·복귀를 같은 시간에 대조한다. 이 동반 반응은 제어 응답 가설과 부합할 수 있지만 설정값·제어 로그가 없어 원인을 확정하지 않는다.
- **정비 대상·원인 불일치:** 캘린더의 정기수리와 WO REMARK는 작업 기록이다. 그 작업이 Z3 제어 또는 CP 변화의 원인을 제거했다는 검증 기록은 없다. 수리 후 관측됐다는 이유만으로 수리 실패·재고장으로 부르지 않는다.
- **추가 검증:** 같은 강종·설정·속도·두께에서 재현되는지, SV 대비 편차와 OP 응답, 정비 대상 부품·계측 보정·완료 점검 결과를 확인해야 원인을 좁힐 수 있다.

**6월 확대의 핵심:** 승온 이후 Z3의 단발 응답이 있어도 Z4–Z6·CP가 같은 폭으로 지속 이탈하는지 따로 본다. 온도·출력이 돌아오는 모습은 일시 제어 반응의 근거이며, 정상 제어임을 확정하려면 SV·조업 조건이 필요하다.

<a id="repair-november"></a>
#### 11월 정기수리 · 일정 → 전후 원본 → 같은 시각 확대

In [ ]:
# draw_plan_case 생성 함수 정의: 5.5의 6월 코드. 같은 레이아웃을 재사용한다.
plan_compare.append(draw_plan_case(1,240114,240117,'november'))
plan_compare=pd.DataFrame(plan_compare)

**이 그림에서 확인한 사실**

1. **수리 기간과 관측을 분리했다.** 계획수리 안에는 온도 0행이다. 위 파란 날짜 범위에서는 작업 WO와 강종 이벤트를 확인하며, 온도는 수리 전(A) 240114(SPCEN)·수리 후(B) 240117(SPCD)에서만 관측된다. 전체 범위 실제 작업은 7 WO다. 기간 안의 온도·정지 상태를 추정해서 그리지 않았다. ◆는 교체 WO 완료 시각을 대용하며 실제 교체 순간은 아니다.
2. **일정과 맞는 전환은 먼저 설명한다.** 전 LOT 하강 11/04 20:22–20:59, 후 LOT 승온 11/13 07:08–08:37를 회색으로 표시했다. 여러 TC·OP의 동시 변화와 수리 전후라는 위치는 운전 전환 가설을 뒷받침한다. 사후 형태 분류여서 실제 모드·SV·정지 명령 확인은 남는다.
3. **수리 후(B)의 남는 응답을 (B-1)로 확대했다.** (B-1) 11/13 12:46–13:16의 TC·OP·CP는 같은 시간축이다. 확인 시각 전 10~5분과 후 5~10분 TC-Z3 중앙값은 869.33℃ → 855.14℃다. 원본의 복귀·지속과 함께 읽는다. 이 창의 CP 최대 절대 편차는 0.0829다. 국소 응답 근거이며 정상 범위의 검증 자료는 아니다.

**어떻게 이상이 아닐 가능성을 검토하는가?** 일정과 다신호 동시 하강·승온 → 기준 수준으로 접근·복귀 → 같은 TC–OP의 반복 제어 응답 → 별도 CP 이탈 잔존 여부 순서로 본다. 근거가 있는 전환은 **고장 후보와 분리한 설명 가능 구간**으로 둔다. 구간 전체의 정상 보증으로 바꾸지 않는다. 전환 뒤의 단발 응답·CP 사건도 별도 검토한다.

**수리 효과로 결론 내릴 수 없는 이유:** 강종 SPCEN→SPCD, 관측 시점과 운전 조건이 다르고 수리 중 온도도 없다. 출력·온도 높낮이를 수리의 개선·악화 효과로 해석하지 않는다.

**왜 (B-1)을 골랐는가?** CBM 완료 뒤 CP 원본 이탈이 남는 창을 선택했다. 온도 최대점만 보고 정한 창이 아니다.

**수리를 했는데 왜 다시 변화가 있는가? — 현재 자료로 나눠 볼 가설**

- **재가동 전환:** 공통 승온은 재가동 형태로 설명할 수 있어 수준 비교에서 제외했다. (B-1)은 그 이후의 반응을 따로 확인한다.
- **국소 제어 응답:** TC-Z3의 단발 상승·하강과 OP-Z3의 저하·복귀를 같은 시간에 대조한다. 이 동반 반응은 제어 응답 가설과 부합할 수 있지만 설정값·제어 로그가 없어 원인을 확정하지 않는다.
- **정비 대상·원인 불일치:** 캘린더의 정기수리와 WO REMARK는 작업 기록이다. 그 작업이 Z3 제어 또는 CP 변화의 원인을 제거했다는 검증 기록은 없다. 수리 후 관측됐다는 이유만으로 수리 실패·재고장으로 부르지 않는다.
- **추가 검증:** 같은 강종·설정·속도·두께에서 재현되는지, SV 대비 편차와 OP 응답, 정비 대상 부품·계측 보정·완료 점검 결과를 확인해야 원인을 좁힐 수 있다.

**11월 확대의 핵심:** CP 하강은 CBM 12:44 완료 후에 발생한다. 승온이라는 설명만으로 이 CP 사건까지 정상 처리하지 않으며, 해당 CBM 전 경보라는 해석도 시간 순서와 맞지 않는다.

<a id="relation-events"></a>
### 5.6 Q1. CP 사건마다 Z3~Z6의 변화 방향이 반복되는가?

**질문:** 240069에서 본 관계가 다른 CP 사건과 존에서도 반복되는가?

**선정 근거:** 14 LOT 전체를 같은 CP 기준으로 검사한 뒤, 운전 전환 후보를 제외한 **6 LOT·14사건을 모두** 비교한다. CP 양·음 방향을 그대로 둔다. CP 피크 전후 15초의 신호 중앙값에서 **해당 사건 시작 전 5~1분 중앙값**을 뺀다. TC는 ℃, OP는 %p로 따로 표시한다. 최고 편차를 골라 색칠하는 방식과 달리 같은 CP 시각에 얼마나 변했는지 보는 그림이다.

**확인하려는 것:** 어느 존이 같은 방향으로 반복되는지, CP 크기·부호가 다를 때 반응도 달라지는지다. 앞 기준 구간은 신호 190초 이상, 피크 주변은 31초 중 24초 이상 실제 기록이 있을 때만 표시한다. 0은 변화량의 기준이며 정상 한계가 아니다.

**구간 연결:** `240069 (a-1)`은 기존 상세 그림 (a)의 첫 CP 펄스, `240069 (c-1)`은 (c)의 CP 사건이다. 다른 LOT도 같은 상세 구간 문자와 CP 순번을 사용한다.


In [ ]:
# CP 사건 목록 → 사건별 기준·같은 CP 시각 변화량 → 직접 그리는 히트맵
REL_SENSORS=[f'{kind}-Z{z}' for kind in ['TC','OP'] for z in range(3,7)]
relation_raw={k:g.set_index('MEAS_DT').asfreq('s') for k,g in raw.items()}
relation_events=cp_real.sort_values(['LOT','PEAK']).reset_index(drop=True).copy()
relation_events['EVENT_NO']=relation_events.groupby('LOT').cumcount()+1
# 원본 상세 그림의 a/b/c 구간 ID에 CP 시간순 번호를 붙여 같은 사건을 추적한다.
relation_event_source=cp_time_features[['LOT','PEAK','EVENT_ID']].copy()
relation_events=relation_events.merge(relation_event_source,on=['LOT','PEAK'],how='left',validate='one_to_one')
relation_events['ID']=relation_events.EVENT_ID+(relation_events.groupby('EVENT_ID').cumcount()+1).astype(str)
assert relation_events.ID.is_unique and relation_events.EVENT_ID.notna().all()
relation_events['ONSET']=[at(int(r.LOT),r.A).round('s') for r in relation_events.itertuples()]
relation_rows=[]
for ev in relation_events.itertuples():
    frame=relation_raw[int(ev.LOT)]
    for sensor in REL_SENSORS+['CP-Z4M']:
        pre=frame[sensor].loc[(frame.index>=ev.ONSET-pd.Timedelta(minutes=5))&(frame.index<ev.ONSET-pd.Timedelta(minutes=1))]
        center=frame[sensor].loc[ev.PEAK-pd.Timedelta(seconds=15):ev.PEAK+pd.Timedelta(seconds=15)]
        valid=pre.count()>=190 and center.count()>=24
        relation_rows.append(dict(ID=ev.ID,LOT=int(ev.LOT),SENSOR=sensor,CP_DEV=ev.DEV,PEAK=ev.PEAK,
            BASE=pre.median(),DELTA=center.median()-pre.median() if valid else np.nan,
            N_BASE=int(pre.count()),N_PEAK=int(center.count()),VALID=valid))
relation_event_delta=pd.DataFrame(relation_rows)
relation_repetition=[]
for sensor,g in relation_event_delta.loc[relation_event_delta.SENSOR.isin(REL_SENSORS)].groupby('SENSOR',sort=False):
    v=g.DELTA.dropna()
    relation_repetition.append({'신호':sensor,'유효 사건':len(v),'증가':int(v.gt(0).sum()),'감소':int(v.lt(0).sum()),
        '0':int(v.eq(0).sum()),'사건 변화량 중앙값':v.median(),
        'LOT 동일 가중 변화량 중앙값':g.groupby('LOT').DELTA.median().median()})
relation_repetition=pd.DataFrame(relation_repetition)
from matplotlib.colors import TwoSlopeNorm
fig,cells=report_page('CP 14사건 | 같은 CP 피크 시각의 Z3~Z6 변화량',
    '행=CP 사건 · 열=존. 앞 기준 대비 증가(빨강)·감소(파랑)와 실제 변화량을 함께 본다. CP 크기만으로 TC·OP 반응을 대신 판단하지 않는다.',
    [dict(panels=[('TC-Z3~Z6 | CP 피크 부근 온도 변화',[f'TC-Z{i}' for i in range(3,7)]),
                  ('OP-Z3~Z6 | CP 피크 부근 출력 변화',[f'OP-Z{i}' for i in range(3,7)])],height=5.6)],width=24,
    info=['기준=사건 시작 전 5~1분 중앙값 · 표시=CP 피크 ±15초 중앙값 - 기준 · 같은 LOT의 가까운 사건은 독립 표본이 아님.'])
for (ax,head,_),kind in zip(cells[0],['TC','OP']):
    sensors=[f'{kind}-Z{i}' for i in range(3,7)]
    matrix=relation_event_delta.pivot(index='ID',columns='SENSOR',values='DELTA').reindex(index=relation_events.ID,columns=sensors)
    limit=max(float(matrix.abs().max().max()),.001)
    cmap=plt.get_cmap('RdBu_r').copy();cmap.set_bad('#e5e9ed')
    ax.imshow(np.ma.masked_invalid(matrix.to_numpy()),aspect='auto',cmap=cmap,norm=TwoSlopeNorm(vmin=-limit,vcenter=0,vmax=limit))
    ax.set_xticks(range(4),sensors);ax.set_yticks(range(len(matrix)),
        [f'{event_text(ev.ID)}\nCP {ev.DEV:+.3f}' for ev in relation_events.itertuples()],fontsize=9)
    ax.set_xlabel('신호 · 색 범위 ±'+f'{limit:.2f}'+('℃' if kind=='TC' else '%p'))
    ax.grid(False)
    for j,row in enumerate(matrix.to_numpy()):
        for i,value in enumerate(row):
            ax.text(i,j,(f'{value:+.3f}' if kind=='TC' else f'{value:+.2f}') if np.isfinite(value) else '보류',ha='center',va='center',fontsize=9.5,
                color='white' if np.isfinite(value) and abs(value)>limit*.6 else '#17344d')
    panel_legend(head,ax,extra=[Patch(facecolor='#b2182b',label='증가'),Patch(facecolor='#2166ac',label='감소'),Patch(facecolor='#e5e9ed',label='기록 부족')])
emit(fig,'05_관계검토/01_CP사건별_Z3-Z6_변화량.png')
show_table(relation_repetition)
table(relation_event_delta)

parts=[]
for sensor in REL_SENSORS:
 r=relation_repetition.loc[relation_repetition['신호'].eq(sensor)].iloc[0]
 parts.append(f"{sensor}: 증가 {r['증가']}/감소 {r['감소']}(유효 {r['유효 사건']})")
note('**실제 계산 결과:** '+ ' · '.join(parts)+'. 이는 CP 피크 시각의 앞 기준 대비 변화 방향이다. 신호별 방향이 섞이는지 표·원본과 함께 읽으며, 증가·감소 횟수를 고장 확률로 바꾸지 않는다.')

small=relation_event_delta.loc[relation_event_delta.SENSOR.isin(['TC-Z4','TC-Z5','TC-Z6']),'DELTA'].abs().max()
z3=relation_event_delta.loc[relation_event_delta.SENSOR.eq('TC-Z3'),'DELTA'].abs().max()
note(f'**존 간 크기 차이:** 같은 CP 피크 시각의 앞 기준 대비 최대 절대 온도 변화는 TC-Z3 {z3:.2f}℃, TC-Z4~Z6 {small:.2f}℃다. Z4~Z6를 생략할 근거가 아니라 작은 동반 반응도 별도 축·시차·대조로 확인할 이유다.')


**관찰 결과·반례·해석**

- 색의 방향이 사건마다 같은지 먼저 보고, 그다음 숫자로 진폭을 확인한다. 동일한 CP 상승이라도 온도·출력이 매번 같은 방향·크기로 움직여야 단순한 공통 반응 가설을 지지할 수 있다.
- 이 그림의 값은 CP 피크 부근의 변화다. 앞뒤에 더 큰 변화가 있었는지와 어느 신호가 먼저 움직였는지는 5.7의 같은 시간축 원본·앞뒤 파형으로 따로 확인한다. CP 상승·하강을 섞지 않고 방향별로 본다. 작은 수치나 0에 가까운 변화만으로 긴 지연 관계까지 배제하지 않는다.
- 같은 LOT에서 가까이 발생한 펄스는 앞 기준 창이 겹친다. 14사건을 독립 실험 14회로 취급하지 않고 사건 수와 LOT 수를 나눠 표시한다. 운전 설정과 CP 정의가 없으므로 반복 방향만으로 고장 원인을 선정하지 않는다.


<a id="relation-lags"></a>
### 5.7 Q2. CP가 떨어질 때, 주변 존의 온도·출력은 어떻게 움직이는가?

**질문:** CP 하강을 기준으로 앞뒤에 TC-Z3~Z6·OP-Z3~Z6가 어떤 모양을 보이는가? CP 최저점 전에 이미 변화가 있었는가?

**선정 근거:** 전환 후보를 제외한 14 CP 사건 중 하강은 **240024 (a-1)·240117 (b-1), 2사건**이다. 여기서 하강은 LOT 중앙값보다 0.05 이상 낮아진 이탈을 뜻한다. 양의 CP 피크가 내려와 복귀하는 과정은 상승 사건의 원본에서 따로 확인한다. 상승 12사건과 섞어 평균내지 않고 두 하강 사건을 먼저 나란히 본다. 상승 사건도 아래에서 모두 같은 형식으로 펼쳐볼 수 있다.

**그림 읽기:** 각 열은 한 CP 사건이다. 위에서 아래로 **CP → TC-Z3 실제 온도 → OP-Z3 실제 출력 → TC-Z4~Z6 변화 → OP-Z4~Z6 변화**를 읽는다. 모든 행은 같은 실제 시각으로 맞추며, 가로축 **0은 그 사건의 CP 최저점**, 앞뒤 15분이다. 붉은 음영은 해당 CP 이탈 기록 범위이고 세로 점선은 CP 최저점이다. CP 가로 점선은 LOT 중앙값 ±0.05의 탐색 기준이며 물리적 정상 한계가 아니다. 선을 결측 구간에 이어 붙이지 않는다.

Z4~Z6는 작은 움직임도 보이도록 **각 신호의 CP 이탈 시작 전 5~1분 중앙값을 뺀 변화량**을 그린다. 0은 비교 기준이다. CP 최저점 ±15초의 중앙값 변화는 패널 위 숫자, 그 시각의 1초 실제 값은 점으로 구분한다. Z3는 원본 값으로 그려 온도 상승·하강, 출력 저하·복귀가 직접 보이게 했다. 기준창 자체가 정상 운전이라는 뜻은 아니다.

**확인할 것:** CP 최저점의 위치에서 각 선을 위아래로 읽고, 앞선 TC 최고·OP 최저점과 비교한다. 서로 다른 신호의 극값 시각은 반응 시작 시각·소재 전달 시간·물리적 원인으로 해석하지 않는다. 기존 시차 상관 표는 접힌 계산 참고에 두었다.


In [ ]:
# CP 하강과 상승을 분리하고, 같은 사건의 CP·온도·출력을 실제 파형으로 비교한다.
# 이 셀의 matplotlib 코드가 바로 아래 본문 그림과 접힌 사건별 그림을 만든다.
REL_WAVE_MIN=15
relation_wave_checks=[]
relation_wave_rows=[]
for ev in relation_events.itertuples():
    frame=relation_raw[int(ev.LOT)]
    for sensor in REL_SENSORS:
        source=relation_event_delta.loc[relation_event_delta.ID.eq(ev.ID)&relation_event_delta.SENSOR.eq(sensor)].iloc[0]
        q=frame[sensor].loc[ev.PEAK-pd.Timedelta(minutes=REL_WAVE_MIN):ev.PEAK+pd.Timedelta(minutes=REL_WAVE_MIN)].dropna()
        extreme=q.idxmax() if sensor.startswith('TC') else q.idxmin()
        before=frame[sensor].loc[ev.PEAK-pd.Timedelta(minutes=10):ev.PEAK-pd.Timedelta(minutes=5)]
        after=frame[sensor].loc[ev.PEAK+pd.Timedelta(minutes=5):ev.PEAK+pd.Timedelta(minutes=10)]
        relation_wave_rows.append(dict(ID=ev.ID,LOT=int(ev.LOT),DIRECTION='하강' if ev.DEV<0 else '상승',SENSOR=sensor,
            BASE=source.BASE,AT_CP=source.BASE+source.DELTA if source.VALID else np.nan,DELTA=source.DELTA,
            N_BASE=int(source.N_BASE),N_PEAK=int(source.N_PEAK),
            BEFORE=before.median(),AFTER=after.median(),N_BEFORE=int(before.count()),N_AFTER=int(after.count()),
            EXTREME_AT=extreme,EXTREME=float(q.loc[extreme]),OFFSET_MIN=(extreme-ev.PEAK).total_seconds()/60))
relation_wave_metrics=pd.DataFrame(relation_wave_rows)
relation_down=relation_events.loc[relation_events.DEV.lt(0)].copy()
relation_up=relation_events.loc[relation_events.DEV.gt(0)].copy()
assert len(relation_down)==2 and len(relation_up)==12


def cp_wave_caption(ev,kind):
    d=relation_wave_metrics.loc[relation_wave_metrics.ID.eq(ev.ID)].set_index('SENSOR')
    if kind=='CP':
        value=relation_raw[int(ev.LOT)].loc[ev.PEAK,'CP-Z4']
        return f'{ev.PEAK:%m/%d %H:%M:%S} · 원본 {value:.4f} · LOT 중앙값 대비 {ev.DEV:+.4f}'
    if kind in ['TC3','OP3']:
        sensor='TC-Z3' if kind=='TC3' else 'OP-Z3';r=d.loc[sensor]
        when=f'CP보다 {abs(r.OFFSET_MIN):.2f}분 '+('앞' if r.OFFSET_MIN<0 else '뒤')
        label='온도 최고점' if kind=='TC3' else '출력 최저점'
        return f'{r.EXTREME_AT:%H:%M:%S} {label} {r.EXTREME:.2f}'+('℃' if kind=='TC3' else '%')+f' · {when} (반응 시작 시각 아님)'
    prefix='TC' if kind=='TC456' else 'OP';unit='℃' if prefix=='TC' else '%p'
    words=[f'{prefix}-Z{z} {d.loc[f"{prefix}-Z{z}","DELTA"]:+.2f}' for z in [4,5,6]]
    return ('CP 최저 부근의 앞 기준 대비 변화: ' if ev.DEV<0 else 'CP 최고 부근의 앞 기준 대비 변화: ')+' / '.join(words)+' '+unit


def draw_cp_zone_patterns(group,heading):
    # 2열=서로 다른 CP 사건, 5행=CP / TC-Z3 / OP-Z3 / TC-Z4~6 / OP-Z4~6.
    # 모든 존의 같은 실제 시각을 맞춘다. 숫자 시차 r 대신 원본의 높낮이·모양을 읽는다.
    import textwrap
    from matplotlib.ticker import MaxNLocator
    events=list(group.itertuples());count=len(events);width=24 if count==2 else 14
    specs=[('CP',['CP-Z4'],1.45),('TC3',['TC-Z3'],1.3),('OP3',['OP-Z3'],1.2),
           ('TC456',[f'TC-Z{i}' for i in [4,5,6]],1.3),('OP456',[f'OP-Z{i}' for i in [4,5,6]],1.3)]
    head_h,tick_h,gap,top_h=.88,.43,.16,1.60
    height=top_h+sum(head_h+h+tick_h+gap for _,_,h in specs)+.15
    fig=plt.figure(figsize=(width,height),dpi=150,facecolor='white')
    fig.suptitle(heading,x=.07,y=1-.14/height,ha='left',va='top',fontsize=19,fontweight='bold',color='#17344d')
    fig.text(.07,1-.55/height,'각 열=한 CP 사건 · 위에서 아래로 같은 시각의 CP → TC-Z3 → OP-Z3 → TC-Z4~Z6 → OP-Z4~Z6.',
             ha='left',va='top',fontsize=11,color='#17344d')
    fig.text(.07,1-.89/height,'0=선택 CP의 최저점/최고점 · 전후 15분 · 붉은 음영=해당 CP 이탈 기록 · 회색 음영=다른 CP 사건/LOT 관측 밖.',
             ha='left',va='top',fontsize=10,color='#526d80')
    fig.text(.07,1-1.18/height,'Z3는 실제 온도·출력, Z4~Z6는 CP 이탈 시작 전 5~1분 중앙값을 뺀 변화량이다. 비교 기준은 정상 한계가 아니다.',
             ha='left',va='top',fontsize=10,color='#526d80')
    cells=[];cursor=top_h;left,right,hgap=.075,.98,.075;panel_w=(right-left-(count-1)*hgap)/count
    for kind,sensors,data_h in specs:
        row=[]
        for j,ev in enumerate(events):
            tag=event_text(ev.ID,include_lot=False);x=left+j*(panel_w+hgap)
            head=fig.add_axes([x,1-(cursor+head_h)/height,panel_w,head_h/height]);head.axis('off');head._has_note_gutter=False
            titles={'CP':event_text(ev.ID)+' | CP-Z4 '+('하강' if ev.DEV<0 else '상승'),
                    'TC3':tag+' TC-Z3 | 가열 초입의 실제 온도',
                    'OP3':tag+' OP-Z3 | 같은 시각의 실제 출력',
                    'TC456':tag+' TC-Z4·TC-Z5·TC-Z6 | 기준에서 움직인 온도',
                    'OP456':tag+' OP-Z4·OP-Z5·OP-Z6 | 기준에서 움직인 출력'}
            head.text(0,.98,titles[kind],ha='left',va='top',fontsize=12,fontweight='bold',color='#17344d')
            head.text(0,.66,textwrap.fill(cp_wave_caption(ev,kind),width=100 if count==1 else 87),
                      ha='left',va='top',fontsize=9.5,color='#526d80',linespacing=1.2)
            ax=fig.add_axes([x,1-(cursor+head_h+data_h)/height,panel_w,data_h/height]);ax._note_gutter=0
            style(ax,'');ax.ticklabel_format(axis='y',style='plain',useOffset=False);ax.tick_params(labelsize=9,pad=3)
            ax.yaxis.set_major_locator(MaxNLocator(4));row.append((ax,head,sensors))
        cells.append(row);cursor+=head_h+data_h+tick_h+gap
    fig._report_cells=cells
    for i,((kind,sensors,_),row) in enumerate(zip(specs,cells)):
        pooled=[]
        for (ax,head,_),ev in zip(row,events):
            k=int(ev.LOT);t=ev.PEAK;a,b=t-pd.Timedelta(minutes=15),t+pd.Timedelta(minutes=15)
            frame=relation_raw[k];metrics=relation_wave_metrics.loc[relation_wave_metrics.ID.eq(ev.ID)].set_index('SENSOR')
            for sensor in sensors:
                y=frame[sensor].loc[a:b]
                if kind in ['TC456','OP456']:
                    y=y-metrics.loc[sensor,'BASE'] if metrics.loc[sensor,'N_BASE']>=190 else y*np.nan
                ax.plot(y.index,y,color=sensor_color(sensor),lw=.95 if sensor.startswith('OP') else 1.15,label=sensor+' · '+('변화량' if kind in ['TC456','OP456'] else '1초 원본'))
                pooled.append(y.dropna())
            # 다른 CP 사건이 같은 창에 있음을 표시한다. 서로 독립된 반응이라고 가정하지 않는다.
            for other in relation_events.loc[relation_events.LOT.eq(k)&relation_events.ID.ne(ev.ID)].itertuples():
                lo,hi=at(k,other.A).round('s'),at(k,other.B).round('s')
                if lo<=b and hi>=a:ax.axvspan(max(lo,a),min(hi,b),color='#8496a5',alpha=.10,lw=0)
            start,end=at(k,ev.A).round('s'),at(k,ev.B).round('s')
            ax.axvspan(start,end,color=EVENT_COLORS['CP'],alpha=.13,lw=0)
            ax.axvline(t,color=EVENT_COLORS['CP'],ls=':',lw=1.2)
            if kind=='CP':
                base=frame['CP-Z4'].median()
                for bound,y in enumerate([base-.05,base+.05]):ax.axhline(y,color=EVENT_COLORS['CP'],ls='--',lw=.8,label='LOT 중앙값 ±0.05 · 탐색선' if bound==0 else '_nolegend_')
                selected=frame['CP-Z4'].loc[start:end]
                ax.plot(selected.index,selected,color=EVENT_COLORS['CP'],lw=1.8,zorder=5)
            if kind in ['TC456','OP456']:ax.axhline(0,color='#8496a5',ls='--',lw=.7)
            # LOT 관측 밖으로 선을 연장하지 않고, 실제 관측 범위 밖만 옅은 회색으로 표시한다.
            lo,hi=lot_meta.loc[k,'START'],lot_meta.loc[k,'END']
            if a<lo:ax.axvspan(a,min(lo,b),color='#8496a5',alpha=.15,lw=0)
            if b>hi:ax.axvspan(max(hi,a),b,color='#8496a5',alpha=.15,lw=0)
            ticks=[t+pd.Timedelta(minutes=m) for m in [-15,-10,-5,0,5,10,15]]
            ax.set_xticks(ticks,['-15','-10','-5','0','+5','+10','+15']);ax.set_xlim(a,b)
            if i==4:ax.set_xlabel('CP '+('최저점' if ev.DEV<0 else '최고점')+f' 대비 [분] · 0 = {t:%H:%M:%S}',fontsize=10)
            else:ax.set_xlabel('CP '+('최저점' if ev.DEV<0 else '최고점')+' 대비 [분]',fontsize=9)
            ax.set_ylabel({'CP':'CP-Z4 [기록값]','TC3':'온도 [℃]','OP3':'출력 [%]',
                           'TC456':'앞 기준 대비 온도 변화 [℃]','OP456':'앞 기준 대비 출력 변화 [%p]'}[kind],fontsize=10)
            panel_legend(head,ax)
        values=pd.concat(pooled).dropna();lo,hi=values.min(),values.max();pad=max((hi-lo)*.32,.01 if kind=='CP' else .2)
        limits=(-35,115) if kind=='OP3' else (lo-pad,hi+pad)
        for (ax,head,_),ev in zip(row,events):
            ax.set_ylim(*limits)
            if kind=='OP3':ax.set_yticks([0,25,50,75,100])
            if kind in ['CP','TC3','OP3']:
                if kind=='CP':
                    point=ev.PEAK;value=relation_raw[int(ev.LOT)].loc[point,'CP-Z4'];label=('CP 최저 ' if ev.DEV<0 else 'CP 최고 ')+f'{value:.4f}'
                else:
                    sensor=sensors[0];r=relation_wave_metrics.loc[relation_wave_metrics.ID.eq(ev.ID)&relation_wave_metrics.SENSOR.eq(sensor)].iloc[0]
                    point,value=r.EXTREME_AT,r.EXTREME
                    label=('온도 최고 ' if kind=='TC3' else '출력 최저 ')+f'{value:.2f}'+('℃' if kind=='TC3' else '%')
                ax.plot(point,value,'o',ms=4,mfc='white',mec=EVENT_COLORS['CP'],zorder=6)
                if ev.DEV<0:near_note(ax,point,value,label,EVENT_COLORS['CP'])
            if kind in ['TC456','OP456']:
                for sensor in sensors:
                    v=relation_raw[int(ev.LOT)].loc[ev.PEAK,sensor]-relation_wave_metrics.loc[relation_wave_metrics.ID.eq(ev.ID)&relation_wave_metrics.SENSOR.eq(sensor),'BASE'].iloc[0]
                    ax.plot(ev.PEAK,v,'o',ms=3,color=sensor_color(sensor),zorder=6)
    for col,ev in enumerate(events):
        axes=[row[col][0] for row in cells]
        assert all(np.array_equal(ax.get_xticks(),axes[0].get_xticks()) for ax in axes)
        assert all(np.allclose(ax.get_xlim(),axes[0].get_xlim(),rtol=0,atol=1e-10) for ax in axes)
        shown={sensor for _,_,sensors in [row[col] for row in cells] for sensor in sensors}
        assert shown==set(REL_SENSORS+['CP-Z4'])
        relation_wave_checks.append(dict(ID=ev.ID,LOT=int(ev.LOT),DIRECTION='하강' if ev.DEV<0 else '상승',N_PANELS=5,SENSORS=shown))
    return fig


fig=draw_cp_zone_patterns(relation_down,'CP 하강 2사건 | CP가 낮아진 전후, 주변 존에서는 무엇이 움직였는가?')
emit(fig,'05_관계검토/02_CP하강_주변존_동시변화.png')
# 해석은 그림의 실제 시각·동일 기준 수치로 작성한다.
for ev in relation_down.itertuples():
    d=relation_wave_metrics.loc[relation_wave_metrics.ID.eq(ev.ID)].set_index('SENSOR');tc=d.loc['TC-Z3'];op=d.loc['OP-Z3']
    values=' / '.join(f'TC-Z{z} {d.loc[f"TC-Z{z}","DELTA"]:+.2f}℃, OP-Z{z} {d.loc[f"OP-Z{z}","DELTA"]:+.2f}%p' for z in [4,5,6])
    note(f'**{event_text(ev.ID)}에서 읽는 순서:** CP 최저 {ev.PEAK:%H:%M:%S}보다 TC-Z3 원본 최고점은 {abs(tc.OFFSET_MIN):.2f}분 '+('앞' if tc.OFFSET_MIN<0 else '뒤')+
         f', OP-Z3 원본 최저점은 {abs(op.OFFSET_MIN):.2f}분 '+('앞' if op.OFFSET_MIN<0 else '뒤')+
         f'이다. CP 최저 부근의 TC-Z3는 앞 기준 대비 {tc.DELTA:+.2f}℃, OP-Z3는 {op.DELTA:+.2f}%p다. 다른 존의 같은 시각 변화는 **{values}**다. 극값의 순서이며 반응 시작·원인을 뜻하지 않는다.')
show_table(event_frame(relation_wave_metrics.loc[relation_wave_metrics.DIRECTION.eq('하강'),['ID','SENSOR','BASE','AT_CP','DELTA']]).rename(columns={
    'ID':'CP 사건','SENSOR':'신호','BASE':'CP 이탈 앞 기준','AT_CP':'CP 최저 ±15초 중앙값','DELTA':'기준 대비 변화'}))

# 상승 12사건도 같은 5행·같은 30분 폭으로 전부 남긴다. 파일을 따로 늘리지 않는다.
from IPython.utils.capture import capture_output
for first in range(0,len(relation_up),2):
    group=relation_up.iloc[first:first+2]
    with capture_output() as captured:
        fig=draw_cp_zone_patterns(group,'CP 상승 사건 | 같은 CP 시각의 주변 존 원본·변화량')
        checked_emit(fig,'CP상승_사건별_파형',save=False,record=False)
    captured_details('CP 상승 사건 원본 대조: '+' / '.join(event_text(v) for v in group.ID),captured)
table(relation_wave_metrics.rename(columns={'ID':'CP 사건','SENSOR':'신호','BEFORE':'CP 전 10~5분 중앙값','AFTER':'CP 후 5~10분 중앙값',
    'EXTREME_AT':'창 안의 TC 최고/OP 최저 시각','OFFSET_MIN':'그 극값의 CP 대비 위치[분]','N_BEFORE':'앞 창 기록수','N_AFTER':'뒤 창 기록수'}))
assert len(relation_wave_checks)==14 and {r['ID'] for r in relation_wave_checks}==set(relation_events.ID)
assert len(relation_wave_metrics)==14*8
legacy=OUT_DIR/'05_관계검토/02_사건별_시차와_OP_TC.png'
if legacy.is_file():
    assert legacy.resolve().parent==(OUT_DIR/'05_관계검토').resolve()
    assert (OUT_DIR/'05_관계검토/02_CP하강_주변존_동시변화.png').is_file()
    legacy.unlink()


# 5초 유효 중앙값의 차분을 만들고 시차를 바꾸어 직접 사건별 r을 계산한다.
from scipy.stats import spearmanr
REL_LAGS=np.arange(-120,121,10)
relation_five={};relation_diff={};relation_phase={}
for k,frame in relation_raw.items():
    p=frame[REL_SENSORS+['CP-Z4','CP-Z4M']].resample('5s',origin=lot_meta.loc[k,'START']).agg(['median','count'])
    v=p.xs('median',axis=1,level=1).where(p.xs('count',axis=1,level=1).ge(4))
    minute=np.floor((v.index-lot_meta.loc[k,'START']).total_seconds()/60).astype(int)
    stable=pd.Series(comparison_states[k].reindex(minute).eq('안정 참고').to_numpy(),index=v.index)
    relation_five[k]=v;relation_diff[k]=v.diff();relation_phase[k]=stable&stable.shift(1,fill_value=False)

def observed_lag(x,y,min_pairs):
    rows=[]
    for lag in REL_LAGS:
        # 양의 lag: x(t)와 나중의 y(t+lag)를 같은 행에 맞춘다.
        q=pd.concat([x.rename('x'),y.shift(-int(lag//5)).rename('y')],axis=1).dropna()
        valid=len(q)>=min_pairs and q.x.nunique()>1 and q.y.nunique()>1
        r=float(spearmanr(q.x,q.y).statistic) if valid else np.nan
        rows.append(dict(LAG_SEC=int(lag),R=r,N_PAIRS=len(q)))
    return pd.DataFrame(rows)

relation_lag_rows=[]
for ev in relation_events.itertuples():
    k=int(ev.LOT);d=relation_diff[k].where(relation_phase[k],axis=0)
    window=d.loc[ev.PEAK-pd.Timedelta(minutes=5):ev.PEAK+pd.Timedelta(minutes=5)]
    for sensor in REL_SENSORS+['CP-Z4M']:
        curve=observed_lag(window['CP-Z4'],window[sensor],40)
        for r in curve.to_dict('records'):relation_lag_rows.append(dict(PAIR='CP',ID=ev.ID,LOT=k,SENSOR=sensor,**r))
for k,d in relation_diff.items():
    d=d.where(relation_phase[k],axis=0)
    for zone in range(3,7):
        curve=observed_lag(d[f'OP-Z{zone}'],d[f'TC-Z{zone}'],200)
        for r in curve.to_dict('records'):relation_lag_rows.append(dict(PAIR='OP_TC',ID=str(k),LOT=k,SENSOR=f'TC-Z{zone}',**r))
relation_lags=pd.DataFrame(relation_lag_rows)

def lag_ensemble(frame):
    per_lot=frame.groupby(['LOT','LAG_SEC']).R.median().unstack('LOT').reindex(REL_LAGS)
    line=per_lot.median(axis=1)
    if per_lot.shape[1]>1:
        omitted=pd.concat([per_lot.drop(columns=[k]).median(axis=1) for k in per_lot],axis=1)
        low,high=omitted.min(axis=1),omitted.max(axis=1)
    else:low=high=line*np.nan
    return line,low,high,per_lot.notna().sum(axis=1)

relation_lag_summary=[]
for (pair,sensor),g in relation_lags.groupby(['PAIR','SENSOR'],sort=False):
    med,low,high,count=lag_ensemble(g);zero=g.loc[g.LAG_SEC.eq(0)].dropna(subset=['R'])
    peak=med.abs().idxmax() if med.notna().any() else np.nan
    relation_lag_summary.append({'비교':pair,'신호':sensor,'0초 유효 사건/LOT':len(zero),'0초 r 중앙값(LOT 동일 가중)':med.loc[0],
        '0초 양의 r':int(zero.R.gt(0).sum()),'0초 음의 r':int(zero.R.lt(0).sum()),
        '중앙 곡선 최대절대 r 시차[초]':peak,'그 시차 r':med.loc[peak] if pd.notna(peak) else np.nan,
        '그 시차 유효 LOT':int(count.loc[peak]) if pd.notna(peak) else 0})
relation_lag_summary=pd.DataFrame(relation_lag_summary)


# 보조 통계: CP-Z4M 대조 절에서도 쓰는 차분·시차 계산을 보존한다.
# 아래 계산은 위 파형의 변화 순서를 확정하거나 물리적 인과로 판정하지 않는다.
with capture_output() as lag_reference:
    note('**계산 참고:** CP와 각 TC·OP의 짧은 변동을 비교한 통계다. 동일 존 OP–TC는 예를 들어 OP-Z4와 TC-Z4의 비교이며 CP 하강 사건 자체의 파형이 아니다. 본문은 위 실제 사건 파형을 먼저 읽는다.')
    shown=relation_lag_summary.copy();shown['비교']=shown['비교'].replace({'CP':'CP와 TC·OP','OP_TC':'동일 존 OP와 TC'})
    show_table(shown)
    note('실제 시각의 5초 중앙값(5개 중 4개 이상)의 차분, 보간 없음. CP 피크 전후 5분은 40쌍 이상, LOT 전환 제외 구간은 200쌍 이상일 때 Spearman r을 계산한다. -120~+120초를 10초 간격으로 검사한다. 양의 시차는 CP 또는 OP 기록이 먼저인 방향이다. 사후 선정·자기상관·시차 탐색이 있어 원인 검정으로 쓰지 않는다.')
    table(relation_lags)
captured_details('보조 통계 표·계산 정의 펼치기 — 원본 파형과 별도로 확인',lag_reference)


**이 그림으로 확인한 결과**

1. **두 CP 하강 모두 Z3의 변화가 먼저 보인다.** TC-Z3는 CP 최저점보다 약 6분 앞에서 높은 온도를 보인 뒤 내려온다. OP-Z3도 먼저 낮은 출력을 보인 뒤 CP 최저점 부근에서는 올라온다. 따라서 “CP가 떨어진 뒤 처음 Z3가 움직였다”는 순서와는 맞지 않는다. 극값 위치를 비교한 관찰이며 실제 제어 동작 시작을 추정한 것은 아니다.
2. **Z4·Z5·Z6가 같은 방향으로 한꺼번에 변하지 않는다.** 두 사건의 CP 최저점 부근에서 TC-Z4·TC-Z6는 앞 기준보다 낮고 TC-Z5는 높다. 출력은 OP-Z4·OP-Z6가 높고 OP-Z5가 낮은 방향이다. Z3와 같은 축에 묻히지 않도록 다른 존도 별도 크기로 확인했다. 정확한 크기는 각 패널 위와 표에 표시한다.
3. **반복되는 양상은 원인 후보를 좁히는 단서다.** 두 LOT의 강종과 관측 날짜가 다르고 하강 사례도 2개뿐이다. 반복 모양만으로 Z3 또는 특정 존이 CP를 떨어뜨렸다고 확정할 수 없다. CP 없는 Z3 반응 및 CP 상승 사건과도 대조해야 한다.
4. **정비 후 CP 하강과 제어 반응을 구분한다.** 240117의 CP 최저는 CBM 완료 뒤에 있지만, 이 사실만으로 정비가 CP 하강을 만들었거나 수리 실패라고 판단하지 않는다. 동일 시각의 CP 계측·SV·제어 로그, 라인 조건과 실제 작업 대상을 확인할 근거다.

CP 두 채널의 정의가 미확인이라 “왜 CP가 떨어졌는가”의 물리적 원인은 아직 정할 수 없다. 이 그림은 어떤 신호·시간대를 함께 확인할지 구체적으로 정하는 관측 근거다.


<a id="relation-tc-op"></a>
### 5.7.1 TC-OP 관계 — 같은 존의 온도와 출력은 어떻게 함께 움직였는가?

**분석 질문:** `TC-Z3 ↔ OP-Z3`, `TC-Z4 ↔ OP-Z4`, `TC-Z5 ↔ OP-Z5`, `TC-Z6 ↔ OP-Z6`를 각각 비교한다. 온도가 오를 때 출력도 오르는가, 반대로 움직이는가? CP와의 관계와 구분해 읽는다.

**본문 파형:** 5.1.2에서 확인한 **240069의 두 차례 Z3 반응(07/07 22:45~23:50)**을 같은 시간 범위로 다시 사용한다. 각 열이 한 존이고, 위는 TC(℃), 아래는 대응 OP(%)다. 다른 존도 같은 시각으로 맞춰 함께 본다. 특정 존·시차의 상관이 가장 큰 창을 새로 골라낸 것은 아니다.

회색은 1초 원본, 색 선은 **5개 중 4개 이상 실제 값이 있는 5초 중앙값**이다. 원 표시는 원본 값이다. 세로 점선 ①·②는 Z3 온도 최고점의 실제 시각이며, 다른 존에서는 같은 시각의 참고선이다. 존마다 세로축 범위가 다르므로 값·단위를 함께 읽는다.

**전체 LOT의 반복성:** 예시 파형 한 장으로 일반화하지 않는다. 아래 표는 **14 LOT 각각의 전환 제외 구간**에서 TC·OP의 5초 중앙값이 직전 5초보다 어느 방향으로 바뀌었는지 비교한 Spearman r이다. 0초는 같은 시각끼리 비교했다는 뜻이다. 온도 수준과 출력 수준의 상관계수와 구분한다. 결측은 채우지 않고 200쌍 이상 있을 때만 계산한다.


In [ ]:
# 같은 존의 TC–OP 파형을 직접 그리고, 기존 전체 LOT 차분 통계를 별도 표로 연결한다.
TCOP_LOT=240069
case=relation_raw[TCOP_LOT]
z_event=next(e for e in LOT_EVENTS[TCOP_LOT] if e['kind']=='Z' and e['sensor']=='TC-Z3')
first_end=z_event['point']
first_peak=case['TC-Z3'].loc[first_end-pd.Timedelta(minutes=1):first_end].idxmax()
second_peak=case['TC-Z3'].loc[first_end:first_end+pd.Timedelta(hours=1)].idxmax()
tcop_a=first_end.floor('min')-pd.Timedelta(minutes=14)
tcop_b=second_peak.ceil('min')+pd.Timedelta(minutes=17)
tcop_points=[first_peak,second_peak]
tcop_zero=relation_lags.loc[relation_lags.PAIR.eq('OP_TC')&relation_lags.LAG_SEC.eq(0)].copy()
tcop_zero['ZONE']=tcop_zero.SENSOR.str.extract(r'Z(\d+)',expand=False).astype(int)
tcop_summary=[]
for zone in range(3,7):
    values=tcop_zero.loc[tcop_zero.ZONE.eq(zone)].dropna(subset=['R'])
    tcop_summary.append({'같은 존의 신호':f'TC-Z{zone} / OP-Z{zone}','유효 LOT':values.LOT.nunique(),
        '동시 변화가 역방향인 LOT':int(values.R.lt(0).sum()),'0초 변화 r 중앙값':values.R.median(),
        '그 LOT들의 최소 유효 비교 수':int(values.N_PAIRS.min())})
tcop_summary=pd.DataFrame(tcop_summary)


def draw_tc_op_relations():
    rows=[dict(panels=[(f'TC-Z{z} | '+('가열 초입 온도' if z==3 else '가열·균열 온도'),[f'TC-Z{z}']) for z in range(3,7)],height=2.1),
          dict(panels=[(f'OP-Z{z} | 같은 존의 출력',[f'OP-Z{z}']) for z in range(3,7)],height=2.1)]
    fig,cells=report_page('TC–OP 관계 | 같은 존의 온도와 출력을 위아래로 읽는다',
        f'대표 파형: LOT {TCOP_LOT} · {lot_meta.loc[TCOP_LOT,"GRADE"]} · {tcop_a:%Y/%m/%d %H:%M}–{tcop_b:%H:%M}. 각 열은 하나의 존이고 모든 패널의 실제 시간축이 같다.',
        rows,width=26,info=['회색=1초 원본 · 색=5초 중앙값(4개 이상 실측) · 원 표시=1초 원본 값. 각 존의 세로축 범위·단위는 별도로 읽는다.',
        f'세로선 ① {first_peak:%H:%M:%S} / ② {second_peak:%H:%M:%S} = Z3 원본 온도 최고점. 다른 존에는 같은 시각을 대조하는 참고선이다.',
        '아래 14 LOT 표는 짧은 변화 방향의 반복성을 요약한다. 온도·출력 중 어느 것이 원인인지 정한 그림은 아니다.'])
    ticks=list(pd.date_range(tcop_a,tcop_b,freq='20min'))
    if (tcop_b-ticks[-1]).total_seconds()<10*60:ticks=ticks[:-1]
    if ticks[-1]!=tcop_b:ticks.append(tcop_b)
    tcop_axes=[]
    for row,kind in zip(cells,['TC','OP']):
        for (ax,head,_),zone in zip(row,range(3,7)):
            sensor=f'{kind}-Z{zone}'
            raw_y=case[sensor].loc[tcop_a:tcop_b]
            short=relation_five[TCOP_LOT][sensor].loc[tcop_a:tcop_b]
            ax.plot(raw_y.index,raw_y,color='#aebac3',alpha=.45,lw=.55,label='1초 원본')
            ax.plot(short.index,short,color=sensor_color(sensor),lw=1.4,label='5초 중앙값')
            for point in tcop_points:ax.axvline(point,color='#64748b',ls=':',lw=.9,alpha=.75)
            values=raw_y.dropna();lo,hi=values.min(),values.max();pad=max((hi-lo)*.35,.15 if kind=='TC' else .5)
            ax.set_ylim(lo-pad,hi+pad)
            if kind=='OP' and zone==3:ax.set_ylim(-35,115);ax.set_yticks([0,25,50,75,100])
            ax.set_xlim(tcop_a,tcop_b);ax.set_xticks(ticks);ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
            ax.set_ylabel('온도 [℃]' if kind=='TC' else '출력 [%]',fontsize=10)
            ax.set_xlabel('07/07 실제 측정 시각' if kind=='OP' else '',fontsize=10)
            panel_legend(head,ax)
            if zone==3:
                for number,point in enumerate(tcop_points,1):
                    value=float(case.loc[point,sensor]);ax.plot(point,value,'o',ms=4,mfc='white',mec=sensor_color(sensor),zorder=6)
                    near_note(ax,point,value,f'{number} · {value:.2f}'+('℃' if kind=='TC' else '%'),sensor_color(sensor))
            tcop_axes.append(ax)
    assert len(tcop_axes)==8
    assert all(np.array_equal(ax.get_xticks(),tcop_axes[0].get_xticks()) for ax in tcop_axes)
    assert all(np.allclose(ax.get_xlim(),tcop_axes[0].get_xlim(),atol=1e-10,rtol=0) for ax in tcop_axes)
    return fig


fig=draw_tc_op_relations()
emit(fig,'05_관계검토/05_TC_OP_동시파형.png')
show_table(tcop_summary)
tcop_point_results=[]
for number,point in enumerate(tcop_points,1):
    tcop_point_results.append({'확인 지점':number,'실제 시각':point,'TC-Z3 원본[℃]':case.loc[point,'TC-Z3'],'같은 시각 OP-Z3 원본[%]':case.loc[point,'OP-Z3']})
show_table(pd.DataFrame(tcop_point_results))
negative=tcop_zero.R.lt(0).sum();ranges=tcop_summary['0초 변화 r 중앙값']
note(f'**읽은 결과:** Z3 온도가 높은 두 지점의 OP-Z3 원본은 각각 {case.loc[first_peak,"OP-Z3"]:.2f}%·{case.loc[second_peak,"OP-Z3"]:.2f}%다. 파형에서는 온도 상승·하강과 출력의 반대 움직임을 같은 시각에 확인한다. 전체 데이터에서도 유효 {len(tcop_zero)}개 LOT·존 조합 중 {negative}개가 동시 5초 변화의 음의 r이며, 존별 중앙값은 {ranges.min():.3f}~{ranges.max():.3f}다. 예시 한 창과 전체 LOT의 결과를 구분해서 읽는다.')
all_lots=tcop_zero[['LOT','ZONE','R','N_PAIRS']].copy();all_lots['신호']=all_lots.ZONE.map(lambda z:f'TC-Z{z} / OP-Z{z}')
table(all_lots[['LOT','신호','R','N_PAIRS']].rename(columns={'R':'같은 시각 5초 변화 r','N_PAIRS':'유효 비교 수'}))
assert len(tcop_zero)==14*4 and set(tcop_zero.LOT)==set(raw)
assert tcop_zero.N_PAIRS.ge(200).all() and tcop_zero.R.between(-1,1).all()


**관찰 결과와 해석**

- **같은 존의 TC와 OP는 짧은 시간에서 주로 반대로 움직였다.** Z3의 두 반응에서는 온도가 올라간 동안 출력이 낮아지고, 이후 출력이 올라오는 동안 온도가 내려오는 모습이 보인다. Z4~Z6도 대응하는 TC·OP를 각각 읽으며 작은 변화는 각 세로축의 실제 수치로 확인한다.
- **14 LOT에서도 같은 방향성이 반복됐다.** 전환 제외 구간의 동시 5초 변화 r은 Z3~Z6의 56개 LOT·존 조합 모두 음수였고, 존별 중앙값은 약 −0.61~−0.76이다. 이는 온도 변화와 출력 변화가 반대로 기록되는 경향이다. 모든 시간·운전 조건에서 같은 강도로 반복된다는 뜻은 아니다.
- **선후·원인 방향은 별도 문제다.** 기존 시차 탐색에서 집계 최대 절대 r은 0초에 있었으므로, 이 해상도·시차 간격에서 OP가 먼저인지 TC가 먼저인지 분리하지 못했다. 온도에 맞춰 출력을 조절했을 수 있고, 출력과 온도가 서로 반응하거나 공통 설정을 따를 수도 있다. SV·제어 모드·로그 없이 정상 제어 또는 특정 신호의 원인으로 확정하지 않는다.
- **CP 관계와 합쳐서 읽는 방법:** TC–OP의 반대 움직임 자체는 CP 이탈이 없을 때에도 관측된다. 이 절은 온도·출력의 관계이고, [다음 절](#relation-controls)은 그 변화가 CP 이탈 때에만 더 커지는지 확인한다. TC–OP 관계가 관측됐다고 CP 원인이 입증된 것은 아니다.


<a id="tc-op-response"></a>
### 5.7.2 출력을 높인 뒤에도 온도가 내려가는 지점과 이전 수준 복귀

**질문:** OP가 높아졌는데 TC가 계속 내려가는가? 내려가는 동안 다른 존도 같은 모양인가? 온도가 이전 수준에 다시 머물기까지 얼마나 걸렸는가?

**같은 존의 TC–OP는 서로 연결된 제어 루프일 수 있다.** 가열 출력이라면 온도 오차에 따라 출력을 낮추거나 높이는 피드백 설명이 가능하지만, 이 자료만으로 CAL01의 제어 방식·작동 방향을 확인한 것은 아니다. 일반적인 제어에서도 TC·목표값(SV)·실제 구동량을 함께 보아야 한다. OP 명령이 높다는 사실과 실제 열이 투입됐다는 사실도 구분한다. [Eurotherm 제어 점검 설명](https://www.eurotherm.com/en/temperature-control-en/trouble-shooting-temperature-control-equipment/)

**계산 기준 — 고장 기준이나 목표 온도가 아니다.**

- 14 LOT의 Z3~Z6를 모두 검사한다. 기존 승온·하강 전환을 제외하고 **5초 중앙값 13개가 연속 유효**하면서, OP가 모두 90% 이상이고 TC가 60초 동안 2℃ 이상 내려간 창을 찾는다. 90%는 실제 출력 상한·포화값으로 확인된 값이 아니다. 겹치는 창의 시간 범위는 합쳐서 같은 변화의 중복 집계를 줄인다. 빈 구간을 메우지 않는다.
- 본문은 앞 절에서 이미 본 **240069 (b-1), (b-2)**를 이어서 설명한다. 각각 TC 최고점 뒤 12분 안에 처음 나타난 **30초 연속 OP-Z3 ≥90%** 구간을 고른다. t₀는 그 구간 첫 5초창의 끝 시각이다. OP가 최초로 조절된 시각이나 제어 명령 시각과 다르다.
- 온도 기준은 **각 TC 최고점 전 10~5분의 TC 중앙값**을 고정한다. 이 기준은 SV나 정상 판정값이 아니다. t₀ 뒤 10분 내 TC 최저점부터 관측 종료(t₀+15분)까지, 기준 ±1℃ 안의 **12개 연속 유효 5초창(60초)**이 처음 확인된 시각을 찾는다. 5초창은 1초 실측 4개 이상일 때만 유효하다. 도중 결측창이 있으면 60초 확인을 다시 시작한다.
- ±0.5/1/2℃로 복귀 기준을 바꾼 결과도 함께 남긴다. 기존 **시간순 |Z|≥3 탐지와 별개인 복귀 묘사**다. 이전 수준에 잠깐 닿은 것과 60초 머문 것은 구분하며, 15분 내 미확인은 고장·영구 미복귀 판정이 아니다.


In [ ]:
from IPython.utils.capture import capture_output
# 모든 LOT·존의 같은 모양을 찾고, 기존 두 반응의 고정 기준 복귀를 계산한다.
# relation_five는 결측을 채우지 않은 5초 중앙값이다. 이 절에서는 창의 끝 시각으로 표시한다.
response_five={k:v.copy() for k,v in relation_five.items()}
response_phase={k:v.copy() for k,v in relation_phase.items()}
for k in response_five:
    response_five[k].index+=pd.Timedelta(seconds=5)
    response_phase[k].index+=pd.Timedelta(seconds=5)


def output_falling_windows(output_floor=90.,fall_c=2.):
    found=[];coverage=[]
    for k,v in response_five.items():
        for zone in range(3,7):
            tc,op=v[f'TC-Z{zone}'],v[f'OP-Z{zone}']
            eligible=(tc.notna()&op.notna()&response_phase[k]).rolling(13,min_periods=13).sum().eq(13)
            hit=eligible&op.ge(output_floor).rolling(13,min_periods=13).sum().eq(13)&tc.diff(12).le(-fall_c)
            coverage.append(dict(LOT=k,ZONE=zone,VALID_END_POINTS=int(eligible.sum())))
            blocks=[]
            for end in hit.index[hit]:
                start=end-pd.Timedelta(seconds=60)
                if blocks and start<=blocks[-1][1]:blocks[-1][1]=end
                else:blocks.append([start,end])
            for start,end in blocks:
                q=v.loc[start:end];ends=hit.loc[start:end];ends=ends.index[ends]
                found.append(dict(LOT=k,ZONE=zone,START=start,END=end,
                    MIN_OP=float(q[f'OP-Z{zone}'].min()),
                    WORST_60S_C=float(tc.diff(12).loc[ends].min()),
                    HIT_WINDOWS=len(ends),OUTPUT_FLOOR=output_floor))
    return pd.DataFrame(found),pd.DataFrame(coverage)


output_fall_intervals,output_fall_coverage=output_falling_windows()
output_fall_summary=[]
for zone in range(3,7):
    subset=output_fall_intervals.loc[output_fall_intervals.ZONE.eq(zone)]
    output_fall_summary.append({'존':f'TC-Z{zone} / OP-Z{zone}',
        '검사 LOT':output_fall_coverage.loc[output_fall_coverage.ZONE.eq(zone),'LOT'].nunique(),
        '이 모양이 있는 LOT':subset.LOT.nunique(),'겹치는 창을 합친 구간 수':len(subset)})
output_fall_summary=pd.DataFrame(output_fall_summary)

response_returns=[];response_cases=[]
for j,peak in enumerate(tcop_points,1):
    k=TCOP_LOT;v=response_five[k];tc,op=v['TC-Z3'],v['OP-Z3']
    pre=(v.index>peak-pd.Timedelta(minutes=10))&(v.index<=peak-pd.Timedelta(minutes=5))
    assert tc.loc[pre].count()>=48  # 5분 기준 창 60개 중 48개 이상 실측 요약
    base=float(tc.loc[pre].median())
    high=op.loc[peak:peak+pd.Timedelta(minutes=12)].ge(90)
    confirms=high.rolling(6,min_periods=6).sum().eq(6)
    if not confirms.any():raise AssertionError('이 확인 구간에는 30초 연속 높은 출력이 없음')
    high_confirm=confirms.index[confirms][0]
    t0=high_confirm-pd.Timedelta(seconds=25)
    end=t0+pd.Timedelta(minutes=15)
    search=tc.loc[t0:t0+pd.Timedelta(minutes=10)]
    assert search.count()>=108 and tc.loc[t0:end].count()>=162
    assert response_phase[k].loc[peak-pd.Timedelta(minutes=10):end].all()
    trough=search.idxmin()
    rec=dict(TAG=f'(b-{j})',LOT=k,PEAK=peak,BASE=base,T0=t0,END=end,
        HIGH_CONFIRMED=high_confirm,TROUGH=trough,LOW=float(tc.loc[trough]),
        FALL_C=float(tc.loc[trough]-tc.loc[t0]),
        FALL_MIN=(trough-t0).total_seconds()/60,
        OTHER_BASE={s:float(v.loc[pre,s].median()) for s in REL_SENSORS})
    for band in [.5,1.,2.]:
        after=tc.loc[trough:end];inside=after.sub(base).abs().le(band)
        ok=inside.rolling(12,min_periods=12).sum().eq(12)
        returned=ok.index[ok][0] if ok.any() else pd.NaT
        entry=inside.index[inside][0] if inside.any() else pd.NaT
        response_returns.append(dict(TAG=rec['TAG'],LOT=k,BASE=base,BAND_C=band,T0=t0,
            TROUGH=trough,FIRST_ENTRY=entry,CONFIRMED=returned,
            DELAY_MIN=(returned-t0).total_seconds()/60 if pd.notna(returned) else np.nan,
            OBSERVATION_END=end))
        if band==1.:
            rec['CONFIRMED']=returned
            rec['RETURN_START']=returned-pd.Timedelta(seconds=60) if pd.notna(returned) else pd.NaT
    response_cases.append(rec)
response_returns=pd.DataFrame(response_returns)

with capture_output() as response_tables:
    show_table(output_fall_summary)
    note('90% 이상·60초 하강이라는 모양의 집계다. 구간 수는 고장 수가 아니다. Z4~Z6에서 이 조건이 없다고 정상으로 판정하지 않는다. 존마다 평소 출력 수준과 한계가 다르다.')
    table(output_fall_intervals.rename(columns={'LOT':'LOT','ZONE':'존','START':'관측 창 시작','END':'관측 창 끝','MIN_OP':'최저 OP [%]','WORST_60S_C':'최대 60초 하강 [℃]'}))
    show_table(response_returns.rename(columns={'TAG':'구간','BASE':'이전 TC 기준 [℃]','BAND_C':'복귀 폭 ±℃','T0':'고출력 기준 시각','TROUGH':'TC 최저 창 끝','FIRST_ENTRY':'최저 뒤 첫 진입','CONFIRMED':'60초 복귀 확인','DELAY_MIN':'고출력 기준부터 확인까지 [분]','OBSERVATION_END':'관측 종료'}))
captured_details('14 LOT·모든 존 검사와 복귀 기준 민감도 표',response_tables)
assert len(response_returns)==6 and len(output_fall_coverage)==14*4
for r in response_cases:
    assert r['T0']<r['TROUGH']<r['END']
    v=response_five[r['LOT']]
    assert v.loc[r['T0']:r['HIGH_CONFIRMED'],'OP-Z3'].ge(90).all()
    if pd.notna(r['CONFIRMED']):
        inside=v['TC-Z3'].loc[(v.index>r['RETURN_START'])&(v.index<=r['CONFIRMED'])]
        assert len(inside)==12 and inside.sub(r['BASE']).abs().le(1.).all()


**아래 그림에서 바로 볼 점:** 두 열은 서로 다른 반응이다. 위 두 행에서 **고출력 시작 → TC 추가 하강 → 이전 수준 복귀 여부**를 읽고, 아래 두 행에서 같은 실제 시각의 Z4~Z6를 대조한다. 한 열의 네 패널은 시간 범위·눈금이 같다. 주황 음영은 t₀부터 TC 최저점까지, 회색 띠는 TC의 이전 기준 ±1℃다. 파란 세로선은 60초 복귀 확인이며, 목표 온도 도달을 뜻하지 않는다.


In [ ]:
# 실제 시각과 같은 존의 TC·OP를 맞추고, 다른 존의 반응을 아래에서 대조한다.
def draw_tc_op_return():
    rows=[dict(panels=[(r['TAG']+' TC-Z3 | 고출력 이후 추가 하강과 복귀',['TC-Z3']) for r in response_cases],height=2.65),
          dict(panels=[(r['TAG']+' OP-Z3 | 같은 존의 실제 출력',['OP-Z3']) for r in response_cases],height=1.6),
          dict(panels=[(r['TAG']+' TC-Z4·Z5·Z6 | 이전 기준 대비 변화',[f'TC-Z{z}' for z in range(4,7)]) for r in response_cases],height=1.35),
          dict(panels=[(r['TAG']+' OP-Z4·Z5·Z6 | 이전 기준 대비 변화',[f'OP-Z{z}' for z in range(4,7)]) for r in response_cases],height=1.35)]
    first,second=response_cases
    one=response_returns.loc[response_returns.TAG.eq(first['TAG'])&response_returns.BAND_C.eq(1.)].iloc[0]
    fig,cells=report_page('240069 | OP를 높인 뒤 TC는 언제 이전 수준으로 돌아왔는가?',
        f'07/07 · SPHC · 왼쪽 {first["TAG"]} 고출력 기준 {first["T0"]:%H:%M:%S} / 오른쪽 {second["TAG"]} {second["T0"]:%H:%M:%S}. 각 열의 모든 행은 같은 실제 시각이다.',
        rows,width=24,info=[
        f'{first["TAG"]} 추가 하강 {first["FALL_MIN"]:.1f}분, ±1℃ 60초 복귀 확인까지 {one.DELAY_MIN:.1f}분 / {second["TAG"]} 추가 하강 {second["FALL_MIN"]:.1f}분, 15분 내 같은 복귀 미확인.',
        '회색 띠=TC 최고점 전 10~5분 중앙값 ±1℃ · 주황 음영=고출력 시작~TC 최저 · 파란 선=60초 복귀 확인. SV·고장 한계가 아니다.',
        '색 선=유효 5초 중앙값(창 끝 시각) · 회색 가는 선=1초 원본. OP≥90%는 비교용 기준이며 실제 포화·전력 투입은 미확인.'])
    response_axis_checks=[]
    context_limits={}
    for kind in ['TC','OP']:
        peak=0.
        for r in response_cases:
            v=response_five[r['LOT']].loc[r['T0']-pd.Timedelta(minutes=12):r['END']]
            for z in range(4,7):
                sensor=f'{kind}-Z{z}'
                peak=max(peak,float((v[sensor]-r['OTHER_BASE'][sensor]).abs().max()))
        context_limits[kind]=max(2. if kind=='TC' else 8.,peak*1.25)
    for col,rec in enumerate(response_cases):
        v=response_five[rec['LOT']];a=rec['T0']-pd.Timedelta(minutes=12);b=rec['END']
        ticks=[rec['T0']+pd.Timedelta(minutes=m) for m in [-10,0,5,10,15]]
        for row,(ax,head,sensors) in enumerate([cells[q][col] for q in range(4)]):
            for sensor in sensors:
                y=v[sensor].loc[a:b]
                offset=rec['OTHER_BASE'][sensor] if row>=2 else 0
                if row<2:
                    raw_y=relation_raw[rec['LOT']][sensor].loc[a:b]
                    ax.plot(raw_y.index,raw_y,color='#acb8c2',alpha=.4,lw=.55)
                ax.plot(y.index,y-offset,lw=1.35,color=sensor_color(sensor),label=sensor)
            ax.axvspan(rec['T0'],rec['TROUGH'],color=EVENT_COLORS['Z'],alpha=.13,zorder=0)
            ax.axvline(rec['T0'],color=EVENT_COLORS['Z'],ls='--',lw=1.)
            if pd.notna(rec['CONFIRMED']):ax.axvline(rec['CONFIRMED'],color='#166b8a',ls='-.',lw=1.)
            ax.set_xlim(a,b);ax.set_xticks(ticks);ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
            ax.set_xlabel('07/07 실제 측정 시각' if row==3 else '')
            ax.set_ylabel(['TC-Z3 [℃]','OP-Z3 [%]','TC 변화 [℃]','OP 변화 [%p]'][row])
            if row==0:
                ax.axhspan(rec['BASE']-1,rec['BASE']+1,color='#8ca2b2',alpha=.15,zorder=0)
                ax.axhline(rec['BASE'],color='#64748b',ls=':',lw=.8)
                ax.set_ylim(846,877)
                ax.plot(rec['TROUGH'],rec['LOW'],'o',mfc='white',mec=sensor_color('TC-Z3'),ms=4,zorder=6)
                near_note(ax,rec['TROUGH'],rec['LOW'],f'{rec["LOW"]:.2f}℃\n추가 하강 {rec["FALL_MIN"]:.1f}분',sensor_color('TC-Z3'))
                if pd.notna(rec['CONFIRMED']):
                    val=float(v.loc[rec['CONFIRMED'],'TC-Z3']);ax.plot(rec['CONFIRMED'],val,'o',mfc='white',mec='#166b8a',ms=4,zorder=6)
                    near_note(ax,rec['CONFIRMED'],val,'60초 복귀 확인','#166b8a')
                else:
                    val=float(v.loc[b,'TC-Z3']);ax.plot(b,val,'s',mfc='white',mec='#166b8a',ms=4,zorder=6)
                    near_note(ax,b,val,'15분 내 미확인','#166b8a')
                extra=[Patch(facecolor='#8ca2b2',alpha=.3,label=f'이전 {rec["BASE"]:.2f}℃ ±1℃')]
            elif row==1:
                ax.axhline(90,color='#526d80',ls=':',lw=.9,label='비교 기준 90%');ax.set_ylim(-8,112)
                ax.set_yticks([0,50,90]);extra=[]
            else:
                ax.axhline(0,color='#64748b',ls=':',lw=.8);limit=context_limits['TC' if row==2 else 'OP'];ax.set_ylim(-limit,limit);extra=[]
            panel_legend(head,ax,extra=extra)
            for line in ax.lines:
                yy=np.asarray(line.get_ydata(),dtype=float)
                if len(yy)>2 and np.isfinite(yy).any():
                    lo,hi=ax.get_ylim();assert np.nanmin(yy)>=lo and np.nanmax(yy)<=hi,'파형이 세로축 밖으로 잘림'
            response_axis_checks.append((col,ax.get_xlim(),ax.get_xticks().copy()))
    for col in range(2):
        axes=[x for x in response_axis_checks if x[0]==col]
        assert len(axes)==4 and all(np.allclose(v[1],axes[0][1],atol=1e-10,rtol=0) and np.array_equal(v[2],axes[0][2]) for v in axes)
    return fig


fig=draw_tc_op_return()
emit(fig,'05_관계검토/06_TC_OP_고출력후_하강과복귀.png')
return_summary=[]
for rec in response_cases:
    rr=response_returns.loc[response_returns.TAG.eq(rec['TAG'])&response_returns.BAND_C.eq(1.)].iloc[0]
    return_summary.append({'확인 구간':'240069 '+rec['TAG'],'고출력 기준 시각':rec['T0'],
        '고출력 뒤 추가 하강 [℃]':rec['FALL_C'],'TC 최저 창 끝':rec['TROUGH'],
        '추가 하강 [분]':rec['FALL_MIN'],'이전 수준 ±1℃ 60초 확인':rr.CONFIRMED if pd.notna(rr.CONFIRMED) else '15분 내 미확인',
        '고출력부터 확인까지 [분]':round(rr.DELAY_MIN,2) if pd.notna(rr.DELAY_MIN) else '미확인'})
show_table(pd.DataFrame(return_summary))
pattern_lots=output_fall_intervals.LOT.nunique()
note(f'**이 모양만으로 이상 판정할 수 없는 이유:** 높은 출력과 온도 하강의 동시 기록은 14 LOT 중 {pattern_lots}개에서 관측됐다. 두 반응 모두 TC-Z3의 추가 하강과 OP-Z3의 높은 값이 함께 남지만, 복귀 양상은 다르다. 다른 존의 온도·출력은 아래 같은 시간축에서 개별 크기를 비교한다. 고출력 이전의 출력 감소, 열적 지연, 제어 설정·부하 변화 또는 구동·센서 문제는 확인할 가설이며 현재 자료로 구분하지 못한다.')


**관찰 결과 — 무엇을 확인했고 어디를 다시 볼 것인가?**

1. **TC에 따라 OP가 움직였는가?** 동시 변화는 주로 역방향이며, 일반적인 피드백 조절과 양립한다. 그러나 SV·자동/수동 모드가 없으므로 TC가 OP 변화를 유발했다고 확정하지 않는다. 관측된 0초 차분 상관을 ‘OP의 효과가 즉시 나타났다’고 해석하지 않는다.
2. **OP 뒤 몇 분 만에 돌아왔는가?** 이 절의 시간은 고출력 구간 기준 시각부터 **과거 관측 수준에 60초 머문 확인 시각**까지다. 목표 온도 복귀 시간·설비의 고유 지연이 아니다. 첫 반응은 복귀가 관측됐고 두 번째는 15분 창에서 확인되지 않았다. ±0.5/1/2℃의 표를 함께 읽어 임의의 복귀 폭에 결과가 얼마나 달라지는지 확인한다.
3. **우선 확인할 지점:** 240069 **07/07 23:36경~23:51경의 TC-Z3 / OP-Z3**, 특히 OP가 높아진 뒤에도 TC가 추가 하강하는 23:36경~23:40경이다. 같은 시각의 TC-Z4~Z6·OP-Z4~Z6를 함께 보아 Z3의 큰 반응이 공정 전체의 동일한 변화인지 구분한다. 앞선 반응과의 복귀 차이는 점검 질문을 구체화하지만 정상/고장의 라벨은 주지 않는다.
4. **‘출력을 줬는데 제어가 안 됐다’는 판단에 필요한 조건:** SV가 일정하고, OP가 무엇을 구동하며 어느 방향으로 작동하는지 확인되고, 수동·인터록·정지·부하 전환이 없어야 한다. 그 조건에서 허용 오차 밖 TC가 평소 응답시간보다 오래 지속되는지, 실제 전류·전력·유량·밸브 피드백도 명령을 따랐는지를 확인한다. OP 명령만 크면 실제 열 전달 불량·명령 미실행·센서 문제를 구분할 수 없다. 이때에야 제어 성능 저하 후보를 정의할 수 있다. [Eurotherm 점검 기준](https://www.eurotherm.com/en/temperature-control-en/trouble-shooting-temperature-control-equipment/)

**정비와 연결할 때:** 이 23시대 반응은 CP의 큰 피크(07/08 02:31)보다 앞서며, BM 01094의 감지(07/08 08:49)와도 떨어져 있다. 시간상 앞선다는 사실은 고장 전조의 증거가 아니다. BM의 ‘1호기 재발, 절연 저하’가 Z3 구동 계통인지 확인되지 않아 원인으로 연결하지 않는다. 관련 설비 태그·제어 로그를 확보한 뒤 이 시각을 대조해야 한다.


<a id="tc-quality-evidence"></a>
### 5.7.3 존 온도에서 제품 품질의 근거로 연결하려면

**현재는 존 온도만으로 제품 품질을 결정할 수 없다.** 연속 소둔의 열 이력과 재질 사이에는 근거가 있지만, 이 자료에서 그 관계를 검증한 것은 아니다. JFE의 기술 보고는 강종·성분에 따른 열처리 경로와 냉각·과시효 조건이 기계적 성질에 영향을 주는 과정을 설명한다. 논문의 조건을 CAL01의 정상 온도나 불량 임계값으로 옮겨 쓰지 않는다. [연속 소둔 기술과 제품 개발](https://www.jfe-steel.co.jp/archives/en/ksc_giho/no.22/tobira048.html), [열 사이클·탄소량과 기계적 성질](https://www.jfe-steel.co.jp/archives/ksc_giho/16-4/tobira264.htm)

| 확인할 품질 가설 | 연결할 공정 정보 | 필요한 검증 근거 |
| --- | --- | --- |
| 가열·균열 조건이 강도·연신율 차이와 연결되는가? | 소재의 가열 최고 온도·승온 속도·유지시간, 강종·성분·두께 | 소재 온도 측정 위치·정의, 해당 소재의 인장강도·항복강도·연신율 및 판정 규격 |
| 냉각 조건이 재질 차이와 연결되는가? | 소재의 실제 냉각 속도·경로·도달 온도 | 소재 이동 경로·라인 속도, 냉각 설비 상태, 같은 위치의 재질 시험 결과 |
| 과시효 조건이 품질 차이와 연결되는가? | 강종별 실제 유지 온도·시간 | 강종별 승인 레시피, 관련 시험 항목·결과와 소재 위치 연결 |
| Z3의 큰 변동이 제품까지 전달됐는가? | 같은 소재가 Z3에서 이후 존을 지나간 열 이력 | 존 간 거리·체류시간·속도 변화와 코일/소재 위치. 같은 시계 시각의 존끼리는 같은 소재를 뜻하지 않음 |

**지금 만들 수 있는 것은 검증용 요약이다.** 기존 LOT·존별 중앙값·평균·IQR·변화 크기·지속시간을 보존하고, 소재가 추적되면 강종별 허용 범위 밖 유지시간과 냉각 속도를 추가한다. 이후 동일 소재 위치의 품질 결과와 연결해 조건별 차이 및 반례를 확인한다. 현재 LOT 기준 평균만으로 짧은 구간의 품질 영향을 평가하지 않는다.

현재 CAL01 CSV에는 TC·OP·CP와 LOT·시각이 있으며 **SV, CAL01 라인 속도, 소재 온도 정의, 두께·성분, 품질 시험값·규격이 없다.** 다른 `WLD01_용접.csv`의 `SPD`는 WLD01 신호로, CAL01 속도로 사용할 연결 근거가 없다. 따라서 이번 결과는 **TC–OP 관계 및 점검할 구간을 제시한 탐색 분석**이며, 제어 불능이나 품질 불량을 확정한 결과가 아니다.


<a id="relation-controls"></a>
### 5.8 Q3. CP가 튄 때에만 주변 존도 더 크게 움직였는가?

**질문:** CP 이탈이 있던 때의 TC·OP 변화가, 같은 LOT에서 CP 이탈이 없던 다른 때보다 큰가? 다른 때에도 비슷하거나 더 큰 움직임이 있는가?

**먼저 구분할 것:** 아래의 차이는 **두 시간대에서 관측한 변화폭의 차이**다. CP가 주변 존에 준 영향의 크기가 아니다. CP가 존을 바꾸었는지, 존의 변화가 CP와 연결되는지, 공통 운전 조건이 둘을 바꾸었는지는 이 비교만으로 결정할 수 없다.

**이전 절과의 연결:** 5.7은 한 CP 구간 안에서 신호가 어떤 순서로 움직였는지 본다. 여기서는 비슷한 움직임이 CP 이탈이 없는 때에도 나오는지 확인한다.

**바꾼 읽기 방식:** 연결점 대신 두 시간대의 **CP 원본 → TC-Z3/OP-Z3 → TC-Z4/OP-Z4 → TC-Z5/OP-Z5 → TC-Z6/OP-Z6** 파형을 겹쳐 본다. 빨강은 CP 이탈이 있던 때, 회색 점선은 이탈이 없던 다른 때다. **두 선은 서로 다른 실제 시각**이며 각 시작을 0분으로 정렬했다. 원래 날짜·시각은 맨 위에 표시한다. 회색도 CP가 실제 측정된 구간이다.

**변화폭의 뜻:** 각 시간대의 시작 전 5~1분 중앙값을 기준으로, 시작 뒤 0~4분 중 **위아래로 가장 멀어진 값의 절댓값**이다. 온도는 ℃, 출력은 %p로 따로 읽는다. 예를 들어 앞 기준 860℃에서 뒤 4분에 858℃까지 내려가면 최대폭은 2℃다. 선을 보면 방향도 알 수 있고, 점은 그 최대폭이 기록된 위치를 뜻한다. 존마다 세로축 범위가 다르므로 최대폭 숫자를 함께 읽는다. 정상·고장 임계값이나 지속시간은 아니다.

**시간대 선정:** 같은 LOT에서 CP가 LOT 중앙값 ±0.02 안에 기록된 다른 시간대를 찾는다. 5분 간격 후보 중 CP 이탈 시작~끝 ±10분과 겹치는 창을 제외하고, 전환 참고 구간도 제외했다. CP 실측 90% 이상, TC·OP의 앞·뒤 기록 각각 216/240초 이상을 요구한다. 앞 기준 TC-Z4~Z6가 각각 2.5℃ 이내인 후보 중 온도 차이가 작은 시간을 선택하고 시간 거리도 약하게 반영한다. 다른 시간대는 겹치거나 중복되지 않는다. 세부 선정 수치는 아래 접힌 표에 둔다.

**대표 선정·누락 방지:** 14 CP 구간 중 이런 비교가 가능한 것은 12개·5 LOT이다. 가장 큰 CP 피크인 **240069 (c-1)**을 본문에서 먼저 보이고, 나머지 11개도 모두 같은 파형 형식으로 펼쳐볼 수 있다. 같은 강종·LOT이지만 SV·속도·두께·부하까지 같은 운전 조건은 아니다. 12개 중 7개가 240069에 속하므로 독립된 실험 12회로 일반화하지 않는다.


In [ ]:
# CP 사건과 같은 LOT의 CP 없는 대조 창을 짝짓고 8개 신호를 같은 정의로 계산한다.
def relation_window(k,t,need_quiet=False):
    frame=relation_raw[k];a=t-pd.Timedelta(minutes=5);b=t+pd.Timedelta(minutes=4)
    if a<frame.index.min() or b>frame.index.max()+pd.Timedelta(seconds=1):return None,'관측 범위 부족'
    mins=np.arange(int((a-lot_meta.loc[k,'START']).total_seconds()//60),int(np.ceil((b-lot_meta.loc[k,'START']).total_seconds()/60)))
    if not comparison_states[k].reindex(mins).eq('안정 참고').all():return None,'전환 구간 포함'
    pre=frame.loc[(frame.index>=a)&(frame.index<t-pd.Timedelta(minutes=1)),REL_SENSORS]
    post=frame.loc[(frame.index>=t)&(frame.index<b),REL_SENSORS]
    if pre.count().min()<216 or post.count().min()<216:return None,'TC·OP 기록 부족'
    cp=frame['CP-Z4'].loc[(frame.index>=a)&(frame.index<b)]
    if cp.count()<.9*len(cp):return None,'CP 기록 부족'
    cp_dev=(cp-frame['CP-Z4'].median()).abs().max()
    if need_quiet and cp_dev>CP_GROUP:return None,'CP 0.02 범위 초과'
    base=pre.median();amp=(post-base).abs().max()
    return dict(BASE=base,AMP=amp,CP_MAX=float(cp_dev),TIME=t),''

relation_control_pool={}
for k in relation_events.LOT.astype(int).unique():
    frame=relation_raw[k];pool=[]
    first=frame.index.min()+pd.Timedelta(minutes=5);last=frame.index.max()-pd.Timedelta(minutes=4)
    for t in pd.date_range(first.ceil('5min'),last,freq='5min'):
        occupied=False
        for ev in relation_events.loc[relation_events.LOT.eq(k)].itertuples():
            end=at(k,ev.B).round('s')
            if t+pd.Timedelta(minutes=4)>=ev.ONSET-pd.Timedelta(minutes=10) and t-pd.Timedelta(minutes=5)<=end+pd.Timedelta(minutes=10):occupied=True;break
        if occupied:continue
        features,reason=relation_window(k,t,need_quiet=True)
        if features is not None:pool.append(features)
    relation_control_pool[k]=pool
relation_matches=[];relation_match_coverage=[];used={k:[] for k in relation_control_pool}
for ev in relation_events.itertuples():
    k=int(ev.LOT);features,reason=relation_window(k,ev.ONSET)
    if features is None:
        relation_match_coverage.append(dict(ID=ev.ID,LOT=k,STATUS=reason));continue
    candidates=[]
    for control in relation_control_pool[k]:
        if any(abs((control['TIME']-t).total_seconds())<9*60 for t in used[k]):continue
        diff=(control['BASE'][HEAT[1:]]-features['BASE'][HEAT[1:]]).abs()
        if diff.max()>2.5:continue
        cost=float(diff.sum()/.5)+.001*abs((control['TIME']-ev.ONSET).total_seconds()/60)
        candidates.append((cost,control,float(diff.max())))
    if not candidates:
        relation_match_coverage.append(dict(ID=ev.ID,LOT=k,STATUS='조건을 만족하는 별도 대조 창 없음'));continue
    _,control,gap=min(candidates,key=lambda x:x[0]);used[k].append(control['TIME'])
    relation_match_coverage.append(dict(ID=ev.ID,LOT=k,STATUS='대조 가능'))
    for sensor in REL_SENSORS:
        relation_matches.append(dict(ID=ev.ID,LOT=k,SENSOR=sensor,EVENT_START=ev.ONSET,CONTROL_START=control['TIME'],
            EVENT_AMP=features['AMP'][sensor],CONTROL_AMP=control['AMP'][sensor],
            BASE_TC456_MAX_GAP=gap,CONTROL_CP_MAX=control['CP_MAX']))
relation_matches=pd.DataFrame(relation_matches);relation_match_coverage=pd.DataFrame(relation_match_coverage)
relation_match_summary=[]
for sensor in REL_SENSORS:
    g=relation_matches.loc[relation_matches.SENSOR.eq(sensor)]
    difference=g.EVENT_AMP-g.CONTROL_AMP
    relation_match_summary.append({'신호':sensor,'비교 쌍':len(g),'사건>대조':int(difference.gt(0).sum()),
        '사건 최대변화 중앙값':g.EVENT_AMP.median(),'대조 최대변화 중앙값':g.CONTROL_AMP.median(),
        '짝 차이 중앙값':difference.median(),'해당 LOT 수':g.LOT.nunique()})
relation_match_summary=pd.DataFrame(relation_match_summary)
matched=relation_match_coverage.STATUS.eq('대조 가능')
# 점 두 개 대신 서로 다른 두 시간대의 실제 파형을 같은 경과시간에 겹친다.
# 선색은 신호 종류가 아니라 두 시간대를 뜻하며, 원본 값·기준·최대폭을 확인한다.
PAIR_RED='#B11226';PAIR_GREY='#688092'
relation_pair_wave_checks=[];relation_pair_amplitude_checks=[]
matched_events=relation_events.loc[relation_events.ID.isin(relation_matches.ID.unique())].copy()
representative_id=matched_events.set_index('ID').DEV.abs().idxmax()  # 비교 가능한 CP 중 절대 피크 최대


def draw_cp_present_absent(ident):
    from matplotlib.patches import Patch
    from matplotlib.ticker import MaxNLocator
    ev=relation_events.loc[relation_events.ID.eq(ident)].iloc[0];k=int(ev.LOT)
    records=relation_matches.loc[relation_matches.ID.eq(ident)].set_index('SENSOR')
    quiet=records.CONTROL_START.iloc[0];frame=relation_raw[k]
    first,_=relation_window(k,ev.ONSET);second,_=relation_window(k,quiet,need_quiet=True)
    assert first is not None and second is not None
    times=[ev.ONSET,quiet];colors=[PAIR_RED,PAIR_GREY]
    periods=['CP 이탈이 있던 때','CP 이탈이 없던 다른 때']
    tag=event_text(ident);width=20;head_h,tick_h,gap,top=.65,.40,.14,2.1
    layout=[(['CP-Z4'],None,1.45)]+[([f'TC-Z{z}'],[f'OP-Z{z}'],1.25) for z in range(3,7)]
    height=top+sum(head_h+h+tick_h+gap for _,_,h in layout)+.15
    fig=plt.figure(figsize=(width,height),dpi=150,facecolor='white')
    fig.suptitle(tag+' | CP가 튄 때, 주변 존도 항상 더 크게 움직였는가?',x=.07,y=1-.14/height,
                 ha='left',va='top',fontsize=18,fontweight='bold',color='#17344d')
    fig.text(.07,1-.54/height,f'빨강: {ev.ONSET:%Y/%m/%d %H:%M:%S} 시작 · 회색: {quiet:%Y/%m/%d %H:%M:%S} 시작 · 같은 LOT {k} / {lot_meta.loc[k,"GRADE"]}',
             ha='left',va='top',fontsize=10.5,color='#17344d')
    fig.text(.07,1-.86/height,'서로 다른 실제 시각의 두 파형이다. 가로축은 각 시작에서의 경과 분이며, 같은 가로 위치가 실제 동시 관측이라는 뜻은 아니다.',
             ha='left',va='top',fontsize=10,color='#526d80')
    fig.text(.07,1-1.16/height,'TC·OP는 각 시간대의 앞 5~1분 중앙값을 뺀 값 · 점=뒤 4분 중 기준에서 가장 멀어진 위치 · 수치=그 폭의 절댓값.',
             ha='left',va='top',fontsize=10,color='#526d80')
    fig.text(.07,1-1.46/height,'크기뿐 아니라 선의 위아래 방향·변화 시점도 읽는다. 같은 LOT·비슷한 Z4~Z6 시작 온도이며, 설정·속도·부하까지 같다는 근거는 없다.',
             ha='left',va='top',fontsize=9.5,color='#526d80')
    handles=[Line2D([0],[0],color=PAIR_RED,lw=1.8,label=periods[0]),Line2D([0],[0],color=PAIR_GREY,lw=1.6,ls='--',label=periods[1]),
             Line2D([0],[0],marker='o',color='#475569',ls='',label='비교한 4분의 최대폭 위치'),Patch(facecolor='#e9f1f5',label='시작 뒤 4분')]
    fig.legend(handles=handles,loc='upper left',bbox_to_anchor=(.075,1-1.74/height),ncol=4,fontsize=9.5,frameon=False,borderaxespad=0)
    cursor=top;cells=[]
    for left,right,data_h in layout:
        row=[];signals=[left] if right is None else [left,right]
        ncols=len(signals);hgap=.075;w=(.98-.075-(ncols-1)*hgap)/ncols
        for col,sensors in enumerate(signals):
            sensor=sensors[0];x=.075+col*(w+hgap)
            head=fig.add_axes([x,1-(cursor+head_h)/height,w,head_h/height]);head.axis('off')
            ax=fig.add_axes([x,1-(cursor+head_h+data_h)/height,w,data_h/height]);style(ax,'')
            ax.yaxis.set_major_locator(MaxNLocator(4));ax.tick_params(labelsize=9,pad=3)
            title='CP-Z4 | 실제 원본값' if sensor=='CP-Z4' else sensor+' | '+('온도 변화' if sensor.startswith('TC') else '출력 변화')
            head.text(0,.98,title,ha='left',va='top',fontsize=12,fontweight='bold',color='#17344d')
            if sensor=='CP-Z4':
                caption=f'선택 CP 피크의 LOT 중앙값 편차 {ev.DEV:+.4f} · 다른 때 9분 전체의 최대 절대편차 {second["CP_MAX"]:.4f} · 가로 점선=중앙값 ±0.05 탐색선'
            else:
                r=records.loc[sensor];unit='℃' if sensor.startswith('TC') else '%p'
                caption=f'기준에서 가장 멀어진 폭: CP 이탈 때 {r.EVENT_AMP:.2f}{unit} / 이탈 없던 다른 때 {r.CONTROL_AMP:.2f}{unit}'
            head.text(0,.49,caption,ha='left',va='top',fontsize=9.5,color='#526d80')
            values=[]
            for number,(start,color,label,features) in enumerate(zip(times,colors,periods,[first,second])):
                a,b=start-pd.Timedelta(minutes=5),start+pd.Timedelta(minutes=4)
                y=frame[sensor].loc[(frame.index>=a)&(frame.index<b)]
                if sensor!='CP-Z4':y=y-features['BASE'][sensor]
                elapsed=(y.index-start).total_seconds()/60
                ax.plot(elapsed,y,color=color,lw=1.25 if number==0 else 1.15,ls='-' if number==0 else '--',label=label)
                values.append(y.dropna())
                post=y.loc[(y.index>=start)&(y.index<b)]
                if sensor=='CP-Z4':
                    if number==0:point=ev.PEAK
                    else:point=(post-frame['CP-Z4'].median()).abs().idxmax()
                else:
                    point=post.abs().idxmax();amp=float(abs(post.loc[point]))
                    expected=float(records.loc[sensor,'EVENT_AMP' if number==0 else 'CONTROL_AMP'])
                    assert np.isclose(amp,expected,rtol=0,atol=1e-9),(ident,sensor,number,amp,expected)
                    relation_pair_amplitude_checks.append((ident,sensor,number,amp))
                ax.plot((point-start).total_seconds()/60,y.loc[point],'o',ms=4,mfc='white',mec=color,mew=1.2,zorder=6)
            ax.axvspan(0,4,color='#e9f1f5',alpha=.55,lw=0,zorder=0)
            ax.axvline(0,color='#8496a5',ls=':',lw=.9)
            if sensor=='CP-Z4':
                base=frame[sensor].median()
                for bound in [base-.05,base+.05]:ax.axhline(bound,color=PAIR_RED,ls=':',lw=.8,alpha=.6)
            else:ax.axhline(0,color='#8496a5',lw=.7,ls=':')
            pooled=pd.concat(values);lo,hi=pooled.min(),pooled.max();pad=max((hi-lo)*.25,.01 if sensor=='CP-Z4' else .1)
            ax.set_ylim(lo-pad,hi+pad);ax.set_xlim(-5,4);ax.set_xticks([-5,-3,-1,0,2,4])
            ax.set_ylabel('CP 기록값' if sensor=='CP-Z4' else '앞 기준 대비 [℃]' if sensor.startswith('TC') else '앞 기준 대비 [%p]',fontsize=10)
            if cursor==top or sensor in ['TC-Z6','OP-Z6']:ax.set_xlabel('각 시간대의 시작 대비 [분] · 0은 서로 다른 실제 시각',fontsize=9.5)
            row.append((ax,head,sensors))
        cells.append(row);cursor+=head_h+data_h+tick_h+gap
    fig._report_cells=cells
    axes=[ax for row in cells for ax,_,_ in row]
    assert len(axes)==9 and all(np.array_equal(ax.get_xticks(),axes[0].get_xticks()) for ax in axes)
    assert all(ax.get_xlim()==(-5,4) for ax in axes)
    relation_pair_wave_checks.append(dict(ID=ident,LOT=k,CP_TIME=ev.ONSET,OTHER_TIME=quiet,N_PANELS=9))
    return fig


fig=draw_cp_present_absent(representative_id)
emit(fig,'05_관계검토/03_CP이탈_유무_실제파형.png')
primary=relation_matches.loc[relation_matches.ID.eq(representative_id)].set_index('SENSOR')
note(f'**이 사례에서 읽는 결과:** 비교 가능한 CP 중 절대 피크가 가장 큰 {event_text(representative_id)}를 먼저 보였다. TC-Z3의 4분 최대폭은 CP 이탈 때 **{primary.loc["TC-Z3","EVENT_AMP"]:.2f}℃**, CP 이탈이 없던 다른 때 **{primary.loc["TC-Z3","CONTROL_AMP"]:.2f}℃**다. OP-Z3도 각각 {primary.loc["OP-Z3","EVENT_AMP"]:.2f}%p·{primary.loc["OP-Z3","CONTROL_AMP"]:.2f}%p로 다른 때가 더 크다. 반면 Z4~Z6는 신호별 방향·크기가 다르므로 아래 각 패널을 함께 읽는다.')
summary_rows=[]
for r in relation_match_summary.to_dict('records'):
    nlarger=int(r['사건>대조']);total=int(r['비교 쌍'])
    summary_rows.append({'신호':r['신호'],'CP 이탈 때 더 크게 움직임':f'{total}번 중 {nlarger}번','다른 때가 크거나 같음':f'{total}번 중 {total-nlarger}번',
                         '두 시간대 최대폭 차이의 중앙값':r['짝 차이 중앙값'],'단위':'℃' if r['신호'].startswith('TC') else '%p'})
show_table(pd.DataFrame(summary_rows))
note('**이 표의 답:** CP 이탈이 있는 때에 더 큰 변화가 반복되는지 살펴본 결과다. 영향의 크기·고장 확률·정비 효과를 계산한 표가 아니다. 변화량이 더 큰 경우에도 CP→존, 존→CP, 공통 운전 조건의 영향을 나누어 검증해야 한다.')
# 모든 비교를 남기되 대표 파형만 본문에 둔다. 추가 PNG 파일은 만들지 않는다.
from IPython.utils.capture import capture_output
ordered=matched_events.assign(ABS=matched_events.DEV.abs()).sort_values('ABS',ascending=False)
for ident in ordered.ID:
    if ident==representative_id:continue
    with capture_output() as captured:
        fig=draw_cp_present_absent(ident)
        checked_emit(fig,'CP이탈유무_시간대별_파형',save=False,record=False)
    captured_details('다른 CP 구간도 같은 방식으로 보기: '+event_text(ident),captured)
coverage=relation_match_coverage.rename(columns={'ID':'CP 구간','LOT':'LOT','STATUS':'비교할 시간대 확보 결과'}).copy()
coverage['비교할 시간대 확보 결과']=coverage['비교할 시간대 확보 결과'].replace({'대조 가능':'서로 다른 두 시간대 비교 가능','조건을 만족하는 별도 대조 창 없음':'조건을 맞춘 다른 시간대 없음'})
table(coverage)
table(relation_matches.rename(columns={'ID':'CP 구간','LOT':'LOT','SENSOR':'신호','EVENT_START':'CP 이탈 때 시작','CONTROL_START':'이탈 없던 다른 때 시작',
    'EVENT_AMP':'CP 이탈 때 최대폭','CONTROL_AMP':'이탈 없던 다른 때 최대폭','BASE_TC456_MAX_GAP':'앞 기준 TC-Z4~Z6 최대 차이','CONTROL_CP_MAX':'다른 때 CP 최대 절대편차'}))
assert len(relation_pair_wave_checks)==int(matched.sum())==12
assert len(relation_pair_amplitude_checks)==12*8*2
assert {v['ID'] for v in relation_pair_wave_checks}==set(relation_matches.ID)
legacy=OUT_DIR/'05_관계검토/03_CP사건과_비사건_짝대조.png'
if legacy.is_file():
    assert legacy.resolve().parent==(OUT_DIR/'05_관계검토').resolve()
    assert (OUT_DIR/'05_관계검토/03_CP이탈_유무_실제파형.png').is_file()
    legacy.unlink()


**이 비교로 알 수 있는 것**

- **Z3의 큰 움직임은 CP 이탈 때에만 나타나지 않았다.** 가장 큰 CP 피크인 240069 (c-1)에서 TC-Z3 최대폭은 CP 이탈 때 2.70℃, 이탈이 없던 다른 때 7.19℃다. OP-Z3도 42.95%p 대 61.59%p로 다른 때가 더 크다. 따라서 “CP가 크게 튀면 Z3도 항상 더 크게 움직인다”는 설명과는 맞지 않는다.
- **Z4~Z6에는 함께 더 커지는 경향을 확인할 단서가 있다.** CP 이탈 때 온도 최대폭이 더 큰 경우는 TC-Z4 12번 중 10번, TC-Z5 9번, TC-Z6 8번이었다. 다만 두 시간대 최대폭 차이의 중앙값은 각각 약 0.200℃·0.243℃·0.200℃다. 빈도와 실제 크기를 함께 읽는다. 이를 CP의 인과적 영향량으로 부르지 않는다.
- **이 비교가 하지 못하는 판단:** CP가 원인이라는 확정, 반대로 존이 CP를 바꾸었다는 확정, 고장·정상 분류, 수리 효과 평가다. 설정·속도·부하·제어 경로가 다를 수 있고 각 최대값이 같은 순간에 나타난 것도 아니다. 파형에서 시각·방향을 확인한 뒤 공통 조건과 계측·제어 로그를 검토할 근거로 쓴다.
- **다른 시간대를 찾지 못한 CP 구간은 판단 보류다.** 기록 부족·전환 또는 범위 부족을 정상이나 관계없음으로 처리하지 않는다. 시작 후 4분 밖의 느린 반응도 이 최대폭으로 배제하지 않는다.


<a id="relation-maintenance"></a>
### 5.9 Q4. CP 두 채널과 정비 기록은 어디까지 연결되는가?

**질문:** CP-Z4와 CP-Z4M은 같은 사건에서 같이 변하는가? CP 사건은 정비 감지·착수·완료의 어느 쪽에 위치하는가?

**선정 근거:** CP-Z4M도 5.6과 같은 앞 기준·피크 창으로 계산한다. 왼쪽은 14사건 각각의 같은 시각 변화량, 오른쪽은 5.7의 접힌 보조 통계와 같은 차분 시차다. 표시를 위해 CP-Z4M 기록값에 10¹⁰을 곱하며 물리적 단위를 새로 정의하지 않는다.

정비 표는 각 CP 피크에서 감지·착수·완료 중 하나라도 ±72시간 안에 있는 WO를 모두 보여준다. 72시간은 대조 표의 범위이며 고장 전조 임계값이 아니다. 작업이 있으면 REMARK·교체 기록도 함께 보되, 가까운 WO를 원인으로 자동 연결하지 않는다.


In [ ]:
# 두 CP 기록의 관측 관계를 직접 그리고, 각 사건과 작업 시각·REMARK를 대조한다.
g=relation_event_delta.loc[relation_event_delta.SENSOR.eq('CP-Z4M')].copy()
curve=relation_lags.loc[relation_lags.PAIR.eq('CP')&relation_lags.SENSOR.eq('CP-Z4M')]
fig,cells=report_page('CP 두 기록의 관계 | 동반 변화와 계측 정의 확인을 나누어 본다',
    'CP-Z4·CP-Z4M의 물리량·단위·측정/계산 경로는 미정의 상태다. 관측상의 동반 변화가 같은 센서·물리적 원인임을 뜻하지 않는다.',
    [dict(panels=[('CP 사건별 | 피크 부근의 CP-Z4M 변화',['CP-Z4M']),('CP-Z4 → CP-Z4M | 차분 시차',['CP-Z4M'])],height=3.0)],width=24,
    info=['왼쪽 각 점=CP 사건 · 가로=CP-Z4 피크 편차 · 세로=CP-Z4M 피크 ±15초 중앙값 - 사건 시작 전 5~1분 중앙값(×10^10).',
          '오른쪽=사건별 회색 곡선, LOT 동일 가중 중앙값, LOT 하나씩 제외한 범위. 정의 미확인 상태에서 독립 센서 검증·단위 환산으로 쓰지 않음.'])
ax,head,_=cells[0][0]
ax.scatter(g.CP_DEV,g.DELTA*1e10,s=38,color=sensor_color('CP-Z4M'),alpha=.8,label='유효 CP 사건')
ax.axhline(0,color='#aebac3',lw=.7);ax.axvline(0,color='#aebac3',lw=.7)
ax.set_xlabel('CP-Z4 · LOT 중앙값 대비 피크 편차 [기록값]');ax.set_ylabel('CP-Z4M · 앞 기준 대비 변화량 [기록값 ×10^10]')
ax.margins(x=.2,y=.45);panel_legend(head,ax)
selected=g.loc[g.LOT.eq(240069)].sort_values('CP_DEV').iloc[-1]
ax.annotate(event_text(selected.ID)+' · 최대 CP 사건',xy=(selected.CP_DEV,selected.DELTA*1e10),xytext=(-6,8),textcoords='offset points',ha='right',va='bottom',fontsize=9.5,color='#477B94',bbox=dict(boxstyle='round,pad=.3',fc='white',ec='#477B94',lw=.8),arrowprops=dict(arrowstyle='->',lw=.8,color='#477B94'))
ax,head,_=cells[0][1]
for _,v in curve.groupby('ID'):ax.plot(v.LAG_SEC,v.R,color='#b8c2cc',alpha=.5,lw=.7)
med,lo,hi,count=lag_ensemble(curve)
ax.fill_between(REL_LAGS,lo,hi,color='#64748b',alpha=.15,lw=0)
ax.plot(REL_LAGS,med,color='#64748b',lw=1.8,label='LOT 동일 가중 중앙값')
ax.axhline(0,color='#aebac3',lw=.7);ax.axvline(0,color='#aebac3',ls=':',lw=.7)
ax.set(xlim=(-120,120),ylim=(-1.05,1.05),xlabel='시차 [초] · +는 CP-Z4 먼저',ylabel='차분 Spearman r');panel_legend(head,ax)
emit(fig,'05_관계검토/04_CP두채널_관측관계.png')
relation_work_rows=[]
for ev in relation_events.itertuples():
    near=cal_work.loc[cal_work[['DETC_DT','STRT_DT','END_DT']].sub(ev.PEAK).abs().min(axis=1).le(pd.Timedelta(hours=72))]
    if near.empty:
        relation_work_rows.append({'사건':ev.ID,'CP 피크':ev.PEAK,'WO':'±72h 연결 작업 없음','REMARK':'-','시간 관계':'해당 범위 비교 보류'})
    for w in near.itertuples():
        phase='착수 전' if ev.PEAK<w.STRT_DT else '작업 중' if ev.PEAK<=w.END_DT else '완료 뒤'
        relation_work_rows.append({'사건':ev.ID,'CP 피크':ev.PEAK,'WO':w.WO_NO,'유형':w.MNT_TYPE,
            '감지':w.DETC_DT,'착수':w.STRT_DT,'완료':w.END_DT,'CP→감지[h]':(w.DETC_DT-ev.PEAK).total_seconds()/3600,
            '완료→CP[분]':(ev.PEAK-w.END_DT).total_seconds()/60,'시간 관계':phase,
            'REMARK':remark_text(w),'교체 기록':' / '.join(parts_for(w.WO_NO)) or '-(미기재)'})
relation_work_links=pd.DataFrame(relation_work_rows)
show_table(relation_work_links.loc[relation_work_links.사건.str.startswith(('240069','240117')),
    ['사건','WO','유형','시간 관계','CP→감지[h]','완료→CP[분]','REMARK']])
table(relation_work_links)
# 7월 25일 교체는 ±72h 밖의 별도 후속 단서로 기존 30일 표를 유지한다.
show_table(pd.DataFrame([
 {'질문':'CP 두 채널은 같은 물리량인가?','현재 근거':'그림의 기록 변화·시차만 계산','보류·추가 정보':'측정/계산 정의·배선·변환식·보정·INS-006 대응'},
 {'질문':'240117의 CP가 해당 CBM을 미리 알렸는가?','현재 근거':'CBM 01854 완료 12:44 → CP 최저 13:04:50 (+20.83분)','보류·추가 정보':'해당 작업의 사전 경보로 해석 불가; 작업 대상·완료 점검·후속 조건 확인'},
 {'질문':'240069의 이후 교체가 CP 원인인가?','현재 근거':'07/08 절연 저하 BM, 07/25 가스 분석계 셀 손상 교체','보류·추가 정보':'동일 설비 후속 기록; CP 채널 연결·진단·보정 미확인, 조치 직전직후 같은 조건 관측 부족'},
 {'질문':'정기수리로 개선됐는가?','현재 근거':'전후 강종 다름, 수리 중 온도 0행','보류·추가 정보':'같은 강종·SV·속도·두께·부하에서 횟수·크기·지속시간을 재비교'},
 {'질문':'온도 변화가 품질에 영향을 주었는가?','현재 근거':'품질 결과·소재 이동 정렬 조건 없음','보류·추가 정보':'라인 속도·존 간 거리·소재 추적·품질 데이터로 다음 단계 검증'}]))

v=g.DELTA.dropna()
note(f'**CP-Z4M의 실제 관측:** 유효 {len(v)}사건에서 앞 기준 대비 증가 {int(v.gt(0).sum())}건·감소 {int(v.lt(0).sum())}건이다. 이는 기록값 변화의 집계이며 CP-Z4와 같은 물리량인지 확인한 결과가 아니다. 정비의 원인·효과 판단은 위 표의 시간 순서와 채널 정의 확인이 먼저다.')


**관찰 결과·반례·해석**

- 두 CP 기록의 같은 시각 변화량과 차분 시차는 각각 다른 질문이다. 일부 점이나 일부 시차에서 함께 움직여도 공통 계산식·같은 센서·고장 원인의 증거로 바로 바꾸지 않는다. 부호·시차가 사건마다 다른 경우에도 미정의 채널의 전체 관계가 없다고 단정하지 않는다.
- 정비 표의 시간 부호를 먼저 읽는다. `CP→감지[h]`가 양수면 CP 뒤에 감지가 기록됐고, `완료→CP[분]`가 양수면 완료 뒤 CP가 기록됐다. 감지 시각은 실제 고장 시작 시각과 같다고 가정하지 않는다.
- 240069의 7월 25일 가스 분석계 셀 손상 교체는 CP 최고에서 약 16.9일 뒤의 단서다. CP–INS-006 대응이 없어 인과를 제시할 수 없다. 그 뒤 다른 LOT 관측이 있어도 조치 직전·직후의 같은 조건 비교가 아니므로 수리 효과가 검증된 것은 아니다.
- 정상 제어 반응인지 점검 신호인지는 SV·제어 모드·운전 조건, 실제 작업 진단·보정, 같은 조건에서의 재현과 조치 후 변화로 구분한다. 현재 자료에서는 **관측 관계와 원인 가설의 근거·반례**까지 제시한다.


<a id="process-summary"></a>
## 06 공정별 시계열 특성 종합

동일한 임계값 하나로 모든 공정을 판정하지 않는다. 온도 추세·제어 출력·CP 펄스·주기 신호를 서로 다른 질문으로 정리한다.

In [ ]:
def stable_z_count(signals):
    return sum(int((zstats[k,sensor].Z.abs().ge(3)&states[k].eq('안정 참고')).sum()) for k in LOT_EVENTS for sensor in signals)
def gap_value(k,sensor):return float(level_gap.loc[level_gap.LOT.eq(k)&level_gap.SENSOR.eq(sensor),'GAP'].iloc[0])
show_table(pd.DataFrame([
 {'공정':'세정','주요 신호':'CLN-IN','관측 결과':f'전환 제외 |Z|≥3 {stable_z_count(["CLN-IN"])}개 신호·1분창','해석·확인':'LOT 내부 수준 변화와 반복 제어 형태를 대조. 강종 정상 한계로 사용하지 않음'},
 {'공정':'건조','주요 신호':'TC-Z1·TC-Z2 / OP-Z1·OP-Z2','관측 결과':f'전환 제외 온도 |Z|≥3 {stable_z_count(["TC-Z1","TC-Z2"])}개 신호·1분창','해석·확인':'온도와 대응 출력의 시각·복귀를 구분. 초기·종료 전환 문맥 확인'},
 {'공정':'가열 초입','주요 신호':'TC-Z3 / OP-Z3','관측 결과':'단발 온도 반응과 큰 출력 변동이 함께 관측됨','해석·확인':'반복 제어 응답 가설. SV·제어 로그 없이 정상 제어·센서 고장 확정 불가'},
 {'공정':'가열·균열','주요 신호':'TC-Z4–TC-Z6 / OP-Z4–OP-Z6','관측 결과':f'240061 OP-Z4 차이 {gap_value(240061,"OP-Z4"):+.2f}%p, TC-Z4 {gap_value(240061,"TC-Z4"):+.3f}℃','해석·확인':'전환 제외 후 분포 중심 차이. 설정·부하·속도·두께 확인'},
 {'공정':'CP 계측','주요 신호':'CP-Z4 / CP-Z4M','관측 결과':f'원본 CP {len(cp_real)}사건 / {cp_real.LOT.nunique()} LOT','해석·확인':'1초 펄스의 폭·반복·복귀를 확인. 두 CP 기록의 측정 정의를 같게 가정하지 않음'},
 {'공정':'과시효','주요 신호':'TC-Z7·TC-Z8 / OP-Z7·OP-Z8','관측 결과':f'240061 OP-Z8 차이 {gap_value(240061,"OP-Z8"):+.2f}%p; 다른 LOT의 온도 수준 차이도 관측','해석·확인':'동일 강종 분포로 확인. 어느 LOT이 정상인지 정할 독립 기준은 없음'},
 {'공정':'Z9·냉각','주요 신호':'TC-Z9 / TC-OUT1·TC-OUT2','관측 결과':'반복 파형 관측; 비주기 온도 Z 선별에서는 분리','해석·확인':'주기·위상·운전 조건을 따로 확인. 전 신호에 자기상관·시차 분석을 억지로 적용하지 않음'}]))
note('**표의 창 수는 고장 수가 아니다.** 여러 신호·시간창이 중첩될 수 있다. 지속·반복·복귀와 정비의 실제 시각으로 재검토한다.')

**공정 간 시간 관계의 한계:** 같은 시각에 기록된 가열존·과시효·냉각 값이 반드시 같은 소재 위치를 뜻하지 않는다. 품질과 연결하려면 라인 속도·존 간 거리로 소재 이동 시차를 정렬하고 품질 결과와 맞춰야 한다. 여기서는 동시 기록의 반응을 기술했으며 인과나 품질 영향을 검증하지 않았다.

<a id="conclusion"></a>
## 07 종합 판단 및 현장 적용 방안

| 분석 대상 | 확인된 사실 | 현재 판단 | 후속 점검·확보 기록 |
|---|---|---|---|
| 240069 | CP 7사건·시작 전후 2분 TC/OP 최대편차 방향 불일치; 큰 TC 변화에 CP 이탈이 없는 대조 사례 | 단일 TC 원인 설명 부족; 정비와의 인과 미확정 | CP-Z4·CP-Z4M 정의, INS-006 채널 연결, 07/08·07/25 진단·보정 및 제어 로그 |
| 240037 | CP 반복 3사건 | 짧은 원본 펄스와 동반 반응 확인 필요 | 동시 TC·OP, CP 계측 정의·수집 로그 |
| 240117 | CBM 완료 후 20.83분에 CP 최저 | 해당 CBM의 사전 경보 아님 | 작업 대상·완료 점검·후속 운전 조건·CP 로그 |
| 240061 | SPHC의 OP-Z4·OP-Z8 분포 중심 차이 | 운전 조건 차이 가능, 고장 미확정 | SV·라인 속도·부하·두께·보정·같은 조건의 반복 LOT |
| 계획수리 전후 | 공통 하강·승온과 남는 국소 반응 | 운전 전환 가설 우선, 수리 효과 미확정 | 운전 모드·재가동 기록·정비 대상 부품·완료 확인 |

**현장 적용 순서:** 먼저 기록·계측 정의와 시각 정합성을 확인하고, 사례별 계측·제어·운전 조건 점검으로 범위를 좁힌다. 추가 기록이 확보되면 같은 강종·같은 운전 조건에서 재현 여부와 조치 후 변화를 검증한다.

**현재 판단 불가:** 정상 기준 방향, 고장 정답, 수리의 인과 효과, 품질 영향·예측 성능. 제공 자료에 품질 결과와 소재 이동 정렬 조건이 없어 품질 영향은 후속 검증 가설이다.

**종합 메시지:** 14 LOT의 운전 전환, 반복적 순간 이탈, 강종별 운전 수준 차이와 정비 후 변화를 구분해 추가 점검 신호와 필요한 검증 항목을 도출했다.

**원인 해석을 제한하는 우선 정보:** CP-Z4·CP-Z4M의 정의와 INS-006 연결은 분석팀에서도 미확인이다. 현재 결론은 관측 관계까지이며 물리적 원인·고장 부품 판정은 보류한다. 정의가 확보되더라도 인과 판단에는 운전 조건·제어/계측 로그와 조치 전후의 비교가 추가로 필요하다.
**새 관계 분석을 반영한 판단:** CP 이탈이 없던 시간대에도 큰 Z3 변화가 관측된다. Z4~Z6는 작은 진폭이지만 이탈 때 변화폭이 더 큰 경우가 있어 후속 점검에서 제외하지 않는다. CP 하강 2사건에서 Z3 온도·출력 변화가 CP 최저점보다 앞서 보이고 Z4~Z6의 동반 방향도 나뉜다는 점을 근거로, CP 계측·계산 경로와 당시 제어 응답을 함께 확인한다. 이는 관측 관계의 근거이며 특정 존·부품이 CP 원인이라는 결론은 아니다.


In [ ]:
table(plan_compare)

# 새 관계 분석의 계산값을 종합 판단에 연결한다.
matched=relation_match_coverage.STATUS.eq('대조 가능')
show_table(relation_match_summary[['신호','비교 쌍','사건>대조','짝 차이 중앙값','해당 LOT 수']].rename(columns={'비교 쌍': '비교한 시간대 수', '사건>대조': 'CP 이탈 때 최대폭이 더 큰 횟수', '짝 차이 중앙값': '두 시간대 최대폭 차이 중앙값'}))
note(f'**관계 검토의 범위:** CP {len(relation_events)}사건/{relation_events.LOT.nunique()} LOT의 같은 시각 변화량, 사건별 CP 시차와 14 LOT OP–TC 시차를 계산했다. CP 이탈이 없던 같은 LOT의 다른 시간대는 {int(matched.sum())}번 확보했다. 사건 선별·겹치는 창·미측정 운전 조건이 있어 이 관측 비교를 인과·고장 확률로 해석하지 않는다.')


<a id="detail"></a>
## 08 부록 — 14 LOT 전체 검토 및 재현 코드

본문의 핵심 사례로 결론을 읽은 뒤 원본 근거를 확인하는 부분이다. 모든 LOT의 전체 22신호·모든 (a)(b)(c)·강종 대조·정비 시각 검토를 보존했다. **고정 구간 번호·시각·선정 이유가 전체 그림과 확대 그림에 반복되며, 확대 상단의 전체 신호가 현재 위치를 다시 보여 준다.**

**표시 기준:** CP 0.05 사건은 전부 표시하고, 45분 안의 사건을 한 확대 묶음으로 합친다. 온도 |Z|≥3 중 전환 후보 밖의 가장 큰 1구간을 추가한다. CLN-IN의 이탈은 별도로 표시한다. Z9·냉각 주기 파형을 온도 Z 선별에 넣지 않는다. 수리 전후 전환 후보와 관측 중 DETC_DT도 별도 표시한다. 같은 강종 수준 차이는 (L)로 표시하고 시간 펄스와 구분한다.

**정비 대조 기준:** 실제 작업·감지 교집합 WO를 우선한다. 교집합이 없으면 감지 또는 작업 구간이 가장 가까운 1 WO를 고른다. 전·후 관측은 착수 전 마지막 45분, 완료 후 첫 45분 중 **72시간 이내에 실제 기록된 범위**만 보여 준다. 작업 중은 실제 교집합을 쓴다. 관측이 없으면 해당 칸을 비워 이유를 표시한다. 시간상 가까움은 원인·효과의 증거가 아니다.
**확대 그림의 색·시간:** 전체 위치와 확대의 선은 같은 센서색이다. 음영·테두리는 구간 유형색이고, 위 음영은 확대 범위, 아래 음영은 실제 CP 기록 범위·Z 1분창·감지 시각 대조를 뜻한다. 확대 패널끼리 실제 시각·범위·눈금을 고정한다.


**LOT 바로가기:** [240001](#lot-240001) · [240024](#lot-240024) · [240037](#lot-240037) · [240038](#lot-240038) · [240044](#lot-240044) · [240058](#lot-240058) · [240061](#lot-240061) · [240069](#lot-240069) · [240091](#lot-240091) · [240108](#lot-240108) · [240114](#lot-240114) · [240117](#lot-240117) · [240121](#lot-240121) · [240133](#lot-240133)

In [ ]:
lot_execution=[]

<a id="lot-240001"></a>
### 8.1 LOT 240001 · SGCC · 01/02

**① 전체 위치 확인** → **② 핵심 변화 확대 → 같은 시각 공정 대조 → 나머지 구간** → **③ 강종 수준 차이(해당 LOT)** → **④ 관련 WO 전·중·후 관측과 판정**

In [ ]:
# 이 셀에서 아래 그래프의 matplotlib 생성 함수를 정의하고 바로 실행한다.
def focus_event(k):
    """표시 순서만 정한다. CP 내 최대 절대편차, Z 내 최대 절대점수이며 고장 순위가 아니다."""
    events=LOT_EVENTS[k]
    cp=[e for e in events if e['kind']=='CP']
    if cp:return max(cp,key=lambda e:abs(e['deviation']))
    zs=[e for e in events if e['kind']=='Z' and not e.get('transition_context')]
    if zs:return max(zs,key=lambda e:abs(e['score']))
    return next((e for e in events if e['kind']=='감지'),events[0])

def lot_cards(k):
    r=lot_meta.loc[k];e=focus_event(k);ws=lot_works(k);w=primary_work(k)
    evs=LOT_EVENTS[k];cp=sum(len(t['pulses']) for t in evs if t['kind']=='CP')
    if e['kind']=='CP':
        base=raw[k]['CP-Z4'].median();value=f"CP-Z4 {e['deviation']:+.3f} · {e['tag']}"
        facts=[f"{e['point']:%m/%d %H:%M:%S} · 원본 {base+e['deviation']:.4f}",f'LOT 중앙값 {base:.4f} 대비 · 전체 {cp}사건',
               '확인: 탐색 기준 ±0.05 이탈','고장·원인은 추가 확인 필요']
    elif e['kind']=='Z':
        value=f"{e['sensor']} Z={e['score']:+.2f} · {e['tag']}"
        facts=[f"{e['point']:%m/%d %H:%M:%S} 종료 1분창",'확인: 이전 30분 기준에서 |Z| ≥ 3','지속 변화·일시 제어 반응을 아래에서 대조','Z 초과는 고장 판정이 아님']
    else:
        value=e['title'];facts=[f"{e['point']:%m/%d %H:%M:%S}",'시각 기록과 다신호 원본 대조','감지 기록은 온도 자동 경보 정의가 아님']
    direct=len(linked_work(k));det=int(cal_work.DETC_DT.between(r.START,r.END).sum())
    worklines=[f'{w.MNT_TYPE} {w.WO_NO[-5:]} 감지 {w.DETC_DT:%m/%d %H:%M}',f'작업 {w.STRT_DT:%m/%d %H:%M}–{w.END_DT:%m/%d %H:%M}']
    for other in ws.loc[ws.WO_NO.ne(w.WO_NO)].itertuples():
        worklines.extend([f'{other.MNT_TYPE} {other.WO_NO[-5:]} · {remark_text(other)}',f'감지 {other.DETC_DT:%m/%d %H:%M} · 작업 {other.STRT_DT:%H:%M}–{other.END_DT:%H:%M}'])
    if k==240117:worklines.append(f"CP 최저: CBM 완료 후 {(e['point']-w.END_DT).total_seconds()/60:.2f}분")
    elif parts_for(w.WO_NO):worklines.append('교체: '+' / '.join(parts_for(w.WO_NO)))
    if not direct:worklines.append('이 LOT 관측 중 실제 작업 겹침 없음')
    c=context.set_index('LOT').loc[k]
    obs=[f'{r.START:%Y/%m/%d %H:%M}–{r.END:%m/%d %H:%M}',f'{r.MINUTES:.1f}분 · 22신호 · {len(raw[k]):,}행',
         f'작업 교집합 {direct} WO · 감지 {det}건',str(c.PLAN_RELATION) if c.PLAN_RELATION else '정기수리 경계 LOT 아님']
    return [dict(title='먼저 볼 근거 · '+EVENT_LABELS[e['kind']],value=value,lines=facts,color=e['color']),
            dict(title='정비 REMARK · '+('관측과 작업 겹침' if direct else '가까운 정비 기록'),
                 value=f'{w.MNT_TYPE} · {remark_text(w)}',lines=worklines,color='#7540A8',face='#f7f3fb'),
            dict(title='LOT 관측 정보 · 실제 시각',value=f'{k} · {r.GRADE}',lines=obs,color='#477B94')]

def event_line(k,e):return f"{event_text(event_id(k,e))} | {e['point']:%m/%d %H:%M:%S} | {event_reason(k,e)}"

def is_trigger(e,sensor):
    return sensor==e['sensor'] if e['kind'] in ['CP','Z'] else sensor in HEAT+[f'OP-Z{i}' for i in range(3,7)] if e['kind']=='전환' else False

def show_event_index(k):
    from IPython.display import HTML
    from html import escape
    rows=''.join('<tr><td><a href="#event-'+event_id(k,e)+'"><b>'+event_text(event_id(k,e))+'</b></a></td><td>'+escape(event_reason(k,e))+'</td><td>'+e['point'].strftime('%m/%d %H:%M:%S')+'</td></tr>' for e in LOT_EVENTS[k])
    display(HTML('<p><b>이 LOT에서 확인할 지점 — 같은 번호가 전체·확대·해석에 반복됩니다.</b></p><table><thead><tr><th>고정 번호</th><th>왜 보는가</th><th>기준 시각</th></tr></thead><tbody>'+rows+'</tbody></table>'))

def draw_lot_overview(k):
    evs=LOT_EVENTS[k];r=lot_meta.loc[k]
    labels=' / '.join(e['tag']+' '+('전환' if e['kind']=='전환' else '국소 변화(Z)' if e['kind']=='Z' else e['kind']) for e in evs)
    rows=[{'panels':[('관측·정비 시각과 확대 위치 | '+labels,[])],'height':.85}]
    for name,left,right in FLOW_ROWS:
        panels=[(name+' · 온도',left)] if right is None else [(name+' · '+('기록값' if name.startswith('CP') else '온도'),left),(name+' · '+('기록값' if name.startswith('CP') else '온도' if name.startswith('Z9') else '출력'),right)]
        rows.append({'panels':panels,'height':1.3})
    fig,cells=report_page(f'{k} · {r.GRADE} | ① LOT 전체 22신호',
        '진한 원·굵은 선·테두리=이 구간을 선택한 신호. 다른 공정의 옅은 음영은 같은 시각의 대조이며 이상 판정이 아니다.',rows,cards=lot_cards(k),info=[event_line(k,e) for e in evs],event_key=True)
    show_event_index(k)
    ax,head,_=cells[0][0];work_clock(ax,head,k)
    for e in evs:
        ax.axvspan(at(k,e['a']),at(k,e['b']),color=e['color'],alpha=.22 if e['kind']!='전환' else .12,lw=0)
        ax.text(at(k,(e['a']+e['b'])/2),-.4,e['tag'],ha='center',va='center',fontsize=10,fontweight='bold',color=e['color'])
    for row in cells[1:]:
        for ax,head,sensors in row:
            for s in sensors:report_signal(ax,k,s)
            selected=[]
            for e in evs:
                primary=any(is_trigger(e,sensor) for sensor in sensors)
                ax.axvspan(at(k,e['a']),at(k,e['b']),color=e['color'],alpha=.18 if primary else .035,lw=0)
                ax.axvline(e['point'],color=e['color'],lw=1.1 if primary else .55,ls=EVENT_LINES[e['kind']],alpha=1 if primary else .35)
                if primary:
                    selected.append(e)
                    for sensor in sensors:
                        if not is_trigger(e,sensor):continue
                        if e['kind']=='CP':
                            y=raw[k].set_index('MEAS_DT')[sensor]
                            for pulse in e['pulses'].itertuples():
                                q=y.loc[at(k,pulse.A).round('s'):at(k,pulse.B).round('s')]
                                ax.plot(q.index,q,color=e['color'],lw=2.4,zorder=6)
                            value=y.loc[e['point']];point=e['point']
                        elif e['kind']=='Z':value=zstats[k,sensor].loc[e['minute'],'X'];point=at(k,e['minute']+.5)
                        else:continue
                        ax.scatter(point,value,s=55,facecolors='white',edgecolors=e['color'],lw=2.1,zorder=8)
                        overview_trigger_checks.append((k,e['tag'],sensor))
            if selected:
                color=selected[0]['color']
                head.text(1,.98,'확인 '+', '.join(e['tag']+' '+('동반 전환' if e['kind']=='전환' else e['sensor']) for e in selected),ha='right',va='top',fontsize=11,fontweight='bold',color=color,transform=head.transAxes)
                for spine in ax.spines.values():spine.set_edgecolor(color);spine.set_linewidth(1.2)

            for w in linked_work(k).itertuples():
                for t,ls in [(w.STRT_DT,'--'),(w.END_DT,'-.')]:ax.axvline(t,color=COL[w.MNT_TYPE],lw=.8,ls=ls,alpha=.7)
            if 'CP-Z4' in sensors:
                base=raw[k]['CP-Z4'].median()
                for v in [base-.05,base+.05]:ax.axhline(v,color=EVENT_COLORS['CP'],ls='--',lw=.75)
            ax.set_ylabel(display_unit(sensors[0]));clock_in_lot(ax,k);panel_legend(head,ax)
    fig._folded=f'{k} {r.GRADE}: 전체 22신호·LOT 정보·정비 REMARK 펼치기'
    emit(fig,f'03_상세/01_LOT전체/{k}_{r.GRADE}.png')
    note('**확대 위치:** '+' / '.join(f"{e['tag']} {at(k,e['a']):%m/%d %H:%M}–{at(k,e['b']):%H:%M}: {e['title']}" for e in evs))

draw_lot_overview(240001)

**② 모든 확인 구간을 각각 검토했다.** 아래 펼침 블록에서 (a)(b)(c) 각각을 같은 크기로 확인한다. 매 구간은 주된 신호 → 같은 시각 다른 공정 → 정비 REMARK → 해석으로 이어진다. 최대값 하나로 나머지 구간을 정상 처리하지 않는다.

In [ ]:
# 이 셀에서 아래 그래프의 matplotlib 생성 함수를 정의하고 바로 실행한다.
def event_relation(k,e):
    r=lot_meta.loc[k];w=primary_work(k);t=e['point']
    dt=lambda x:f'{x:%m/%d %H:%M}'
    if w.STRT_DT<=t<=w.END_DT:
        return f'{w.MNT_TYPE} {w.WO_NO[-5:]} 작업 중 관측 · REMARK: {remark_text(w)}'
    if t>w.END_DT:
        return f'{w.MNT_TYPE} {w.WO_NO[-5:]} 완료 뒤 {(t-w.END_DT).total_seconds()/60:.2f}분 · REMARK: {remark_text(w)}'
    return f'{w.MNT_TYPE} {w.WO_NO[-5:]} 감지 {dt(w.DETC_DT)} / 착수 {dt(w.STRT_DT)} · REMARK: {remark_text(w)} · '+('작업 겹침 있음' if len(linked_work(k)) else '작업 겹침 없음')

def response_stats(k,e,s):
    """동시 움직임을 기술한다. 정상 범위나 다른 공정의 이상 판정에 사용하지 않는다."""
    t=e['point'];g=raw[k].set_index('MEAS_DT')[s]
    before=g.loc[t-pd.Timedelta(minutes=10):t-pd.Timedelta(minutes=5)].dropna()
    during=g.loc[t-pd.Timedelta(minutes=1):t+pd.Timedelta(minutes=1)].dropna()
    after=g.loc[t+pd.Timedelta(minutes=5):t+pd.Timedelta(minutes=10)].dropna()
    center=before.median();dev=during-center
    peak=dev.loc[dev.abs().idxmax()] if len(dev) and pd.notna(center) else np.nan
    return dict(sensor=s,before=center,after=after.median(),change=after.median()-center,
                peak=peak,n_before=len(before),n_focus=len(during),n_after=len(after))

def response_caption(k,e,sensors):
    metrics=[response_stats(k,e,s) for s in sensors]
    unit='℃' if sensors[0] in TC else '%p' if sensors[0] in OP else '기록값'
    factor=1e10 if sensors==['CP-Z4M'] else 1
    if sensors==['CP-Z4M']:unit='기록값 x1e-10'
    show=lambda v:f'{v*factor:+.3f}' if pd.notna(v) else '자료 없음'
    return '표시 시각 ±1분 최대편차: '+' / '.join(f"{d['sensor']} {show(d['peak'])}" for d in metrics)+f' {unit}'

def interval_main(e):
    if e['kind']=='전환':return 3,[('TC-Z3–TC-Z6',HEAT),('OP-Z3–OP-Z6',[f'OP-Z{i}' for i in range(3,7)])]
    if e['kind'] in ['CP','감지']:return 4,[('CP-Z4',['CP-Z4']),('CP-Z4M',['CP-Z4M'])]
    zone=degree_zone(e['sensor'])
    if not zone:return 0,[(e['sensor'],[e['sensor']])]
    index=1 if zone<=2 else 2 if zone==3 else 3 if zone<=6 else 5
    name,left,right=FLOW_ROWS[index]
    return index,[(name+' · 온도',left),(name+' · 출력',right)]


def detail_clock(ax,a,b):
    # 모든 확대 패널에 같은 실제 시각·범위·눈금을 적용한다. 양 끝도 눈금으로 표시한다.
    span=(b-a).total_seconds()
    seconds=next((step for step in [1,2,5,10,15,20,30,60,120,300,600,900,1200,1800,3600,7200] if span/step<=4),7200)
    inner=pd.date_range(a.ceil(f'{seconds}s'),b.floor(f'{seconds}s'),freq=f'{seconds}s')
    ticks=[a]+[t for t in inner if (t-a).total_seconds()>=span*.10 and (b-t).total_seconds()>=span*.10]+[b]
    ticks=list(dict.fromkeys(ticks));values=mdates.date2num(ticks)
    seconds_visible=any(t.second or t.microsecond or t.nanosecond for t in ticks)
    fmt=('%H:%M:%S' if seconds_visible else '%H:%M')
    if a.date()!=b.date():fmt='%m/%d\n'+fmt
    ax.set_xticks(values);ax.xaxis.set_major_formatter(mdates.DateFormatter(fmt))
    ax.set_xlim(a,b);ax.tick_params(axis='x',labelsize=10,pad=5)
    return values


def draw_lot_zoom(k):
    r=lot_meta.loc[k];pictures=[];naxes=0;interpretations=[]
    for e in LOT_EVENTS[k]:
        a,b=at(k,e['a']).round('s'),at(k,e['b']).round('s');main_index,main=interval_main(e)
        flow=[i for i in range(6) if i!=main_index and (i in [2,3,4,5] or e['kind']=='전환')]
        labels=[]
        for label,sigs in main:
            role='감지 시각 대조' if e['kind']=='감지' else '보조 채널 대조' if sigs==['CP-Z4M'] else '선정 근거'
            labels.append((e['tag']+' '+role+' | '+label,sigs))
        captions=[e['title']+(' · 점선=LOT 중앙값 ±0.05' if sigs==['CP-Z4'] else '') for _,sigs in main]
        if e['kind']=='Z':
            start,end=event_focus_ranges(k,e)[0]
            captions=[text+f' · 음영={start:%H:%M}–{end:%H:%M} 1분창' for text in captions]
        rows=[{'panels':[(f'전체 LOT → 아래 확대창 | {event_id(k,e)} · {a:%H:%M:%S}–{b:%H:%M:%S}',[])],'height':.9},
              {'panels':labels,'height':2.1,'captions':captions}]
        for i in flow:
            name,left,right=FLOW_ROWS[i];panels=[(name+' | '+', '.join(left),left)]
            if right:panels.append((name+' | '+', '.join(right),right))
            rows.append({'panels':panels,'height':1.3,'captions':[response_caption(k,e,sigs) for _,sigs in panels]})
        meaning={'Z':'아래 주황 음영=선정된 실제 1분창 · 마름모=그 창의 중앙값(원본 순간값과 구분).',
                 'CP':'아래 빨강 음영=CP 원본 이탈 기록 범위 · 원=해당 사건의 실제 피크.',
                 '감지':'아래 보라 음영=감지 시각 ±30초 대조 · 세로선=DETC_DT. CP 이상을 탐지했다는 뜻이 아님.',
                 '전환':'아래 회색 음영=전환 참고 구간. 실제 운전 모드·정상 여부는 별도 확인.'}[e['kind']]
        fig,cells=report_page(f'{event_id(k,e)} · {r.GRADE} | {EVENT_LABELS[e["kind"]]} · 구간 상세',
            f'{a:%Y/%m/%d %H:%M:%S}–{b:%m/%d %H:%M:%S} · 위 음영은 아래 그래프의 확대 범위. 아래 모든 패널의 x축은 같은 실제 시각이다.',
            rows,width=20,info=[event_line(k,e),'정비 대조: '+event_relation(k,e),
                              '선=센서색(위·아래 동일) · 음영·테두리=구간 유형색. '+meaning],event_key=True)
        ax,head,_=cells[0][0];draw_event_locator(ax,head,k,e)
        bodies=[]
        for index,row in enumerate(cells[1:],1):
            for ax,head,sensors in row:
                bodies.append(ax)
                can_primary=(e['kind'] in ['CP','Z'] and e['sensor'] in sensors)
                if can_primary and len(sensors)==1:plot_event_trace(ax,head,k,e,a,b,large=True)
                else:
                    for sensor in sensors:report_signal(ax,k,sensor,a,b,fine=True)
                    shade_event_focus(ax,k,e)
                    if 'CP-Z4' in sensors:
                        base=raw[k]['CP-Z4'].median()
                        for v in [base-.05,base+.05]:ax.axhline(v,color=EVENT_COLORS['CP'],ls='--',lw=.8)
                    ax.set_ylabel(display_unit(sensors[0]));zoom_clock(ax,a,b)
                    if can_primary:z_window_marker(ax,k,e,annotate=True)
                    panel_legend(head,ax)
                if index==1:
                    head.texts[0].set_color(e['color'])
                    for spine in ax.spines.values():spine.set_edgecolor(e['color']);spine.set_linewidth(1.1)
                detail_clock(ax,a,b)
                if index==len(cells)-1:
                    ax.set_xlabel(f'{a:%Y/%m/%d} 실제 측정 시각' if a.date()==b.date() else '실제 측정 날짜·시각')
                if sensors==['TC-Z3'] and e['kind']=='CP':
                    y=raw_window(k,'TC-Z3',a,b).dropna();point=y.idxmax()
                    near_note(ax,point,y.loc[point],f'{point:%H:%M:%S} TC-Z3 최고 {y.loc[point]:.2f}℃')
                if sensors==['OP-Z3'] and e['kind']=='CP':
                    y=raw_window(k,'OP-Z3',a,b).dropna();point=y.idxmin()
                    near_note(ax,point,y.loc[point],f'{point:%H:%M:%S} OP-Z3 최저 {y.loc[point]:.2f}%')
        expected_limits=mdates.date2num([a,b]);ticks=bodies[0].get_xticks()
        for ax in bodies:
            assert np.allclose(ax.get_xlim(),expected_limits,atol=1e-10,rtol=0),event_id(k,e)
            assert np.array_equal(ax.get_xticks(),ticks),event_id(k,e)
        focus=event_focus_ranges(k,e)
        if e['kind']=='Z':assert focus[0][1]==e['point'] and focus[0][1]-focus[0][0]==pd.Timedelta(minutes=1)
        zoom_alignment_checks.append(dict(LOT=k,TAG=e['tag'],START=a,END=b,N_PANELS=len(bodies),N_TICKS=len(ticks),COLOR=e['color'],FOCUS=focus))
        naxes+=len(fig.axes);fig._silent=True
        pictures.append(checked_emit(fig,f'{k}_{e["tag"]}',save=False,record=False))
        interpretations.append(event_result_markdown(k,e))
        zoom_presentations.append(dict(LOT=k,TAG=e['tag'],KIND=e['kind'],PRIMARY_SENSOR=e['sensor']))
    save_lot_sheet(pictures,f'03_상세/02_구간확대/{k}_전체구간_공정순서.png',naxes)
    from io import BytesIO
    from base64 import b64encode
    from html import escape
    from PIL import Image
    from IPython.display import HTML
    import mistune
    pieces=[]
    for e,array,text in zip(LOT_EVENTS[k],pictures,interpretations):
        data=BytesIO();Image.fromarray(array).save(data,format='PNG')
        pieces.append('<section id="event-'+event_id(k,e)+'"><div style="position:sticky;top:0;background:white;border-left:5px solid '+e['color']+';padding:8px;z-index:2"><b>'+escape(event_line(k,e))+'</b></div><img style="width:100%;height:auto" src="data:image/png;base64,'+b64encode(data.getvalue()).decode()+'">'+mistune.html(text)+'</section>')
    display(HTML('<details><summary><b>'+str(k)+' 전체 '+str(len(pictures))+'개 구간: (a)(b)(c) 각각의 동일 크기 상세·해석 펼치기</b></summary>'+''.join(pieces)+'</details>'))
    table(iqr_intervals.loc[iqr_intervals.LOT.eq(k)])


def event_result_markdown(k,e):
    a,b=at(k,e['a']),at(k,e['b']);t=e['point'];lines=[f'**{event_text(event_id(k,e))}에서 무엇을 확인했는가?**','']
    if e['kind']=='CP':
        base=raw[k]['CP-Z4'].median();after=raw_window(k,'CP-Z4',t+pd.Timedelta(minutes=5),t+pd.Timedelta(minutes=10)).median()-base
        facts=' / '.join(f'{r.PEAK:%H:%M:%S} {r.DEV:+.4f}' for r in e['pulses'].itertuples())
        lines += [f'- **선정 근거:** CP 원본 {len(e["pulses"])}사건. 시각·LOT 중앙값 편차: {facts}. 탐색 경계 ±0.05를 넘은 기록이다.',
                  f'- **복귀·지속:** 가장 큰 사건 후 5~10분 CP 중앙값 편차는 {after:+.4f}. '+('이 창의 중앙값은 경계 안으로 복귀했다.' if abs(after)<.05 else '이 창의 중앙값은 경계 밖에 남았다.')+' 모든 순간값의 정상 또는 고장 부재를 뜻하지 않는다.']
    elif e['kind']=='Z':
        d=zstats[k,e['sensor']].loc[e['minute']]
        lines += ['- **시각 표시:** 음영은 종료 시각 직전의 실제 1분창이며, 마름모는 그 창 중앙에 놓은 1분 중앙값이다.', f'- **선정 근거:** {e["sensor"]}의 {t:%H:%M} 종료 1분 중앙값 {d.X:.3f}, 과거 중앙값 {d.MEDIAN:.3f}, SD {d.SD:.3f} → Z={d.Z:+.2f}. 과거 기준 ±3 SD를 벗어난 창이다.',
                  '- **확인할 가설:** 해당 TC의 변화와 대응 OP의 저하·증가·복귀를 같은 시각에 대조한다. 반복 제어 응답인지 지속 수준 변화인지는 파형과 아래 전후 수치를 함께 본다.']
    elif e['kind']=='전환':lines += ['- **선정 근거:** 수리 전후의 여러 TC·OP가 함께 하강 또는 승온하는 구간이다. 한 센서만 튀는지와 구분하기 위해 온도·출력·CP를 모두 대조했다.', '- **판단:** 일정과 다신호의 함께 움직임은 운전 전환 가설과 부합한다. 실제 모드·SV·정지 명령을 확인하지 못해 정상 정지·재가동을 확정한 것은 아니다.']
    else:lines += [f'- **선정 근거:** {t:%H:%M}는 정비 DETC_DT 기록이다. 자동 온도 이상 경보의 정의는 없다. 그 시각에 실제 TC·OP·CP 변화가 있는지 비교한다.']
    details=[]
    for sensor in ['TC-Z3','OP-Z3','TC-Z4','TC-Z5','TC-Z6','TC-Z7','TC-Z8']:
        d=response_stats(k,e,sensor);unit='℃' if sensor in TC else '%p'
        if pd.notna(d['peak']):details.append(f"{sensor} {d['peak']:+.2f}{unit}")
    lines += ['- **다른 공정의 동반 반응:** 표시 시각 ±1분의 기준 대비 최대 편차는 '+' / '.join(details)+'. 기준은 전 10~5분 중앙값이며 정상 한계가 아니다.',
              '- **정비 REMARK·시각:** '+event_relation(k,e)+'.']
    if k==240117 and e['kind']=='CP':lines.append('- **가설 판정:** 마모량 초과 CBM 완료 후 20.83분에 CP 최저점이 발생했다. 이 CBM의 사전 경보라는 설명과 시간 순서가 맞지 않는다. 정비 대상과 CP 이탈 원인이 같은지는 미확인이다.')
    if k==240069 and e['kind']=='CP':lines.append('- **가설 판정:** CP 사건은 이후 BM 감지보다 먼저지만 작업 뒤 관측이 없다. REMARK “1호기 재발, 절연 저하”의 원인 신호이거나 수리로 개선됐다고 입증할 수 없다.')
    overlap=iqr_intervals.loc[iqr_intervals.LOT.eq(k)&iqr_intervals.START.lt(b)&iqr_intervals.END.gt(a)]
    lines.append(f'- **IQR 대조:** 이 시간 범위와 겹치는 신호별 IQR 후보 구간은 {len(overlap)}개다. 서로 겹치는 신호·창을 합친 고장 사건 수가 아니다. 전환 제외·기록 수·LOT별 분포를 적용한 시간표는 아래에 제공한다.')
    return '\n'.join(lines)

draw_lot_zoom(240001)

**240001 구간 검토:** 모든 표시 구간을 같은 크기의 그림·해석으로 확인했다. 접힌 상세에서 주된 근거·다른 공정의 동반 반응·정비 REMARK·복귀 여부를 각각 대조한다.

In [ ]:
# 이 셀에서 아래 그래프의 matplotlib 생성 함수를 정의하고 바로 실행한다.
def event_evidence(k):
    from IPython.display import HTML
    from html import escape
    data=raw[k].set_index('MEAS_DT');pieces=[]
    for e in LOT_EVENTS[k]:
        t=e['point'];pieces.append('<h4>'+escape(e['tag']+' '+e['title'])+'</h4>')
        if e['kind']=='CP':
            base=data['CP-Z4'].median()
            frame=pd.DataFrame([{'피크':f"({e['tag'][1]}-{number})",'확인 시각':p.PEAK,'CP 원본':base+p.DEV,'LOT 중앙값':base,'편차':p.DEV,'사건 폭[초]':(p.B-p.A)*60} for number,p in enumerate(e['pulses'].itertuples(),1)])
            pieces.append(frame.to_html(index=False,float_format=lambda v:f'{v:.4f}'))
        elif e['kind']=='Z':
            d=zstats[k,e['sensor']].loc[e['minute']]
            frame=pd.DataFrame([{'신호':e['sensor'],'1분창 종료':t,'현재 중앙값':d.X,'이전 중앙값':d.MEDIAN,'SD(하한 적용)':d.SD,'Z':d.Z}])
            pieces.append(frame.to_html(index=False,float_format=lambda v:f'{v:.4f}'))
        zone=degree_zone(e['sensor'])
        sensors=SIGNALS  # 대조 그림에서 생략한 신호·나머지 구간도 22신호를 모두 수치로 확인한다.
        windows=[('표시 전 10~5분',t-pd.Timedelta(minutes=10),t-pd.Timedelta(minutes=5)),('표시 전후 5분',t-pd.Timedelta(minutes=5),t+pd.Timedelta(minutes=5)),('표시 후 5~10분',t+pd.Timedelta(minutes=5),t+pd.Timedelta(minutes=10))]
        evidence=[]
        for s in sensors:
            rec={'신호':s,'단위':display_unit(s)}
            for label,a,b in windows:
                y=data.loc[(data.index>=a)&(data.index<b),s].dropna()
                rec[label+' 중앙값']=y.median()*(1e10 if s=='CP-Z4M' else 1);rec[label+' 기록수']=len(y)
            evidence.append(rec)
        pieces.append(pd.DataFrame(evidence).to_html(index=False,float_format=lambda v:f'{v:.4f}'))
    display(HTML('<details><summary><b>구간별 선정 수치와 전·주변·후 원본 중앙값 펼치기</b></summary><p>각 창의 유효 기록수도 표시한다. 값이 없는 창은 판정 보류이며, 이 표는 정비 효과 검증표가 아니다.</p>'+''.join(pieces)+'</details>'))

event_evidence(240001)

**④ 이 LOT과 관련 WO의 감지·착수·완료를 대조한다.** 정기수리 경계 LOT은 위 공통 일정으로 연결하고, 실제 작업·감지 대조가 가능한 LOT만 추가 그림을 그린다. 관측이 없는 단계를 정상이나 개선으로 채우지 않는다.

In [ ]:
# 이 셀에서 아래 그래프의 matplotlib 생성 함수를 정의하고 바로 실행한다.
def phase_sensors(k):
    if k in [240091,240114,240121]:return [['TC-Z7','TC-Z8'],['OP-Z7','OP-Z8'],['CP-Z4']]
    evs=LOT_EVENTS[k];z=next((degree_zone(e['sensor']) for e in evs if e['kind']=='Z'),3) or 3
    return [[f'TC-Z{z}'],[f'OP-Z{z}'],['CP-Z4']]

def draw_lot_maintenance(k):
    w=primary_work(k);phases=LOT_PHASES[k].copy();ws=lot_works(k);r=lot_meta.loc[k];sensors=phase_sensors(k)
    if k not in [240069,240117,240133]:
        if k in [240058,240061,240114]:
            anchor='repair-june' if k in [240058,240061] else 'repair-november'
            note(f'**수리 일정 대조:** [공통 정기수리 일정·전후 원본](#{anchor})에서 이 LOT을 확인한다. 이 LOT 관측 중 실제 작업·감지 교집합은 없다. 전환 형태를 정비 발생 또는 고장으로 표시하지 않는다.')
        else:
            note(f'**정비와의 관계:** 관측 중 실제 작업·감지 교집합이 없다. 가장 가까운 {w.MNT_TYPE} {w.WO_NO} 감지 {w.DETC_DT:%m/%d %H:%M}, 작업 {w.STRT_DT:%m/%d %H:%M}–{w.END_DT:%m/%d %H:%M}. 근접했다는 이유만으로 연결하지 않는다. 비교 가능한 전후가 없어 빈 그림은 생성하지 않는다.')
        show_table(pd.DataFrame([{'WO':t.WO_NO,'유형':t.MNT_TYPE,'감지':t.DETC_DT,'착수':t.STRT_DT,'완료':t.END_DT,'내용':remark_text(t)} for t in ws.itertuples()]))
        return
    if k in [240069,240133]:phases['착수 전']=raw[k].copy()
    plotted={key:g for key,g in phases.items() if len(g)}
    labels={}
    for phase,g in phases.items():
        if len(g):
            keys=g.LOT_NO.unique();labels[phase]=' / '.join(f'{int(key)} {lot_meta.loc[int(key),"GRADE"]}' for key in keys)
        else:labels[phase]='근접 관측 없음'
    rows=[{'panels':[('LOT 관측창·실제 WO·REMARK',[])],'height':1.5}]
    for sigs in sensors:
        rows.append({'panels':[(f'{phase} · {labels[phase]}',sigs) for phase in plotted],'height':1.4})
    fig,cells=report_page(f'{k} | ④ 정비 감지·착수·완료 전후 · {w.MNT_TYPE} {w.WO_NO[-5:]}',
        '실제 관측이 있는 단계만 비교한다. 작업 행 위에는 REMARK, 파형 부근에는 실제 특이점과 시각을 표시한다.',rows,width=22,event_key=True,info=[f'기준 WO {w.WO_NO} · REMARK: {remark_text(w)} · 실제 작업 {w.STRT_DT:%m/%d %H:%M}–{w.END_DT:%m/%d %H:%M}'])
    ax,head,_=cells[0][0];work_clock(ax,head,k,expanded=True);add_work_remarks(ax,ws)
    for i,sigs in enumerate(sensors,1):
        populated=[]
        for (phase,g),(ax,head,_) in zip(plotted.items(),cells[i]):
            if not len(g):
                ax.axis('off');ax.text(.5,.5,'실제 관측 없음\n작업 경계 ±72시간 안에\n비교할 센서 기록이 없음',transform=ax.transAxes,ha='center',va='center',fontsize=11,color=GREY,linespacing=1.7)
                continue
            for kk,gg in g.groupby('LOT_NO'):
                gg=gg.set_index('MEAS_DT').sort_index()
                for s in sigs:
                    y=gg[s].asfreq('s')
                    ax.plot(y.index,y.to_numpy()*(1e10 if s=='CP-Z4M' else 1),color=sensor_color(s),lw=1.1,label=s)
            for e in LOT_EVENTS[k]:
                if g.MEAS_DT.min()<=e['point']<=g.MEAS_DT.max():ax.axvline(e['point'],color=e['color'],ls=EVENT_LINES[e['kind']],lw=1.15)
            clock_ticks(ax,g.MEAS_DT.min(),g.MEAS_DT.max());ax.set_ylabel(display_unit(sigs[0]));panel_legend(head,ax);populated.append(ax)
        if populated:
            low=min(ax.get_ylim()[0] for ax in populated);high=max(ax.get_ylim()[1] for ax in populated)
            for ax in populated:ax.set_ylim(low-(high-low)*.5,high+(high-low)*.5)
    for row,sigs in zip(cells[1:],sensors):
        for (phase,g),(ax,head,_) in zip(plotted.items(),row):
            if not len(g):continue
            events=cp_real.loc[cp_real.LOT.eq(k)&cp_real.PEAK.between(g.MEAS_DT.min(),g.MEAS_DT.max())]
            if 'CP-Z4' in sigs and len(events):
                ev=events.loc[events.DEV.abs().idxmax()];value=raw[k]['CP-Z4'].median()+ev.DEV
                near_note(ax,ev.PEAK,value,f'{ev.PEAK:%H:%M:%S} CP {ev.DEV:+.3f}',EVENT_COLORS['CP'])
            elif k==240133 and 'CP-Z4' in sigs and g.MEAS_DT.min()<=w.DETC_DT<=g.MEAS_DT.max():
                value=raw_window(k,'CP-Z4',w.DETC_DT,w.DETC_DT).iloc[0]
                near_note(ax,w.DETC_DT,value,f'{w.DETC_DT:%H:%M} 감지\n{remark_text(w)}',EVENT_COLORS['감지'])
    if k!=240117:fig._folded=f'{k}: REMARK·실제 작업 시각과 관측 가능한 전후 확인 펼치기'
    emit(fig,f'03_상세/04_정비대조/{k}_WO전중후.png')
    show_table(pd.DataFrame([{'WO':t.WO_NO,'유형':t.MNT_TYPE,'감지 DETC':t.DETC_DT,'착수 STRT':t.STRT_DT,'완료 END':t.END_DT,'정지[분]':t.DOWN_MIN,'REMARK':remark_text(t),'교체 기록':' / '.join(parts_for(t.WO_NO)) or '없음'} for t in ws.itertuples()]))
    records=[];tc,op=sensors[0][0],sensors[1][0]
    for phase,g in phases.items():
        records.append({'구분':phase,'관측 LOT·강종':labels[phase],'관측 시작':g.MEAS_DT.min(),'관측 끝':g.MEAS_DT.max(),'원본 행':len(g),
                        f'{tc} 중앙값[℃]':g[tc].median(),f'{op} 중앙값[%]':g[op].median(),'CP 중앙값':g['CP-Z4'].median(),'CP 최저':g['CP-Z4'].min(),'CP 최고':g['CP-Z4'].max()})
    show_table(pd.DataFrame(records).set_index('구분').T.reset_index().rename(columns={'index':'비교 항목'}))
    direct=cal_work.loc[cal_work.STRT_DT.lt(r.END)&cal_work.END_DT.gt(r.START)]
    det=cal_work.loc[cal_work.DETC_DT.between(r.START,r.END)]
    if len(direct):
        overlap=sum(b-a for a,b in work_ranges[k]);fact=f'실제 작업 {len(direct)} WO가 이 LOT과 {overlap:.1f}분(합집합) 겹친다.'
    elif len(det):fact=f'감지 {len(det)}건은 관측 중이지만 실제 작업 교집합은 없다. 착수는 관측 종료 뒤다.'
    else:fact=f'이 LOT에는 실제 작업·감지의 교집합이 없다. 선택 WO와 가장 가까운 기록 간격은 {wo_distance(w,r)/3600:.1f}시간이다.'
    available=[phase for phase,g in phases.items() if len(g)]
    if len(phases['착수 전']) and len(phases['완료 후']):
        before,after=phases['착수 전'],phases['완료 후'];dtc=after[tc].median()-before[tc].median();dop=after[op].median()-before[op].median()
        grades={lot_meta.loc[int(key),'GRADE'] for key in pd.concat([before,after]).LOT_NO.unique()}
        difference=f'완료 후-착수 전 중앙값은 {tc} {dtc:+.2f}℃, {op} {dop:+.2f}%p다. '+('전후 강종이 달라 수리 효과로 비교할 수 없다.' if len(grades)>1 else '같은 강종이어도 설정·운전 상태가 같다는 증거는 없어 수리 효과를 확정하지 않는다.')
    else:difference=('관측 가능한 단계는 '+', '.join(available)+'뿐이다. ' if available else '작업 경계 72시간 안의 전·중·후 관측이 모두 없다. ')+'자료가 없는 단계를 가정해 회복·개선을 판정할 수 없다.'
    note('**정비 전후 확인:** '+fact+' **관측된 양상:** '+difference)
    if k==240117:
        note('**주의:** CBM 착수 전 45분에도 TBM 01859가 진행 중이다. 정비가 없는 정상 기준과의 비교가 아니다.')
        cbm=ws.loc[ws.WO_NO.eq('WO-2024-01854')].iloc[0];e=cp_real.loc[cp_real.LOT.eq(k)].iloc[0]
        note(f'**이 LOT의 핵심:** CP 최저점 {e.PEAK:%H:%M:%S}은 CBM 완료 {cbm.END_DT:%H:%M} 후 {(e.PEAK-cbm.END_DT).total_seconds()/60:.2f}분이다. 완료 후 패널의 하강을 확인할 수 있으나, 해당 CBM을 유발한 사전 경보는 아니다.')
    if k==240069:note('**이 LOT의 핵심:** 반복 CP는 이후 BM 감지보다 앞서지만 BM 작업 뒤 관측이 없다. REMARK의 절연 저하와 CP의 원인 연결, 조치 효과는 확인되지 않는다.')
    if k==240133:note('**이 LOT의 핵심:** DETC 15:54는 관측 중, 실제 작업 16:54부터는 관측 종료 16:44 뒤다. 셀 교체 후 CP·온도는 제공되지 않아 효과 비교가 불가능하다.')

def lot_conclusion(k):
    evs=LOT_EVENTS[k];cp=sum(len(e['pulses']) for e in evs if e['kind']=='CP');z=[e for e in evs if e['kind']=='Z'];tr=[e for e in evs if e['kind']=='전환']
    lines=[]
    if cp:lines.append(f'CP {cp}개 사건의 원본 이탈 확인')
    if z:lines.append('국소 기준을 벗어난 '+', '.join(e['sensor'] for e in z)+' 반응 확인')
    if tr:lines.append('수리 전후 하강·승온은 운전 전환 가설을 우선 검토')
    if k in LEVEL_TARGETS:lines.append('같은 강종 수준 차이 확인, 정상 기준 방향·운전 조건 추가 확인')
    if not lines:lines.append('현재 CP·온도 Z 기준으로 추가 고장 근거를 선정하지 않음')
    note('**이 LOT의 판정:** '+' / '.join(lines)+'. 고장 확정이나 정상 보증으로 쓰지 않는다. 정비 관련 판단은 바로 위 WO 교집합·전후 관측 유무를 따른다.')

draw_lot_maintenance(240001)
lot_conclusion(240001)
lot_execution.append({'LOT':240001,'확대 구간':len(LOT_EVENTS[240001]),'강종 대조':240001 in LEVEL_TARGETS,'WO 전중후':'완료'})

<a id="lot-240024"></a>
### 8.2 LOT 240024 · SPFH590 · 03/07

**① 전체 위치 확인** → **② 핵심 변화 확대 → 같은 시각 공정 대조 → 나머지 구간** → **③ 강종 수준 차이(해당 LOT)** → **④ 관련 WO 전·중·후 관측과 판정**

In [ ]:
# draw_lot_overview 생성 함수 정의: 8.1 전체 LOT 코드. 같은 레이아웃을 재사용한다.
draw_lot_overview(240024)

**② 모든 확인 구간을 각각 검토했다.** 아래 펼침 블록에서 (a)(b)(c) 각각을 같은 크기로 확인한다. 매 구간은 주된 신호 → 같은 시각 다른 공정 → 정비 REMARK → 해석으로 이어진다. 최대값 하나로 나머지 구간을 정상 처리하지 않는다.

In [ ]:
# draw_lot_zoom 생성 함수 정의: 8.1 통합 확대 코드. 같은 레이아웃을 재사용한다.
draw_lot_zoom(240024)

**240024 구간 검토:** 모든 표시 구간을 같은 크기의 그림·해석으로 확인했다. 접힌 상세에서 주된 근거·다른 공정의 동반 반응·정비 REMARK·복귀 여부를 각각 대조한다.

In [ ]:
# event_evidence 생성 함수 정의: 8.1 사건 근거 코드. 같은 레이아웃을 재사용한다.
event_evidence(240024)

**④ 이 LOT과 관련 WO의 감지·착수·완료를 대조한다.** 정기수리 경계 LOT은 위 공통 일정으로 연결하고, 실제 작업·감지 대조가 가능한 LOT만 추가 그림을 그린다. 관측이 없는 단계를 정상이나 개선으로 채우지 않는다.

In [ ]:
# draw_lot_maintenance 생성 함수 정의: 8.1 정비 대조 코드. 같은 레이아웃을 재사용한다.
draw_lot_maintenance(240024)
lot_conclusion(240024)
lot_execution.append({'LOT':240024,'확대 구간':len(LOT_EVENTS[240024]),'강종 대조':240024 in LEVEL_TARGETS,'WO 전중후':'완료'})

<a id="lot-240037"></a>
### 8.3 LOT 240037 · SPCE · 04/13

**① 전체 위치 확인** → **② 핵심 변화 확대 → 같은 시각 공정 대조 → 나머지 구간** → **③ 강종 수준 차이(해당 LOT)** → **④ 관련 WO 전·중·후 관측과 판정**

In [ ]:
# draw_lot_overview 생성 함수 정의: 8.1 전체 LOT 코드. 같은 레이아웃을 재사용한다.
draw_lot_overview(240037)

**② 모든 확인 구간을 각각 검토했다.** 아래 펼침 블록에서 (a)(b)(c) 각각을 같은 크기로 확인한다. 매 구간은 주된 신호 → 같은 시각 다른 공정 → 정비 REMARK → 해석으로 이어진다. 최대값 하나로 나머지 구간을 정상 처리하지 않는다.

In [ ]:
# draw_lot_zoom 생성 함수 정의: 8.1 통합 확대 코드. 같은 레이아웃을 재사용한다.
draw_lot_zoom(240037)

**240037 구간 검토:** 모든 표시 구간을 같은 크기의 그림·해석으로 확인했다. 접힌 상세에서 주된 근거·다른 공정의 동반 반응·정비 REMARK·복귀 여부를 각각 대조한다.

In [ ]:
# event_evidence 생성 함수 정의: 8.1 사건 근거 코드. 같은 레이아웃을 재사용한다.
event_evidence(240037)

**④ 이 LOT과 관련 WO의 감지·착수·완료를 대조한다.** 정기수리 경계 LOT은 위 공통 일정으로 연결하고, 실제 작업·감지 대조가 가능한 LOT만 추가 그림을 그린다. 관측이 없는 단계를 정상이나 개선으로 채우지 않는다.

In [ ]:
# draw_lot_maintenance 생성 함수 정의: 8.1 정비 대조 코드. 같은 레이아웃을 재사용한다.
draw_lot_maintenance(240037)
lot_conclusion(240037)
lot_execution.append({'LOT':240037,'확대 구간':len(LOT_EVENTS[240037]),'강종 대조':240037 in LEVEL_TARGETS,'WO 전중후':'완료'})

<a id="lot-240038"></a>
### 8.4 LOT 240038 · SPCEN · 04/16

**① 전체 위치 확인** → **② 핵심 변화 확대 → 같은 시각 공정 대조 → 나머지 구간** → **③ 강종 수준 차이(해당 LOT)** → **④ 관련 WO 전·중·후 관측과 판정**

In [ ]:
# draw_lot_overview 생성 함수 정의: 8.1 전체 LOT 코드. 같은 레이아웃을 재사용한다.
draw_lot_overview(240038)

**② 모든 확인 구간을 각각 검토했다.** 아래 펼침 블록에서 (a)(b)(c) 각각을 같은 크기로 확인한다. 매 구간은 주된 신호 → 같은 시각 다른 공정 → 정비 REMARK → 해석으로 이어진다. 최대값 하나로 나머지 구간을 정상 처리하지 않는다.

In [ ]:
# draw_lot_zoom 생성 함수 정의: 8.1 통합 확대 코드. 같은 레이아웃을 재사용한다.
draw_lot_zoom(240038)

**240038 구간 검토:** 모든 표시 구간을 같은 크기의 그림·해석으로 확인했다. 접힌 상세에서 주된 근거·다른 공정의 동반 반응·정비 REMARK·복귀 여부를 각각 대조한다.

In [ ]:
# event_evidence 생성 함수 정의: 8.1 사건 근거 코드. 같은 레이아웃을 재사용한다.
event_evidence(240038)

**③ (L) 같은 강종의 수준 차이.** 시간 펄스와 달리 수리 전후 승온·하강을 제외한 유효 구간만 대표 수준을 비교한다. 축 아래 회색 표식은 제외한 전환이며, 남은 차이도 고장 표시가 아니다.

In [ ]:
# 이 셀에서 아래 그래프의 matplotlib 생성 함수를 정의하고 바로 실행한다.
def draw_lot_level(k):
    if k not in LEVEL_TARGETS:return
    grade=lot_meta.loc[k,'GRADE'];zones=sorted({degree_zone(s) for s in LEVEL_TARGETS[k]})
    ids=lots.loc[lots.GRADE.eq(grade)].sort_values('START').LOT_NO.astype(int).tolist()
    rows=[]
    for z in zones:
        rows.append({'panels':[(f'TC-Z{z} · 평균·중앙값·IQR',[f'TC-Z{z}']),(f'OP-Z{z} · 평균·중앙값·IQR',[f'OP-Z{z}'])],'height':1.8})
        rows.append({'panels':[(f'TC-Z{z} · 전환 제외 실제 추세',[f'TC-Z{z}']),(f'OP-Z{z} · 전환 제외 실제 추세',[f'OP-Z{z}'])],'height':1.5})
    fig,cells=report_page(f'{k} · {grade} | ③ 같은 강종 분포 → 실제 추세',
        '신호별 상자와 실제 추세를 함께 본다. 파랑=이 LOT · 회색=다른 LOT. 수리 전환은 온도값 대신 축 아래 회색 표식으로 표시한다.',rows,
        info=['IQR 수염 밖 점은 해당 LOT 내부 분포 기준이다. LOT 간 수준 차이는 같은 강종의 상자 위치·중앙값으로 따로 확인한다.',
              f'상세 신호: 기존 16신호 차이에서 |TC| ≥ {LEVEL_TC_LIMIT:g}℃ 또는 |OP| ≥ {LEVEL_OP_LIMIT:g}%p인 신호와 대응 TC·OP. 고장 한계가 아님.'])
    for i,z in enumerate(zones):
        for ax,head,sensors in cells[2*i]:draw_boxes(ax,head,sensors[0],ids,highlight=k)
        for ax,head,sensors in cells[2*i+1]:
            sensor=sensors[0]
            for kk in ids:
                y=profile[kk][(sensor,'median')].where(comparison_states[kk].eq('안정 참고')&profile[kk][(sensor,'count')].ge(54))
                ax.plot(y.index,y,color='#086788' if kk==k else '#8496a5',lw=1.5 if kk==k else .9,label=str(kk))
                excluded=profile[kk].index[comparison_states[kk].ne('안정 참고')]
                ax.plot(excluded,np.full(len(excluded),.025),transform=ax.get_xaxis_transform(),color='#bec7ce',marker='|',ms=3,lw=0)
            limits=iqr_summary.loc[iqr_summary.LOT.eq(k)&iqr_summary.SENSOR.eq(sensor)].iloc[0]
            y=profile[k][(sensor,'median')];mask=iqr_masks[k,sensor]
            ax.scatter(y.index[mask],y[mask],s=10,color='#086788',zorder=5)
            ax.set_ylabel(display_unit(sensor));ax.set_xlabel('각 LOT 관측 시작 후 경과시간[분]');ax.xaxis.set_major_locator(MaxNLocator(5));panel_legend(head,ax)
    fig._folded=f'{k} {grade}: 동일 강종 분포·IQR·실제 추세와 전환 제외 범위 펼치기'
    emit(fig,f'03_상세/03_강종비교/{k}_{grade}_수준대조.png')
    table(iqr_summary.loc[iqr_summary.LOT.isin(ids)])
    note('**이 그림에서 확인할 것:** 다른 LOT보다 상자 전체가 이동했는지, 상자 폭이 커졌는지, 일부 1분창만 수염 밖인지 구분한다. 같은 강종이어도 설정·부하·보정이 같다는 증거는 없어 차이를 고장으로 확정하지 않는다.')

draw_lot_level(240038)

<details><summary><b>같은 강종 비교의 선정 신호·제외 범위·판정 펼치기</b></summary>

**같은 강종에서 무엇을 비교했고, 왜 이 신호를 확대했는가?**

같은 강종 대조 LOT이 있는 경우 22개 신호의 LOT 중앙값 차이를 모두 계산했다. 단독 강종은 수준 비교를 보류한다. 수준 비교의 표시 대상은 **TC-Z1–TC-Z8·OP-Z1–OP-Z8 16신호**다. CLN-IN은 LOT 내부 변화, TC-Z9·TC-OUT1·TC-OUT2는 주기·위상, CP는 원본 사건과 계측 정의를 별도로 확인한다. 전체 22신호의 수치는 4.2절 접힌 표에 남겼다.

이 그림은 기존 우선 검토 LOT 240038(SPCEN)의 상세다. 4.3절의 전체 16신호 분포를 확인한 뒤, **절대 차이 2.5℃ 또는 4%p 이상**인 신호와 대응 TC·OP를 아래에 표시했다. 이는 기존 문서의 표시 보조 경계를 적용한 것이며, 통계적 이상값 검정이나 고장 임계값은 아니다.

**표시 경계를 넘은 신호:**

- TC-Z7: 이 LOT 330.187 - 대조 333.233 = **-3.046℃**, 다른 1 LOT과 대조.
- TC-Z8: 이 LOT 330.200 - 대조 333.607 = **-3.407℃**, 다른 1 LOT과 대조.
- OP-Z4: 이 LOT 70.460 - 대조 53.331 = **+17.129%p**, 다른 1 LOT과 대조.
- OP-Z5: 이 LOT 58.724 - 대조 54.505 = **+4.219%p**, 다른 1 LOT과 대조.
- OP-Z7: 이 LOT 71.180 - 대조 66.910 = **+4.270%p**, 다른 1 LOT과 대조.

**대응 신호로 함께 보여 준 것:**

- TC-Z4: 차이 -0.058℃. 자체 차이는 표시 경계 미만이며, 같은 구간의 TC–OP 관계를 보기 위해 포함했다.
- TC-Z5: 차이 -0.148℃. 자체 차이는 표시 경계 미만이며, 같은 구간의 TC–OP 관계를 보기 위해 포함했다.
- OP-Z8: 차이 +3.022%p. 자체 차이는 표시 경계 미만이며, 같은 구간의 TC–OP 관계를 보기 위해 포함했다.

**전환을 제외한 범위:** 승온은 사후 형태 분류를 따른다. 수리 전 TC-Z4·TC-Z5·TC-Z6가 중간 참고보다 모두 2℃ 낮은 상태가 3개 1분창 이어지면 시작부터 이후를 추가 제외했다. 이 2℃도 대조 구간을 고르는 기술 기준이다.

- 240038: 전환 제외 없음.
- 240114: 11/04 20:08–20:59.

**판정:** 상자·평균·중앙값과 실제 추세는 같은 강종에서 관측된 수준 차이를 보여 준다. 운전 조건과 설정이 일치하거나 정상 기준이 확보된 것은 아니므로, 차이가 크다는 사실만으로 고장·센서 이상·에너지 증가로 결론 내리지 않는다.
같은 강종이 두 LOT뿐이므로 차이의 방향은 대칭적이다. 어느 LOT이 정상인지 정할 수 없다.

</details>

**④ 이 LOT과 관련 WO의 감지·착수·완료를 대조한다.** 정기수리 경계 LOT은 위 공통 일정으로 연결하고, 실제 작업·감지 대조가 가능한 LOT만 추가 그림을 그린다. 관측이 없는 단계를 정상이나 개선으로 채우지 않는다.

In [ ]:
# draw_lot_maintenance 생성 함수 정의: 8.1 정비 대조 코드. 같은 레이아웃을 재사용한다.
draw_lot_maintenance(240038)
lot_conclusion(240038)
lot_execution.append({'LOT':240038,'확대 구간':len(LOT_EVENTS[240038]),'강종 대조':240038 in LEVEL_TARGETS,'WO 전중후':'완료'})

<a id="lot-240044"></a>
### 8.5 LOT 240044 · SPHC · 05/03

**① 전체 위치 확인** → **② 핵심 변화 확대 → 같은 시각 공정 대조 → 나머지 구간** → **③ 강종 수준 차이(해당 LOT)** → **④ 관련 WO 전·중·후 관측과 판정**

In [ ]:
# draw_lot_overview 생성 함수 정의: 8.1 전체 LOT 코드. 같은 레이아웃을 재사용한다.
draw_lot_overview(240044)

**② 모든 확인 구간을 각각 검토했다.** 아래 펼침 블록에서 (a)(b)(c) 각각을 같은 크기로 확인한다. 매 구간은 주된 신호 → 같은 시각 다른 공정 → 정비 REMARK → 해석으로 이어진다. 최대값 하나로 나머지 구간을 정상 처리하지 않는다.

In [ ]:
# draw_lot_zoom 생성 함수 정의: 8.1 통합 확대 코드. 같은 레이아웃을 재사용한다.
draw_lot_zoom(240044)

**240044 구간 검토:** 모든 표시 구간을 같은 크기의 그림·해석으로 확인했다. 접힌 상세에서 주된 근거·다른 공정의 동반 반응·정비 REMARK·복귀 여부를 각각 대조한다.

In [ ]:
# event_evidence 생성 함수 정의: 8.1 사건 근거 코드. 같은 레이아웃을 재사용한다.
event_evidence(240044)

**④ 이 LOT과 관련 WO의 감지·착수·완료를 대조한다.** 정기수리 경계 LOT은 위 공통 일정으로 연결하고, 실제 작업·감지 대조가 가능한 LOT만 추가 그림을 그린다. 관측이 없는 단계를 정상이나 개선으로 채우지 않는다.

In [ ]:
# draw_lot_maintenance 생성 함수 정의: 8.1 정비 대조 코드. 같은 레이아웃을 재사용한다.
draw_lot_maintenance(240044)
lot_conclusion(240044)
lot_execution.append({'LOT':240044,'확대 구간':len(LOT_EVENTS[240044]),'강종 대조':240044 in LEVEL_TARGETS,'WO 전중후':'완료'})

<a id="lot-240058"></a>
### 8.6 LOT 240058 · DP590 · 06/09

**① 전체 위치 확인** → **② 핵심 변화 확대 → 같은 시각 공정 대조 → 나머지 구간** → **③ 강종 수준 차이(해당 LOT)** → **④ 관련 WO 전·중·후 관측과 판정**

In [ ]:
# draw_lot_overview 생성 함수 정의: 8.1 전체 LOT 코드. 같은 레이아웃을 재사용한다.
draw_lot_overview(240058)

**② 모든 확인 구간을 각각 검토했다.** 아래 펼침 블록에서 (a)(b)(c) 각각을 같은 크기로 확인한다. 매 구간은 주된 신호 → 같은 시각 다른 공정 → 정비 REMARK → 해석으로 이어진다. 최대값 하나로 나머지 구간을 정상 처리하지 않는다.

In [ ]:
# draw_lot_zoom 생성 함수 정의: 8.1 통합 확대 코드. 같은 레이아웃을 재사용한다.
draw_lot_zoom(240058)

**240058 구간 검토:** 모든 표시 구간을 같은 크기의 그림·해석으로 확인했다. 접힌 상세에서 주된 근거·다른 공정의 동반 반응·정비 REMARK·복귀 여부를 각각 대조한다.

In [ ]:
# event_evidence 생성 함수 정의: 8.1 사건 근거 코드. 같은 레이아웃을 재사용한다.
event_evidence(240058)

**④ 이 LOT과 관련 WO의 감지·착수·완료를 대조한다.** 정기수리 경계 LOT은 위 공통 일정으로 연결하고, 실제 작업·감지 대조가 가능한 LOT만 추가 그림을 그린다. 관측이 없는 단계를 정상이나 개선으로 채우지 않는다.

In [ ]:
# draw_lot_maintenance 생성 함수 정의: 8.1 정비 대조 코드. 같은 레이아웃을 재사용한다.
draw_lot_maintenance(240058)
lot_conclusion(240058)
lot_execution.append({'LOT':240058,'확대 구간':len(LOT_EVENTS[240058]),'강종 대조':240058 in LEVEL_TARGETS,'WO 전중후':'완료'})

<a id="lot-240061"></a>
### 8.7 LOT 240061 · SPHC · 06/15

**① 전체 위치 확인** → **② 핵심 변화 확대 → 같은 시각 공정 대조 → 나머지 구간** → **③ 강종 수준 차이(해당 LOT)** → **④ 관련 WO 전·중·후 관측과 판정**

In [ ]:
# draw_lot_overview 생성 함수 정의: 8.1 전체 LOT 코드. 같은 레이아웃을 재사용한다.
draw_lot_overview(240061)

**② 모든 확인 구간을 각각 검토했다.** 아래 펼침 블록에서 (a)(b)(c) 각각을 같은 크기로 확인한다. 매 구간은 주된 신호 → 같은 시각 다른 공정 → 정비 REMARK → 해석으로 이어진다. 최대값 하나로 나머지 구간을 정상 처리하지 않는다.

In [ ]:
# draw_lot_zoom 생성 함수 정의: 8.1 통합 확대 코드. 같은 레이아웃을 재사용한다.
draw_lot_zoom(240061)

**240061 구간 검토:** 모든 표시 구간을 같은 크기의 그림·해석으로 확인했다. 접힌 상세에서 주된 근거·다른 공정의 동반 반응·정비 REMARK·복귀 여부를 각각 대조한다.

In [ ]:
# event_evidence 생성 함수 정의: 8.1 사건 근거 코드. 같은 레이아웃을 재사용한다.
event_evidence(240061)

**③ (L) 같은 강종의 수준 차이.** 시간 펄스와 달리 수리 전후 승온·하강을 제외한 유효 구간만 대표 수준을 비교한다. 축 아래 회색 표식은 제외한 전환이며, 남은 차이도 고장 표시가 아니다.

In [ ]:
# draw_lot_level 생성 함수 정의: 8.4 동일 강종 코드. 같은 레이아웃을 재사용한다.
draw_lot_level(240061)

<details><summary><b>같은 강종 비교의 선정 신호·제외 범위·판정 펼치기</b></summary>

**같은 강종에서 무엇을 비교했고, 왜 이 신호를 확대했는가?**

같은 강종 대조 LOT이 있는 경우 22개 신호의 LOT 중앙값 차이를 모두 계산했다. 단독 강종은 수준 비교를 보류한다. 수준 비교의 표시 대상은 **TC-Z1–TC-Z8·OP-Z1–OP-Z8 16신호**다. CLN-IN은 LOT 내부 변화, TC-Z9·TC-OUT1·TC-OUT2는 주기·위상, CP는 원본 사건과 계측 정의를 별도로 확인한다. 전체 22신호의 수치는 4.2절 접힌 표에 남겼다.

이 그림은 기존 우선 검토 LOT 240061(SPHC)의 상세다. 4.3절의 전체 16신호 분포를 확인한 뒤, **절대 차이 2.5℃ 또는 4%p 이상**인 신호와 대응 TC·OP를 아래에 표시했다. 이는 기존 문서의 표시 보조 경계를 적용한 것이며, 통계적 이상값 검정이나 고장 임계값은 아니다.

**표시 경계를 넘은 신호:**

- OP-Z4: 이 LOT 54.132 - 대조 42.121 = **+12.011%p**, 다른 3 LOT과 대조.
- OP-Z8: 이 LOT 62.137 - 대조 55.684 = **+6.454%p**, 다른 3 LOT과 대조.

**대응 신호로 함께 보여 준 것:**

- TC-Z4: 차이 +0.145℃. 자체 차이는 표시 경계 미만이며, 같은 구간의 TC–OP 관계를 보기 위해 포함했다.
- TC-Z8: 차이 -0.583℃. 자체 차이는 표시 경계 미만이며, 같은 구간의 TC–OP 관계를 보기 위해 포함했다.

**이 사례의 핵심:** OP-Z4의 차이가 커서 TC-Z4를 함께 본 것이다. TC-Z4 자체를 이상 온도로 선별한 것이 아니다. OP-Z8도 표시 경계를 넘으므로 TC-Z8·OP-Z8을 함께 표시했다. 비슷한 온도 수준에서 출력 수준이 다르게 관측되는 이유를 설정·속도·두께·부하·보정 기록으로 확인해야 한다.

**전환을 제외한 범위:** 승온은 사후 형태 분류를 따른다. 수리 전 TC-Z4·TC-Z5·TC-Z6가 중간 참고보다 모두 2℃ 낮은 상태가 3개 1분창 이어지면 시작부터 이후를 추가 제외했다. 이 2℃도 대조 구간을 고르는 기술 기준이다.

- 240044: 전환 제외 없음.
- 240061: 06/15 01:04–02:40.
- 240069: 전환 제외 없음.
- 240091: 전환 제외 없음.

**판정:** 상자·평균·중앙값과 실제 추세는 같은 강종에서 관측된 수준 차이를 보여 준다. 운전 조건과 설정이 일치하거나 정상 기준이 확보된 것은 아니므로, 차이가 크다는 사실만으로 고장·센서 이상·에너지 증가로 결론 내리지 않는다.

</details>

**④ 이 LOT과 관련 WO의 감지·착수·완료를 대조한다.** 정기수리 경계 LOT은 위 공통 일정으로 연결하고, 실제 작업·감지 대조가 가능한 LOT만 추가 그림을 그린다. 관측이 없는 단계를 정상이나 개선으로 채우지 않는다.

In [ ]:
# draw_lot_maintenance 생성 함수 정의: 8.1 정비 대조 코드. 같은 레이아웃을 재사용한다.
draw_lot_maintenance(240061)
lot_conclusion(240061)
lot_execution.append({'LOT':240061,'확대 구간':len(LOT_EVENTS[240061]),'강종 대조':240061 in LEVEL_TARGETS,'WO 전중후':'완료'})

<a id="lot-240069"></a>
### 8.8 LOT 240069 · SPHC · 07/07

**① 전체 위치 확인** → **② 핵심 변화 확대 → 같은 시각 공정 대조 → 나머지 구간** → **③ 강종 수준 차이(해당 LOT)** → **④ 관련 WO 전·중·후 관측과 판정**

In [ ]:
# draw_lot_overview 생성 함수 정의: 8.1 전체 LOT 코드. 같은 레이아웃을 재사용한다.
draw_lot_overview(240069)

**② 모든 확인 구간을 각각 검토했다.** 아래 펼침 블록에서 (a)(b)(c) 각각을 같은 크기로 확인한다. 매 구간은 주된 신호 → 같은 시각 다른 공정 → 정비 REMARK → 해석으로 이어진다. 최대값 하나로 나머지 구간을 정상 처리하지 않는다.

In [ ]:
# draw_lot_zoom 생성 함수 정의: 8.1 통합 확대 코드. 같은 레이아웃을 재사용한다.
draw_lot_zoom(240069)

**240069 구간 검토:** 모든 표시 구간을 같은 크기의 그림·해석으로 확인했다. 접힌 상세에서 주된 근거·다른 공정의 동반 반응·정비 REMARK·복귀 여부를 각각 대조한다.

In [ ]:
# event_evidence 생성 함수 정의: 8.1 사건 근거 코드. 같은 레이아웃을 재사용한다.
event_evidence(240069)

**④ 이 LOT과 관련 WO의 감지·착수·완료를 대조한다.** 정기수리 경계 LOT은 위 공통 일정으로 연결하고, 실제 작업·감지 대조가 가능한 LOT만 추가 그림을 그린다. 관측이 없는 단계를 정상이나 개선으로 채우지 않는다.

In [ ]:
# draw_lot_maintenance 생성 함수 정의: 8.1 정비 대조 코드. 같은 레이아웃을 재사용한다.
draw_lot_maintenance(240069)
lot_conclusion(240069)
lot_execution.append({'LOT':240069,'확대 구간':len(LOT_EVENTS[240069]),'강종 대조':240069 in LEVEL_TARGETS,'WO 전중후':'완료'})

<a id="lot-240091"></a>
### 8.9 LOT 240091 · SPHC · 09/08

**① 전체 위치 확인** → **② 핵심 변화 확대 → 같은 시각 공정 대조 → 나머지 구간** → **③ 강종 수준 차이(해당 LOT)** → **④ 관련 WO 전·중·후 관측과 판정**

In [ ]:
# draw_lot_overview 생성 함수 정의: 8.1 전체 LOT 코드. 같은 레이아웃을 재사용한다.
draw_lot_overview(240091)

**② 모든 확인 구간을 각각 검토했다.** 아래 펼침 블록에서 (a)(b)(c) 각각을 같은 크기로 확인한다. 매 구간은 주된 신호 → 같은 시각 다른 공정 → 정비 REMARK → 해석으로 이어진다. 최대값 하나로 나머지 구간을 정상 처리하지 않는다.

In [ ]:
# draw_lot_zoom 생성 함수 정의: 8.1 통합 확대 코드. 같은 레이아웃을 재사용한다.
draw_lot_zoom(240091)

**240091 구간 검토:** 모든 표시 구간을 같은 크기의 그림·해석으로 확인했다. 접힌 상세에서 주된 근거·다른 공정의 동반 반응·정비 REMARK·복귀 여부를 각각 대조한다.

In [ ]:
# event_evidence 생성 함수 정의: 8.1 사건 근거 코드. 같은 레이아웃을 재사용한다.
event_evidence(240091)

**③ (L) 같은 강종의 수준 차이.** 시간 펄스와 달리 수리 전후 승온·하강을 제외한 유효 구간만 대표 수준을 비교한다. 축 아래 회색 표식은 제외한 전환이며, 남은 차이도 고장 표시가 아니다.

In [ ]:
# draw_lot_level 생성 함수 정의: 8.4 동일 강종 코드. 같은 레이아웃을 재사용한다.
draw_lot_level(240091)

<details><summary><b>같은 강종 비교의 선정 신호·제외 범위·판정 펼치기</b></summary>

**같은 강종에서 무엇을 비교했고, 왜 이 신호를 확대했는가?**

같은 강종 대조 LOT이 있는 경우 22개 신호의 LOT 중앙값 차이를 모두 계산했다. 단독 강종은 수준 비교를 보류한다. 수준 비교의 표시 대상은 **TC-Z1–TC-Z8·OP-Z1–OP-Z8 16신호**다. CLN-IN은 LOT 내부 변화, TC-Z9·TC-OUT1·TC-OUT2는 주기·위상, CP는 원본 사건과 계측 정의를 별도로 확인한다. 전체 22신호의 수치는 4.2절 접힌 표에 남겼다.

이 그림은 기존 우선 검토 LOT 240091(SPHC)의 상세다. 4.3절의 전체 16신호 분포를 확인한 뒤, **절대 차이 2.5℃ 또는 4%p 이상**인 신호와 대응 TC·OP를 아래에 표시했다. 이는 기존 문서의 표시 보조 경계를 적용한 것이며, 통계적 이상값 검정이나 고장 임계값은 아니다.

**표시 경계를 넘은 신호:**

- TC-Z7: 이 LOT 332.964 - 대조 329.807 = **+3.157℃**, 다른 3 LOT과 대조.
- TC-Z8: 이 LOT 333.295 - 대조 330.183 = **+3.112℃**, 다른 3 LOT과 대조.

**대응 신호로 함께 보여 준 것:**

- OP-Z7: 차이 -1.554%p. 자체 차이는 표시 경계 미만이며, 같은 구간의 TC–OP 관계를 보기 위해 포함했다.
- OP-Z8: 차이 -0.764%p. 자체 차이는 표시 경계 미만이며, 같은 구간의 TC–OP 관계를 보기 위해 포함했다.

**전환을 제외한 범위:** 승온은 사후 형태 분류를 따른다. 수리 전 TC-Z4·TC-Z5·TC-Z6가 중간 참고보다 모두 2℃ 낮은 상태가 3개 1분창 이어지면 시작부터 이후를 추가 제외했다. 이 2℃도 대조 구간을 고르는 기술 기준이다.

- 240044: 전환 제외 없음.
- 240061: 06/15 01:04–02:40.
- 240069: 전환 제외 없음.
- 240091: 전환 제외 없음.

**판정:** 상자·평균·중앙값과 실제 추세는 같은 강종에서 관측된 수준 차이를 보여 준다. 운전 조건과 설정이 일치하거나 정상 기준이 확보된 것은 아니므로, 차이가 크다는 사실만으로 고장·센서 이상·에너지 증가로 결론 내리지 않는다.

</details>

**④ 이 LOT과 관련 WO의 감지·착수·완료를 대조한다.** 정기수리 경계 LOT은 위 공통 일정으로 연결하고, 실제 작업·감지 대조가 가능한 LOT만 추가 그림을 그린다. 관측이 없는 단계를 정상이나 개선으로 채우지 않는다.

In [ ]:
# draw_lot_maintenance 생성 함수 정의: 8.1 정비 대조 코드. 같은 레이아웃을 재사용한다.
draw_lot_maintenance(240091)
lot_conclusion(240091)
lot_execution.append({'LOT':240091,'확대 구간':len(LOT_EVENTS[240091]),'강종 대조':240091 in LEVEL_TARGETS,'WO 전중후':'완료'})

<a id="lot-240108"></a>
### 8.10 LOT 240108 · SPFH590 · 10/18

**① 전체 위치 확인** → **② 핵심 변화 확대 → 같은 시각 공정 대조 → 나머지 구간** → **③ 강종 수준 차이(해당 LOT)** → **④ 관련 WO 전·중·후 관측과 판정**

In [ ]:
# draw_lot_overview 생성 함수 정의: 8.1 전체 LOT 코드. 같은 레이아웃을 재사용한다.
draw_lot_overview(240108)

**② 모든 확인 구간을 각각 검토했다.** 아래 펼침 블록에서 (a)(b)(c) 각각을 같은 크기로 확인한다. 매 구간은 주된 신호 → 같은 시각 다른 공정 → 정비 REMARK → 해석으로 이어진다. 최대값 하나로 나머지 구간을 정상 처리하지 않는다.

In [ ]:
# draw_lot_zoom 생성 함수 정의: 8.1 통합 확대 코드. 같은 레이아웃을 재사용한다.
draw_lot_zoom(240108)

**240108 구간 검토:** 모든 표시 구간을 같은 크기의 그림·해석으로 확인했다. 접힌 상세에서 주된 근거·다른 공정의 동반 반응·정비 REMARK·복귀 여부를 각각 대조한다.

In [ ]:
# event_evidence 생성 함수 정의: 8.1 사건 근거 코드. 같은 레이아웃을 재사용한다.
event_evidence(240108)

**④ 이 LOT과 관련 WO의 감지·착수·완료를 대조한다.** 정기수리 경계 LOT은 위 공통 일정으로 연결하고, 실제 작업·감지 대조가 가능한 LOT만 추가 그림을 그린다. 관측이 없는 단계를 정상이나 개선으로 채우지 않는다.

In [ ]:
# draw_lot_maintenance 생성 함수 정의: 8.1 정비 대조 코드. 같은 레이아웃을 재사용한다.
draw_lot_maintenance(240108)
lot_conclusion(240108)
lot_execution.append({'LOT':240108,'확대 구간':len(LOT_EVENTS[240108]),'강종 대조':240108 in LEVEL_TARGETS,'WO 전중후':'완료'})

<a id="lot-240114"></a>
### 8.11 LOT 240114 · SPCEN · 11/04

**① 전체 위치 확인** → **② 핵심 변화 확대 → 같은 시각 공정 대조 → 나머지 구간** → **③ 강종 수준 차이(해당 LOT)** → **④ 관련 WO 전·중·후 관측과 판정**

In [ ]:
# draw_lot_overview 생성 함수 정의: 8.1 전체 LOT 코드. 같은 레이아웃을 재사용한다.
draw_lot_overview(240114)

**② 모든 확인 구간을 각각 검토했다.** 아래 펼침 블록에서 (a)(b)(c) 각각을 같은 크기로 확인한다. 매 구간은 주된 신호 → 같은 시각 다른 공정 → 정비 REMARK → 해석으로 이어진다. 최대값 하나로 나머지 구간을 정상 처리하지 않는다.

In [ ]:
# draw_lot_zoom 생성 함수 정의: 8.1 통합 확대 코드. 같은 레이아웃을 재사용한다.
draw_lot_zoom(240114)

**240114 구간 검토:** 모든 표시 구간을 같은 크기의 그림·해석으로 확인했다. 접힌 상세에서 주된 근거·다른 공정의 동반 반응·정비 REMARK·복귀 여부를 각각 대조한다.

In [ ]:
# event_evidence 생성 함수 정의: 8.1 사건 근거 코드. 같은 레이아웃을 재사용한다.
event_evidence(240114)

**③ (L) 같은 강종의 수준 차이.** 시간 펄스와 달리 수리 전후 승온·하강을 제외한 유효 구간만 대표 수준을 비교한다. 축 아래 회색 표식은 제외한 전환이며, 남은 차이도 고장 표시가 아니다.

In [ ]:
# draw_lot_level 생성 함수 정의: 8.4 동일 강종 코드. 같은 레이아웃을 재사용한다.
draw_lot_level(240114)

<details><summary><b>같은 강종 비교의 선정 신호·제외 범위·판정 펼치기</b></summary>

**같은 강종에서 무엇을 비교했고, 왜 이 신호를 확대했는가?**

같은 강종 대조 LOT이 있는 경우 22개 신호의 LOT 중앙값 차이를 모두 계산했다. 단독 강종은 수준 비교를 보류한다. 수준 비교의 표시 대상은 **TC-Z1–TC-Z8·OP-Z1–OP-Z8 16신호**다. CLN-IN은 LOT 내부 변화, TC-Z9·TC-OUT1·TC-OUT2는 주기·위상, CP는 원본 사건과 계측 정의를 별도로 확인한다. 전체 22신호의 수치는 4.2절 접힌 표에 남겼다.

이 그림은 기존 우선 검토 LOT 240114(SPCEN)의 상세다. 4.3절의 전체 16신호 분포를 확인한 뒤, **절대 차이 2.5℃ 또는 4%p 이상**인 신호와 대응 TC·OP를 아래에 표시했다. 이는 기존 문서의 표시 보조 경계를 적용한 것이며, 통계적 이상값 검정이나 고장 임계값은 아니다.

**표시 경계를 넘은 신호:**

- TC-Z7: 이 LOT 333.233 - 대조 330.187 = **+3.046℃**, 다른 1 LOT과 대조.
- TC-Z8: 이 LOT 333.607 - 대조 330.200 = **+3.407℃**, 다른 1 LOT과 대조.
- OP-Z4: 이 LOT 53.331 - 대조 70.460 = **-17.129%p**, 다른 1 LOT과 대조.
- OP-Z5: 이 LOT 54.505 - 대조 58.724 = **-4.219%p**, 다른 1 LOT과 대조.
- OP-Z7: 이 LOT 66.910 - 대조 71.180 = **-4.270%p**, 다른 1 LOT과 대조.

**대응 신호로 함께 보여 준 것:**

- TC-Z4: 차이 +0.058℃. 자체 차이는 표시 경계 미만이며, 같은 구간의 TC–OP 관계를 보기 위해 포함했다.
- TC-Z5: 차이 +0.148℃. 자체 차이는 표시 경계 미만이며, 같은 구간의 TC–OP 관계를 보기 위해 포함했다.
- OP-Z8: 차이 -3.022%p. 자체 차이는 표시 경계 미만이며, 같은 구간의 TC–OP 관계를 보기 위해 포함했다.

**전환을 제외한 범위:** 승온은 사후 형태 분류를 따른다. 수리 전 TC-Z4·TC-Z5·TC-Z6가 중간 참고보다 모두 2℃ 낮은 상태가 3개 1분창 이어지면 시작부터 이후를 추가 제외했다. 이 2℃도 대조 구간을 고르는 기술 기준이다.

- 240038: 전환 제외 없음.
- 240114: 11/04 20:08–20:59.

**판정:** 상자·평균·중앙값과 실제 추세는 같은 강종에서 관측된 수준 차이를 보여 준다. 운전 조건과 설정이 일치하거나 정상 기준이 확보된 것은 아니므로, 차이가 크다는 사실만으로 고장·센서 이상·에너지 증가로 결론 내리지 않는다.
같은 강종이 두 LOT뿐이므로 차이의 방향은 대칭적이다. 어느 LOT이 정상인지 정할 수 없다.

</details>

**④ 이 LOT과 관련 WO의 감지·착수·완료를 대조한다.** 정기수리 경계 LOT은 위 공통 일정으로 연결하고, 실제 작업·감지 대조가 가능한 LOT만 추가 그림을 그린다. 관측이 없는 단계를 정상이나 개선으로 채우지 않는다.

In [ ]:
# draw_lot_maintenance 생성 함수 정의: 8.1 정비 대조 코드. 같은 레이아웃을 재사용한다.
draw_lot_maintenance(240114)
lot_conclusion(240114)
lot_execution.append({'LOT':240114,'확대 구간':len(LOT_EVENTS[240114]),'강종 대조':240114 in LEVEL_TARGETS,'WO 전중후':'완료'})

<a id="lot-240117"></a>
### 8.12 LOT 240117 · SPCD · 11/13

**① 전체 위치 확인** → **② 핵심 변화 확대 → 같은 시각 공정 대조 → 나머지 구간** → **③ 강종 수준 차이(해당 LOT)** → **④ 관련 WO 전·중·후 관측과 판정**

정비 대조의 전·후는 선택한 **CBM 01854** 기준이다. 착수 전 패널에도 **TBM 01859가 진행 중**이므로 이 구간을 정비가 없는 정상 기준으로 해석하지 않는다.

In [ ]:
# draw_lot_overview 생성 함수 정의: 8.1 전체 LOT 코드. 같은 레이아웃을 재사용한다.
draw_lot_overview(240117)

**② 모든 확인 구간을 각각 검토했다.** 아래 펼침 블록에서 (a)(b)(c) 각각을 같은 크기로 확인한다. 매 구간은 주된 신호 → 같은 시각 다른 공정 → 정비 REMARK → 해석으로 이어진다. 최대값 하나로 나머지 구간을 정상 처리하지 않는다.

In [ ]:
# draw_lot_zoom 생성 함수 정의: 8.1 통합 확대 코드. 같은 레이아웃을 재사용한다.
draw_lot_zoom(240117)

**240117 구간 검토:** 모든 표시 구간을 같은 크기의 그림·해석으로 확인했다. 접힌 상세에서 주된 근거·다른 공정의 동반 반응·정비 REMARK·복귀 여부를 각각 대조한다.

In [ ]:
# event_evidence 생성 함수 정의: 8.1 사건 근거 코드. 같은 레이아웃을 재사용한다.
event_evidence(240117)

**④ 이 LOT과 관련 WO의 감지·착수·완료를 대조한다.** 정기수리 경계 LOT은 위 공통 일정으로 연결하고, 실제 작업·감지 대조가 가능한 LOT만 추가 그림을 그린다. 관측이 없는 단계를 정상이나 개선으로 채우지 않는다.

In [ ]:
# draw_lot_maintenance 생성 함수 정의: 8.1 정비 대조 코드. 같은 레이아웃을 재사용한다.
draw_lot_maintenance(240117)
lot_conclusion(240117)
lot_execution.append({'LOT':240117,'확대 구간':len(LOT_EVENTS[240117]),'강종 대조':240117 in LEVEL_TARGETS,'WO 전중후':'완료'})

<a id="lot-240121"></a>
### 8.13 LOT 240121 · SPCE · 11/24

**① 전체 위치 확인** → **② 핵심 변화 확대 → 같은 시각 공정 대조 → 나머지 구간** → **③ 강종 수준 차이(해당 LOT)** → **④ 관련 WO 전·중·후 관측과 판정**

In [ ]:
# draw_lot_overview 생성 함수 정의: 8.1 전체 LOT 코드. 같은 레이아웃을 재사용한다.
draw_lot_overview(240121)

**② 모든 확인 구간을 각각 검토했다.** 아래 펼침 블록에서 (a)(b)(c) 각각을 같은 크기로 확인한다. 매 구간은 주된 신호 → 같은 시각 다른 공정 → 정비 REMARK → 해석으로 이어진다. 최대값 하나로 나머지 구간을 정상 처리하지 않는다.

In [ ]:
# draw_lot_zoom 생성 함수 정의: 8.1 통합 확대 코드. 같은 레이아웃을 재사용한다.
draw_lot_zoom(240121)

**240121 구간 검토:** 모든 표시 구간을 같은 크기의 그림·해석으로 확인했다. 접힌 상세에서 주된 근거·다른 공정의 동반 반응·정비 REMARK·복귀 여부를 각각 대조한다.

In [ ]:
# event_evidence 생성 함수 정의: 8.1 사건 근거 코드. 같은 레이아웃을 재사용한다.
event_evidence(240121)

**③ (L) 같은 강종의 수준 차이.** 시간 펄스와 달리 수리 전후 승온·하강을 제외한 유효 구간만 대표 수준을 비교한다. 축 아래 회색 표식은 제외한 전환이며, 남은 차이도 고장 표시가 아니다.

In [ ]:
# draw_lot_level 생성 함수 정의: 8.4 동일 강종 코드. 같은 레이아웃을 재사용한다.
draw_lot_level(240121)

<details><summary><b>같은 강종 비교의 선정 신호·제외 범위·판정 펼치기</b></summary>

**같은 강종에서 무엇을 비교했고, 왜 이 신호를 확대했는가?**

같은 강종 대조 LOT이 있는 경우 22개 신호의 LOT 중앙값 차이를 모두 계산했다. 단독 강종은 수준 비교를 보류한다. 수준 비교의 표시 대상은 **TC-Z1–TC-Z8·OP-Z1–OP-Z8 16신호**다. CLN-IN은 LOT 내부 변화, TC-Z9·TC-OUT1·TC-OUT2는 주기·위상, CP는 원본 사건과 계측 정의를 별도로 확인한다. 전체 22신호의 수치는 4.2절 접힌 표에 남겼다.

이 그림은 기존 우선 검토 LOT 240121(SPCE)의 상세다. 4.3절의 전체 16신호 분포를 확인한 뒤, **절대 차이 2.5℃ 또는 4%p 이상**인 신호와 대응 TC·OP를 아래에 표시했다. 이는 기존 문서의 표시 보조 경계를 적용한 것이며, 통계적 이상값 검정이나 고장 임계값은 아니다.

**표시 경계를 넘은 신호:**

- TC-Z7: 이 LOT 333.308 - 대조 330.184 = **+3.124℃**, 다른 2 LOT과 대조.
- TC-Z8: 이 LOT 333.532 - 대조 330.528 = **+3.004℃**, 다른 2 LOT과 대조.
- OP-Z4: 이 LOT 56.849 - 대조 52.775 = **+4.073%p**, 다른 2 LOT과 대조.

**대응 신호로 함께 보여 준 것:**

- TC-Z4: 차이 -0.024℃. 자체 차이는 표시 경계 미만이며, 같은 구간의 TC–OP 관계를 보기 위해 포함했다.
- OP-Z7: 차이 -1.294%p. 자체 차이는 표시 경계 미만이며, 같은 구간의 TC–OP 관계를 보기 위해 포함했다.
- OP-Z8: 차이 -0.032%p. 자체 차이는 표시 경계 미만이며, 같은 구간의 TC–OP 관계를 보기 위해 포함했다.

**전환을 제외한 범위:** 승온은 사후 형태 분류를 따른다. 수리 전 TC-Z4·TC-Z5·TC-Z6가 중간 참고보다 모두 2℃ 낮은 상태가 3개 1분창 이어지면 시작부터 이후를 추가 제외했다. 이 2℃도 대조 구간을 고르는 기술 기준이다.

- 240037: 전환 제외 없음.
- 240121: 전환 제외 없음.
- 240133: 전환 제외 없음.

**판정:** 상자·평균·중앙값과 실제 추세는 같은 강종에서 관측된 수준 차이를 보여 준다. 운전 조건과 설정이 일치하거나 정상 기준이 확보된 것은 아니므로, 차이가 크다는 사실만으로 고장·센서 이상·에너지 증가로 결론 내리지 않는다.

</details>

**④ 이 LOT과 관련 WO의 감지·착수·완료를 대조한다.** 정기수리 경계 LOT은 위 공통 일정으로 연결하고, 실제 작업·감지 대조가 가능한 LOT만 추가 그림을 그린다. 관측이 없는 단계를 정상이나 개선으로 채우지 않는다.

In [ ]:
# draw_lot_maintenance 생성 함수 정의: 8.1 정비 대조 코드. 같은 레이아웃을 재사용한다.
draw_lot_maintenance(240121)
lot_conclusion(240121)
lot_execution.append({'LOT':240121,'확대 구간':len(LOT_EVENTS[240121]),'강종 대조':240121 in LEVEL_TARGETS,'WO 전중후':'완료'})

<a id="lot-240133"></a>
### 8.14 LOT 240133 · SPCE · 12/28

**① 전체 위치 확인** → **② 핵심 변화 확대 → 같은 시각 공정 대조 → 나머지 구간** → **③ 강종 수준 차이(해당 LOT)** → **④ 관련 WO 전·중·후 관측과 판정**

In [ ]:
# draw_lot_overview 생성 함수 정의: 8.1 전체 LOT 코드. 같은 레이아웃을 재사용한다.
draw_lot_overview(240133)

**② 모든 확인 구간을 각각 검토했다.** 아래 펼침 블록에서 (a)(b)(c) 각각을 같은 크기로 확인한다. 매 구간은 주된 신호 → 같은 시각 다른 공정 → 정비 REMARK → 해석으로 이어진다. 최대값 하나로 나머지 구간을 정상 처리하지 않는다.

In [ ]:
# draw_lot_zoom 생성 함수 정의: 8.1 통합 확대 코드. 같은 레이아웃을 재사용한다.
draw_lot_zoom(240133)

**240133 구간 검토:** 모든 표시 구간을 같은 크기의 그림·해석으로 확인했다. 접힌 상세에서 주된 근거·다른 공정의 동반 반응·정비 REMARK·복귀 여부를 각각 대조한다.

In [ ]:
# event_evidence 생성 함수 정의: 8.1 사건 근거 코드. 같은 레이아웃을 재사용한다.
event_evidence(240133)

**④ 이 LOT과 관련 WO의 감지·착수·완료를 대조한다.** 정기수리 경계 LOT은 위 공통 일정으로 연결하고, 실제 작업·감지 대조가 가능한 LOT만 추가 그림을 그린다. 관측이 없는 단계를 정상이나 개선으로 채우지 않는다.

In [ ]:
# draw_lot_maintenance 생성 함수 정의: 8.1 정비 대조 코드. 같은 레이아웃을 재사용한다.
draw_lot_maintenance(240133)
lot_conclusion(240133)
lot_execution.append({'LOT':240133,'확대 구간':len(LOT_EVENTS[240133]),'강종 대조':240133 in LEVEL_TARGETS,'WO 전중후':'완료'})

### 8.15 월별 관측창·44 WO 전체 기록

In [ ]:
from matplotlib.patches import Rectangle

months = pd.Index(range(1,13),name="MONTH")
monthly = pd.DataFrame(index=months)
monthly["온도 LOT[개]"] = lots.groupby(lots.START.dt.month).size().reindex(months,fill_value=0)
monthly["온도 관측창 합계[시간]"] = lots.groupby(lots.START.dt.month).MINUTES.sum().reindex(months,fill_value=0)/60
monthly["CAL01 작업[WO]"] = cal_work.groupby(cal_work.STRT_DT.dt.month).size().reindex(months,fill_value=0)
monthly["CR1 강종변경[건]"] = grade_evt.groupby(grade_evt.EVT_DT.dt.month).size().reindex(months,fill_value=0)
for kind in ["BM","CBM","TBM"]:
    work = cal_work.loc[cal_work.MNT_TYPE.eq(kind)]
    monthly[f"{kind} 작업[WO]"] = work.groupby(work.STRT_DT.dt.month).size().reindex(months,fill_value=0)

# 작업 구간을 LOT 관측창으로 자르고, 여러 WO가 동시에 진행된 시간은 한 번만 센다.
def observed_work_minutes(start,end,work):
    intervals = sorted((max(start,t.STRT_DT),min(end,t.END_DT)) for t in work.itertuples())
    total = 0.0
    current_end = start
    for left,right in intervals:
        left = max(left,current_end)
        if right > left:
            total += (right-left).total_seconds()/60
            current_end = right
    return total

links = []
for r in lots.itertuples():
    work = cal_work.loc[cal_work.STRT_DT.lt(r.END) & cal_work.END_DT.gt(r.START)]
    links.append({"LOT":r.LOT_NO,"MONTH":r.START.month,
                  "온도·작업 겹친 LOT[개]":int(len(work)>0),"온도·작업 겹친 WO[개]":len(work),
                  "온도·작업 겹침[분]":observed_work_minutes(r.START,r.END,work)})
monthly_links = pd.DataFrame(links)
for column in monthly_links.columns[2:]:
    monthly[column] = monthly_links.groupby("MONTH")[column].sum().reindex(months,fill_value=0)

# 월별 건수 대신 실제 관측창과 정비 REMARK를 보여준다.
# START–END는 제공된 기록의 범위이며, 실제 가동시간을 뜻하지 않는다.
GRADE_COLOR_MONTHLY = report_viz.GRADE_COLORS.copy()


def monthly_clock_range(start, end, seconds=False):
    clock = "%H:%M:%S" if seconds else "%H:%M"
    left = start.strftime("%m/%d " + clock)
    right = end.strftime(clock if start.date() == end.date() else "%m/%d " + clock)
    return f"{left}–{right}"


def monthly_part_records(wo_no):
    records = cal_parts.loc[cal_parts.WO_NO.eq(wo_no)]
    return [f"{r.PART_NM} [{r.PART_CD}] ×{r.QTY:g} ({r.CHG_RSN})"
            for r in records.itertuples()]


# 같은 월이라는 이유로 연결하지 않는다. 작업 구간과 LOT 관측창을 직접 대조한다.
work_observation_links = []
detection_observation_links = []
for task in cal_work.itertuples():
    for lot in lots.itertuples():
        left, right = max(task.STRT_DT, lot.START), min(task.END_DT, lot.END)
        if right > left:
            work_observation_links.append({
                "WO_NO":task.WO_NO, "LOT":int(lot.LOT_NO), "TYPE":task.MNT_TYPE,
                "START":left, "END":right, "MINUTES":(right-left).total_seconds()/60})
        if lot.START <= task.DETC_DT <= lot.END:
            detection_observation_links.append({
                "WO_NO":task.WO_NO, "LOT":int(lot.LOT_NO), "DETC_DT":task.DETC_DT,
                "WORK_START":task.STRT_DT})
work_observation_links = pd.DataFrame(work_observation_links)
detection_observation_links = pd.DataFrame(detection_observation_links)

# 각 월의 내용 길이에 맞춰 높이를 잡는다. 숫자 집계와 해석 문장은 그림에서 뺀다.
month_cards = []
for month in months:
    records = []
    month_lots = lots.loc[lots.START.dt.month.eq(month)]
    month_work = cal_work.loc[cal_work.STRT_DT.dt.month.eq(month)]
    month_plans = plans.loc[plans.STRT_DT.dt.month.eq(month)]
    for plan in month_plans.itertuples():
        records.append(("plan", f"계획수리 {plan.STRT_DT:%m/%d}–{plan.END_DT:%m/%d}", "#087f9c"))
    records.append(("section", "온도 · 제공된 LOT 관측창", "#17344d"))
    for lot in month_lots.itertuples():
        records.append(("lot", f"{lot.LOT_NO} {lot.GRADE} · {monthly_clock_range(lot.START,lot.END)}",
                        GRADE_COLOR_MONTHLY[lot.GRADE]))
    if month_lots.empty:
        records.append(("empty", "제공된 온도 기록 없음", "#6c7c89"))
    records.append(("space", "", ""))
    records.append(("section", "정비 · CAL01 작업 기록", "#17344d"))
    for task in month_work.itertuples():
        remark = str(task.REMARK).strip() if pd.notna(task.REMARK) else "작업 내용 미기재"
        short_wo = task.WO_NO.rsplit("-",1)[-1]
        records.append(("work", f"{task.STRT_DT:%m/%d} {task.MNT_TYPE} {short_wo} · {remark}",
                        COL[task.MNT_TYPE]))
        for part_record in monthly_part_records(task.WO_NO):
            records.append(("part", "교체: " + part_record, COL["PART"]))
        observed = work_observation_links.loc[work_observation_links.WO_NO.eq(task.WO_NO)]
        for link in observed.itertuples():
            records.append(("focus", f"LOT {link.LOT} 관측 중 · {link.START:%H:%M}–{link.END:%H:%M} · {link.MINUTES:g}분",
                            "#7540a8"))
        detected = detection_observation_links.loc[detection_observation_links.WO_NO.eq(task.WO_NO)]
        if observed.empty:
            for link in detected.itertuples():
                records.append(("focus", f"LOT {link.LOT} 감지 시각 관측 · {link.DETC_DT:%H:%M}", "#34495e"))
    if month_work.empty:
        records.append(("empty", "제공된 정비 작업 기록 없음", "#6c7c89"))
    card_height = 48 + sum(10 if kind=="space" else 23 for kind,_,_ in records)
    month_cards.append({"month":month,"records":records,"height":card_height})

row_heights = [max(card["height"] for card in month_cards[row*3:(row+1)*3]) for row in range(4)]
header_height, row_gap, margin_bottom = 90, 16, 16
figure_height = (header_height + sum(row_heights) + 3*row_gap + margin_bottom)/72
fig = plt.figure(figsize=(24,figure_height),facecolor="white")
fig.suptitle("월별 관측창·정비 작업 내용",x=.027,y=1-18/(72*figure_height),
             ha="left",va="top",fontsize=23,fontweight="bold",color="#17344d")
fig.text(.027,1-52/(72*figure_height),
         "온도: LOT별 관측 날짜·시각  |  정비: 작업일·유형·WO·REMARK  |  교체 부품은 기록이 있는 작업에만 표시",
         ha="left",va="top",fontsize=12.8,color="#526d80")
fig.text(.973,1-25/(72*figure_height),"2024년 · 실제 기록 시각",ha="right",va="top",
         fontsize=12.5,color="#526d80")
row_top = header_height
left_margin, right_margin, column_gap = .027, .027, .019
card_width = (1-left_margin-right_margin-2*column_gap)/3
for row in range(4):
    for column in range(3):
        card = month_cards[row*3+column]
        height = row_heights[row]
        ax = fig.add_axes([left_margin+column*(card_width+column_gap),
                           1-(row_top+height)/(72*figure_height),
                           card_width,height/(72*figure_height)])
        ax.set(xlim=(0,1),ylim=(height,0),xticks=[],yticks=[])
        ax.spines[:].set_visible(False)
        focused = card["month"]==11
        ax.add_patch(Rectangle((0,0),1,height,facecolor="#fafcfd",
                               edgecolor="#7540a8" if focused else "#dce5eb",lw=1.4))
        if card["month"] in [6,11]:
            ax.add_patch(Rectangle((0,0),1,35,facecolor=COL["PLAN"],alpha=.35,edgecolor="none"))
        ax.text(.025,12,f"{card['month']:02d}월",ha="left",va="top",fontsize=17,
                color="#17344d",fontweight="bold")
        cursor = 45
        for kind,text,color in card["records"]:
            if kind=="space":
                cursor += 10
                continue
            fontsize = 11.6 if kind=="part" else 12.4 if kind=="focus" else 12.8
            x = .045 if kind in ["part","focus"] else .025
            artist = ax.text(x,cursor,text,ha="left",va="top",fontsize=fontsize,color=color,
                             fontweight="bold" if kind in ["section","focus"] else "normal")
            artist._monthly_card = card["month"]
            cursor += 23
    row_top += height + row_gap

# 줄을 접거나 글씨를 줄이지 않고, 각 월의 글자가 칸 안에 들어가는지 검사한다.
fig.set_dpi(160)
fig.canvas.draw()
renderer = fig.canvas.get_renderer()
for ax in fig.axes:
    bounds = ax.get_window_extent(renderer)
    for text in ax.texts:
        box = text.get_window_extent(renderer)
        assert bounds.contains(box.x0,box.y0) and bounds.contains(box.x1,box.y1), (
            f"월별 칸의 글자 잘림: {text.get_text()}")
fig._folded='월별·전체 WO 원기록과 시각 대조 펼치기'
finish(fig,"01_개요/03_월별_기록비교.png", save=True)   # 현재 노트북에서 새로 그려 저장한다.

# 정확한 초 단위의 관측창과 WO별 작업 시간을 표로 보존한다.
observation_rows = []
for lot in lots.itertuples():
    work_links = work_observation_links.loc[work_observation_links.LOT.eq(lot.LOT_NO)]
    detection_links = detection_observation_links.loc[detection_observation_links.LOT.eq(lot.LOT_NO)]
    observation_rows.append({
        "월":f"{lot.START.month:02d}월","LOT":int(lot.LOT_NO),"강종":lot.GRADE,
        "첫 측정":lot.START,"마지막 측정":lot.END,"제공된 측정 기록[행]":lot.N_ROWS,
        "관측 중 작업":", ".join(work_links.WO_NO),
        "관측 중 감지 기록":", ".join(detection_links.WO_NO)})
observation_windows = pd.DataFrame(observation_rows)
maintenance_rows = []
for task in cal_work.itertuples():
    observed = work_observation_links.loc[work_observation_links.WO_NO.eq(task.WO_NO)]
    maintenance_rows.append({
        "월":f"{task.STRT_DT.month:02d}월","유형":task.MNT_TYPE,"WO_NO":task.WO_NO,
        "감지 기록 시각":task.DETC_DT,"작업 시작":task.STRT_DT,"작업 완료":task.END_DT,
        "기록된 작업 내용":str(task.REMARK).strip() if pd.notna(task.REMARK) else "작업 내용 미기재",
        "기록된 교체 부품":"; ".join(monthly_part_records(task.WO_NO)),
        "온도 관측 중 작업":"; ".join(
            f"LOT {r.LOT} · {monthly_clock_range(r.START,r.END,seconds=True)} · {r.MINUTES:g}분"
            for r in observed.itertuples())})
monthly_maintenance_details = pd.DataFrame(maintenance_rows)
table(observation_windows)
table(monthly_maintenance_details)

**관찰:** 6월·11월은 계획수리 전후에 관측창이 놓여 있다. 11월 13일에는 작업 중 온도가 기록되지만, 12월 28일에는 감지 시각에만 온도가 기록된다.

**가설 검토:** ‘같은 달의 온도와 부품 교체는 직접 연결된다’는 가설은 시각 교집합으로 제한한다. 12월 셀 교체 뒤 온도·CP는 제공 자료에 없어 교체 효과를 검증할 수 없다.

**판단 범위:** 작업 내용은 REMARK 원문, 교체는 동일 WO의 기록을 사용했다. 작업 중 온도가 있다는 사실이 설비의 계속 가동이나 해당 부품의 실제 교체 순간을 뜻하지 않는다.

### 8.16 CAL01 44 WO의 감지 → 착수 → 완료

**읽을 질문:** 변화는 감지 전인가, 대기 중인가, 작업 중인가, 완료 뒤인가?

In [ ]:
# CAL01 작업 44건을 '그림이 들어간 표'로 보인다: 한 줄 = 작업 1건, 월별로 묶고 칸을 맞춘다.
# 시간 막대는 모양만(6시간 넘는 대기는 //로 끊음) · 정확한 시간은 숫자 칸에서 읽는다.
PAD_H = 72                                              # 가까운 LOT으로 볼 시간
TYPE_C = {"BM": report_viz.MAINTENANCE_COLORS["BM"], "CBM": report_viz.MAINTENANCE_COLORS["CBM"], "TBM": report_viz.MAINTENANCE_COLORS["TBM"]}
w = cal_work.sort_values("DETC_DT").reset_index(drop=True)
w["감지→착수[h]"] = (w.STRT_DT - w.DETC_DT).dt.total_seconds() / 3600
w["작업 시간[h]"] = (w.END_DT - w.STRT_DT).dt.total_seconds() / 3600
parts_of = cal_parts.groupby("WO_NO").apply(lambda d: "\n".join(f"{r.PART_NM} ×{int(r.QTY)} ({r.CHG_RSN})" for r in d.itertuples()), include_groups=False)
w["교체 부품"] = w.WO_NO.map(parts_of).fillna("")
def near_lot(t):
    """작업 기간과 LOT 관측의 시간 관계(겹침 · 관측 끝 뒤 · 관측 시작 전, 72시간 안)."""
    out = []
    for k, r in lot_meta.iterrows():
        if t.DETC_DT <= r.END and t.END_DT >= r.START: out.append(f"{k} 관측 중")
        elif pd.Timedelta(0) <= t.DETC_DT - r.END <= pd.Timedelta(hours=PAD_H): out.append(f"{k} 관측 끝 {(t.DETC_DT - r.END).total_seconds() / 3600:.0f}h 뒤")
        elif pd.Timedelta(0) <= r.START - t.END_DT <= pd.Timedelta(hours=PAD_H): out.append(f"{k} 관측 시작 {(r.START - t.END_DT).total_seconds() / 3600:.0f}h 전")
    return "\n".join(out)
def temp_record(t):
    """그 작업을 온도로 확인할 수 있는가: 작업 중 기록 / 감지 때만 / 앞뒤 LOT이 72시간 안."""
    if any(t.STRT_DT < r.END and t.END_DT > r.START for _, r in lot_meta.iterrows()): return "작업 중 기록"
    if any(r.START <= t.DETC_DT <= r.END for _, r in lot_meta.iterrows()): return "감지 때만"
    before = lot_meta.END.le(t.DETC_DT) & (t.DETC_DT - lot_meta.END).le(pd.Timedelta(hours=PAD_H))
    after = lot_meta.START.ge(t.END_DT) & (lot_meta.START - t.END_DT).le(pd.Timedelta(hours=PAD_H))
    return "앞뒤 LOT" if before.any() and after.any() else ""
w["가까운 LOT"] = [near_lot(t) for t in w.itertuples()]
w["온도 기록"] = [temp_record(t) for t in w.itertuples()]
plan_of = lambda t: next((p for p in plans.itertuples() if p.STRT_DT <= t < p.END_EXCL), None)
hours = lambda h: f"{h / 24:.1f}일" if h >= 24 else f"{h:.1f}h"

# 줄 목록: 월 머리 줄 + 작업 줄
rows = []
for m, d in w.groupby(w.DETC_DT.dt.month):
    ps = [p for p in plans.itertuples() if p.STRT_DT.month == m]
    rows.append(("month", f"{m}월 · {len(d)}건" + "".join(f"   ·   CR1 정기수리 {p.STRT_DT:%m/%d}~{p.END_DT:%m/%d} ({p.SCOPE})" for p in ps)))
    rows += [("wo", t) for _, t in d.iterrows()]
COLS = [(.0, "감지 시각"), (.072, "종류"), (.112, "WO"), (.165, "◆ 감지 · 점선 = 착수까지 대기 · 막대 = 작업"), (.35, "감지→착수"), (.405, "작업 시간"), (.452, "설비 정지"),
        (.5, "REMARK · 고장 유형"), (.645, "교체 부품"), (.79, "온도 기록"), (.86, "가까운 LOT (±72시간)")]
G0, GW, HMAX = .165, .165, 16                            # 시간 막대 칸 시작·폭, 칸 안에서 보이는 최대 시간
gx = lambda h: G0 + GW * min(h, HMAX) / HMAX
H = .5 * len(rows) + 4.0
fig = plt.figure(figsize=(30, H)); ax = fig.add_axes([.02, 1.0 / H, .97, 1 - 3.5 / H]); ax.axis("off")   # 위 2.5인치는 제목·범례, 아래 1인치는 계산식
ax.set_xlim(0, 1); ax.set_ylim(len(rows) - .5, -1.2)
for x, name in COLS:
    ax.text(x, -.95, name, fontsize=13, fontweight="bold", color="#526d80", va="center")
ax.plot([0, 1], [-.55, -.55], color="#9fb3c3", lw=1.2)
for i, (kind, t) in enumerate(rows):
    if kind == "month":
        ax.add_patch(Rectangle((0, i - .42), 1, .84, color="#e9eff4", lw=0, zorder=0))
        ax.text(.004, i, t, fontsize=13.5, fontweight="bold", color="#17344d", va="center")
        continue
    c = TYPE_C.get(t.MNT_TYPE, "#7c8995")
    if plan_of(t.STRT_DT) is not None:
        ax.add_patch(Rectangle((0, i - .45), 1, .9, color="#f3f7fa", lw=0, zorder=0))
    ax.add_patch(Rectangle((0, i - .4), .003, .8, color=c, lw=0))
    ax.text(.008, i, f"{t.DETC_DT:%m/%d %H:%M}", fontsize=12.5, va="center", color="#17344d")
    ax.text(.072, i, t.MNT_TYPE, fontsize=12, va="center", color="white", fontweight="bold", bbox=dict(boxstyle="round,pad=.25", fc=c, ec="none"))
    ax.text(.112, i, t.WO_NO.replace("WO-2024-", ""), fontsize=12, va="center", color="#526d80")
    wait, work = t["감지→착수[h]"], t["작업 시간[h]"]
    wait_s = wait if wait <= 6 else 7                           # 6시간 넘는 대기는 끊어 그림(//)
    ax.plot([gx(0), gx(wait_s)], [i, i], color=c, ls=":", lw=2)
    if wait > 6:
        ax.text(gx(6.5), i, "//", fontsize=13, color=c, ha="center", va="center", fontweight="bold", bbox=dict(boxstyle="square,pad=.05", fc="white", ec="none"))
    ax.add_patch(Rectangle((gx(wait_s), i - .17), gx(wait_s + min(work, HMAX - wait_s)) - gx(wait_s), .34, color=c, lw=0))
    ax.plot(gx(0), i, "D", ms=7, color=c)
    ax.text(.35, i, hours(wait), fontsize=12.5, va="center", color=c if wait >= 24 else "#17344d", fontweight="bold" if wait >= 24 else "normal")
    ax.text(.405, i, hours(work), fontsize=12.5, va="center", color="#17344d")
    ax.text(.452, i, f"{t.DOWN_MIN:.0f}분", fontsize=12.5, va="center", color="#17344d" if t.DOWN_MIN else "#9aa6ae")
    rem = (str(t.REMARK).strip() if pd.notna(t.REMARK) else "내용 미기재") + (f" · {t.FAIL_MODE}" if pd.notna(t.FAIL_MODE) else "")
    ax.text(.5, i, rem, fontsize=12.5, va="center", color="#17344d" if pd.notna(t.REMARK) else "#9aa6ae")
    if t["교체 부품"]:
        ax.text(.645, i, t["교체 부품"], fontsize=11.5, va="center", color=SELECT_COLOR, fontweight="bold", linespacing=1.15)
    if t["온도 기록"]:
        ax.text(.79, i, t["온도 기록"], fontsize=12, va="center", color="#17344d" if t["온도 기록"] == "작업 중 기록" else "#526d80",
                fontweight="bold" if t["온도 기록"] == "작업 중 기록" else "normal")
    if t["가까운 LOT"]:
        ax.text(.86, i, t["가까운 LOT"], fontsize=11.5, va="center", color="#17344d", linespacing=1.15)
n_part = int(w["교체 부품"].ne("").sum()); n_long = int(w["감지→착수[h]"].ge(24).sum()); n_rec = int(w["온도 기록"].eq("작업 중 기록").sum())
header(fig, f"CAL01 정비 이력 {len(w)}건 · 감지 → 착수 → 완료와 작업 내용",
       f"한 줄 = 작업 1건(감지 순, 월별 묶음) · 감지→착수 중앙값 {w['감지→착수[h]'].median():.1f}h, 1일 넘게 기다린 작업 {n_long}건 · 부품 교체 {n_part}건(빨강) · 작업 중 온도 기록 {n_rec}건 · 연한 바탕 = CR1 정기수리 기간",
       [Patch(facecolor=TYPE_C[k], label=f"{k} {int(w.MNT_TYPE.eq(k).sum())}건") for k in TYPE_C], ncol=3)
compact_header(fig, .975 - .6 / H, .975 - 1.1 / H, title_size=25, sub_size=14, legend_size=13)
fig.text(.02, .25 / H, "감지→착수 = 작업 시작(STRT_DT) - 감지(DETC_DT) · 작업 시간 = 완료(END_DT) - 작업 시작 · 설비 정지 = 기록된 정지 시간(DOWN_MIN) · "
         "온도 기록 = 작업하는 동안 LOT 온도 기록이 있음(작업 중 기록) / 감지 시각에만 있음(감지 때만) / 앞뒤 LOT이 모두 72시간 안(앞뒤 LOT)",
         ha="left", va="bottom", fontsize=12.5, color="#526d80")
axis_tag(fig, "작업 1건 = 1줄 · 감지 순서")
fig._folded='월별·전체 WO 원기록과 시각 대조 펼치기'
finish(fig, "01_개요/04_WO44_작업흐름.png", save=True)   # 현재 노트북에서 새로 그려 저장한다.
table(w[["WO_NO", "MNT_TYPE", "DETC_DT", "STRT_DT", "END_DT", "감지→착수[h]", "작업 시간[h]", "DOWN_MIN", "FAIL_MODE", "REMARK", "교체 부품", "온도 기록", "가까운 LOT"]]
      .rename(columns={"DOWN_MIN": "설비 정지[분]"}).round({"감지→착수[h]": 1, "작업 시간[h]": 1}))

**관찰:** 감지(DETC_DT), 착수(STRT_DT), 완료(END_DT)는 서로 다르다. 6시간을 넘는 대기 막대는 `//`로 축약했으므로 숫자 칸에서 정확한 시간을 읽는다. DOWN_MIN은 기록된 정지 시간으로 작업 시간과 별도다.

**가설 검토:** ‘관측 중 DETC_DT가 있으므로 작업 전후 효과를 볼 수 있다’는 가설은 240133에서 성립하지 않는다. 작업 자체가 관측 이후다. 240117은 실제 작업 중과 완료 뒤를 구분해 볼 수 있다.

**판단 범위:** DETC_DT는 정비 자료의 감지 기록 시각이다. 온도 센서 기반 자동 경보 시각이나 최초 고장 발생 시각으로 단정하지 않는다.

<details><summary><b>전체 판정 근거표 펼치기</b></summary>

### 8.17 추가 판정 근거

| 검토 항목 | 관측으로 확인한 결과 | 현재 판정 |
|---|---|---|
| CP 순간 이탈 | 240037·240069에서 반복, 240117에서 작업 완료 뒤 하강 | 원본 확대·공정 반응 확인 대상. 원인은 미확정 |
| 같은 강종의 출력 수준 | 240038(SPCEN)·240061(SPHC)의 OP-Z4·OP-Z8 등의 차이 | 설정·부하·계측 차이 가설을 남김 |
| 같은 강종의 온도 수준 | 240091·240114·240121의 Z7·Z8이 대조 LOT보다 높음 | 과시효 설정·센서 보정 여부 확인 대상 |
| 수리 전후 큰 변화 | 6월·11월 전후의 하강·승온 형태 | 일정과 부합하는 대안 설명이 있음. 고장 증거로 바로 쓰지 않음 |
| 정비와의 직접 비교 | 실제 작업 교집합은 240117, 감지 시각만 관측된 240133은 작업이 관측 종료 뒤 | 감지·착수·완료를 구분해야 함 |

위 8 LOT은 **선택한 기준으로 더 볼 대상**이다. 나머지를 정상으로 확정한 결과가 아니다. 단독 강종·짧은 관측·관측 공백 때문에 비교할 수 없는 범위가 있다.

</details>

# 이 셀에서 아래 그래프의 matplotlib 생성 함수를 정의하고 바로 실행한다.
def zoom_flow(k):
    # 모든 구간 열에 같은 공정 행을 적용한다. 비교 중 공정 위치가 바뀌지 않는다.
    evs=LOT_EVENTS[k];chosen={2,3,4,5}
    if any(e['sensor']=='CLN-IN' for e in evs):chosen.add(0)
    if any(degree_zone(e['sensor']) in [1,2] for e in evs):chosen.add(1)
    out=[]
    for i,(name,left,right) in enumerate(FLOW_ROWS):
        if i not in chosen:continue
        out.append((name+' · '+('CP' if i==4 else '온도'),left))
        if right:out.append((name+' · '+('CP-Z4M' if i==4 else '출력'),right))
    return out

def lot_result_markdown(k):
    return '**구간별 검토 결과**\n\n'+ '\n\n'.join(event_result_markdown(k,e) for e in LOT_EVENTS[k])+ '\n\n**읽은 뒤 판단:** (a)(b)(c)는 서로 다른 시간·근거다. 최대값 하나로 나머지를 정상 처리하지 않는다. 각 구간의 TC–OP 동반 반응·CP 원본·정비 시각을 위의 동일 크기 상세에서 각각 확인한다.'

def plan_result_markdown(idx,pre,post):
    d=plan_case_data(idx,pre,post);pr,po,ws=d['pre'],d['post'],d['ws']
    tp=expected.loc[expected.LOT.eq(pre)].iloc[0];ta=expected.loc[expected.LOT.eq(post)].iloc[0]
    g=raw[post].set_index('MEAS_DT');a,b=d['zoom_a'],d['zoom_b'];q=g.loc[a:b]
    base=g['CP-Z4'].median();cpdev=(q['CP-Z4']-base).abs().max();t=d['point']
    before=g.loc[t-pd.Timedelta(minutes=10):t-pd.Timedelta(minutes=5),'TC-Z3'].median()
    after=g.loc[t+pd.Timedelta(minutes=5):t+pd.Timedelta(minutes=10),'TC-Z3'].median()
    lines=['**이 그림에서 확인한 사실**','',
      f'1. **수리 기간과 관측을 분리했다.** 계획수리 안에는 온도 0행이다. 위 파란 날짜 범위에서는 작업 WO와 강종 이벤트를 확인하며, 온도는 수리 전(A) {pre}({pr.GRADE})·수리 후(B) {post}({po.GRADE})에서만 관측된다. 전체 범위 실제 작업은 {len(ws)} WO다. 기간 안의 온도·정지 상태를 추정해서 그리지 않았다. ◆는 교체 WO 완료 시각을 대용하며 실제 교체 순간은 아니다.',
      f'2. **일정과 맞는 전환은 먼저 설명한다.** 전 LOT 하강 {tp.시작:%m/%d %H:%M}–{tp.끝:%H:%M}, 후 LOT 승온 {ta.시작:%m/%d %H:%M}–{ta.끝:%H:%M}를 회색으로 표시했다. 여러 TC·OP의 동시 변화와 수리 전후라는 위치는 운전 전환 가설을 뒷받침한다. 사후 형태 분류여서 실제 모드·SV·정지 명령 확인은 남는다.',
      f'3. **수리 후(B)의 남는 응답을 (B-1)로 확대했다.** (B-1) {a:%m/%d %H:%M}–{b:%H:%M}의 TC·OP·CP는 같은 시간축이다. 확인 시각 전 10~5분과 후 5~10분 TC-Z3 중앙값은 {before:.2f}℃ → {after:.2f}℃다. 원본의 복귀·지속과 함께 읽는다. 이 창의 CP 최대 절대 편차는 {cpdev:.4f}다. 국소 응답 근거이며 정상 범위의 검증 자료는 아니다.','',
      '**어떻게 이상이 아닐 가능성을 검토하는가?** 일정과 다신호 동시 하강·승온 → 기준 수준으로 접근·복귀 → 같은 TC–OP의 반복 제어 응답 → 별도 CP 이탈 잔존 여부 순서로 본다. 근거가 있는 전환은 **고장 후보와 분리한 설명 가능 구간**으로 둔다. 구간 전체의 정상 보증으로 바꾸지 않는다. 전환 뒤의 단발 응답·CP 사건도 별도 검토한다.','',
      f'**수리 효과로 결론 내릴 수 없는 이유:** 강종 {pr.GRADE}→{po.GRADE}, 관측 시점과 운전 조건이 다르고 수리 중 온도도 없다. 출력·온도 높낮이를 수리의 개선·악화 효과로 해석하지 않는다.']
    reason=('승온 참고 구간 이후 TC-Z3의 절대 Z가 가장 큰 창을 코드로 선택했다. 모든 온도 또는 전체 수리 효과가 이상이라는 뜻은 아니다.' if post==240061 else 'CBM 완료 뒤 CP 원본 이탈이 남는 창을 선택했다. 온도 최대점만 보고 정한 창이 아니다.')
    lines+=['','**왜 (B-1)을 골랐는가?** '+reason,'',
      '**수리를 했는데 왜 다시 변화가 있는가? — 현재 자료로 나눠 볼 가설**','',
      '- **재가동 전환:** 공통 승온은 재가동 형태로 설명할 수 있어 수준 비교에서 제외했다. (B-1)은 그 이후의 반응을 따로 확인한다.',
      '- **국소 제어 응답:** TC-Z3의 단발 상승·하강과 OP-Z3의 저하·복귀를 같은 시간에 대조한다. 이 동반 반응은 제어 응답 가설과 부합할 수 있지만 설정값·제어 로그가 없어 원인을 확정하지 않는다.',
      '- **정비 대상·원인 불일치:** 캘린더의 정기수리와 WO REMARK는 작업 기록이다. 그 작업이 Z3 제어 또는 CP 변화의 원인을 제거했다는 검증 기록은 없다. 수리 후 관측됐다는 이유만으로 수리 실패·재고장으로 부르지 않는다.',
      '- **추가 검증:** 같은 강종·설정·속도·두께에서 재현되는지, SV 대비 편차와 OP 응답, 정비 대상 부품·계측 보정·완료 점검 결과를 확인해야 원인을 좁힐 수 있다.']
    if post==240117:lines+=['','**11월 확대의 핵심:** CP 하강은 CBM 12:44 완료 후에 발생한다. 승온이라는 설명만으로 이 CP 사건까지 정상 처리하지 않으며, 해당 CBM 전 경보라는 해석도 시간 순서와 맞지 않는다.']
    else:lines+=['','**6월 확대의 핵심:** 승온 이후 Z3의 단발 응답이 있어도 Z4–Z6·CP가 같은 폭으로 지속 이탈하는지 따로 본다. 온도·출력이 돌아오는 모습은 일시 제어 반응의 근거이며, 정상 제어임을 확정하려면 SV·조업 조건이 필요하다.']
    return '\n'.join(lines)

def level_result_markdown(k):
    grade=lot_meta.loc[k,'GRADE'];d=level_check.loc[level_check.LOT.eq(k)]
    hit=d.loc[d.DISPLAY_HIT];zones=sorted({degree_zone(s) for s in hit.SENSOR})
    pairs=[f'{prefix}-Z{z}' for z in zones for prefix in ['TC','OP']]
    companions=d.loc[d.SENSOR.isin(pairs)&~d.DISPLAY_HIT]
    lines=['**같은 강종에서 무엇을 비교했고, 왜 이 신호를 확대했는가?**','',
      '같은 강종 대조 LOT이 있는 경우 22개 신호의 LOT 중앙값 차이를 모두 계산했다. 단독 강종은 수준 비교를 보류한다. 수준 비교의 표시 대상은 **TC-Z1–TC-Z8·OP-Z1–OP-Z8 16신호**다. CLN-IN은 LOT 내부 변화, TC-Z9·TC-OUT1·TC-OUT2는 주기·위상, CP는 원본 사건과 계측 정의를 별도로 확인한다. 전체 22신호의 수치는 4.2절 접힌 표에 남겼다.','',
      f'이 그림은 기존 우선 검토 LOT {k}({grade})의 상세다. 4.3절에서 16신호 분포를 확인한 뒤, **절대 차이 {LEVEL_TC_LIMIT:g}℃ 또는 {LEVEL_OP_LIMIT:g}%p 이상**인 신호와 대응 TC·OP를 아래에 표시했다. 이는 기존 문서의 표시 보조 경계를 적용한 것이며, 통계적 이상값 검정이나 고장 임계값은 아니다.','',
      '**표시 경계를 넘은 신호:**','']
    for r in hit.itertuples():
        u='℃' if r.SENSOR in LEVEL_TC else '%p'
        lines.append(f'- {r.SENSOR}: 이 LOT {r.LOT_MEDIAN:.3f} - 대조 {r.PEER_MEDIAN:.3f} = **{r.GAP:+.3f}{u}**, 다른 {r.N_PEERS} LOT과 대조.')
    if len(companions):
        lines+=['','**대응 신호로 함께 보여 준 것:**','']
        for r in companions.itertuples():
            u='℃' if r.SENSOR in LEVEL_TC else '%p'
            lines.append(f'- {r.SENSOR}: 차이 {r.GAP:+.3f}{u}. 자체 차이는 표시 경계 미만이며, 같은 구간의 TC–OP 관계를 보기 위해 포함했다.')
    if k==240061:
        lines+=['','**이 사례의 핵심:** OP-Z4의 차이가 커서 TC-Z4를 함께 본 것이다. TC-Z4 자체를 이상 온도로 선별한 것이 아니다. OP-Z8도 표시 경계를 넘으므로 TC-Z8·OP-Z8을 함께 표시했다. 비슷한 온도 수준에서 출력 수준이 다르게 관측되는 이유를 설정·속도·두께·부하·보정 기록으로 확인해야 한다.']
    lines+=['','**전환을 제외한 범위:** 승온은 사후 형태 분류를 따른다. 수리 전 TC-Z4·TC-Z5·TC-Z6가 중간 참고보다 모두 2℃ 낮은 상태가 3개 1분창 이어지면 시작부터 이후를 추가 제외했다. 이 2℃도 대조 구간을 고르는 기술 기준이다.','']
    for kk in lots.loc[lots.GRADE.eq(grade),'LOT_NO'].astype(int):
        ranges=runs(comparison_states[kk].ne('안정 참고'))
        spans=' / '.join(f'{at(kk,a):%m/%d %H:%M}–{min(at(kk,b),lot_meta.loc[kk,"END"]):%H:%M}' for a,b in ranges) or '전환 제외 없음'
        lines.append(f'- {kk}: {spans}.')
    lines+=['','**판정:** 위는 같은 강종에서 관측된 수준 차이다. 운전 조건과 설정이 일치하거나 정상 기준이 확보된 것은 아니므로, 차이가 크다는 사실만으로 고장·센서 이상·에너지 증가로 결론 내리지 않는다.']
    if int(d.N_PEERS.min())==1:lines+=['같은 강종이 두 LOT뿐이므로 차이의 방향은 대칭적이다. 어느 LOT이 정상인지 정할 수 없다.']
    return '\n'.join(lines)

<details><summary><b>계산 정의·재현 조건 펼치기</b></summary>

- **온도·출력 추세:** 1분에 54개 이상 실제 기록이 있는 창의 중앙값. 평균과 중앙값은 순간 최고·최저가 아니다.
- **시간순 Z:** 이전 30분 중앙값과 이전 표준편차, 최소 20개 과거 유효창. SD 하한 TC 0.5℃·OP 2%p. `|Z|≥3`은 탐색 표시선이다.
- **IQR:** LOT·신호별 전환 제외 1분 중앙값. 상자는 Q1~Q3, 수염은 1.5 IQR 안 실제 끝값. IQR 0은 판정 보류한다.
- **CP:** LOT 원본 중앙값 대비 절대편차 0.02 초과 묶음 중 최대편차 0.05 이상을 검사한다. 짧은 CP가 1분 요약에서 작아질 수 있어 원본을 유지한다.
- **판단 범위:** CP·Z·IQR은 다른 질문을 위한 탐색 기준이다. 정상 확률·고장 라벨·정비 효과로 합치지 않는다.

</details>

### 실행·산출물 검사

14 LOT 각각의 전체 그림, 표시한 모든 확대, 해당 강종 대조, 두 정기수리 일정 그림과 실제 대조 가능한 WO 그림이 만들어졌는지 확인한다. 새 그림은 저장 파일을 다시 읽지 않고 Figure를 직접 출력한다. 제목·범례·시간 눈금의 실제 경계를 검사한다.

In [ ]:
# focus_event: 바로 위 첫 사용 절의 생성 함수를 재사용한다.
assert len(temp)==345390 and len(lots)==14 and len(grade_evt)==134
assert cal_work.MNT_TYPE.value_counts().to_dict()=={'TBM':22,'BM':12,'CBM':10}
assert set(work_observation_links.LOT)=={240117} and len(work_observation_links)==2
assert np.isclose(sum(b-a for a,b in work_ranges[240117]),304)
assert cp_count.loc[240037]==3 and cp_count.loc[240069]==7
assert set(pd.DataFrame(lot_execution).LOT)==set(lots.LOT_NO.astype(int))
assert plan_compare['수리 중 온도 행'].eq(0).all()
checks=pd.DataFrame(figure_checks)
assert not checks.FILE.duplicated().any()
expected_full={f'03_상세/01_LOT전체/{k}_{lot_meta.loc[k,"GRADE"]}.png' for k in LOT_EVENTS}
expected_zoom={f'03_상세/02_구간확대/{k}_전체구간_공정순서.png' for k in LOT_EVENTS}
expected_wo={f'03_상세/04_정비대조/{k}_WO전중후.png' for k in [240069,240117,240133]}
expected_plans={f'03_상세/04_정비대조/정기수리_{month}_일정_전후_확대.png' for month in ['6월','11월']}
assert expected_plans<=set(checks.FILE)
assert expected_full|expected_zoom|expected_wo<=set(checks.FILE)
assert all((OUT_DIR/name).is_file() for name in checks.FILE)
show_table(pd.DataFrame(lot_execution))
show_table(checks[['FILE','TEXT_OVERLAP','LEGEND_BOUNDS']])
note(f'검증 통과 · 전체 LOT {len(expected_full)}개 / LOT별 통합 확대 {len(expected_zoom)}개 / 강종 수준 대조 {len(LEVEL_TARGETS)}개 / 실제 WO 대조 {len(expected_wo)}개 / 정기수리 일정 2개 / 전체 그림 {len(checks)}개')
assert quality_effect['3이상 후보'].sum()==zscreen.MAX_ABS_Z.ge(Z_LIMIT).sum()
assert len(cp_quality_evidence)==len(cp_real) and peak_counts.ge(54).all()

assert len(expected_zoom)==14 and len(expected_plans)==2
assert len(checks)==61
assert all(len(events)<=3 for events in LOT_EVENTS.values())
for k,events in LOT_EVENTS.items():
    expected_peaks=set(cp_real.loc[cp_real.LOT.eq(k),'PEAK'])
    shown_peaks={p.PEAK for e in events if e['kind']=='CP' for p in e['pulses'].itertuples()}
    assert expected_peaks==shown_peaks, f'CP 사건 누락: {k}'
assert len(screening_result)==14
assert set(screening_result['LOT·강종'].str[:6].astype(int))==set(lots.LOT_NO.astype(int))
assert len(cp_quality_evidence)==14
assert len(level_check)==len(lots)*16
assert set(LEVEL_TARGETS[240061])=={'OP-Z4','OP-Z8'}
assert all((level_check.loc[level_check.LOT.eq(k)&level_check.SENSOR.isin(tags),'DISPLAY_HIT']).all() for k,tags in LEVEL_TARGETS.items())
assert len(zoom_presentations)==sum(len(e) for e in LOT_EVENTS.values())
assert {r['LOT'] for r in zoom_presentations}==set(LOT_EVENTS)
for k,events in LOT_EVENTS.items():
    focus=focus_event(k);cp=[e for e in events if e['kind']=='CP']
    if cp:assert abs(focus['deviation'])==max(abs(e['deviation']) for e in cp)
assert focus_event(240069)['tag']=='(c)' and focus_event(240117)['tag']=='(b)'
assert all(len(lot_cards(k))==3 for k in LOT_EVENTS)

assert {(r['LOT'],r['TAG']) for r in zoom_presentations}=={(k,e['tag']) for k,events in LOT_EVENTS.items() for e in events}
assert len(iqr_summary)==224
assert iqr_summary.OUT.sum()==sum(mask.sum() for mask in iqr_masks.values())

assert set(locator_checks)=={(k,e['tag']) for k,events in LOT_EVENTS.items() for e in events}
assert {(k,tag) for k,tag,sensor in overview_trigger_checks}=={(k,e['tag']) for k,events in LOT_EVENTS.items() for e in events if e['kind'] in ['CP','Z']}

assert set(report_case_checks)=={240069,240037,240117,240061} and len(report_case_checks)==4
assert cp_time_features.SPAN_SEC.ge(0).all()
assert cp_time_features.POINTS.ge(1).all()
assert len(cp_time_features)==14 and len(interval_time_features)==32
assert {f'01_개요/사례_{k}_보고서.png' for k in report_case_checks}<=set(checks.FILE)

assert len(cp69_peak_checks)==7 and set(cp69_peak_checks)==set(cp69_pulses.ID)
assert len(cp69_signalstats)==147 and cp69_signalstats.N_BASE.ge(190).all()
assert cp69_b_compare.CP_DEV.lt(.05).all()
assert cp69_b_compare.iloc[1].X>cp69_b_compare.iloc[0].X and cp69_b_compare.iloc[1].Z<cp69_b_compare.iloc[0].Z

assert len(cp69_zone_metrics)==42 and cp69_zone_metrics.N_PEAK.ge(24).all()
assert len(cp69_zone_peak_checks)==36 and set(cp69_zone_peak_checks)=={(p.ID,s) for p in cp69_pulses.iloc[:6].itertuples() for s in CP69_ZONE_SENSORS}

b_evidence=cp69_b_observations()
assert b_evidence["CP_HITS"]==0 and b_evidence["CP_MAX_DEV"]<.05
assert b_evidence["CP_VALID"]+b_evidence["DATA"]["CP-Z4"].isna().sum()==b_evidence["SECONDS"]
assert b_evidence["POINTS"][1]["TC_AT"]==cp69_later_point

# 관계 분석: 14사건×9신호, 모든 LOT·시차·짝의 실제 기록을 검증한다.
assert len(relation_events)==14 and relation_events.LOT.nunique()==6
assert len(relation_event_delta)==14*9
assert relation_event_delta.loc[relation_event_delta.VALID,'N_BASE'].ge(190).all()
assert relation_event_delta.loc[relation_event_delta.VALID,'N_PEAK'].ge(24).all()
assert relation_lags.loc[relation_lags.R.notna()&relation_lags.PAIR.eq('CP'),'N_PAIRS'].ge(40).all()
assert relation_lags.loc[relation_lags.R.notna()&relation_lags.PAIR.eq('OP_TC'),'N_PAIRS'].ge(200).all()
assert relation_lags.R.dropna().between(-1,1).all()
assert set(relation_lags.loc[relation_lags.PAIR.eq('OP_TC'),'LOT'])==set(raw)
assert len(relation_match_coverage)==14
assert len(relation_matches)==8*relation_match_coverage.STATUS.eq('대조 가능').sum()
assert relation_matches.CONTROL_CP_MAX.le(.02).all() and relation_matches.BASE_TC456_MAX_GAP.le(2.5).all()
assert not relation_matches.duplicated(['LOT','SENSOR','CONTROL_START']).any()
assert {f'05_관계검토/{name}' for name in ['01_CP사건별_Z3-Z6_변화량.png','02_CP하강_주변존_동시변화.png','03_CP이탈_유무_실제파형.png','04_CP두채널_관측관계.png']}<=set(checks.FILE)

assert relation_events.ID.is_unique and set(relation_events.EVENT_ID)==set(cp_time_features.EVENT_ID)

# 확대창: 같은 실제 x축·눈금, 유형색, 실제 Z 1분창을 검증한다.
assert len(zoom_alignment_checks)==sum(len(events) for events in LOT_EVENTS.values())
assert {(x['LOT'],x['TAG']) for x in zoom_alignment_checks}=={(k,e['tag']) for k,events in LOT_EVENTS.items() for e in events}
assert all(x['LINE']==sensor_color(x['SENSOR']) for x in locator_visual_checks)

# 계산용 ID·HTML 앵커는 보존하고 표시만 (a-1) 형식으로 통일한다.
assert event_text('240069-a1')=='240069 (a-1)'
assert event_text('240069-a1 · CP-Z4',include_lot=False)=='(a-1) · CP-Z4'
assert event_text('b-1 · TC-Z3',include_lot=False)=='(b-1) · TC-Z3'
assert event_text('(a-1) CP-Z4')=='(a-1) CP-Z4'
assert event_text('#event-240069-a')=='#event-240069-a'
assert len(display_label_checks)>0 and all(event_text(label,include_lot=False)==label for _,label in display_label_checks)

assert event_text('B-1 수리 후 확인 구간')=='(B-1) 수리 후 확인 구간'

# CP 방향별 실제 파형: 하강 2·상승 12사건, 모든 존, 같은 x축을 확인한다.
assert len(relation_wave_checks)==14 and len(relation_wave_metrics)==14*8
assert {r['ID'] for r in relation_wave_checks}==set(relation_events.ID)
assert sum(r['DIRECTION']=='하강' for r in relation_wave_checks)==2
assert all(r['N_PANELS']==5 and r['SENSORS']==set(REL_SENSORS+['CP-Z4']) for r in relation_wave_checks)
assert not (OUT_DIR/'05_관계검토/02_사건별_시차와_OP_TC.png').exists()

# CP 이탈이 있던 때/없던 다른 때의 실제 파형과 원래 4분 최대폭 일치를 확인한다.
assert len(relation_pair_wave_checks)==12
assert len(relation_pair_amplitude_checks)==12*8*2
assert {v['ID'] for v in relation_pair_wave_checks}==set(relation_matches.ID)
assert all(v['N_PANELS']==9 for v in relation_pair_wave_checks)
assert not (OUT_DIR/'05_관계검토/03_CP사건과_비사건_짝대조.png').exists()

assert '05_관계검토/05_TC_OP_동시파형.png' in set(checks.FILE)
assert len(tcop_zero)==56 and tcop_zero.N_PAIRS.ge(200).all()

# TC–OP 고출력·추가 하강·고정 기준 복귀 계산을 확인한다.
assert "05_관계검토/06_TC_OP_고출력후_하강과복귀.png" in set(checks.FILE)
assert len(response_returns)==6 and len(output_fall_coverage)==56
assert set(output_fall_coverage.LOT)==set(raw)
assert set(output_fall_coverage.ZONE)=={3,4,5,6}
assert response_returns.loc[response_returns.TAG.eq("(b-1)")&response_returns.BAND_C.eq(1.),"CONFIRMED"].notna().all()
assert response_returns.loc[response_returns.TAG.eq("(b-2)")&response_returns.BAND_C.eq(1.),"CONFIRMED"].isna().all()
